# Scientific Motion Studio V10 — Agentic Production Control Center (10.2.0)

## 0. Overview and Safety Notice

**Production providers:** OpenAI GPT (reasoning + vision review) and BFL FLUX Kontext (image generation/editing) — locked in `production` mode; failures stop the run instead of falling back. Anthropic/Claude is not a runtime provider.

**Safety:** `Runtime ▸ Run all` is completely safe — every automatically executed cell is offline and makes **no paid API calls**. Paid actions exist only behind an explicit gate: a checkbox, the typed phrase `RUN LIVE`, configured API keys and a clean preflight.

| # | Section | Paid API? | Safe for Run all | Output |
|---|---------|-----------|------------------|--------|
| 1 | Runtime & environment bootstrap | No | Yes | tool/dependency report |
| 2 | Package source generation | No | Yes | `scistudio_v10/` package files |
| 3 | Dependency installation | No | Yes | installed package list |
| 4 | Package import & reload | No | Yes | version banner |
| 5 | Configuration presets | No | Yes | validated presets, `config.example.json` |
| 6 | Colab secrets & storage | No | Yes | secret status (never values) |
| 7 | **Notebook Control Center** | Only via gated buttons | Yes (renders UI only) | interactive console |
| 8 | Headless Python usage | No | Yes | offline plan-only demo |
| 9 | Offline validation | No | Yes | 260+ checks + security sweep |
| 10 | Optional live smoke tests | **Yes (opt-in)** | Yes (skips without opt-in) | per-provider PASS/FAIL/NOT RUN |
| 11 | Validation summary | No | Yes | `final_validation_report.json` |
| 12 | Export & download | No | Yes | artifact listing / ZIP helper |


## 1. Runtime and Environment Bootstrap

Checks Python/FFmpeg/Node, selects a portable workspace (`SCISTUDIO_WORKSPACE` or the current directory — never a hard-coded path), and installs missing Python dependencies once. Offline; safe for Run all.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

assert sys.version_info >= (3, 11), f"Python 3.11+ required, found {sys.version.split()[0]}"

WORKSPACE = Path(os.environ.get("SCISTUDIO_WORKSPACE", Path.cwd()))
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
print(f"Workspace: {WORKSPACE}")

def _ensure(packages):
    missing = []
    for module_name, pip_name in packages:
        try:
            importlib.import_module(module_name)
        except ImportError:
            missing.append(pip_name)
    if missing:
        print("Installing:", " ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
    else:
        print("All Python dependencies already present — nothing reinstalled.")

_ensure([
    ("pydantic", "pydantic>=2.7"), ("PIL", "Pillow>=10.0"),
    ("requests", "requests>=2.31"), ("openai", "openai>=1.68"),
    ("fastapi", "fastapi>=0.115"), ("httpx", "httpx>=0.27"),
    ("ipywidgets", "ipywidgets>=8.0"),
])

for tool, required in (("ffmpeg", True), ("ffprobe", True), ("node", False), ("npm", False)):
    found = shutil.which(tool)
    status = "OK" if found else ("MISSING (required)" if required else "missing (optional, Remotion only)")
    print(f"{tool:8s} {status}")
    if required and not found:
        raise RuntimeError(f"{tool} is required. On Debian/Ubuntu: apt-get install -y ffmpeg")


## 2. Package Source Generation

Each cell writes one package file with `%%writefile`. Re-running overwrites the file with identical content — idempotent, no hidden state. Offline; safe for Run all.

In [ ]:
from pathlib import Path
Path("scistudio_v10").mkdir(exist_ok=True)
print("Package directory ready:", Path("scistudio_v10").resolve())

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "scientific-motion-studio-v10"
version = "10.2.0"
description = "Agentic scientific editorial animation pipeline: OpenAI GPT reasoning, BFL FLUX Kontext art generation, resumable job runtime, hybrid Remotion rendering"
readme = "README.md"
license = { text = "Proprietary" }
requires-python = ">=3.11"
keywords = ["animation", "flux-kontext", "openai", "remotion", "science-communication"]
classifiers = [
    "Development Status :: 4 - Beta",
    "Intended Audience :: Developers",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
    "Topic :: Multimedia :: Video",
]
# Default install covers the documented production path: OpenAI GPT
# reasoning + BFL FLUX Kontext image generation.
dependencies = [
    "pydantic>=2.7",
    "Pillow>=10.0",
    "requests>=2.31",
    "openai>=1.68",
]

[project.optional-dependencies]
api = ["fastapi>=0.115", "uvicorn>=0.30"]
webui = ["streamlit>=1.36"]
notebook = ["ipywidgets>=8.0", "IPython>=8.0"]
render = ["cairosvg>=2.7"]
local-models = [
    "torch>=2.2",
    "transformers>=4.44",
    "diffusers>=0.32",
    "accelerate>=1.2",
    "safetensors>=0.4",
    "sentencepiece",
    "protobuf",
]
dev = [
    "ruff>=0.6",
    "build>=1.2",
    "coverage>=7.4",
    "pytest>=8.0",
]
test = ["nbclient>=0.10", "nbformat>=5.10", "coverage>=7.4", "pytest>=8.0", "fastapi>=0.115", "httpx>=0.27"]

[project.scripts]
scistudio-v10 = "scistudio_v10.cli:main"

[tool.setuptools.packages.find]
include = ["scistudio_v10*"]

[tool.ruff]
line-length = 120
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "W", "B", "UP"]
ignore = [
    "E501",  # long prompt strings are intentional
    "B008",  # FastAPI-style call defaults
    "UP035", "UP006", "UP007",  # keep 3.11-friendly typing forms
    "B905",  # zip truncation across scene lists is intentional
    "UP042",  # (str, Enum) kept for broad pydantic compatibility
]

[tool.coverage.run]
source = ["scistudio_v10"]
omit = ["scistudio_v10/webui.py"]


In [ ]:
%%writefile config.example.json
{
  "workspace": "./scientific_motion_studio_v10",
  "execution_mode": "production",
  "research_search": {"enabled": true},
  "llm": {
    "provider_order": ["openai"],
    "vision_provider_order": ["openai"],
    "openai_model": "gpt-5-mini",
    "openai_reasoning_effort": "low",
    "openai_max_output_tokens": 8000,
    "image_provider": "bfl",
    "bfl_model": "flux-kontext-pro",
    "bfl_base_url": "https://api.bfl.ai/v1",
    "bfl_aspect_ratio": "9:16",
    "bfl_prompt_upsampling": false,
    "bfl_safety_tolerance": 2,
    "bfl_output_format": "png",
    "bfl_timeout": 240,
    "bfl_poll_interval": 1.5,
    "bfl_allowed_url_hosts": ["api.bfl.ai", "*.bfl.ai"],
    "retry": {"max_attempts": 4, "base_delay_s": 1.0, "max_delay_s": 30.0, "jitter": 0.25}
  },
  "limits": {"max_stage_attempts": 5, "max_concurrent_jobs": 1},
  "providers": [
    {"provider_id":"flux-kontext-pro","provider_type":"image","implementation":"bfl","priority":10,"capabilities":[{"name":"reference_edit","available":true,"supports_images":true}]},
    {"provider_id":"vision-director","provider_type":"vision","implementation":"llm-router","priority":10,"capabilities":[{"name":"art_direction","available":true,"supports_images":true}]},
    {"provider_id":"sketch-video-local","provider_type":"temporal_video","implementation":"external-command","priority":20,"enabled":false,"capabilities":[{"name":"two_keyframe_sketch_control","available":true,"supports_video":true}]},
    {"provider_id":"remotion","provider_type":"renderer","implementation":"remotion","priority":10,"capabilities":[{"name":"hybrid_compositing","available":true,"supports_video":true}]}
  ],
  "audio": {"voice":"en-US-GuyNeural","rate":"+6%","build_in_plan_mode":false},
  "drawing": {
    "model":"flux-kontext-pro","aspect_ratio":"9:16","seed_base":240921,
    "prompt_upsampling":false,"safety_tolerance":2,"output_format":"png",
    "min_initial_prompt_words":80,"max_initial_prompt_words":380,
    "max_edit_prompt_words":180,"max_adjustments_per_pass":1
  },
  "candidate_tournament": {"candidate_count":4},
  "references": {"maximum_retrieved_assets":8},
  "flux_studio": {"maximum_director_revisions":3,"require_vision_director":true},
  "semantic": {"require_approved_beauty":true},
  "temporal": {
    "enabled": true,
    "use_for_articulated": false,
    "backend_preference": ["sketch-controlled-video", "deterministic-compositor"],
    "sketch_backend": {"argv": [], "allow_shell": false, "timeout_s": 1800}
  },
  "render": {"backend":"remotion","install_dependencies":true,"crf":18},
  "publishing": {"enabled": false, "provider": "local-archive", "allowed_hosts": []}
}


In [ ]:
%%writefile .env.example
# Scientific Motion Studio V10 — environment variables
# Copy to .env (never commit real values) or export in your shell / secret
# manager. Credentials are read ONLY from the runtime environment or the
# secrets dict passed to the pipeline — never from config files or images.

# REQUIRED in production: OpenAI GPT is the exclusive reasoning, script,
# storyboard, ranking and vision-review provider.
OPENAI_API_KEY=

# REQUIRED in production: Black Forest Labs FLUX Kontext is the exclusive
# image generation and image editing provider.
BFL_API_KEY=

# OPTIONAL (development mode only; ignored and rejected in production):
# GEMINI_API_KEY=
# OPENROUTER_API_KEY=
# HF_TOKEN=

# OPTIONAL: publishing token for the upload-post adapter (only used when
# publishing.enabled=true and the endpoint host is allowlisted).
# UPLOAD_POST_TOKEN=

# Set to 1 to allow the notebook's explicit live smoke tests (may incur
# API charges). Everything else runs fully offline.
# SCISTUDIO_RUN_LIVE_TESTS=0


In [ ]:
%%writefile Dockerfile
# Multi-stage build: build the wheel in one stage, install only the wheel in
# the runtime stage. Credentials are NEVER baked into layers — supply
# OPENAI_API_KEY / BFL_API_KEY at run time (docker run -e / secrets manager).

FROM python:3.12-slim AS build
WORKDIR /src
COPY pyproject.toml README.md /src/
COPY scistudio_v10 /src/scistudio_v10
RUN pip install --no-cache-dir build && python -m build --wheel --outdir /dist

FROM python:3.12-slim AS runtime
# FFmpeg for audio/video compositing; Node/npm for the Remotion renderer.
RUN apt-get update \
    && apt-get install -y --no-install-recommends ffmpeg nodejs npm \
    && rm -rf /var/lib/apt/lists/*
COPY --from=build /dist/*.whl /tmp/
RUN pip install --no-cache-dir /tmp/*.whl[api,render] && rm -f /tmp/*.whl

# Non-root runtime user with a writable workspace.
RUN useradd --create-home --uid 10001 studio \
    && mkdir -p /work \
    && chown -R studio:studio /work
USER studio
WORKDIR /work
ENV SCISTUDIO_WORKSPACE=/work

# Smoke/health command: the CLI must import and respond.
HEALTHCHECK --interval=60s --timeout=10s --retries=3 \
    CMD ["scistudio-v10", "--version"]

ENTRYPOINT ["scistudio-v10"]
CMD ["--help"]


In [ ]:
%%writefile README.md
# Scientific Motion Studio V10 (10.1.0) — Agentic Production Architecture

## Executive summary

Scientific Motion Studio turns a science topic plus a style-reference video
into a short editorial science animation. Version 10.1 is the hardened,
production-finalized revision of the V10 agentic architecture: a resumable
job runtime, retrieval-based visual continuity, multi-candidate FLUX
tournaments, an executive vision art director, semantic layer separation and
hybrid Remotion rendering — now with validated configuration, locked
production providers, classified retries, safe subprocess execution, atomic
state, secret redaction and a layered offline test suite (210 checks).

## Provider responsibility matrix

| Responsibility | Production provider | Fallback in production |
|---|---|---|
| Narrative architecture, research synthesis, script + criticism, storyboard, scene illustration architecture, continuity planning, candidate ranking, structured JSON decisions | **OpenAI GPT** (Responses API, `OPENAI_API_KEY`) | **None — fails clearly** |
| Visual critique / approval (vision) | **OpenAI GPT (vision)** | **None — fails clearly** |
| Master style anchors, beauty frames, reference-conditioned scenes, local revisions, pose/state variants, continuity-preserving edits | **BFL FLUX Kontext** (`BFL_API_KEY`, `flux-kontext-pro`/`-max`) | **None — fails clearly** |
| Narration | Edge-TTS / espeak (local tooling) | Silence with warning |
| Rendering | Remotion (Node) or deterministic PIL preview | Explicit config choice |

Anthropic/Claude is **not** a runtime provider: no SDK dependency, no API
calls, no `ANTHROPIC_API_KEY`. Gemini, OpenRouter and local Qwen/diffusers
code paths exist for **development mode only** and are rejected by
configuration validation and by the runtime router in production mode.
Deterministic/fake providers are limited to `test`/`development` modes; the
pipeline constructor rejects injected providers in production.

## Execution modes

* `production` — provider locks enforced at config-validation time and again
  at runtime. Missing/failed OpenAI or BFL ⇒ `ProviderUnavailableError`; the
  run stops. No silent fallback of any kind.
* `development` — alternative providers permitted if explicitly configured;
  deterministic fallbacks allowed but cached separately and marked.
* `test` — like development; intended for injected deterministic providers.

## Architecture and data flow

```
topic + reference.mp4
  → 01 style reference extraction (ffmpeg frames → style board)
  → 02 public research (Wikipedia/Crossref/OpenAlex/arXiv) + GPT evidence pack
  → 03 script (GPT) → 04 storyboard (GPT) → 05 audio (Edge-TTS)
  → 06 art-direction bible + 07 continuity canon (GPT, locked style canon)
  → 08 scene illustration architectures → 09 shot states (GPT)
  → 10 reference retrieval (asset registry) → 11 drawing briefs (FLUX prompts)
  → 12 candidate tournament (FLUX Kontext × N seeds, GPT-vision ranking)
  → 13 executive art-director revisions (FLUX Kontext local edits)
  → 14 semantic layer separation → 15 overlays → 16 animation plans
  → 17 temporal backends (optional sketch-video / deterministic compositor)
  → 18 hybrid packages → 19 Remotion (or PIL) render → MP4
```

Every stage is executed through the resumable job runtime and recorded in
`job_manifest.json`; a provenance manifest (`provenance_manifest.json`) links
all artifacts by SHA-256.

## Installation

```bash
pip install .                 # core production path (OpenAI + BFL)
pip install .[api]            # + FastAPI service
pip install .[webui]          # + Streamlit UI
pip install .[render]         # + SVG rasterization for PIL fallback
pip install .[local-models]   # optional heavy local model support (dev only)
pip install .[dev,test]       # lint/build/coverage/notebook tooling
```

Runtime requirements: Python ≥ 3.11, FFmpeg on PATH. Remotion rendering
additionally needs Node.js + npm.

Credentials come **only** from environment variables (or an explicit secrets
dict): see `.env.example`. Never commit real keys.

## Notebook usage

`Scientific_Motion_Studio_v10_Agentic_Production_FINAL.ipynb` is the
self-contained source of truth: run it top-to-bottom from a clean kernel and
it (1) validates the environment, (2) materializes this exact package,
(3) validates configuration, (4) runs the full offline suite, (5) optionally
runs explicit paid live smoke tests (`SCISTUDIO_RUN_LIVE_TESTS=1`),
(6) shows usage, and (7) writes `final_validation_report.json`. Rerunning the
notebook is idempotent.

## CLI usage

```bash
scistudio-v10 "What happens if it rains nonstop for a year?" reference.mp4 \
    --config config.json                  # plan-only (default, no paid calls)
scistudio-v10 TOPIC reference.mp4 --config config.json --live --render
scistudio-v10 --version
```

## API usage

```python
from scistudio_v10.service_api import create_app
from scistudio_v10 import ScientificMotionStudioV10
import json
app = create_app(lambda: ScientificMotionStudioV10(json.load(open("config.json"))))
# uvicorn module:app --host 127.0.0.1
```

`POST /jobs` returns `202` + job id and runs generation in a background
thread (single-worker; deploy a real queue for multi-instance production —
distributed execution is not implemented). `GET /jobs/{id}` returns status.
Errors are structured and redacted.

## Security model

* **No shell execution of templated commands.** External temporal backends
  use argv token lists with per-token placeholder substitution. An optional
  shell path exists for advanced users only: disabled by default, marked
  unsafe, and rejected in production without
  `security_override_unsafe_shell=true`.
* **SSRF protection.** Provider-supplied URLs (BFL polling/result URLs,
  publishing endpoints) must be https with hosts on a configured allowlist.
* **Download safety.** `raise_for_status`, Content-Type check, byte cap,
  Pillow verification, atomic temp-file + rename.
* **Secret redaction.** Loaded secrets are registered for exact-match
  scrubbing; realistic patterns (`sk-…`, bearer, `x-key`, UUID keys) are also
  redacted from errors, logs, manifests and API responses. Command failures
  are redacted and length-capped.
* **Path safety.** `ensure_within` guards workspace-relative outputs;
  archives are never extracted from untrusted sources.
* **No credentials in images/layers/config files.**

## Cache semantics

* Success cache is content-addressed over every generation input (provider,
  model, endpoint, prompts, schema, temperature/effort/limits, image
  SHA-256, seed, cache schema version). Filenames/sizes are never used as
  identity.
* Transient provider failures are never cached as successes. Development
  fallback values live in a separate `_fallback` namespace, carry an explicit
  marker + failure reason, and expire (`fallback_cache_ttl_s`, default 1 h).
* Corrupt cache files are quarantined (`*.corrupt-<ts>`) and regenerated.
* All writes are atomic.

## Resume semantics

`job_manifest.json` (schema 10.1) records stage status, input hash, output
path + SHA-256, attempts and redacted errors. Resume trusts a completed stage
only if its artifact exists and matches its checksum. Stages left `running`
by a crashed worker are recovered as `interrupted` and re-run. A config-hash
change marks completed stages `stale`. A pid lock file rejects concurrent
workers on one job (stale locks from dead pids are reclaimed). Stage retries
are budgeted (`limits.max_stage_attempts`).

## Failure behavior

Production failures raise typed exceptions (`ProviderUnavailableError`,
`BFLError`, `DownloadPolicyError`, `JobConcurrencyError`, …) with redacted
messages; the job manifest records the failing stage. Retryable HTTP statuses
(408/409/429/5xx) are retried with exponential backoff + jitter, honouring
`Retry-After`. Auth/validation failures are never retried.

## Testing procedure

```bash
python -m scistudio_v10.tests_final          # 210 offline checks, exit code
python -m coverage run --source=scistudio_v10 -m scistudio_v10.tests_final
python -m coverage report
ruff check scistudio_v10/
python -m build && pip check
```

No test spends live API credits. The only live calls are the notebook's
explicit opt-in smoke cells gated by `SCISTUDIO_RUN_LIVE_TESTS=1`.

## Deployment

```bash
docker build -t scistudio-v10 .
docker run --rm -e OPENAI_API_KEY -e BFL_API_KEY \
    -v "$PWD/work:/work" scistudio-v10 \
    "TOPIC" /work/reference.mp4 --config /work/config.json
```

The image is multi-stage, runs as a non-root user, reads credentials only
from the runtime environment and ships a `--version` healthcheck.

## Limitations and paid-provider boundary

* Live FLUX Kontext output quality and live OpenAI responses are **not**
  validated by the offline suite; run the opt-in live smoke tests.
* The FastAPI shell is single-worker; distributed job execution is not
  implemented.
* Sketch-controlled temporal video requires an external, user-supplied model
  command; without it, high-complexity motion intentionally fails rather
  than degrading.
* Docker build/run is validated only where Docker is available.

## Reproducibility statement

All creative stages are cached content-addressed; identical inputs (prompts,
seeds, models, config) reuse identical cached outputs. The notebook
regenerates this package byte-for-byte, and the provenance manifest links
every published artifact to its input hashes.

## Validation evidence

See `FINALIZATION_REPORT.md` and `final_validation_report.json` for the
exact command matrix (PASS/FAIL/NOT RUN), coverage, and the security sweep
results for this revision.


In [ ]:
%%writefile scistudio_v10/__init__.py
"""Scientific Motion Studio V10 — agentic scientific animation pipeline.

Production providers: OpenAI GPT (reasoning/vision) and BFL FLUX Kontext
(image generation/editing). See ``config_models.StudioConfig`` for the
validated configuration surface and execution modes.
"""

from .asset_registry import AssetRegistry
from .bfl_client import BFLClient, BFLError
from .candidate_tournament import CandidateTournament
from .config_models import ExecutionMode, StudioConfig
from .errors import (
    ConfigurationError,
    DownloadPolicyError,
    JobConcurrencyError,
    JobStateError,
    ProviderError,
    ProviderLockViolationError,
    ProviderUnavailableError,
    StageRetryExhaustedError,
    StudioError,
    UnsafeCommandError,
)
from .flux_prompt_system import FluxPromptSystem
from .pipeline import ScientificMotionStudioV10
from .provider_registry import ProviderRegistry
from .schemas import (
    AnimationPlan,
    ArtDirectionBible,
    AssetRecord,
    CandidateTournamentResult,
    ContinuityCanon,
    DirectorChangeOrder,
    DrawingBrief,
    HybridScenePackage,
    JobManifest,
    ReferenceSelection,
    SceneIllustrationArchitecture,
    SemanticLayerContract,
    ShotState,
    TemporalRequest,
    TemporalResult,
)
from .style_canon import build_hard_coded_canon
from .temporal_backends import TemporalBackendRouter

__version__ = "10.2.0"

__all__ = [
    "AnimationPlan",
    "ArtDirectionBible",
    "AssetRecord",
    "AssetRegistry",
    "BFLClient",
    "BFLError",
    "CandidateTournament",
    "CandidateTournamentResult",
    "ConfigurationError",
    "ContinuityCanon",
    "DirectorChangeOrder",
    "DownloadPolicyError",
    "DrawingBrief",
    "ExecutionMode",
    "FluxPromptSystem",
    "HybridScenePackage",
    "JobConcurrencyError",
    "JobManifest",
    "JobStateError",
    "ProviderError",
    "ProviderLockViolationError",
    "ProviderRegistry",
    "ProviderUnavailableError",
    "ReferenceSelection",
    "SceneIllustrationArchitecture",
    "ScientificMotionStudioV10",
    "SemanticLayerContract",
    "ShotState",
    "StageRetryExhaustedError",
    "StudioConfig",
    "StudioError",
    "TemporalBackendRouter",
    "TemporalRequest",
    "TemporalResult",
    "UnsafeCommandError",
    "build_hard_coded_canon",
    "__version__",
]


In [ ]:
%%writefile scistudio_v10/animation_director.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import (
    AnimationPlan,
    MotionEvent,
    SceneIllustrationArchitecture,
    SceneRequest,
    SemanticLayerContract,
)
from .utils import ensure_dir, save_json


class AnimationDirector:
    SYSTEM = """You are the Animation Director of a scientific editorial studio.
Return JSON only. Animate the approved illustration without making it feel like a presentation.
Default is hold. Every event must express the scene's stated causal change and must target an available semantic layer.
Prefer replacement drawings, local masks, material loops and restrained transforms. Never add automatic fade,
zoom, entrance animation, random parallax, camera shake or decorative motion."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def plan(
        self,
        scene: SceneRequest,
        architecture: SceneIllustrationArchitecture,
        contract: SemanticLayerContract,
        *,
        fps: int,
        audio_timing: dict[str, Any] | None = None,
        force: bool = False,
    ) -> AnimationPlan:
        duration_frames = max(1, round(scene.duration_s * fps))
        fallback = self._fallback(scene, architecture, contract, duration_frames, fps, audio_timing or {})
        available = [layer.layer_id for layer in contract.layers]
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""Scene: {json.dumps(scene.model_dump(mode="json"), ensure_ascii=False)}
Illustration architecture: {json.dumps(architecture.model_dump(mode="json"), ensure_ascii=False)}
Available layers: {json.dumps([layer_item.model_dump(mode="json") for layer_item in contract.layers], ensure_ascii=False)}
Duration: {duration_frames} frames at {fps} fps.
Audio timing: {json.dumps(audio_timing or {}, ensure_ascii=False)}

Return an AnimationPlan. Rules:
- camera_locked=true unless an indispensable camera action is explicitly justified; no camera events otherwise.
- Only target these exact layer IDs: {available}.
- Every MotionEvent needs event_id, reason_id, target_layer, representation, start_frame, end_frame, easing and parameters.
- Maximum one primary event and two secondary events at the same time.
- If no local movement is needed, return events=[] and preserve a deliberate hold.
- Use replacement_pose only when pose_variant_paths exist; use masks for local changes; keep beauty-base locked.
""",
            namespace=f"v10_animation_{scene.scene_id}",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        try:
            plan = AnimationPlan.model_validate(raw)
        except Exception:
            plan = fallback
        plan.scene_id = scene.scene_id
        plan.fps = fps
        plan.duration_frames = duration_frames
        plan.camera_locked = True
        plan.events = self._sanitize(plan.events, contract, duration_frames)
        plan.hold_regions = sorted(
            set([*plan.hold_regions, *[layer_item.layer_id for layer_item in contract.layers if layer_item.locked]])
        )
        save_json(self.root / f"{scene.scene_id}.json", plan)
        return plan

    @staticmethod
    def _sanitize(events: list[MotionEvent], contract: SemanticLayerContract, duration: int) -> list[MotionEvent]:
        available = {layer.layer_id: layer for layer in contract.layers}
        clean: list[MotionEvent] = []
        for event in events:
            if event.target_layer not in available:
                continue
            if event.target_layer == "beauty-base":
                continue
            if event.representation == "replacement_pose" and not available[event.target_layer].pose_variant_paths:
                continue
            event.start_frame = max(0, min(duration - 1, event.start_frame))
            event.end_frame = max(event.start_frame + 1, min(duration, event.end_frame))
            clean.append(event)
        return clean

    def _fallback(
        self,
        scene: SceneRequest,
        architecture: SceneIllustrationArchitecture,
        contract: SemanticLayerContract,
        duration_frames: int,
        fps: int,
        audio_timing: dict[str, Any],
    ) -> AnimationPlan:
        events: list[MotionEvent] = []
        start = max(1, round(duration_frames * 0.20))
        end = max(start + 2, round(duration_frames * 0.82))
        layer_map = {layer.layer_id: layer for layer in contract.layers}
        for index, seam in enumerate(architecture.motion_seams):
            if seam.seam_id not in layer_map:
                continue
            layer = layer_map[seam.seam_id]
            secondary = index > 0
            # Replacement drawings preserve authored art better than synthetic
            # deformation. Whenever approved variants exist, prefer them even if
            # the conceptual seam was described as a local deformation.
            if layer.pose_variant_paths and seam.method != "texture_loop":
                representation = "replacement_pose"
                params = {"pose_paths": layer.pose_variant_paths, "hold_last": True}
            elif seam.method == "texture_loop":
                representation = "texture_loop"
                params = {"axis": "y", "speed_px_per_second": 55, "masked": True}
            elif seam.method == "layer_transform":
                representation = "translate"
                params = {"from": [0, 0], "to": [18, 0], "masked": True}
            elif seam.method == "overlay_only":
                representation = "overlay_draw"
                params = {"progress": [0, 1]}
            else:
                representation = "local_deformation" if seam.method == "local_deformation" else "mask_reveal"
                params = {"progress": [0, 1], "masked": True}
            events.append(
                MotionEvent(
                    event_id=f"EV{index + 1:02d}",
                    reason_id=scene.beat_id or scene.scene_id,
                    target_layer=seam.seam_id,
                    representation=representation,  # type: ignore[arg-type]
                    start_frame=start + index * 2,
                    end_frame=end,
                    easing="ease-in-out" if representation != "texture_loop" else "linear",
                    parameters=params,
                    secondary=secondary,
                )
            )
        holds = [layer.layer_id for layer in contract.layers if layer.locked]
        return AnimationPlan(
            scene_id=scene.scene_id,
            fps=fps,
            duration_frames=duration_frames,
            camera_locked=True,
            events=events,
            audio_sync=audio_timing,
            hold_regions=holds,
        )


In [ ]:
%%writefile scistudio_v10/asset_registry.py
from __future__ import annotations

import json
import sqlite3
from pathlib import Path

from .schemas import AssetRecord
from .utils import ensure_dir


class AssetRegistry:
    """Searchable visual continuity registry backed by SQLite."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        ensure_dir(self.path.parent)
        self.conn = sqlite3.connect(self.path)
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS assets (
            asset_id TEXT PRIMARY KEY,
            path TEXT NOT NULL,
            asset_type TEXT NOT NULL,
            approved INTEGER NOT NULL,
            scene_id TEXT,
            chronology_index INTEGER,
            payload TEXT NOT NULL
        )""")
        self.conn.commit()

    def upsert(self, asset: AssetRecord) -> None:
        payload = json.dumps(asset.model_dump(mode="json"), ensure_ascii=False)
        self.conn.execute(
            "INSERT OR REPLACE INTO assets(asset_id,path,asset_type,approved,scene_id,chronology_index,payload) VALUES(?,?,?,?,?,?,?)",
            (
                asset.asset_id,
                asset.path,
                asset.asset_type,
                int(asset.approved),
                asset.scene_id,
                asset.chronology_index,
                payload,
            ),
        )
        self.conn.commit()

    def get(self, asset_id: str) -> AssetRecord | None:
        row = self.conn.execute("SELECT payload FROM assets WHERE asset_id=?", (asset_id,)).fetchone()
        return AssetRecord.model_validate(json.loads(row[0])) if row else None

    def list(self, *, approved_only: bool = True) -> list[AssetRecord]:
        query = "SELECT payload FROM assets" + (" WHERE approved=1" if approved_only else "")
        return [AssetRecord.model_validate(json.loads(row[0])) for row in self.conn.execute(query)]

    def close(self) -> None:
        self.conn.close()


In [ ]:
%%writefile scistudio_v10/audio.py
"""Audio engine: Edge-TTS/espeak narration and procedural support bed."""

from __future__ import annotations

import shutil
import re
from pathlib import Path
from typing import Any

from .schemas import ScriptPackage, Storyboard
from .utils import ensure_dir, ffprobe_duration, run_command, save_json


class AudioEngine:
    """Notebook-safe audio engine.

    Edge TTS is invoked through its CLI in a separate process. This avoids the
    `asyncio.run() cannot be called from a running event loop` failure in Colab.
    """

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)

    def synthesize(self, text: str, output: str | Path) -> Path:
        output = Path(output)
        if output.exists() and output.stat().st_size > 1000:
            return output
        text_path = output.with_suffix(".txt")
        text_path.write_text(text.strip(), encoding="utf-8")
        edge_binary = shutil.which("edge-tts")
        if edge_binary:
            try:
                run_command(
                    [
                        edge_binary,
                        "--voice",
                        str(self.config.get("voice", "en-US-GuyNeural")),
                        "--rate",
                        str(self.config.get("rate", "+6%")),
                        "--pitch",
                        str(self.config.get("pitch", "+0Hz")),
                        "--file",
                        str(text_path),
                        "--write-media",
                        str(output),
                    ],
                    timeout=180,
                )
            except Exception as exc:
                print(f"[audio] Edge TTS failed; using local fallback: {exc}")
        if not output.exists() or output.stat().st_size < 1000:
            self._fallback_tts(text, output)
        text_path.unlink(missing_ok=True)
        return output

    def _fallback_tts(self, text: str, output: Path) -> None:
        wav = output.with_suffix(".wav")
        if not shutil.which("espeak-ng"):
            # Last-resort preview path: keep the pipeline executable even on a
            # minimal runtime, but mark the voice as silent so publishing QC can
            # never approve it as a finished institutional output.
            duration = max(0.8, len(re.findall(r"[A-Za-z0-9']+", text)) / 2.5)
            run_command(
                [
                    "ffmpeg",
                    "-y",
                    "-loglevel",
                    "error",
                    "-f",
                    "lavfi",
                    "-i",
                    f"anullsrc=r=48000:cl=mono:d={duration}",
                    "-c:a",
                    "libmp3lame",
                    "-b:a",
                    "128k",
                    str(output),
                ]
            )
            output.with_suffix(output.suffix + ".silent").write_text(
                "TTS unavailable; silent preview voice generated.", encoding="utf-8"
            )
            return
        run_command(
            ["espeak-ng", "-v", str(self.config.get("fallback_voice", "en-us")), "-s", "170", "-w", str(wav), text]
        )
        run_command(["ffmpeg", "-y", "-i", str(wav), "-c:a", "libmp3lame", "-b:a", "192k", str(output)])
        wav.unlink(missing_ok=True)

    def build(self, script: ScriptPackage, storyboard: Storyboard) -> tuple[ScriptPackage, Storyboard, dict[str, Any]]:
        voice_root = ensure_dir(self.root / "voice")
        voice_files = {}
        word_timing: dict[str, list[dict[str, Any]]] = {}
        updated_beats = []
        updated_scenes = []
        for index, beat in enumerate(script.beats):
            voice = self.synthesize(beat.spoken_line, voice_root / f"{beat.beat_id.lower()}.mp3")
            actual = ffprobe_duration(voice)
            duration = round(max(beat.duration_s, actual + beat.pause_after_ms / 1000 + 0.2), 3)
            updated_beats.append(beat.model_copy(update={"duration_s": duration}))
            scene = storyboard.scenes[index] if index < len(storyboard.scenes) else storyboard.scenes[-1]
            updated_scenes.append(scene.model_copy(update={"duration_s": duration, "narration": beat.spoken_line}))
            voice_files[beat.beat_id] = str(voice)
            word_timing[beat.beat_id] = self._word_timing(beat.spoken_line, actual)
        total = round(sum(scene.duration_s for scene in updated_scenes), 3)
        script = script.model_copy(update={"beats": updated_beats, "estimated_duration_s": total})
        storyboard = storyboard.model_copy(update={"scenes": updated_scenes, "estimated_duration_s": total})
        scene_starts = []
        _cursor = 0.0
        for scene in updated_scenes:
            scene_starts.append(round(_cursor, 3))
            _cursor += scene.duration_s
        bed = self._support_bed(total, self.root / "support_bed.m4a", scene_starts=scene_starts)
        silent_voice_files = [
            path for path in voice_files.values() if Path(path).with_suffix(Path(path).suffix + ".silent").exists()
        ]
        manifest = {
            "voice": voice_files,
            "support_bed": str(bed),
            "duration_s": total,
            "word_timing": word_timing,
            "voice_silent_preview": bool(silent_voice_files),
            "warnings": ["TTS unavailable; silent preview voice was generated."] if silent_voice_files else [],
        }
        save_json(self.root / "audio_manifest.json", manifest)
        return script, storyboard, manifest

    @staticmethod
    def _word_timing(text: str, duration: float) -> list[dict[str, Any]]:
        """Create deterministic local word timing for choreography.

        Edge-TTS media is measured with ffprobe. When provider word-boundary
        events are unavailable, duration is distributed by spoken-word weight;
        punctuation receives a small pause. The manifest clearly stores these
        as estimated timings so the orchestrator can sync events without
        pretending they are forced-alignment ground truth.
        """
        tokens = re.findall(r"[A-Za-z0-9']+|[.,!?;:]", text)
        words = [token for token in tokens if re.match(r"[A-Za-z0-9']", token)]
        if not words:
            return []
        weights = [max(1.0, len(word) ** 0.55) for word in words]
        total_weight = sum(weights)
        usable = max(0.1, float(duration))
        cursor = 0.0
        output: list[dict[str, Any]] = []
        for index, (word, weight) in enumerate(zip(words, weights)):
            segment = usable * weight / total_weight
            start = cursor
            end = min(usable, cursor + segment)
            output.append(
                {
                    "word": word,
                    "start_s": round(start, 4),
                    "end_s": round(end, 4),
                    "estimated": True,
                    "index": index,
                }
            )
            cursor = end
        if output:
            output[-1]["end_s"] = round(usable, 4)
        return output

    def _support_bed(self, duration: float, output: Path, scene_starts: list[float] | None = None) -> Path:
        """Designed music bed (not a flat drone): a minor-triad pad with slow
        tremolo movement, a filtered 'air' layer, a soft ~0.9 Hz sub pulse, and
        short filtered-noise whooshes synced to each scene start. All procedural
        via FFmpeg lavfi — no external assets, no licensing."""
        if output.exists() and output.stat().st_size > 1000 and not self.config.get("force_audio", False):
            return output
        fade_out = max(0.0, duration - 2.0)
        starts = [s for s in (scene_starts or []) if 0.05 < s < duration - 0.05]

        inputs = [
            "-f",
            "lavfi",
            "-i",
            f"sine=frequency=110:sample_rate=48000:duration={duration}",  # root A2
            "-f",
            "lavfi",
            "-i",
            f"sine=frequency=130.81:sample_rate=48000:duration={duration}",  # minor third C3
            "-f",
            "lavfi",
            "-i",
            f"sine=frequency=164.81:sample_rate=48000:duration={duration}",  # fifth E3
            "-f",
            "lavfi",
            "-i",
            f"sine=frequency=55:sample_rate=48000:duration={duration}",  # sub A1 (pulsed)
            "-f",
            "lavfi",
            "-i",
            f"anoisesrc=color=pink:sample_rate=48000:duration={duration}",  # air
        ]
        parts = [
            "[0:a]volume=0.020,tremolo=f=0.10:d=0.5,lowpass=f=360[p0]",
            "[1:a]volume=0.013,tremolo=f=0.13:d=0.5,lowpass=f=520[p1]",
            "[2:a]volume=0.012,tremolo=f=0.11:d=0.4,lowpass=f=680[p2]",
            "[3:a]volume=0.030,tremolo=f=0.9:d=0.9,lowpass=f=140[sub]",  # heartbeat pulse
            "[4:a]volume=0.006,highpass=f=120,lowpass=f=1400[air]",
            "[p0][p1][p2][sub][air]amix=inputs=5:normalize=0[bed]",
        ]
        # Whooshes at scene transitions.
        whoosh_labels = []
        wh_input_start = 5
        for i, start in enumerate(starts):
            inputs += ["-f", "lavfi", "-i", "anoisesrc=color=white:sample_rate=48000:duration=0.6"]
            lbl = f"wh{i}"
            delay = int(max(0, (start - 0.18) * 1000))
            parts.append(
                f"[{wh_input_start + i}:a]volume=0.10,highpass=f=300,lowpass=f=5000,"
                f"afade=t=in:st=0:d=0.12,afade=t=out:st=0.22:d=0.35,adelay={delay}|{delay}[{lbl}]"
            )
            whoosh_labels.append(f"[{lbl}]")

        if whoosh_labels:
            parts.append("[bed]" + "".join(whoosh_labels) + f"amix=inputs={1 + len(whoosh_labels)}:normalize=0[mixed]")
            final_in = "[mixed]"
        else:
            final_in = "[bed]"
        parts.append(f"{final_in}alimiter=limit=0.72,afade=t=in:st=0:d=1.2,afade=t=out:st={fade_out}:d=2[out]")
        run_command(
            [
                "ffmpeg",
                "-y",
                *inputs,
                "-filter_complex",
                ";".join(parts),
                "-map",
                "[out]",
                "-t",
                str(duration),
                "-c:a",
                "aac",
                "-b:a",
                "160k",
                str(output),
            ]
        )
        return output


In [ ]:
%%writefile scistudio_v10/batch.py
from __future__ import annotations

from pathlib import Path
from typing import Any, Iterable

from .pipeline import ScientificMotionStudioV10
from .utils import save_json


def run_batch(
    studio: ScientificMotionStudioV10, jobs: Iterable[dict[str, Any]], output_manifest: str | Path
) -> list[dict[str, Any]]:
    results = []
    for item in jobs:
        try:
            result = studio.run(
                item["topic"],
                item["reference_video"],
                plan_only=bool(item.get("plan_only", True)),
                render_video=bool(item.get("render_video", False)),
                force=bool(item.get("force", False)),
                job_id=item.get("job_id"),
            )
            results.append({"ok": True, "result": result})
        except Exception as exc:
            results.append(
                {"ok": False, "topic": item.get("topic", ""), "error": str(exc), "error_type": type(exc).__name__}
            )
    save_json(output_manifest, results)
    return results


In [ ]:
%%writefile scistudio_v10/bfl_client.py
"""Hardened Black Forest Labs FLUX Kontext client.

Submission → polling → download with:

* model / base-URL / aspect-ratio validation before any paid call;
* https + host-allowlist policy on the endpoint, the server-supplied polling
  URL and the result ``sample`` URL (SSRF prevention);
* bounded retries with exponential backoff and jitter for 408/409/429/5xx and
  connection failures, honouring ``Retry-After`` hints;
* explicit handling of every polling status — unknown statuses fail loudly;
* ``raise_for_status`` + Content-Type check + size cap + Pillow verification
  on the final download, written atomically (no partial cache artifacts);
* a content-addressed cache keyed on every generation input (provider, model,
  endpoint, prompt, input-image SHA-256, aspect, format, seed, upsampling,
  safety tolerance and cache schema version) — never filename + size;
* failures are never written into the success cache;
* observability records carry request id, model, prompt hash, input-image
  hash, latency and status — never raw base64 payloads or API keys.

Note on HTTP style: calls go through module-level ``requests.post`` /
``requests.get`` (not a private ``Session``) so the offline regression suite
can intercept the exact wire contract with ``unittest.mock.patch``.
"""

from __future__ import annotations

import base64
import time
from pathlib import Path
from typing import Any

import requests

from .config_models import BFL_ALLOWED_MODELS, BFL_MAX_ASPECT, BFL_MIN_ASPECT
from .errors import DownloadPolicyError, ProviderError
from .http_safety import call_with_retries, download_image, is_retryable_status, validate_url
from .security import redacted_exception_text, register_secret
from .utils import atomic_copy, ensure_dir, hash_value, save_json, sha256_file

CACHE_SCHEMA_VERSION = 3

_PENDING_STATUSES = {"pending", "queued", "processing", "running", "task queued"}
_MODERATED_STATUSES = {"content moderated", "request moderated"}
_FAILED_STATUSES = {"error", "failed", "task not found"}


class BFLError(ProviderError):
    """A BFL FLUX Kontext request failed."""

    def __init__(
        self,
        message: str,
        *,
        retryable: bool = False,
        status_code: int | None = None,
        request_id: str = "",
        status: str = "",
    ):
        super().__init__(message, provider="bfl", retryable=retryable, status_code=status_code, request_id=request_id)
        self.status = status


class BFLClient:
    """Submit/poll/download client for the BFL FLUX Kontext API."""

    def __init__(
        self,
        config: dict[str, Any],
        api_key: str,
        cache_root: str | Path,
        *,
        event_logger: Any | None = None,
    ):
        if not api_key:
            raise BFLError("BFL_API_KEY is not configured")
        self.config = config
        self.api_key = api_key
        register_secret(api_key)
        self.cache_dir = ensure_dir(Path(cache_root) / "bfl_images")
        self.events = event_logger

        self.model = str(config.get("bfl_model", "flux-kontext-pro"))
        if self.model not in BFL_ALLOWED_MODELS:
            raise BFLError(f"Unsupported FLUX Kontext model {self.model!r}; allowed: {sorted(BFL_ALLOWED_MODELS)}")
        self.base_url = str(config.get("bfl_base_url", "https://api.bfl.ai/v1")).rstrip("/")
        hosts = list(config.get("bfl_allowed_url_hosts") or ["api.bfl.ai", "*.bfl.ai"])
        from urllib.parse import urlsplit

        base_host = urlsplit(self.base_url).hostname or ""
        if base_host and base_host not in hosts:
            hosts.append(base_host)
        self.allowed_hosts = hosts
        validate_url(self.base_url + "/", allowed_hosts=self.allowed_hosts, purpose="BFL base URL")

        self.timeout_s = float(config.get("bfl_timeout", 240))
        self.poll_interval_s = float(config.get("bfl_poll_interval", 1.5))
        self.max_download_bytes = int(config.get("bfl_max_download_bytes", 32 * 1024 * 1024))
        retry_cfg = config.get("retry") or {}
        self.max_attempts = int(retry_cfg.get("max_attempts", 4))
        self.base_delay_s = float(retry_cfg.get("base_delay_s", 1.0))
        self.max_delay_s = float(retry_cfg.get("max_delay_s", 30.0))
        self.jitter = float(retry_cfg.get("jitter", 0.25))

        self.output_format = str(config.get("bfl_output_format", "png")).lower()
        if self.output_format not in {"png", "jpeg"}:
            raise BFLError("bfl_output_format must be png or jpeg")
        self.aspect_ratio = self._validate_aspect(str(config.get("bfl_aspect_ratio", "9:16")))
        self.prompt_upsampling = bool(config.get("bfl_prompt_upsampling", False))
        self.safety_tolerance = int(config.get("bfl_safety_tolerance", 2))
        self.seed = config.get("bfl_seed")

    @staticmethod
    def _validate_aspect(aspect: str) -> str:
        try:
            left, right = [float(x) for x in str(aspect).split(":", 1)]
            ratio = left / right
        except (ValueError, ZeroDivisionError) as exc:
            raise BFLError(f"Unsupported BFL aspect ratio: {aspect!r}") from exc
        if not (BFL_MIN_ASPECT <= ratio <= BFL_MAX_ASPECT):
            raise BFLError(f"Unsupported BFL aspect ratio {aspect!r}: outside Kontext range 3:7..7:3")
        return aspect

    # -- cache -----------------------------------------------------------
    def cache_key(self, prompt: str, init_hash: str) -> str:
        return hash_value(
            {
                "provider": "bfl",
                "endpoint": self.base_url,
                "model": self.model,
                "prompt": prompt,
                "init": init_hash,
                "aspect": self.aspect_ratio,
                "prompt_upsampling": self.prompt_upsampling,
                "safety_tolerance": self.safety_tolerance,
                "output_format": self.output_format,
                "seed": self.seed,
                "cache_schema": CACHE_SCHEMA_VERSION,
            },
            32,
        )

    def _cached_image(self, key: str) -> Path | None:
        suffix = ".png" if self.output_format == "png" else ".jpg"
        cached = self.cache_dir / f"{key}{suffix}"
        if not cached.exists() or cached.stat().st_size == 0:
            return None
        try:
            import io

            from PIL import Image

            with Image.open(io.BytesIO(cached.read_bytes())) as image:
                image.verify()
            return cached
        except Exception:
            # Corrupt legacy cache entry: quarantine and regenerate.
            from .utils import quarantine_corrupt_file

            quarantine_corrupt_file(cached)
            return None

    # -- events ----------------------------------------------------------
    def _emit(self, event: str, **payload: Any) -> None:
        if self.events is not None:
            try:
                self.events.emit(event, provider="bfl", model=self.model, **payload)
            except Exception:
                pass

    # -- request ---------------------------------------------------------
    def generate(
        self,
        prompt: str,
        output_path: str | Path,
        *,
        init_image: str | Path | None = None,
        force: bool = False,
    ) -> Path:
        """Generate (or reuse from cache) one image; returns the output path.

        Raises :class:`BFLError` or :class:`DownloadPolicyError` on failure —
        failures are never cached as successes.
        """
        output_path = Path(output_path)
        ensure_dir(output_path.parent)
        init_path = Path(init_image) if init_image else None
        init_hash = sha256_file(init_path) if init_path and init_path.exists() else ""
        prompt_hash = hash_value(prompt, 24)
        key = self.cache_key(prompt, init_hash)

        if not force:
            cached = self._cached_image(key)
            if cached is not None:
                atomic_copy(cached, output_path)
                self._emit("bfl.cache_hit", cache_key=key, prompt_hash=prompt_hash)
                return output_path

        payload: dict[str, Any] = {
            "prompt": prompt,
            "output_format": self.output_format,
            "aspect_ratio": self.aspect_ratio,
            "prompt_upsampling": self.prompt_upsampling,
            "safety_tolerance": self.safety_tolerance,
        }
        if self.seed is not None:
            payload["seed"] = int(self.seed)
        if init_path and init_path.exists():
            payload["input_image"] = base64.b64encode(init_path.read_bytes()).decode("ascii")

        safe_payload = {k: v for k, v in payload.items() if k != "input_image"}
        safe_payload["input_image_hash"] = init_hash
        save_json(
            self.cache_dir / f"{key}.request.json",
            {
                "endpoint": f"{self.base_url}/{self.model}",
                "payload": safe_payload,
                "cache_key": key,
            },
        )

        headers = {"x-key": self.api_key, "Content-Type": "application/json", "accept": "application/json"}
        started = time.perf_counter()
        attempts = {"submit": 0, "download": 0}

        def _submit() -> dict[str, Any]:
            attempts["submit"] += 1
            response = requests.post(
                f"{self.base_url}/{self.model}",
                headers=headers,
                json=payload,
                timeout=60,
            )
            status_code = getattr(response, "status_code", 200)
            if is_retryable_status(status_code):
                raise BFLError(f"BFL submit returned {status_code}", retryable=True, status_code=status_code)
            response.raise_for_status()
            body = response.json()
            if not isinstance(body, dict) or not (body.get("id") or body.get("polling_url")):
                raise BFLError("BFL submit response missing id/polling_url")
            return body

        try:
            submitted = call_with_retries(
                _submit,
                max_attempts=self.max_attempts,
                base_delay=self.base_delay_s,
                max_delay=self.max_delay_s,
                jitter=self.jitter,
                on_retry=lambda n, exc, d: self._emit(
                    "bfl.retry", phase="submit", attempt=n, delay_s=round(d, 2), error=redacted_exception_text(exc, 300)
                ),
            )
        except BFLError:
            raise
        except Exception as exc:
            raise BFLError(f"BFL submit failed: {redacted_exception_text(exc)}", retryable=False) from exc

        request_id = str(submitted.get("id", ""))
        polling_url = submitted.get("polling_url") or f"{self.base_url}/get_result"
        validate_url(polling_url, allowed_hosts=self.allowed_hosts, purpose="BFL polling URL")

        sample_url = self._poll(polling_url, request_id)
        validate_url(sample_url, allowed_hosts=self.allowed_hosts, purpose="BFL result URL")

        suffix = ".png" if self.output_format == "png" else ".jpg"
        cached_target = self.cache_dir / f"{key}{suffix}"

        def _download() -> Path:
            attempts["download"] += 1
            response = requests.get(sample_url, timeout=120)
            return download_image(response, cached_target, max_bytes=self.max_download_bytes)

        try:
            call_with_retries(
                _download,
                max_attempts=self.max_attempts,
                base_delay=self.base_delay_s,
                max_delay=self.max_delay_s,
                jitter=self.jitter,
                is_retryable=lambda exc: (
                    not isinstance(exc, DownloadPolicyError)
                    and is_retryable_status(getattr(getattr(exc, "response", None), "status_code", None))
                ),
                on_retry=lambda n, exc, d: self._emit(
                    "bfl.retry",
                    phase="download",
                    attempt=n,
                    delay_s=round(d, 2),
                    error=redacted_exception_text(exc, 300),
                ),
            )
        except DownloadPolicyError:
            raise
        except Exception as exc:
            raise BFLError(f"BFL download failed: {redacted_exception_text(exc)}", request_id=request_id) from exc

        atomic_copy(cached_target, output_path)
        latency = round(time.perf_counter() - started, 3)
        save_json(
            self.cache_dir / f"{key}.meta.json",
            {
                "request_id": request_id,
                "model": self.model,
                "prompt_hash": prompt_hash,
                "input_image_hash": init_hash,
                "latency_s": latency,
                "status": "Ready",
                "attempts": attempts,
                "cache_key": key,
            },
        )
        self._emit(
            "bfl.generated",
            request_id=request_id,
            prompt_hash=prompt_hash,
            input_image_hash=init_hash,
            latency_s=latency,
            cache_key=key,
            attempts=attempts,
        )
        return output_path

    def _poll(self, polling_url: str, request_id: str) -> str:
        """Poll until Ready; returns the result sample URL."""
        deadline = time.time() + self.timeout_s
        params = None if "get_result" not in polling_url else {"id": request_id}
        last_status = ""
        while time.time() < deadline:
            try:
                response = requests.get(
                    polling_url,
                    headers={"x-key": self.api_key, "accept": "application/json"},
                    params=params,
                    timeout=30,
                )
            except Exception as exc:
                # Transient poll failure: wait and try again within the deadline.
                self._emit("bfl.poll_error", request_id=request_id, error=redacted_exception_text(exc, 300))
                time.sleep(self.poll_interval_s)
                continue
            status_code = getattr(response, "status_code", 200)
            if is_retryable_status(status_code):
                time.sleep(self.poll_interval_s)
                continue
            response.raise_for_status()
            body = response.json()
            if not isinstance(body, dict):
                raise BFLError("BFL polling response is not an object", request_id=request_id)
            status = str(body.get("status", ""))
            low = status.lower()
            last_status = status
            if low == "ready":
                result = body.get("result") or {}
                sample = result.get("sample") if isinstance(result, dict) else None
                if not sample:
                    raise BFLError("BFL Ready response missing result.sample URL", request_id=request_id, status=status)
                return str(sample)
            if low in _MODERATED_STATUSES:
                raise BFLError(f"BFL request was moderated (status={status})", request_id=request_id, status=status)
            if low in _FAILED_STATUSES:
                raise BFLError(f"BFL generation failed (status={status})", request_id=request_id, status=status)
            if low not in _PENDING_STATUSES:
                raise BFLError(f"BFL returned unknown status {status!r}", request_id=request_id, status=status)
            time.sleep(self.poll_interval_s)
        raise BFLError(
            f"BFL timed out after {self.timeout_s:.0f}s (last status={last_status or 'none'})",
            retryable=True,
            request_id=request_id,
            status=last_status,
        )


In [ ]:
%%writefile scistudio_v10/cache_store.py
"""Content-addressed artifact cache.

* Keys are derived from the full input payload hash (never filename+size).
* All writes are atomic (temp file + rename) — a killed process cannot leave
  a partial cache artifact.
* Corrupt JSON entries are quarantined and treated as cache misses.
* Fallback entries live in a dedicated namespace so degraded results can
  never masquerade as live provider output.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

from .utils import atomic_copy, ensure_dir, hash_value, load_json, save_json

FALLBACK_PREFIX = "fallback-"


class ArtifactCache:
    """Content-addressed cache for expensive LLM, FLUX and render stages."""

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)

    def key(self, namespace: str, payload: Any) -> str:
        """Cache key: namespace + SHA-256 over the canonical payload JSON."""
        return f"{namespace}-{hash_value(payload, 32)}"

    def fallback_key(self, namespace: str, payload: Any) -> str:
        """Separate namespace for degraded/fallback results."""
        return f"{FALLBACK_PREFIX}{self.key(namespace, payload)}"

    def json_path(self, key: str) -> Path:
        return self.root / key[:2] / f"{key}.json"

    def file_path(self, key: str, suffix: str) -> Path:
        return self.root / key[:2] / f"{key}{suffix}"

    def get_json(self, key: str) -> Any | None:
        """Read a cached JSON value; corrupt entries are quarantined and
        reported as misses instead of raising."""
        path = self.json_path(key)
        if not path.exists():
            return None
        return load_json(path, None)

    def put_json(self, key: str, value: Any) -> str:
        path = self.json_path(key)
        ensure_dir(path.parent)
        save_json(path, value)
        return str(path)

    def get_file(self, key: str, suffix: str) -> str | None:
        path = self.file_path(key, suffix)
        return str(path) if path.exists() and path.stat().st_size > 0 else None

    def put_file(self, key: str, source: str | Path, suffix: str | None = None) -> str:
        source_path = Path(source)
        extension = suffix or source_path.suffix
        destination = self.file_path(key, extension)
        ensure_dir(destination.parent)
        atomic_copy(source_path, destination)
        return str(destination)


In [ ]:
%%writefile scistudio_v10/candidate_tournament.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

from PIL import Image, ImageDraw, ImageOps, ImageStat

from .schemas import (
    CandidateFrame,
    CandidateScore,
    CandidateTournamentResult,
    DrawingBrief,
    SceneIllustrationArchitecture,
    ShotState,
)
from .utils import ensure_dir, hash_value, save_json


class CandidateTournament:
    """Generate multiple low-resolution candidates, rank them, then revise the winner."""

    def __init__(self, llm: Any, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def run(
        self,
        brief: DrawingBrief,
        architecture: SceneIllustrationArchitecture,
        shot_state: ShotState,
        generator: Callable[[DrawingBrief], str | Path | None],
        *,
        force: bool = False,
    ) -> CandidateTournamentResult:
        count = max(2, int(self.config.get("candidate_count", 4)))
        candidates = []
        for i in range(count):
            variant = brief.model_copy(deep=True)
            variant.brief_id = f"{brief.brief_id}-C{i + 1:02d}"
            variant.seed = (brief.seed or 0) + i * 9973
            variant.output_path = str(self.root / "candidates" / f"{brief.scene_id}_C{i + 1:02d}.{brief.output_format}")
            variant.request_metadata = {**variant.request_metadata, "candidate_index": i, "tournament": True}
            path = generator(variant)
            if path:
                candidates.append(
                    CandidateFrame(
                        candidate_id=f"C{i + 1:02d}",
                        scene_id=brief.scene_id,
                        image_path=str(path),
                        seed=variant.seed,
                        prompt_hash=hash_value(variant.compiled_prompt or variant.positive_prompt, 20),
                        metrics=self._image_metrics(path),
                    )
                )
        if len(candidates) < 2:
            raise RuntimeError(f"Candidate tournament needs at least two generated frames for {brief.scene_id}")
        board = self._board(candidates, brief.scene_id)
        scores, source = self._rank(board, candidates, architecture, shot_state, force=force)
        scores.sort(key=lambda x: (-x.total_score, x.candidate_id))
        winner = scores[0]
        winner_path = next(c.image_path for c in candidates if c.candidate_id == winner.candidate_id)
        result = CandidateTournamentResult(
            scene_id=brief.scene_id,
            candidates=candidates,
            scores=scores,
            winner_id=winner.candidate_id,
            winner_path=winner_path,
            comparison_board=board,
            ranking_source=source,
        )
        save_json(self.root / f"{brief.scene_id}.json", result)
        return result

    def _rank(
        self,
        board: str,
        candidates: list[CandidateFrame],
        architecture: SceneIllustrationArchitecture,
        shot_state: ShotState,
        *,
        force: bool,
    ) -> tuple[list[CandidateScore], str]:
        fallback = [
            CandidateScore(
                candidate_id=c.candidate_id,
                total_score=self._fallback_score(c),
                style_consistency=0.7,
                subject_consistency=0.7,
                composition_fitness=0.7,
                motion_readiness=0.7,
                causal_clarity=0.7,
                rationale="Deterministic fallback ranking from image information and motion-readiness priors.",
            )
            for c in candidates
        ]
        raw = self.llm.critique_image(
            image_path=board,
            namespace=f"v10_candidate_rank_{architecture.scene_id}",
            force=force,
            fallback={"scores": [x.model_dump(mode="json") for x in fallback]},
            prompt=f"""Rank candidate panels for one scientific editorial shot. Return JSON with scores only.
Architecture: {json.dumps(architecture.model_dump(mode="json"), ensure_ascii=False)}
Shot state: {json.dumps(shot_state.model_dump(mode="json"), ensure_ascii=False)}
Judge studio-style continuity, subject identity, coherent perspective, causal clarity, adult authored illustration,
and whether the first frame can reach the specified last frame without destructive redraw. Penalize AI artifacts,
maskot anatomy, sticker composition, random detail and hidden motion seams that will tear.""",
        )
        try:
            values = raw.get("scores", raw) if isinstance(raw, dict) else raw
            parsed = [CandidateScore.model_validate(x) for x in values]
            if {x.candidate_id for x in parsed} == {x.candidate_id for x in candidates}:
                return parsed, "vision-director"
        except Exception:
            pass
        return fallback, "deterministic-fallback"

    @staticmethod
    def _image_metrics(path: str | Path) -> dict[str, float]:
        image = Image.open(path).convert("RGB").resize((128, 128))
        stat = ImageStat.Stat(image)
        contrast = sum(stat.stddev) / 3 / 128
        entropy = sum(image.getchannel(c).entropy() for c in range(3)) / 3 / 8
        return {"contrast": float(contrast), "entropy": float(entropy)}

    @staticmethod
    def _fallback_score(c: CandidateFrame) -> float:
        return (
            0.55
            + 0.22 * c.metrics.get("entropy", 0)
            + 0.18 * c.metrics.get("contrast", 0)
            + (0.001 * (c.seed or 0) % 0.04)
        )

    def _board(self, candidates: list[CandidateFrame], scene_id: str) -> str:
        cols = 2
        rows = (len(candidates) + 1) // 2
        cw, ch = 600, 900
        canvas = Image.new("RGB", (cols * cw, rows * ch), "#E6E8E7")
        draw = ImageDraw.Draw(canvas)
        for i, c in enumerate(candidates):
            x = (i % cols) * cw
            y = (i // cols) * ch
            im = Image.open(c.image_path).convert("RGB")
            im = ImageOps.contain(im, (cw - 30, ch - 80))
            canvas.paste(im, (x + (cw - im.width) // 2, y + 45))
            draw.rectangle((x + 5, y + 5, x + cw - 5, y + ch - 5), outline="#475157", width=3)
            draw.text((x + 18, y + 15), f"{c.candidate_id} | SEED {c.seed}", fill="#20282D")
        out = self.root / "boards" / f"{scene_id}.png"
        ensure_dir(out.parent)
        canvas.save(out)
        return str(out)


In [ ]:
%%writefile scistudio_v10/character_registry.py
from __future__ import annotations

from pathlib import Path

from .schemas import CharacterProfile, CharacterView
from .utils import ensure_dir, load_json, save_json


class CharacterRegistry:
    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)
        self.path = self.root / "characters.json"
        self.characters: dict[str, CharacterProfile] = {}
        if self.path.exists():
            self.characters = {x["subject_id"]: CharacterProfile.model_validate(x) for x in load_json(self.path)}

    def upsert(self, profile: CharacterProfile) -> None:
        self.characters[profile.subject_id] = profile
        self._save()

    def add_view(self, subject_id: str, view: CharacterView) -> None:
        if subject_id not in self.characters:
            self.characters[subject_id] = CharacterProfile(subject_id=subject_id, display_name=subject_id)
        profile = self.characters[subject_id]
        profile.views = [x for x in profile.views if x.view_id != view.view_id] + [view]
        self._save()

    def get(self, subject_id: str) -> CharacterProfile | None:
        return self.characters.get(subject_id)

    def _save(self) -> None:
        save_json(self.path, [x.model_dump(mode="json") for x in self.characters.values()])


In [ ]:
%%writefile scistudio_v10/cli.py
"""Command-line interface.

``scistudio-v10 TOPIC REFERENCE_VIDEO --config config.json`` runs plan-only
by default (no paid image generation); pass ``--live`` to run the paid FLUX
studio path and ``--render`` to render the final video.
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

from .config_models import StudioConfig
from .security import redact_secrets, redacted_exception_text

__version__ = "10.2.0"


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="scistudio-v10",
        description="Agentic scientific animation studio (OpenAI GPT reasoning, BFL FLUX Kontext art).",
        epilog="Plan-only is the default; use --live for paid generation.",
    )
    parser.add_argument("topic", nargs="?", help="Video topic / question")
    parser.add_argument("reference_video", nargs="?", help="Path to the style reference MP4")
    parser.add_argument("--config", help="Path to the studio config JSON")
    parser.add_argument("--live", action="store_true", help="Run live generation (paid provider calls)")
    parser.add_argument("--render", action="store_true", help="Render the final video")
    parser.add_argument("--force", action="store_true", help="Ignore stage caches")
    parser.add_argument("--job-id", default=None, help="Explicit job identifier")
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
    return parser


def main(argv=None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if not args.topic or not args.reference_video or not args.config:
        parser.print_usage(sys.stderr)
        print("error: topic, reference_video and --config are required", file=sys.stderr)
        return 2

    config_path = Path(args.config)
    if not config_path.is_file():
        print(f"error: config file not found: {config_path}", file=sys.stderr)
        return 2
    try:
        settings = StudioConfig.from_dict(json.loads(config_path.read_text(encoding="utf-8")))
    except Exception as exc:
        print(f"error: invalid configuration: {redacted_exception_text(exc, 500)}", file=sys.stderr)
        return 2

    reference = Path(args.reference_video)
    if not reference.is_file():
        print(f"error: reference video not found: {reference}", file=sys.stderr)
        return 2

    from .pipeline import ScientificMotionStudioV10

    try:
        result = ScientificMotionStudioV10(settings).run(
            args.topic,
            str(reference),
            plan_only=not args.live,
            render_video=args.render,
            force=args.force,
            job_id=args.job_id,
        )
    except Exception as exc:
        print(f"error [{type(exc).__name__}]: {redacted_exception_text(exc, 800)}", file=sys.stderr)
        return 1

    print(json.dumps(redact_secrets(result), ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scistudio_v10/config_models.py
"""Strongly-validated configuration models and execution modes.

``StudioConfig`` is the single entry point: it validates the loose nested
dictionary that earlier prototypes passed around, enforces the production
provider boundary, and can be converted back into the runtime dictionary
shape (`as_runtime_dict`) that the pipeline modules consume, preserving any
extra keys the user supplied.

Execution modes
---------------
* ``development`` — permissive: alternative providers and deterministic
  fallbacks are allowed (fallbacks are still marked and cached separately).
* ``test`` — like development, intended for injected fake providers and
  offline suites. Never performs paid live calls by itself.
* ``production`` — the provider boundary is locked and validated:
  OpenAI GPT is the exclusive reasoning/vision provider and BFL FLUX
  Kontext is the exclusive image engine. Configurations that enable any
  unauthorized fallback are rejected, and runtime failures raise
  ``ProviderUnavailableError`` instead of degrading silently.
"""

from __future__ import annotations

import re
from enum import Enum
from typing import Any

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

from .errors import ProviderLockViolationError

BFL_ALLOWED_MODELS = frozenset(
    {
        "flux-kontext-pro",
        "flux-kontext-max",
    }
)
BFL_MIN_ASPECT = 3 / 7
BFL_MAX_ASPECT = 7 / 3

_ASPECT_PATTERN = re.compile(r"^\s*(\d+(?:\.\d+)?)\s*:\s*(\d+(?:\.\d+)?)\s*$")


class ExecutionMode(str, Enum):
    development = "development"
    test = "test"
    production = "production"


class RetryPolicy(BaseModel):
    """Shared retry policy for provider HTTP calls."""

    model_config = ConfigDict(extra="allow")

    max_attempts: int = Field(default=4, ge=1, le=10)
    base_delay_s: float = Field(default=1.0, ge=0.05, le=30.0)
    max_delay_s: float = Field(default=30.0, ge=0.1, le=300.0)
    jitter: float = Field(default=0.25, ge=0.0, le=1.0)


class LLMConfig(BaseModel):
    """Text/vision/image provider routing configuration."""

    model_config = ConfigDict(extra="allow")

    provider_order: list[str] | str = Field(default_factory=lambda: ["openai"])
    vision_provider_order: list[str] | str | None = None
    execution_mode: ExecutionMode = ExecutionMode.development

    openai_model: str = "gpt-5-mini"
    openai_vision_model: str = ""
    openai_reasoning_effort: str = "low"
    openai_max_output_tokens: int = Field(default=8000, ge=256, le=200_000)
    openai_connect_timeout_s: float = Field(default=15.0, ge=1.0, le=120.0)
    openai_read_timeout_s: float = Field(default=180.0, ge=5.0, le=1200.0)
    openai_json_repair_attempts: int = Field(default=1, ge=0, le=3)

    image_provider: str = "bfl"
    bfl_model: str = "flux-kontext-pro"
    bfl_base_url: str = "https://api.bfl.ai/v1"
    bfl_aspect_ratio: str = "9:16"
    bfl_timeout: float = Field(default=240.0, ge=10.0, le=3600.0)
    bfl_poll_interval: float = Field(default=1.5, ge=0.1, le=30.0)
    bfl_prompt_upsampling: bool = False
    bfl_safety_tolerance: int = Field(default=2, ge=0, le=6)
    bfl_output_format: str = "png"
    bfl_seed: int | None = None
    bfl_allowed_url_hosts: list[str] = Field(default_factory=lambda: ["api.bfl.ai", "*.bfl.ai"])
    bfl_max_download_bytes: int = Field(default=32 * 1024 * 1024, ge=1024, le=512 * 1024 * 1024)

    retry: RetryPolicy = Field(default_factory=RetryPolicy)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    fallback_cache_ttl_s: float = Field(default=3600.0, ge=0.0)

    enable_local_fallback: bool = False
    enable_local_image_generation: bool = False

    @field_validator("openai_reasoning_effort")
    @classmethod
    def _effort(cls, value: str) -> str:
        allowed = {"minimal", "low", "medium", "high"}
        if value not in allowed:
            raise ValueError(f"openai_reasoning_effort must be one of {sorted(allowed)}")
        return value

    @field_validator("openai_model")
    @classmethod
    def _openai_model(cls, value: str) -> str:
        if not value or not value.strip():
            raise ValueError("openai_model must be a non-empty model identifier")
        return value.strip()

    @field_validator("bfl_model")
    @classmethod
    def _bfl_model(cls, value: str) -> str:
        if value not in BFL_ALLOWED_MODELS:
            raise ValueError(
                f"bfl_model {value!r} is not a supported FLUX Kontext model; allowed: {sorted(BFL_ALLOWED_MODELS)}"
            )
        return value

    @field_validator("bfl_base_url")
    @classmethod
    def _bfl_base_url(cls, value: str) -> str:
        if not value.startswith("https://"):
            raise ValueError("bfl_base_url must be https")
        return value.rstrip("/")

    @field_validator("bfl_aspect_ratio")
    @classmethod
    def _aspect(cls, value: str) -> str:
        match = _ASPECT_PATTERN.match(str(value))
        if not match:
            raise ValueError(f"bfl_aspect_ratio {value!r} must look like '9:16'")
        left, right = float(match.group(1)), float(match.group(2))
        if right <= 0 or left <= 0:
            raise ValueError("bfl_aspect_ratio terms must be positive")
        ratio = left / right
        if not (BFL_MIN_ASPECT <= ratio <= BFL_MAX_ASPECT):
            raise ValueError(f"bfl_aspect_ratio {value!r} outside the FLUX Kontext supported range 3:7..7:3")
        return f"{match.group(1)}:{match.group(2)}"

    @field_validator("bfl_output_format")
    @classmethod
    def _fmt(cls, value: str) -> str:
        if str(value).lower() not in {"png", "jpeg"}:
            raise ValueError("bfl_output_format must be png or jpeg")
        return str(value).lower()

    def ordered_providers(self) -> list[str]:
        order = self.provider_order
        if isinstance(order, str):
            order = [item.strip() for item in order.split(",") if item.strip()]
        return [str(item).lower() for item in order]

    def ordered_vision_providers(self) -> list[str]:
        order = self.vision_provider_order
        if order is None:
            return self.ordered_providers()
        if isinstance(order, str):
            order = [item.strip() for item in order.split(",") if item.strip()]
        return [str(item).lower() for item in order]


class TemporalCommandConfig(BaseModel):
    """External temporal-backend command configuration.

    The safe form is ``argv``: a list of tokens where ``{placeholders}`` are
    substituted per token (never joined through a shell). The legacy string
    ``command`` is parsed with ``shlex`` into the same argv form.
    Shell execution is disabled by default, marked unsafe, and rejected in
    production mode unless ``security_override_unsafe_shell`` is set.
    """

    model_config = ConfigDict(extra="allow")

    argv: list[str] = Field(default_factory=list)
    command: str = ""  # legacy single-string form; shlex-split, never shell-run
    timeout_s: float = Field(default=1800.0, ge=1.0, le=24 * 3600.0)
    allow_shell: bool = False
    unsafe_shell_command: str = ""
    security_override_unsafe_shell: bool = False


class TemporalConfig(BaseModel):
    model_config = ConfigDict(extra="allow")

    enabled: bool = True
    use_for_articulated: bool = False
    backend_preference: list[str] = Field(
        default_factory=lambda: ["sketch-controlled-video", "deterministic-compositor"]
    )
    sketch_backend: TemporalCommandConfig = Field(default_factory=TemporalCommandConfig)


class RenderConfig(BaseModel):
    model_config = ConfigDict(extra="allow")

    backend: str = "remotion"
    install_dependencies: bool = True
    crf: int = Field(default=18, ge=0, le=51)

    @field_validator("backend")
    @classmethod
    def _backend(cls, value: str) -> str:
        allowed = {"remotion", "pil", "deterministic", "preview"}
        low = str(value).lower()
        if low not in allowed:
            raise ValueError(f"render backend must be one of {sorted(allowed)}")
        return low


class LimitsConfig(BaseModel):
    model_config = ConfigDict(extra="allow")

    max_file_bytes: int = Field(default=512 * 1024 * 1024, ge=1024)
    max_concurrent_jobs: int = Field(default=1, ge=1, le=64)
    max_stage_attempts: int = Field(default=5, ge=1, le=50)


class PublishingConfig(BaseModel):
    model_config = ConfigDict(extra="allow")

    enabled: bool = False
    provider: str = "local-archive"
    archive_dir: str = ""
    endpoint: str = ""
    allowed_hosts: list[str] = Field(default_factory=list)


class CandidateTournamentConfig(BaseModel):
    model_config = ConfigDict(extra="allow")

    candidate_count: int = Field(default=4, ge=1, le=12)
    seed_stride: int = Field(default=9973, ge=1)


class FluxStudioConfig(BaseModel):
    model_config = ConfigDict(extra="allow")

    maximum_director_revisions: int = Field(default=3, ge=1, le=10)
    require_vision_director: bool = True
    min_valid_image_bytes: int = Field(default=1024, ge=1)


class StudioConfig(BaseModel):
    """Top-level validated configuration for the studio pipeline."""

    model_config = ConfigDict(extra="allow")

    workspace: str = "./scientific_motion_studio_v10"
    execution_mode: ExecutionMode = ExecutionMode.development
    job_id: str | None = None

    llm: LLMConfig = Field(default_factory=LLMConfig)
    temporal: TemporalConfig = Field(default_factory=TemporalConfig)
    render: RenderConfig = Field(default_factory=RenderConfig)
    limits: LimitsConfig = Field(default_factory=LimitsConfig)
    publishing: PublishingConfig = Field(default_factory=PublishingConfig)
    candidate_tournament: CandidateTournamentConfig = Field(default_factory=CandidateTournamentConfig)
    flux_studio: FluxStudioConfig = Field(default_factory=FluxStudioConfig)

    @field_validator("workspace")
    @classmethod
    def _workspace(cls, value: str) -> str:
        if not str(value).strip():
            raise ValueError("workspace must be a non-empty path")
        return str(value)

    @model_validator(mode="after")
    def _propagate_and_lock(self) -> StudioConfig:
        # The LLM router reads its own execution mode; keep it in sync.
        self.llm.execution_mode = self.execution_mode
        if self.execution_mode == ExecutionMode.production:
            self._enforce_production_locks()
        return self

    def _enforce_production_locks(self) -> None:
        problems: list[str] = []
        order = self.llm.ordered_providers()
        if order != ["openai"]:
            problems.append(
                f"llm.provider_order must be ['openai'] in production (got {order}); "
                "Gemini/OpenRouter/local fallbacks are not authorized"
            )
        vision = self.llm.ordered_vision_providers()
        if vision != ["openai"]:
            problems.append(f"llm.vision_provider_order must be ['openai'] in production (got {vision})")
        if self.llm.image_provider != "bfl":
            problems.append(
                f"llm.image_provider must be 'bfl' in production (got {self.llm.image_provider!r}); "
                "BFL FLUX Kontext is the exclusive production image engine"
            )
        if self.llm.enable_local_fallback:
            problems.append("llm.enable_local_fallback must be false in production")
        if self.llm.enable_local_image_generation:
            problems.append("llm.enable_local_image_generation must be false in production")
        extras = self.llm.model_extra or {}
        for banned in ("gemini_image_model", "openai_image_model", "local_image_model"):
            if extras.get(banned):
                problems.append(
                    f"llm.{banned} must not be set in production; only BFL FLUX Kontext may generate images"
                )
        sketch = self.temporal.sketch_backend
        if sketch.allow_shell and not sketch.security_override_unsafe_shell:
            problems.append(
                "temporal.sketch_backend.allow_shell requires security_override_unsafe_shell=true in production"
            )
        if problems:
            raise ProviderLockViolationError("Production provider-lock validation failed:\n- " + "\n- ".join(problems))

    @classmethod
    def from_dict(cls, raw: dict[str, Any] | StudioConfig) -> StudioConfig:
        if isinstance(raw, StudioConfig):
            return raw
        return cls.model_validate(raw or {})

    def as_runtime_dict(self) -> dict[str, Any]:
        """The nested-dict shape the pipeline modules consume.

        Extra keys supplied by the user are preserved verbatim.
        """
        data = self.model_dump(mode="json")
        data["execution_mode"] = self.execution_mode.value
        data["llm"]["execution_mode"] = self.execution_mode.value
        return data


In [ ]:
%%writefile scistudio_v10/director.py
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import Beat, ResearchPack, SceneRequest, ScriptPackage, Storyboard
from .utils import ensure_dir, hash_value, load_json, save_json


class NarrativeDirector:
    SCRIPT_SYSTEM = """You are the narrative director of an institutional science-animation studio.
Return JSON only. Answer one hypothetical question through a direct, escalating causal chain.
Use short spoken sentences, concrete physical consequences, explicit uncertainty where inputs are unspecified,
and a concise payoff. Do not imitate or name another channel or studio."""

    STORY_SYSTEM = """You are a scene-level scientific storyboard director.
Return JSON only. Think in persistent worlds and causal state changes, not slides and not isolated asset lists.
For each beat, define the physical visual event, the scientific claim, the attention goal, and the exact change the
audience should notice. Do not choose an art style here; the Executive Art Director owns style."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def script(self, research: ResearchPack, *, force: bool = False) -> ScriptPackage:
        cache = self.root / f"script-{research.research_hash or hash_value(research.topic)}.json"
        if cache.exists() and not force:
            return ScriptPackage.model_validate(load_json(cache))
        fallback = self._fallback_script(research.topic)
        raw = self.llm.generate_json(
            system=self.SCRIPT_SYSTEM,
            prompt=f"""Topic: {research.topic}
Research summary: {research.summary[:1400]}
Limitations: {json.dumps(research.limitations, ensure_ascii=False)}
Useful facts: {json.dumps([f.claim for f in research.facts[:10]], ensure_ascii=False)}

Create 7-9 beats for a 40-60 second science short. Each spoken line should generally stay below 18 words.
Each beat must include beat_id, time_stage, spoken_line, visual_event, retention_function and optional sfx.
Do not state a precise global outcome when the hypothetical omits intensity, location or boundary conditions.
Return a ScriptPackage-shaped JSON object.""",
            namespace="v10_script",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        script = self._normalize_script(raw, fallback, research.topic)
        save_json(cache, script)
        return script

    def storyboard(self, script: ScriptPackage, research: ResearchPack, *, force: bool = False) -> Storyboard:
        cache = self.root / f"storyboard-{script.script_hash}.json"
        if cache.exists() and not force:
            return Storyboard.model_validate(load_json(cache))
        fallback = self._fallback_storyboard(script)
        raw = self.llm.generate_json(
            system=self.STORY_SYSTEM,
            prompt=f"""Topic: {script.topic}
Script: {json.dumps(script.model_dump(mode="json"), ensure_ascii=False)}
Evidence constraints: {json.dumps(research.limitations, ensure_ascii=False)}

Return one scene for every beat. Required fields per scene:
scene_id, beat_id, duration_s, narration, headline, visual_event, scientific_claim,
time_stage, attention_goal, desired_change, transition.

Rules:
- A scene is one integrated physical world, not a list of icons.
- Preserve useful locations and subjects between adjacent scenes when continuity improves comprehension.
- desired_change names only the causal state change that should animate.
- Do not prescribe SVG, isolated object assets, art style, camera gimmicks or decorative movement.
- Use cut or motivated match-cut by default.
""",
            namespace="v10_storyboard",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        board = self._normalize_storyboard(raw, fallback, script)
        save_json(cache, board)
        return board

    def _normalize_script(self, raw: Any, fallback: ScriptPackage, topic: str) -> ScriptPackage:
        if not isinstance(raw, dict):
            raw = fallback.model_dump(mode="json")
        raw_beats = raw.get("beats", fallback.model_dump(mode="json")["beats"])
        if isinstance(raw_beats, dict):
            raw_beats = [raw_beats[k] for k in sorted(raw_beats)]
        beats: list[Beat] = []
        for i, item in enumerate(raw_beats if isinstance(raw_beats, list) else [], 1):
            if isinstance(item, str):
                item = {"spoken_line": item}
            if not isinstance(item, dict):
                continue
            beats.append(
                Beat(
                    beat_id=str(item.get("beat_id", item.get("id", f"B{i:02d}"))),
                    duration_s=item.get("duration_s", item.get("duration", 4.5)),
                    spoken_line=str(item.get("spoken_line", item.get("narration", ""))),
                    visual_event=str(item.get("visual_event", item.get("visual", item.get("spoken_line", "")))),
                    retention_function=str(item.get("retention_function", item.get("purpose", "information_gain"))),
                    sfx=str(item.get("sfx", "none")),
                    raw={**item, "time_stage": item.get("time_stage", "")},
                )
            )
        if not beats:
            beats = fallback.beats
        words_per_second = float(self.config.get("words_per_second", 2.65))
        for beat in beats:
            words = max(1, len(re.findall(r"\b\w+\b", beat.spoken_line)))
            beat.duration_s = max(2.6, beat.duration_s, words / words_per_second + 0.3)
        result = ScriptPackage(
            topic=topic,
            title=str(raw.get("title", fallback.title)),
            hook=str(raw.get("hook", beats[0].spoken_line)),
            beats=beats,
            closing=str(raw.get("closing", beats[-1].spoken_line)),
            raw_llm_output=raw,
        )
        result.total_words = sum(len(re.findall(r"\b\w+\b", b.spoken_line)) for b in beats)
        result.estimated_duration_s = round(sum(b.duration_s for b in beats), 3)
        result.script_hash = hash_value(result.model_dump(exclude={"script_hash", "raw_llm_output"}))
        return result

    def _normalize_storyboard(self, raw: Any, fallback: Storyboard, script: ScriptPackage) -> Storyboard:
        if not isinstance(raw, dict):
            return fallback
        scenes_raw = raw.get("scenes", raw.get("storyboard", []))
        if isinstance(scenes_raw, dict):
            scenes_raw = [scenes_raw[k] for k in sorted(scenes_raw)]
        if not isinstance(scenes_raw, list):
            scenes_raw = []
        scenes: list[SceneRequest] = []
        for i, beat in enumerate(script.beats):
            item = scenes_raw[i] if i < len(scenes_raw) and isinstance(scenes_raw[i], dict) else {}
            scenes.append(
                SceneRequest(
                    scene_id=str(item.get("scene_id", f"SC{i + 1:02d}")),
                    beat_id=beat.beat_id,
                    duration_s=item.get("duration_s", beat.duration_s),
                    narration=str(item.get("narration", beat.spoken_line)),
                    headline=str(item.get("headline", self._headline(beat.spoken_line))),
                    visual_event=str(item.get("visual_event", beat.visual_event)),
                    scientific_claim=str(item.get("scientific_claim", beat.spoken_line)),
                    time_stage=str(item.get("time_stage", beat.raw.get("time_stage", ""))),
                    attention_goal=str(item.get("attention_goal", beat.visual_event)),
                    desired_change=str(item.get("desired_change", beat.visual_event)),
                    transition=str(item.get("transition", "cut")),
                    raw_llm_output=item,
                )
            )
        board = Storyboard(topic=script.topic, width=1080, height=1920, fps=30, scenes=scenes, raw_llm_output=raw)
        board.estimated_duration_s = round(sum(s.duration_s for s in scenes), 3)
        board.storyboard_hash = hash_value(board.model_dump(exclude={"storyboard_hash", "raw_llm_output"}))
        return board

    @staticmethod
    def _fallback_script(topic: str) -> ScriptPackage:
        subject = re.sub(r"^\s*what\s+(would\s+happen\s+)?if\s+", "", topic.strip().rstrip("?"), flags=re.I)
        lines = [
            (f"What if {subject}?", "hook", "Establish the changed world in one readable image."),
            (
                "First, the system loses the balance that normally resets it.",
                "instant",
                "Show the first broken equilibrium.",
            ),
            (
                "The nearest materials respond before the larger environment catches up.",
                "seconds",
                "Show local physical response.",
            ),
            ("Then the effect spreads through every connected pathway.", "minutes", "Show the causal path widening."),
            (
                "Accumulation turns a temporary disturbance into a persistent state.",
                "hours",
                "Compare storage before and after accumulation.",
            ),
            (
                "Infrastructure and living systems begin failing at different thresholds.",
                "days",
                "Show multiple thresholds inside one environment.",
            ),
            (
                "The final outcome depends on intensity, geography, and how long recovery is denied.",
                "long_term",
                "Show conditional outcomes, not false precision.",
            ),
            (
                "The real danger is not one event—it is a system that never gets time to reset.",
                "payoff",
                "Resolve the causal chain in one final composition.",
            ),
        ]
        beats = [
            Beat(beat_id=f"B{i:02d}", spoken_line=line, visual_event=visual, duration_s=4.6, raw={"time_stage": stage})
            for i, (line, stage, visual) in enumerate(lines, 1)
        ]
        result = ScriptPackage(topic=topic, title=topic, hook=lines[0][0], beats=beats, closing=lines[-1][0])
        result.total_words = sum(len(x[0].split()) for x in lines)
        result.estimated_duration_s = sum(b.duration_s for b in beats)
        result.script_hash = hash_value(result.model_dump(exclude={"script_hash"}))
        return result

    def _fallback_storyboard(self, script: ScriptPackage) -> Storyboard:
        scenes = []
        for i, beat in enumerate(script.beats, 1):
            scenes.append(
                SceneRequest(
                    scene_id=f"SC{i:02d}",
                    beat_id=beat.beat_id,
                    duration_s=beat.duration_s,
                    narration=beat.spoken_line,
                    headline=self._headline(beat.spoken_line),
                    visual_event=beat.visual_event,
                    scientific_claim=beat.spoken_line,
                    time_stage=str(beat.raw.get("time_stage", "")),
                    attention_goal=beat.visual_event,
                    desired_change=beat.visual_event,
                    transition="cut",
                )
            )
        board = Storyboard(topic=script.topic, scenes=scenes)
        board.estimated_duration_s = sum(s.duration_s for s in scenes)
        board.storyboard_hash = hash_value(board.model_dump(exclude={"storyboard_hash"}))
        return board

    @staticmethod
    def _headline(text: str) -> str:
        words = re.findall(r"\b[\w'-]+\b", text)
        return " ".join(words[:6]).upper() if words else "EXPERIMENT"


In [ ]:
%%writefile scistudio_v10/drawing_brief.py
from __future__ import annotations

from pathlib import Path
from typing import Any

from .flux_prompt_system import FluxPromptSystem
from .schemas import (
    ArtDirectionBible,
    ContinuityCanon,
    DirectorChangeOrder,
    DrawingBrief,
    ReferencePack,
    SceneIllustrationArchitecture,
)


class DrawingBriefCompiler:
    """Compatibility facade around the V10 FLUX prompting system."""

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = Path(root)
        self.prompts = FluxPromptSystem(config, root)
        self.seed_base = self.prompts.policy.seed_base

    def master_anchor(self, topic: str, bible: ArtDirectionBible, reference_board_path: str) -> DrawingBrief:
        return self.prompts.master_anchor(topic, bible, reference_board_path)

    def beauty_frame(
        self,
        architecture: SceneIllustrationArchitecture,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        references: ReferencePack,
        narration: str,
        headline: str,
        index: int,
        *,
        style_source_path: str | None = None,
        init_strategy: str | None = None,
    ) -> DrawingBrief:
        source = style_source_path or references.previous_approved_scene or references.board_path
        strategy = init_strategy or (
            "previous_approved_scene"
            if references.previous_approved_scene
            else "master_style_anchor"
            if style_source_path
            else "reference_board"
        )
        return self.prompts.beauty_frame(
            architecture,
            bible,
            continuity,
            references,
            narration,
            headline,
            index,
            style_source_path=source,
            init_strategy=strategy,
        )

    def revision_passes(
        self,
        original: DrawingBrief,
        order: DirectorChangeOrder,
        current_path: str,
        revision_cycle: int,
    ) -> list[DrawingBrief]:
        return self.prompts.revision_passes(original, order, current_path, revision_cycle)

    def revision(
        self,
        original: DrawingBrief,
        revision_number: int,
        revised_instruction: str,
        draft_path: str,
    ) -> DrawingBrief:
        # Legacy callers are mapped to a single explicit local-edit pass.
        from .schemas import ConcreteAdjustment, DirectorChangeOrder

        order = DirectorChangeOrder(
            scene_id=original.scene_id,
            revision_number=revision_number,
            status="revise",
            adjustments=[
                ConcreteAdjustment(
                    adjustment_id=f"LEGACY_{revision_number:02d}",
                    target_region="director-specified region",
                    problem="legacy revision instruction",
                    instruction=revised_instruction,
                    preserve=original.preserve,
                    priority="high",
                )
            ],
            immutable_preserve_list=original.preserve,
        )
        return self.revision_passes(original, order, draft_path, revision_number)[0]

    def pose_variant(
        self,
        original: DrawingBrief,
        approved_frame_path: str,
        variant_name: str,
        exact_change: str,
    ) -> DrawingBrief:
        return self.prompts.pose_variant(original, approved_frame_path, variant_name, exact_change)


In [ ]:
%%writefile scistudio_v10/errors.py
"""Structured exception hierarchy for Scientific Motion Studio V10.

Every provider, cache, job and security failure raises a subclass of
:class:`StudioError` so callers can distinguish configuration mistakes,
retryable provider incidents and hard security violations without string
matching. Messages are pre-redacted at the boundary that raises them.
"""

from __future__ import annotations


class StudioError(RuntimeError):
    """Base class for all first-party errors raised by the studio."""


class ConfigurationError(StudioError):
    """The supplied configuration is invalid or violates the execution mode."""


class ProviderError(StudioError):
    """A provider call failed.

    Attributes:
        provider: short provider identifier (``openai``, ``bfl``...).
        retryable: whether the failure class is worth retrying.
        status_code: HTTP-ish status when one exists.
        request_id: provider request id when one was returned.
    """

    def __init__(
        self,
        message: str,
        *,
        provider: str = "",
        retryable: bool = False,
        status_code: int | None = None,
        request_id: str = "",
    ):
        super().__init__(message)
        self.provider = provider
        self.retryable = retryable
        self.status_code = status_code
        self.request_id = request_id


class ProviderUnavailableError(ProviderError):
    """No authorized provider could satisfy the request.

    In ``production`` mode this is raised instead of any silent fallback.
    """


class ProviderLockViolationError(ConfigurationError):
    """A configuration or runtime injection attempted to bypass the
    production provider boundary (OpenAI for reasoning, BFL FLUX Kontext
    for image generation)."""


class DownloadPolicyError(ProviderError):
    """A provider-supplied URL or payload violated the download policy
    (scheme, host allowlist, content type, size bound or image validity)."""


class JobConcurrencyError(StudioError):
    """Another live worker already holds the job lock."""


class JobStateError(StudioError):
    """An invalid job/stage status transition was attempted."""


class StageRetryExhaustedError(StudioError):
    """A stage exceeded its configured attempt budget."""


class UnsafeCommandError(StudioError):
    """A temporal/external command configuration was rejected as unsafe."""


class PreflightError(StudioError):
    """A preflight check with FAIL status blocked the requested action."""


class ProviderAuthenticationError(ProviderError):
    """The provider rejected the credentials (401/403); never retried."""


class ProviderRateLimitError(ProviderError):
    """The provider rate-limited the request (429) beyond the retry budget."""


class ProviderTimeoutError(ProviderError):
    """The provider did not answer within the configured deadline."""


class ReferenceVideoError(StudioError):
    """The reference video is missing, unreadable or has an unsupported format."""


class RenderDependencyError(PreflightError):
    """A rendering dependency (Node/npm/npx/FFmpeg) required by the selected
    backend is unavailable."""


class PipelineStageError(StudioError):
    """A pipeline stage failed; carries the failing stage id."""

    def __init__(self, message: str, *, stage_id: str = ""):
        super().__init__(message)
        self.stage_id = stage_id


class PublishingError(StudioError):
    """Publishing the final artifact failed."""


class JobCancelledError(StudioError):
    """The run was cancelled by the user; it stopped after the last safe
    stage and can be resumed with the same job id."""


def classify_provider_error(exc: BaseException) -> BaseException:
    """Translate a generic ProviderError into a more specific class for
    user-facing display, based on its status code. Returns the original
    exception when no better classification exists."""
    status = getattr(exc, "status_code", None)
    provider = getattr(exc, "provider", "")
    if status in (401, 403):
        return ProviderAuthenticationError(str(exc), provider=provider, status_code=status)
    if status == 429:
        return ProviderRateLimitError(str(exc), provider=provider, retryable=True, status_code=status)
    if status == 408 or "timed out" in str(exc).lower():
        return ProviderTimeoutError(str(exc), provider=provider, retryable=True, status_code=status)
    return exc


In [ ]:
%%writefile scistudio_v10/flux_prompt_system.py
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Iterable

from .schemas import (
    ArtDirectionBible,
    ContinuityCanon,
    DirectorChangeOrder,
    DrawingBrief,
    FluxGenerationPolicy,
    FluxPromptBlock,
    FluxPromptDiagnostics,
    FluxPromptStack,
    FluxStyleFingerprint,
    ReferencePack,
    SceneIllustrationArchitecture,
)
from .utils import ensure_dir, hash_value, save_json


class FluxPromptSystem:
    """Production prompt compiler for FLUX.1 Kontext [pro].

    It keeps a short immutable style fingerprint verbatim across the production,
    then describes only the scene delta. For edits it follows a strict local
    contract: name the target, state the drawable change, and explicitly preserve
    everything else. The prompt is natural language rather than a JSON dump.
    """

    VAGUE = (
        "make it better",
        "make it professional",
        "improve it",
        "fix the image",
        "more dynamic",
        "more beautiful",
        "less childish",
        "enhance quality",
    )
    CONTRADICTIONS = (
        ("camera locked", "camera movement"),
        ("flat paper field", "glossy 3d"),
        ("adult proportion", "oversized head"),
        ("preserve everything", "redesign everything"),
    )

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)
        self.policy = FluxGenerationPolicy(
            model=str(config.get("model", "flux-kontext-pro")),
            aspect_ratio=str(config.get("aspect_ratio", "9:16")),
            prompt_upsampling=bool(config.get("prompt_upsampling", False)),
            safety_tolerance=int(config.get("safety_tolerance", 2)),
            output_format=str(config.get("output_format", "png")),
            seed_base=int(config.get("seed_base", 240921)),
            max_initial_prompt_words=int(config.get("max_initial_prompt_words", 380)),
            min_initial_prompt_words=int(config.get("min_initial_prompt_words", 80)),
            max_edit_prompt_words=int(config.get("max_edit_prompt_words", 180)),
            max_adjustments_per_pass=int(config.get("max_adjustments_per_pass", 1)),
        )
        save_json(self.root / "flux_generation_policy.json", self.policy)

    def fingerprint(self, bible: ArtDirectionBible) -> FluxStyleFingerprint:
        canon = bible.locked_canon
        fp = FluxStyleFingerprint(
            canon_id=canon.canon_id,
            medium_sentence=(
                "Create a mature scientific editorial ink illustration for adults as one authored scene on warm paper, never as assembled icons."
            ),
            contour_sentence=(
                "Use fluid charcoal contours with variable pressure, purposeful breaks and tapered ends; lighter interior lines explain form, overlap, perspective and force."
            ),
            anatomy_sentence=("Use believable adult proportions, weight, hands and joints with natural asymmetry."),
            palette_sentence=(
                f"Keep paper {canon.color.paper}, charcoal {canon.color.ink}, slate {canon.color.slate} and blue-gray {canon.color.blue_primary}, with sparse red and yellow causal accents."
            ),
            composition_sentence=(
                "Use one perspective, three to five depth planes, intentional negative space, restrained values and material-specific marks."
            ),
            texture_sentence="",
            anti_ai_sentence=(
                "It must feel drawn by one skilled illustrator, never clip-art, Office shapes, stickers or generic AI concept art."
            ),
        )
        fp.immutable_prompt = self._clean(
            " ".join(
                x
                for x in [
                    fp.medium_sentence,
                    fp.contour_sentence,
                    fp.anatomy_sentence,
                    fp.palette_sentence,
                    fp.composition_sentence,
                    fp.texture_sentence,
                    fp.anti_ai_sentence,
                ]
                if x
            )
        )
        fp.fingerprint_hash = hash_value(fp.immutable_prompt, 20)
        save_json(self.root / "style_fingerprint.json", fp)
        return fp

    def master_anchor(
        self,
        topic: str,
        bible: ArtDirectionBible,
        reference_board_path: str,
    ) -> DrawingBrief:
        fp = self.fingerprint(bible)
        blocks = [
            self._block("image", "image_type", fp.immutable_prompt, immutable=True),
            self._block(
                "subject",
                "subject_action",
                f"Create an original vertical style specimen for the scientific topic family '{topic}'. Show one integrated "
                "physical environment containing an atmospheric flow, a material structure, a measured phenomenon and one "
                "small experimental status panel. It is a studio visual canon, not a final narrative shot.",
            ),
            self._block(
                "composition",
                "composition",
                "Arrange the visual mass asymmetrically through foreground, midground and background, with a clear diagonal "
                "or curved eye path and enough clean paper for later scientific labels.",
            ),
            self._block(
                "edit",
                "edit_scope",
                "Use the input reference only to inherit the high-level experiment-ledger visual language. Replace all subject "
                "matter and composition with original content. Do not reproduce its globe, buildings, waves, labels or layout.",
            ),
            self._negative_block(bible, []),
        ]
        return self._brief(
            brief_id="MASTER_STYLE_ANCHOR",
            scene_id="MASTER",
            purpose="master_anchor",
            blocks=blocks,
            init_image_path=reference_board_path,
            init_strategy="reference_board",
            output_path=self.root / "master_style_anchor.png",
            seed=self.policy.seed_base,
            preserve=["line rhythm", "paper field", "palette roles", "scientific UI grammar"],
            change=["all content and composition"],
            fingerprint=fp,
        )

    def beauty_frame(
        self,
        architecture: SceneIllustrationArchitecture,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        references: ReferencePack,
        narration: str,
        headline: str,
        index: int,
        *,
        style_source_path: str,
        init_strategy: str,
    ) -> DrawingBrief:
        fp = self.fingerprint(bible)
        subject = self._scene_subject(architecture)
        environment = self._environment(architecture)
        composition = self._composition(architecture)
        construction = self._construction(architecture)
        continuity_text = self._continuity(continuity, references)
        animation = self._animation_prep(architecture)
        negatives = self._negative_block(bible, architecture.prohibited_visual_shortcuts)
        blocks = [
            self._block("image", "image_type", fp.immutable_prompt, immutable=True),
            self._block("subject", "subject_action", subject),
            self._block("environment", "environment", environment),
            self._block("composition", "composition", composition),
        ]
        if construction:
            blocks.append(self._block("construction", "anatomy_material", construction))
        blocks.extend(
            [
                self._block("continuity", "continuity", continuity_text, immutable=True),
                self._block("animation", "animation_preparation", animation),
                self._block(
                    "overlay",
                    "lighting_color",
                    "Reserve clean space for later scientific overlays. Draw no narration or final labels in the beauty art.",
                ),
                negatives,
            ]
        )
        brief = self._brief(
            brief_id=f"{architecture.scene_id}_BEAUTY",
            scene_id=architecture.scene_id,
            purpose="beauty_frame",
            blocks=blocks,
            init_image_path=style_source_path,
            init_strategy=init_strategy,
            output_path=self.root / "beauty" / f"{architecture.scene_id}_draft_00.png",
            seed=self.policy.seed_base + index,
            preserve=[
                "exact house line rhythm",
                "paper field and palette roles",
                "adult construction language",
                "continuity subjects explicitly named in the brief",
            ],
            change=["scene content", "pose", "environment state", "causal phenomenon"],
            fingerprint=fp,
            semantic_requirements=[m.seam_id + ": " + m.region for m in architecture.motion_seams],
            motion_requirements=architecture.animation_representation,
            anchor_board_path=references.board_path,
        )
        brief.request_metadata.update(
            {
                "narration_context": narration,
                "headline_context": headline,
                "reference_requirements": [r.model_dump(mode="json") for r in references.requirements],
            }
        )
        self._save(brief)
        return brief

    def revision_passes(
        self,
        original: DrawingBrief,
        order: DirectorChangeOrder,
        current_path: str,
        revision_cycle: int,
    ) -> list[DrawingBrief]:
        ranked = sorted(order.adjustments, key=lambda a: self._priority(a.priority))
        if not ranked and order.status == "revise":
            raise ValueError("Director revision requires concrete adjustments")
        groups: list[list[Any]] = []
        for adjustment in ranked:
            if (
                groups
                and len(groups[-1]) < self.policy.max_adjustments_per_pass
                and self._region_root(groups[-1][0].target_region) == self._region_root(adjustment.target_region)
            ):
                groups[-1].append(adjustment)
            else:
                groups.append([adjustment])
        briefs: list[DrawingBrief] = []
        source = current_path
        for pass_index, adjustments in enumerate(groups, 1):
            target_regions = [a.target_region for a in adjustments]
            preserve = self._unique(
                [
                    *order.immutable_preserve_list,
                    *[item for a in adjustments for item in a.preserve],
                    "all pixels, line rhythm, palette, camera and geometry outside the named target region",
                ]
            )
            change_sentences = [f"In {a.target_region}, {a.instruction.rstrip('.')}" for a in adjustments]
            instruction = (
                f"Edit only {', '.join(target_regions)}. " + " ".join(change_sentences) + ". "
                f"Preserve unchanged: {'; '.join(preserve)}. Do not restyle, recompose, crop, relight or redesign any unaffected region."
            )
            blocks = [
                self._block("edit", "edit_scope", instruction),
                self._block(
                    "consistency",
                    "continuity",
                    "Keep the exact approved authored ink language, adult anatomy, line pressure, palette, paper texture and camera from the input image.",
                    immutable=True,
                ),
            ]
            brief_id = f"{original.scene_id}_REV_{revision_cycle:02d}_{pass_index:02d}"
            output = self.root / "beauty" / f"{original.scene_id}_draft_{revision_cycle:02d}_{pass_index:02d}.png"
            brief = self._brief(
                brief_id=brief_id,
                scene_id=original.scene_id,
                purpose="revision",
                blocks=blocks,
                init_image_path=source,
                init_strategy="current_revision",
                output_path=output,
                seed=original.seed,
                preserve=preserve,
                change=change_sentences,
                fingerprint_hash=original.style_fingerprint_hash,
                semantic_requirements=original.semantic_requirements,
                motion_requirements=original.motion_requirements,
            )
            brief.request_metadata["revision_cycle"] = revision_cycle
            brief.request_metadata["revision_pass"] = pass_index
            brief.request_metadata["target_regions"] = target_regions
            briefs.append(brief)
            source = str(output)
        return briefs

    def pose_variant(
        self,
        original: DrawingBrief,
        approved_frame_path: str,
        variant_name: str,
        exact_change: str,
    ) -> DrawingBrief:
        safe = self._safe(variant_name)
        instruction = (
            f"Edit only the local animation state named '{variant_name}': {exact_change.rstrip('.')}. "
            "Keep the exact composition, camera, background, face identity, line pressure, palette, material marks and every pixel outside that local region unchanged. "
            "The unchanged area must register with the approved input frame for replacement animation; do not redesign or improve anything else."
        )
        blocks = [
            self._block("edit", "edit_scope", instruction),
            self._block(
                "style",
                "continuity",
                "Preserve the exact approved illustrated style from the input image; this is a local state edit, not a new generation.",
                immutable=True,
            ),
        ]
        return self._brief(
            brief_id=f"{original.scene_id}_POSE_{safe.upper()}",
            scene_id=original.scene_id,
            purpose="pose_variant",
            blocks=blocks,
            init_image_path=approved_frame_path,
            init_strategy="approved_beauty_frame",
            output_path=self.root / "poses" / original.scene_id / f"{safe}.png",
            seed=original.seed,
            preserve=["all unaffected pixels", *original.preserve],
            change=[exact_change],
            fingerprint_hash=original.style_fingerprint_hash,
            semantic_requirements=original.semantic_requirements,
            motion_requirements=original.motion_requirements,
        )

    def lint(self, brief: DrawingBrief) -> FluxPromptDiagnostics:
        prompt = brief.compiled_prompt or (brief.prompt_stack.compiled_prompt if brief.prompt_stack else "")
        low = prompt.lower()
        errors: list[str] = []
        warnings: list[str] = []
        vague = [x for x in self.VAGUE if x in low]
        contradictions = [f"{a} <> {b}" for a, b in self.CONTRADICTIONS if a in low and b in low]
        words = self._word_count(prompt)
        edit = brief.purpose in {"revision", "pose_variant", "layer_isolation"}
        if not prompt.strip():
            errors.append("compiled prompt is empty")
        if "{" in prompt or "}" in prompt:
            errors.append("prompt contains JSON-like braces")
        if vague:
            errors.append("vague edit language: " + ", ".join(vague))
        if contradictions:
            errors.append("contradictory instructions: " + ", ".join(contradictions))
        if edit:
            if words > self.policy.max_edit_prompt_words:
                errors.append("edit prompt is too long")
            if not any(x in low for x in ("edit only", "change only", "local animation state")):
                errors.append("edit prompt lacks a local target scope")
            if not any(x in low for x in ("preserve", "keep the exact", "unchanged")):
                errors.append("edit prompt lacks explicit preservation")
        else:
            if words < self.policy.min_initial_prompt_words:
                warnings.append("initial prompt may be under-specified")
            if words > self.policy.max_initial_prompt_words:
                errors.append("initial prompt is bloated")
            first = " ".join(prompt.split()[:55]).lower()
            if not any(x in first for x in ("scientific editorial", "editorial ink", "illustration")):
                errors.append("style and image type are not stated early")
        diagnostics = FluxPromptDiagnostics(
            valid=not errors,
            errors=errors,
            warnings=warnings,
            word_count=words,
            immutable_style_hash=brief.style_fingerprint_hash,
            has_explicit_subject=any(x in low for x in ("focal", "show", "scene", "subject")),
            has_explicit_preservation=any(x in low for x in ("preserve", "keep the exact", "unchanged")),
            has_local_edit_scope=any(x in low for x in ("edit only", "change only", "local animation state")),
            vague_language_found=vague,
            contradictory_language_found=contradictions,
        )
        return diagnostics

    def _fit_initial_budget(self, blocks: list[FluxPromptBlock], max_words: int) -> list[FluxPromptBlock]:
        """Trim descriptive blocks so the assembled initial prompt fits the
        word budget. Immutable blocks (style/image type, continuity) and the
        negative-constraints block are preserved in full; the remaining
        LLM-authored blocks are shortened proportionally to their length, each
        keeping at least a short head so no scene element is dropped entirely.
        Deterministic, so a resumed job reproduces the same prompt.
        """
        protected_roles = {"negative_constraints"}

        def word_count(text: str) -> int:
            return len(text.split())

        def is_fixed(block: FluxPromptBlock) -> bool:
            return block.immutable or block.role in protected_roles

        fixed_words = sum(word_count(b.text) for b in blocks if is_fixed(b))
        trimmable = [b for b in blocks if not is_fixed(b)]
        trimmable_total = sum(word_count(b.text) for b in trimmable) or 1
        budget = max(0, max_words - fixed_words)

        result: list[FluxPromptBlock] = []
        for block in blocks:
            if is_fixed(block):
                result.append(block)
                continue
            share = max(8, int(budget * word_count(block.text) / trimmable_total))
            words = block.text.split()
            if len(words) > share:
                trimmed = " ".join(words[:share]).rstrip(",;: ") + "."
                result.append(
                    self._block(block.block_id, block.role, trimmed, immutable=block.immutable, priority=block.priority)
                )
            else:
                result.append(block)
        return result

    def _brief(
        self,
        *,
        brief_id: str,
        scene_id: str,
        purpose: str,
        blocks: list[FluxPromptBlock],
        init_image_path: str,
        init_strategy: str,
        output_path: str | Path,
        seed: int | None,
        preserve: list[str],
        change: list[str],
        fingerprint: FluxStyleFingerprint | None = None,
        fingerprint_hash: str = "",
        semantic_requirements: list[str] | None = None,
        motion_requirements: list[str] | None = None,
        anchor_board_path: str = "",
    ) -> DrawingBrief:
        # Fit an over-long INITIAL prompt to the word budget by trimming the
        # descriptive LLM-authored blocks — never the immutable style/continuity
        # blocks or the negative constraints — instead of aborting an expensive
        # production run. Edit prompts keep their own (stricter) length gate.
        edit = purpose in {"revision", "pose_variant", "layer_isolation"}
        if not edit and self._word_count(self._compile(blocks)) > self.policy.max_initial_prompt_words:
            blocks = self._fit_initial_budget(blocks, self.policy.max_initial_prompt_words)
        compiled = self._compile(blocks)
        style_hash = fingerprint.fingerprint_hash if fingerprint else fingerprint_hash
        stack = FluxPromptStack(
            purpose=purpose,
            blocks=blocks,
            compiled_prompt=compiled,
            immutable_style_hash=style_hash,
            scene_delta_hash=hash_value([b.text for b in blocks if not b.immutable], 20),
            word_count=self._word_count(compiled),
            init_strategy=init_strategy,
        )
        negative_text = next((b.text for b in blocks if b.role == "negative_constraints"), "")
        edit_text = next((b.text for b in blocks if b.role == "edit_scope"), "")
        brief = DrawingBrief(
            brief_id=brief_id,
            scene_id=scene_id,
            purpose=purpose,
            positive_prompt=compiled,
            negative_prompt=negative_text,
            kontext_instruction=edit_text or compiled,
            anchor_board_path=anchor_board_path,
            init_image_path=init_image_path,
            output_path=str(output_path),
            aspect_ratio=self.policy.aspect_ratio,
            seed=seed,
            preserve=preserve,
            change=change,
            semantic_requirements=semantic_requirements or [],
            motion_requirements=motion_requirements or [],
            prompt_stack=stack,
            compiled_prompt=compiled,
            style_fingerprint_hash=style_hash,
            init_strategy=init_strategy,
            prompt_upsampling=self.policy.prompt_upsampling,
            safety_tolerance=self.policy.safety_tolerance,
            output_format=self.policy.output_format,
            request_metadata={"model": self.policy.model, "policy": self.policy.model_dump(mode="json")},
        )
        brief.prompt_diagnostics = self.lint(brief)
        if not brief.prompt_diagnostics.valid:
            raise ValueError(f"Invalid FLUX prompt {brief_id}: {'; '.join(brief.prompt_diagnostics.errors)}")
        self._save(brief)
        return brief

    def _save(self, brief: DrawingBrief) -> None:
        ensure_dir(Path(brief.output_path).parent)
        save_json(self.root / "briefs" / f"{brief.brief_id}.json", brief)
        if brief.prompt_diagnostics:
            save_json(self.root / "diagnostics" / f"{brief.brief_id}.json", brief.prompt_diagnostics)

    @staticmethod
    def _block(block_id: str, role: str, text: str, immutable: bool = False, priority: int = 1) -> FluxPromptBlock:
        return FluxPromptBlock(
            block_id=block_id, role=role, text=FluxPromptSystem._clean(text), immutable=immutable, priority=priority
        )

    def _negative_block(self, bible: ArtDirectionBible, extra: list[str]) -> FluxPromptBlock:
        items = self._unique(
            [
                "child mascot anatomy",
                "oversized round heads",
                "tube or capsule limbs",
                "mitten or oval hands",
                "uniform sticker outlines",
                "isolated icon collage",
                "Microsoft Word shape assembly",
                "Office-shape geometry",
                "glossy generic AI rendering",
                "random ornamental detail",
                "incoherent perspective",
                "visible animation cutout seams",
                *bible.locked_canon.forbidden,
                *extra,
            ]
        )
        # Keep negatives concise. FLUX responds better to a clear positive image description;
        # this final sentence only blocks the most damaging failure modes.
        return self._block("negative", "negative_constraints", "Avoid " + ", ".join(items[:8]) + ".")

    @staticmethod
    def _compile(blocks: Iterable[FluxPromptBlock]) -> str:
        # Preserve the director-authored order. FLUX benefits from image type and
        # main subject appearing early; alphabetical sorting previously buried them.
        return FluxPromptSystem._clean(" ".join(b.text for b in blocks if b.text.strip()))

    @staticmethod
    def _scene_subject(a: SceneIllustrationArchitecture) -> str:
        focus = FluxPromptSystem._strip_directive(
            FluxPromptSystem._clean(a.focal_subject or a.visual_thesis).rstrip(" .;:")
        )
        claim = FluxPromptSystem._strip_directive(
            FluxPromptSystem._clean(a.narrative_claim or a.visual_thesis).rstrip(" .;:")
        )
        secondaries = FluxPromptSystem._unique(
            FluxPromptSystem._clean(x).rstrip(" .;:")
            for x in a.secondary_subjects
            if FluxPromptSystem._clean(x).lower() not in {focus.lower(), claim.lower()}
        )[:3]
        text = f"Show {focus}. Make {claim} the dominant causal event."
        if secondaries:
            text += f" Integrate {', '.join(secondaries)} into that same scene, never as separate icons."
        return text

    @staticmethod
    def _environment(a: SceneIllustrationArchitecture) -> str:
        focus = FluxPromptSystem._clean(a.focal_subject or a.visual_thesis).lower()
        parts = FluxPromptSystem._unique(
            FluxPromptSystem._clean(p.contents).rstrip(" .;:")
            for p in a.depth_planes
            if p.depth != "overlay" and FluxPromptSystem._clean(p.contents).lower() != focus
        )[:3]
        contents = ", ".join(parts) if parts else "a coherent physical setting"
        return f"Build one integrated environment from {contents}. Connect foreground, subject and atmosphere through overlap, scale and perspective; keep the causal action legible on a phone."

    @staticmethod
    def _composition(a: SceneIllustrationArchitecture) -> str:
        route_items = [
            FluxPromptSystem._route_phrase(x)
            for x in (a.composition_route[:3] or ["focal subject", "causal consequence"])
        ]
        route = " to ".join(route_items)
        p = a.perspective
        view = re.split(r"\bwhen\b", FluxPromptSystem._clean(p.view).split(";")[0], maxsplit=1, flags=re.I)[0].rstrip(
            " ."
        )
        height = re.split(
            r"\baccording\b", FluxPromptSystem._clean(p.camera_height).split(";")[0], maxsplit=1, flags=re.I
        )[0].rstrip(" .")
        negative = FluxPromptSystem._clean(a.negative_space or "Keep deliberate negative space beside the focal route.")
        return f"Use {view} from {height}, one horizon near {p.horizon_y:.2f} of frame height, and guide the eye from {route}. {negative}"

    @staticmethod
    def _construction(a: SceneIllustrationArchitecture) -> str:
        chunks: list[str] = []
        for f in a.figure_construction[:2]:
            chunks.append(
                f"Construct {f.figure_id} at {f.proportion_heads:.1f} adult heads, {f.body_orientation}, with {f.weight_distribution}; "
                f"hands use palm, knuckle and thumb structure and clothing follows tension and gravity."
            )
        for m in a.material_marks[:3]:
            cues = ", ".join(m.visual_cues[:2])
            marks = ", ".join(m.line_marks[:2])
            chunks.append(f"Describe {m.material} through {cues}, using {marks} and restrained values.")
        if a.contour_architecture:
            chunks.append("Contour rule: " + "; ".join(a.contour_architecture[:2]) + ".")
        return " ".join(chunks)

    @staticmethod
    def _continuity(c: ContinuityCanon, r: ReferencePack) -> str:
        prior = (
            "Use the previous approved scene as visual DNA"
            if r.previous_approved_scene
            else "Use the master anchor as visual DNA"
        )
        recurring_names = [
            getattr(x, "description", "") or getattr(x, "subject_id", "") or str(x) for x in c.recurring_subjects[:3]
        ]
        recurring = (
            ", ".join(FluxPromptSystem._clean(x) for x in recurring_names if FluxPromptSystem._clean(x))
            or "recurring subjects"
        )
        return (
            f"Continuity lock: {prior}. Preserve house line rhythm, palette roles, paper field, value hierarchy and the construction of {recurring}. "
            "Change only this shot's narrative content; never drift into another illustrator, anatomy system or finish."
        )

    @staticmethod
    def _animation_prep(a: SceneIllustrationArchitecture) -> str:
        seams = "; ".join(f"{m.region} via {m.method}" for m in a.motion_seams[:3]) or "no local moving region"
        return f"Prepare later motion at {seams}. Hide seams inside natural overlaps, material boundaries or atmosphere; never show puppet joints or sticker edges."

    @staticmethod
    def _strip_directive(text: str) -> str:
        return re.sub(
            r"^(show|establish|depict|illustrate|visualize|create|resolve|conclude|summarize)\s+", "", text, flags=re.I
        ).strip()

    @staticmethod
    def _route_phrase(text: str) -> str:
        value = FluxPromptSystem._clean(text).rstrip(" .;:")
        value = re.split(
            r"\b(?:establishes|reveals|shows|leads|guides|indicates|explains)\b", value, maxsplit=1, flags=re.I
        )[0]
        return FluxPromptSystem._strip_directive(value).strip() or "causal consequence"

    @staticmethod
    def _clean(text: str) -> str:
        return re.sub(r"\s+", " ", str(text)).strip()

    @staticmethod
    def _word_count(text: str) -> int:
        return len(re.findall(r"\b[\w#./'-]+\b", text))

    @staticmethod
    def _safe(value: str) -> str:
        return "-".join("".join(ch.lower() if ch.isalnum() else " " for ch in value).split())

    @staticmethod
    def _unique(items: Iterable[str]) -> list[str]:
        out: list[str] = []
        for item in items:
            item = str(item).strip()
            if item and item not in out:
                out.append(item)
        return out

    @staticmethod
    def _priority(value: str) -> int:
        return {"critical": 0, "high": 1, "medium": 2, "low": 3}.get(str(value), 2)

    @staticmethod
    def _region_root(value: str) -> str:
        return re.split(r"[./:_-]", value.lower())[0]


In [ ]:
%%writefile scistudio_v10/flux_studio.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

from PIL import Image, ImageDraw, ImageOps

from .drawing_brief import DrawingBriefCompiler
from .llm import LLMRouter
from .schemas import (
    ArtDirectionBible,
    BeautyFrame,
    ContinuityCanon,
    DirectorChangeOrder,
    DrawingBrief,
    RevisionRecord,
    SceneIllustrationArchitecture,
)
from .utils import ensure_dir, hash_value, save_json, sha256_file


class FluxKontextStudio:
    """Executive-art-directed FLUX.1 Kontext [pro] production studio.

    The model receives one visual input at a time, matching the Kontext Pro API.
    A master anchor establishes the house style. Each scene then uses either that
    anchor or the previous approved scene as its single input. Edits are local,
    explicit and sequential rather than vague multi-region rewrites.
    """

    def __init__(
        self,
        llm: LLMRouter,
        brief_compiler: DrawingBriefCompiler,
        config: dict[str, Any],
        root: str | Path,
        image_generator: Callable[[DrawingBrief], Path | None] | None = None,
    ):
        self.llm = llm
        self.briefs = brief_compiler
        self.config = config
        self.root = ensure_dir(root)
        self.image_generator = image_generator

    def generate(self, brief: DrawingBrief, *, force: bool = False) -> Path | None:
        output = Path(brief.output_path)
        ensure_dir(output.parent)
        if output.exists() and output.stat().st_size > 1024 and not force:
            return output

        prompt = brief.compiled_prompt or (
            brief.prompt_stack.compiled_prompt if brief.prompt_stack else brief.positive_prompt
        )
        if not prompt.strip():
            raise ValueError(f"Drawing brief {brief.brief_id} has no compiled FLUX prompt")
        if brief.prompt_diagnostics and not brief.prompt_diagnostics.valid:
            raise ValueError(f"Drawing brief {brief.brief_id} failed prompt diagnostics")
        init = brief.init_image_path if brief.init_image_path and Path(brief.init_image_path).exists() else None
        manifest = {
            "brief_id": brief.brief_id,
            "scene_id": brief.scene_id,
            "purpose": brief.purpose,
            "model": brief.request_metadata.get(
                "model", getattr(self.llm, "config", {}).get("bfl_model", "flux-kontext-pro")
            ),
            "prompt": prompt,
            "prompt_hash": hash_value(prompt, 24),
            "prompt_word_count": len(prompt.split()),
            "style_fingerprint_hash": brief.style_fingerprint_hash,
            "init_strategy": brief.init_strategy,
            "init_image_path": str(init or ""),
            "init_image_hash": sha256_file(init) if init else "",
            "aspect_ratio": brief.aspect_ratio,
            "seed": brief.seed,
            "prompt_upsampling": brief.prompt_upsampling,
            "safety_tolerance": brief.safety_tolerance,
            "output_format": brief.output_format,
            "output_path": str(output),
        }
        save_json(self.root / "requests" / f"{brief.brief_id}.json", manifest)

        if self.image_generator is not None:
            return self.image_generator(brief)

        # Context manager behavior without introducing a global mutable provider
        # state after the call returns.
        keys = {
            "bfl_aspect_ratio": brief.aspect_ratio,
            "bfl_seed": brief.seed,
            "bfl_prompt_upsampling": brief.prompt_upsampling,
            "bfl_safety_tolerance": brief.safety_tolerance,
            "bfl_output_format": brief.output_format,
        }
        previous = {k: self.llm.config.get(k, None) for k in keys}
        for key, value in keys.items():
            if value is None:
                self.llm.config.pop(key, None)
            else:
                self.llm.config[key] = value
        try:
            return self.llm.generate_reference_image(prompt=prompt, output_path=output, init_image=init, force=force)
        finally:
            for key, value in previous.items():
                if value is None:
                    self.llm.config.pop(key, None)
                else:
                    self.llm.config[key] = value

    def create_master_anchor(self, brief: DrawingBrief, *, force: bool = False) -> str:
        result = self.generate(brief, force=force)
        if result is None:
            raise RuntimeError("FLUX Kontext Pro could not create the master style anchor")
        save_json(
            self.root / "master_anchor_manifest.json",
            {
                "path": str(result),
                "brief_id": brief.brief_id,
                "canon_id": brief.canon_id,
                "seed": brief.seed,
                "provider": "flux-kontext-pro",
                "style_fingerprint_hash": brief.style_fingerprint_hash,
            },
        )
        return str(result)

    def direct_scene(
        self,
        brief: DrawingBrief,
        architecture: SceneIllustrationArchitecture,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        *,
        initial_path: str | None = None,
        force: bool = False,
    ) -> BeautyFrame:
        first = Path(initial_path) if initial_path else self.generate(brief, force=force)
        if first is None or not Path(first).exists():
            raise RuntimeError(f"FLUX Kontext Pro could not create scene {brief.scene_id}")
        revisions: list[RevisionRecord] = []
        current = str(first)
        max_cycles = int(self.config.get("maximum_director_revisions", 3))
        strict_director = bool(self.config.get("require_vision_director", True))
        approval_source = ""

        for revision_cycle in range(1, max_cycles + 1):
            comparison = self._comparison_board(current, brief.scene_id, continuity, revision_cycle)
            change = self._director_review(
                comparison,
                brief,
                architecture,
                bible,
                continuity,
                revision_number=revision_cycle,
                force=force,
            )
            save_json(self.root / "change_orders" / f"{brief.scene_id}_{revision_cycle:02d}.json", change)
            if change.status == "approve":
                approval_source = "vision-llm-art-director"
                break
            if change.status == "requires_human_or_vision_director":
                if strict_director:
                    raise RuntimeError(
                        f"Scene {brief.scene_id} requires an available vision art director. "
                        "The pipeline will not substitute procedural or unreviewed hero illustration."
                    )
                approval_source = "unreviewed-preview"
                break

            pass_briefs = self.briefs.revision_passes(brief, change, current, revision_cycle)
            for pass_number, revision_brief in enumerate(pass_briefs, 1):
                revised = self.generate(revision_brief, force=force)
                if revised is None:
                    raise RuntimeError(f"FLUX revision {revision_cycle}.{pass_number} failed for {brief.scene_id}")
                revisions.append(
                    RevisionRecord(
                        scene_id=brief.scene_id,
                        revision_number=len(revisions) + 1,
                        draft_path=current,
                        change_order=change,
                        output_path=str(revised),
                    )
                )
                current = str(revised)
        else:
            raise RuntimeError(f"Scene {brief.scene_id} exhausted revision budget without approval")

        frame = BeautyFrame(
            scene_id=brief.scene_id,
            image_path=current,
            approved=approval_source in {"vision-llm-art-director", "human-director"},
            approval_source=approval_source,
            style_anchor_ids=[brief.init_image_path, brief.anchor_board_path],
            revision_records=revisions,
            prompt_hash=hash_value(brief.model_dump(mode="json"), 20),
            seed=brief.seed,
        )
        save_json(self.root / "beauty_frames" / f"{brief.scene_id}.json", frame)
        return frame

    def create_pose_variants(
        self,
        beauty: BeautyFrame,
        original_brief: DrawingBrief,
        architecture: SceneIllustrationArchitecture,
        *,
        force: bool = False,
    ) -> dict[str, str]:
        if not beauty.approved:
            raise RuntimeError("Pose variants may only be derived from an approved beauty frame")
        variants: dict[str, str] = {}
        descriptions = self._variant_descriptions(architecture)
        for name, change in descriptions.items():
            brief = self.briefs.pose_variant(original_brief, beauty.image_path, name, change)
            result = self.generate(brief, force=force)
            if result is None:
                raise RuntimeError(f"Pose/state variant generation failed: {architecture.scene_id}/{name}")
            variants[name] = str(result)
        save_json(self.root / "pose_variants" / f"{architecture.scene_id}.json", variants)
        return variants

    def _comparison_board(
        self,
        current_path: str,
        scene_id: str,
        continuity: ContinuityCanon,
        revision_number: int,
    ) -> str:
        master = next(
            (a.path for a in continuity.anchors if a.role == "master_style_anchor" and Path(a.path).exists()), ""
        )
        approved = [a.path for a in continuity.anchors if a.role == "approved_scene" and Path(a.path).exists()]
        previous = approved[-1] if approved else ""
        items = [("MASTER STYLE", master), ("PREVIOUS APPROVED", previous), ("CURRENT DRAFT", current_path)]
        output = self.root / "director_boards" / f"{scene_id}_{revision_number:02d}.png"
        ensure_dir(output.parent)
        canvas = Image.new("RGB", (1536, 1024), "#E6E8E7")
        draw = ImageDraw.Draw(canvas)
        cells = [(0, 0, 512, 1024), (512, 0, 1024, 1024), (1024, 0, 1536, 1024)]
        for (label, path), box in zip(items, cells):
            x0, y0, x1, y1 = box
            draw.rectangle(box, fill="#FAFAF7", outline="#475157", width=3)
            if path and Path(path).exists():
                image = Image.open(path).convert("RGB")
                fitted = ImageOps.contain(image, (x1 - x0 - 28, y1 - y0 - 90))
                canvas.paste(fitted, (x0 + (x1 - x0 - fitted.width) // 2, y0 + 50))
            draw.rectangle((x0 + 8, 8, x1 - 8, 42), fill="#20282D")
            draw.text((x0 + 18, 16), label, fill="white")
        canvas.save(output)
        return str(output)

    def _director_review(
        self,
        comparison_path: str,
        brief: DrawingBrief,
        architecture: SceneIllustrationArchitecture,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        *,
        revision_number: int,
        force: bool,
    ) -> DirectorChangeOrder:
        fallback = DirectorChangeOrder(
            scene_id=brief.scene_id,
            revision_number=revision_number,
            status="requires_human_or_vision_director",
            diagnosis="No vision director was available; aesthetic approval cannot be inferred from technical metrics.",
            adjustments=[],
            immutable_preserve_list=brief.preserve,
        )
        raw = self.llm.critique_image(
            image_path=comparison_path,
            namespace=f"v10_1_art_director_{brief.scene_id}_{revision_number:02d}",
            force=force,
            fallback=fallback.model_dump(mode="json"),
            prompt=f"""You are the Executive Art Director reviewing a three-panel comparison board.
Left: MASTER STYLE. Center: PREVIOUS APPROVED SCENE when available. Right: CURRENT DRAFT.
Return JSON only as a DirectorChangeOrder. You are directing the next drawing pass, not assigning a score.

LOCKED BIBLE:
{json.dumps(bible.model_dump(mode="json"), ensure_ascii=False)}

SCENE ARCHITECTURE:
{json.dumps(architecture.model_dump(mode="json"), ensure_ascii=False)}

CONTINUITY:
{json.dumps(continuity.model_dump(mode="json"), ensure_ascii=False)}

FLUX PROMPT DIAGNOSTICS:
{json.dumps(brief.prompt_diagnostics.model_dump(mode="json") if brief.prompt_diagnostics else {}, ensure_ascii=False)}

Compare the CURRENT DRAFT against the MASTER and PREVIOUS panels. Inspect composition, coherent perspective,
fluid authored contour rhythm, line hierarchy, adult anatomy and weight, hand/joint construction, material marks,
palette roles, negative space, recurring subject identity and hidden motion seams. Detect generic AI smoothness,
mascot construction, sticker outlines, clip-art isolation, random details and style drift.

If ready, return status=approve with no adjustments. If revision is needed, return status=revise with one to three
ConcreteAdjustments only. Prioritize the most damaging visible issue. Each adjustment must name a precise target region,
state the visible problem, give a measurable or drawable instruction, list what must remain unchanged, and assign priority.
Do not say only 'make it better', 'more professional', 'less childish' or 'more dynamic'. Do not ask Kontext to redesign
unaffected regions. The prompt compiler will convert your adjustments into sequential local edit passes.""",
        )
        try:
            return DirectorChangeOrder.model_validate(raw)
        except Exception:
            return fallback

    @staticmethod
    def _variant_descriptions(architecture: SceneIllustrationArchitecture) -> dict[str, str]:
        variants: dict[str, str] = {}
        for seam in architecture.motion_seams:
            for variant in seam.required_variants:
                key = f"{seam.seam_id}-{variant}"
                variants[key] = (
                    f"Change {seam.region} to the '{variant}' state of {seam.subject}; "
                    f"representation method {seam.method}. Preserve all unaffected pixels and the natural overlap at the seam."
                )
        return variants


In [ ]:
%%writefile scistudio_v10/http_safety.py
"""HTTP safety primitives shared by every network client in the studio.

Provides:

* :func:`validate_url` — scheme + host-allowlist policy for provider URLs
  (submission endpoints, polling URLs, result/download URLs). Prevents SSRF
  through provider-supplied links.
* :func:`is_retryable_status` / :func:`classify_exception` — retry
  classification (408, 409, 429 and 5xx are retryable; 4xx auth/validation
  failures are not).
* :func:`backoff_delays` — exponential backoff with jitter, honouring a
  server ``Retry-After`` hint when supplied.
* :func:`download_image` — bounded, validated, atomic image download
  (``raise_for_status``, Content-Type check, byte cap, Pillow verification,
  temp-file + atomic rename).
"""

from __future__ import annotations

import io
import os
import random
import tempfile
import time
from pathlib import Path
from typing import Any, Callable, Iterable
from urllib.parse import urlsplit

from .errors import DownloadPolicyError

RETRYABLE_STATUS_CODES = frozenset({408, 409, 429, 500, 502, 503, 504})
NON_RETRYABLE_STATUS_CODES = frozenset({400, 401, 403, 404, 422})

DEFAULT_MAX_DOWNLOAD_BYTES = 32 * 1024 * 1024
_DOWNLOAD_CHUNK_BYTES = 256 * 1024


def _host_matches(host: str, pattern: str) -> bool:
    host = host.lower().rstrip(".")
    pattern = pattern.lower().rstrip(".")
    if pattern.startswith("*."):
        suffix = pattern[1:]  # ".bfl.ai"
        return host.endswith(suffix) and host != suffix.lstrip(".")
    return host == pattern


def validate_url(url: str, *, allowed_hosts: Iterable[str], purpose: str = "request") -> str:
    """Validate that *url* is https and its host is in the allowlist.

    Returns the URL unchanged when valid; raises DownloadPolicyError otherwise.
    """
    if not url or not isinstance(url, str):
        raise DownloadPolicyError(f"Empty or non-string URL for {purpose}", provider="http")
    parts = urlsplit(url)
    if parts.scheme != "https":
        raise DownloadPolicyError(
            f"URL scheme {parts.scheme!r} rejected for {purpose}; only https is allowed",
            provider="http",
        )
    host = parts.hostname or ""
    patterns = [p for p in allowed_hosts if p]
    if not patterns:
        raise DownloadPolicyError(
            f"No allowed hosts configured for {purpose}; refusing outbound request",
            provider="http",
        )
    if not any(_host_matches(host, pattern) for pattern in patterns):
        raise DownloadPolicyError(
            f"Host {host!r} is not in the allowed host list for {purpose}",
            provider="http",
        )
    return url


def is_retryable_status(status_code: int | None) -> bool:
    """408/409/429 and 5xx responses are retryable; other 4xx are not."""
    if status_code is None:
        return False
    if status_code in RETRYABLE_STATUS_CODES:
        return True
    return 500 <= status_code <= 599


def classify_exception(exc: BaseException) -> bool:
    """Best-effort retryability classification for arbitrary client errors.

    Connection/timeout errors are retryable. Errors exposing a
    ``status_code`` (or a ``response.status_code``) follow HTTP rules.
    Authentication/validation failures are never retried.
    """
    explicit = getattr(exc, "retryable", None)
    if isinstance(explicit, bool):
        return explicit
    status = getattr(exc, "status_code", None)
    if status is None:
        response = getattr(exc, "response", None)
        status = getattr(response, "status_code", None)
    if status is not None:
        try:
            return is_retryable_status(int(status))
        except (TypeError, ValueError):
            return False
    name = type(exc).__name__.lower()
    if any(marker in name for marker in ("timeout", "connection", "protocol", "chunked")):
        return True
    if any(marker in name for marker in ("authentication", "permission", "badrequest", "notfound")):
        return False
    return isinstance(exc, (ConnectionError, TimeoutError, OSError))


def backoff_delays(
    attempts: int,
    *,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    jitter: float = 0.25,
) -> list[float]:
    """Exponential backoff schedule with multiplicative jitter."""
    delays = []
    for index in range(max(0, attempts - 1)):
        delay = min(max_delay, base_delay * (2**index))
        delay *= 1.0 + random.uniform(-jitter, jitter)
        delays.append(max(0.05, delay))
    return delays


def retry_after_hint(headers: Any) -> float | None:
    """Parse a server Retry-After header (seconds form) when present."""
    try:
        raw = (headers or {}).get("Retry-After")
    except AttributeError:
        return None
    if raw is None:
        return None
    try:
        return max(0.0, float(raw))
    except (TypeError, ValueError):
        return None


def call_with_retries(
    fn: Callable[[], Any],
    *,
    max_attempts: int = 4,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    jitter: float = 0.25,
    is_retryable: Callable[[BaseException], bool] = classify_exception,
    on_retry: Callable[[int, BaseException, float], None] | None = None,
    sleep: Callable[[float], None] = time.sleep,
) -> Any:
    """Run *fn* with bounded retries for retryable failures.

    Non-retryable failures propagate immediately with their original type.
    """
    delays = backoff_delays(max_attempts, base_delay=base_delay, max_delay=max_delay, jitter=jitter)
    attempt = 0
    while True:
        attempt += 1
        try:
            return fn()
        except BaseException as exc:  # noqa: BLE001 - classified below
            if attempt >= max_attempts or not is_retryable(exc):
                raise
            hint = retry_after_hint(getattr(getattr(exc, "response", None), "headers", None))
            delay = hint if hint is not None else delays[min(attempt - 1, len(delays) - 1)]
            if on_retry is not None:
                on_retry(attempt, exc, delay)
            sleep(delay)


def download_image(
    response: Any,
    destination: str | Path,
    *,
    max_bytes: int = DEFAULT_MAX_DOWNLOAD_BYTES,
    require_image_content_type: bool = True,
) -> Path:
    """Persist an already-issued image response safely and atomically.

    The response must expose ``raise_for_status()``, ``headers`` and either
    ``iter_content(chunk_size)`` or ``content``. The payload is size-capped,
    verified with Pillow (rejecting HTML/JSON bodies, empty files and corrupt
    images) and written to a temporary file that is atomically renamed only
    after validation — no partial artifacts are left on failure.
    """
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    response.raise_for_status()
    headers = getattr(response, "headers", {}) or {}
    content_type = str(headers.get("Content-Type", "")).lower().split(";")[0].strip()
    if require_image_content_type and content_type and not content_type.startswith("image/"):
        raise DownloadPolicyError(
            f"Download rejected: Content-Type {content_type!r} is not an image",
            provider="http",
        )

    declared = headers.get("Content-Length")
    if declared is not None:
        try:
            if int(declared) > max_bytes:
                raise DownloadPolicyError(
                    f"Download rejected: declared size {declared} exceeds cap {max_bytes}",
                    provider="http",
                )
        except (TypeError, ValueError):
            pass

    chunks: list[bytes] = []
    total = 0
    iterator = None
    if hasattr(response, "iter_content"):
        iterator = response.iter_content(_DOWNLOAD_CHUNK_BYTES)
    if iterator is not None:
        for chunk in iterator:
            if not chunk:
                continue
            total += len(chunk)
            if total > max_bytes:
                raise DownloadPolicyError(
                    f"Download rejected: payload exceeded cap of {max_bytes} bytes",
                    provider="http",
                )
            chunks.append(chunk)
        payload = b"".join(chunks)
    else:
        payload = getattr(response, "content", b"") or b""
        if len(payload) > max_bytes:
            raise DownloadPolicyError(
                f"Download rejected: payload exceeded cap of {max_bytes} bytes",
                provider="http",
            )

    if not payload:
        raise DownloadPolicyError("Download rejected: empty response body", provider="http")

    try:
        from PIL import Image

        with Image.open(io.BytesIO(payload)) as image:
            image.verify()
    except DownloadPolicyError:
        raise
    except Exception as exc:
        raise DownloadPolicyError(
            f"Download rejected: payload is not a valid image ({type(exc).__name__})",
            provider="http",
        ) from exc

    handle = tempfile.NamedTemporaryFile(
        dir=str(destination.parent), prefix=f".{destination.name}.", suffix=".part", delete=False
    )
    try:
        with handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(handle.name, destination)
    except BaseException:
        try:
            os.unlink(handle.name)
        except OSError:
            pass
        raise
    return destination


In [ ]:
%%writefile scistudio_v10/hybrid_package.py
from __future__ import annotations

from pathlib import Path
from typing import Any

from PIL import Image, ImageChops, ImageOps

from .schemas import (
    AnimationPlan,
    HybridLayer,
    HybridScenePackage,
    SceneRequest,
    SemanticLayerContract,
)
from .utils import ensure_dir, save_json


class HybridPackageBuilder:
    """Converts an approved beauty frame and masks into renderable layers.

    Visible art is always sampled from the approved beauty/pose frames. Masks
    only control alpha. This avoids vector restyling and preserves the studio's
    authored line quality.
    """

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)

    def build(
        self,
        scene: SceneRequest,
        contract: SemanticLayerContract,
        animation: AnimationPlan,
        overlay_path: str,
        voice_path: str = "",
        temporal_clip_path: str = "",
        temporal_backend: str = "",
    ) -> HybridScenePackage:
        out_dir = ensure_dir(self.root / scene.scene_id)
        beauty = Image.open(contract.beauty_frame_path).convert("RGBA")
        moving_masks: list[Image.Image] = []
        layers: list[HybridLayer] = []

        if temporal_clip_path and Path(temporal_clip_path).exists():
            layers.append(
                HybridLayer(layer_id="temporal-beauty", kind="video_clip", path=temporal_clip_path, z_index=0)
            )
            if overlay_path:
                layers.append(
                    HybridLayer(layer_id="scientific-overlay", kind="svg_overlay", path=overlay_path, z_index=100)
                )
            package = HybridScenePackage(
                scene_id=scene.scene_id,
                duration_frames=animation.duration_frames,
                fps=animation.fps,
                canvas=(int(self.config.get("width", 1080)), int(self.config.get("height", 1920))),
                narration=scene.narration,
                headline=scene.headline,
                layers=layers,
                animation=animation,
                voice_path=voice_path,
                transition=scene.transition,
                temporal_clip_path=temporal_clip_path,
                temporal_backend=temporal_backend,
            )
            save_json(out_dir / "hybrid_scene.json", package)
            return package

        semantic = {layer.layer_id: layer for layer in contract.layers}
        animated_targets = {event.target_layer for event in animation.events}
        for target in sorted(animated_targets):
            layer = semantic.get(target)
            if layer is None or not layer.mask_path:
                continue
            mask = Image.open(layer.mask_path).convert("L").resize(beauty.size)
            moving_masks.append(mask)
            cutout = Image.new("RGBA", beauty.size, (0, 0, 0, 0))
            cutout.paste(beauty, (0, 0), mask)
            cutout_path = out_dir / f"{target}_cutout.png"
            cutout.save(cutout_path)
            pose_cutouts: list[str] = []
            for i, pose_path in enumerate(layer.pose_variant_paths):
                pose = Image.open(pose_path).convert("RGBA").resize(beauty.size)
                pose_cutout = Image.new("RGBA", beauty.size, (0, 0, 0, 0))
                pose_cutout.paste(pose, (0, 0), mask)
                pose_out = out_dir / f"{target}_pose_{i:02d}.png"
                pose_cutout.save(pose_out)
                pose_cutouts.append(str(pose_out))
            layers.append(
                HybridLayer(
                    layer_id=target,
                    kind="pose_sequence" if pose_cutouts else "raster",
                    path=str(cutout_path),
                    z_index=20,
                    pose_paths=pose_cutouts,
                )
            )

        if moving_masks:
            union = moving_masks[0]
            for mask in moving_masks[1:]:
                union = ImageChops.lighter(union, mask)
            inverse = ImageOps.invert(union)
            base = Image.new("RGBA", beauty.size, (0, 0, 0, 0))
            base.paste(beauty, (0, 0), inverse)
        else:
            base = beauty
        base_path = out_dir / "beauty_base.png"
        base.save(base_path)
        layers.insert(0, HybridLayer(layer_id="beauty-base", kind="raster", path=str(base_path), z_index=0))
        if overlay_path:
            layers.append(
                HybridLayer(layer_id="scientific-overlay", kind="svg_overlay", path=overlay_path, z_index=100)
            )

        package = HybridScenePackage(
            scene_id=scene.scene_id,
            duration_frames=animation.duration_frames,
            fps=animation.fps,
            canvas=(int(self.config.get("width", 1080)), int(self.config.get("height", 1920))),
            narration=scene.narration,
            headline=scene.headline,
            layers=layers,
            animation=animation,
            voice_path=voice_path,
            transition=scene.transition,
        )
        save_json(out_dir / "hybrid_scene.json", package)
        return package


In [ ]:
%%writefile scistudio_v10/hybrid_render.py
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path
from typing import Any

from PIL import Image

from .schemas import HybridLayer, HybridScenePackage, MotionEvent
from .utils import ensure_dir, save_json


class PILHybridRenderer:
    """Deterministic fallback renderer for hybrid raster/SVG scene packages.

    It is intended for regression and preview. Production can use the generated
    Remotion project for higher-quality interpolation and compositing.
    """

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)

    def render(self, scenes: list[HybridScenePackage], output: str | Path) -> Path:
        output = Path(output)
        ensure_dir(output.parent)
        fps = scenes[0].fps if scenes else int(self.config.get("fps", 30))
        frame_dir = ensure_dir(self.root / "frames")
        for old in frame_dir.glob("*.png"):
            old.unlink()
        global_index = 0
        for scene in scenes:
            assets = self._load_layers(scene)
            for frame in range(scene.duration_frames):
                canvas = Image.new("RGBA", scene.canvas, scene.background)
                for layer in sorted(scene.layers, key=lambda x: x.z_index):
                    image = self._image_for_layer(layer, assets, scene.animation.events, frame, scene.duration_frames)
                    if image is None:
                        continue
                    canvas.alpha_composite(image.resize(scene.canvas))
                canvas.convert("RGB").save(frame_dir / f"frame_{global_index:06d}.png")
                global_index += 1
        if not shutil.which("ffmpeg"):
            raise RuntimeError("ffmpeg is required for deterministic preview rendering")
        command = [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-framerate",
            str(fps),
            "-i",
            str(frame_dir / "frame_%06d.png"),
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-crf",
            str(self.config.get("crf", 18)),
            str(output),
        ]
        subprocess.run(command, check=True)
        return output

    def _load_layers(self, scene: HybridScenePackage) -> dict[str, list[Image.Image]]:
        result: dict[str, list[Image.Image]] = {}
        for layer in scene.layers:
            images: list[Image.Image] = []
            if layer.kind == "video_clip":
                clip_dir = ensure_dir(self.root / "video_cache" / f"{scene.scene_id}_{layer.layer_id}")
                existing = sorted(clip_dir.glob("frame_*.png"))
                if not existing and shutil.which("ffmpeg") and Path(layer.path).exists():
                    subprocess.run(
                        [
                            "ffmpeg",
                            "-y",
                            "-v",
                            "error",
                            "-i",
                            layer.path,
                            "-vf",
                            f"fps={scene.fps},scale={scene.canvas[0]}:{scene.canvas[1]}",
                            str(clip_dir / "frame_%06d.png"),
                        ],
                        check=True,
                    )
                    existing = sorted(clip_dir.glob("frame_*.png"))
                images.extend(Image.open(x).convert("RGBA") for x in existing)
            elif layer.kind == "svg_overlay":
                png = self.root / "svg_cache" / (Path(layer.path).stem + ".png")
                ensure_dir(png.parent)
                try:
                    import cairosvg

                    cairosvg.svg2png(
                        url=layer.path, write_to=str(png), output_width=scene.canvas[0], output_height=scene.canvas[1]
                    )
                    images.append(Image.open(png).convert("RGBA"))
                except Exception:
                    continue
            else:
                if Path(layer.path).exists():
                    images.append(Image.open(layer.path).convert("RGBA"))
                for path in layer.pose_paths:
                    if Path(path).exists():
                        images.append(Image.open(path).convert("RGBA"))
            result[layer.layer_id] = images
        return result

    def _image_for_layer(
        self,
        layer: HybridLayer,
        assets: dict[str, list[Image.Image]],
        events: list[MotionEvent],
        frame: int,
        duration: int,
    ) -> Image.Image | None:
        images = assets.get(layer.layer_id, [])
        if not images:
            return None
        if layer.kind == "video_clip":
            return images[min(len(images) - 1, frame)].copy()
        event = next((e for e in events if e.target_layer == layer.layer_id), None)
        image = images[0].copy()
        if event is None or frame < event.start_frame:
            return image
        progress = min(1.0, max(0.0, (frame - event.start_frame) / max(1, event.end_frame - event.start_frame)))
        if event.representation == "replacement_pose" and len(images) > 1:
            index = min(len(images) - 1, int(progress * len(images)))
            return images[index].copy()
        if event.representation == "opacity" or event.representation == "mask_reveal":
            alpha = image.getchannel("A").point(lambda p: int(p * progress))
            image.putalpha(alpha)
            return image
        if event.representation == "translate":
            start = event.parameters.get("from", [0, 0])
            end = event.parameters.get("to", [0, 0])
            x = round(float(start[0]) + (float(end[0]) - float(start[0])) * progress)
            y = round(float(start[1]) + (float(end[1]) - float(start[1])) * progress)
            moved = Image.new("RGBA", image.size, (0, 0, 0, 0))
            moved.alpha_composite(image, (x, y))
            return moved
        if event.representation == "rotate":
            angle = float(event.parameters.get("degrees", 5)) * progress
            return image.rotate(angle, resample=Image.Resampling.BICUBIC, expand=False)
        if event.representation == "scale":
            amount = 1 + (float(event.parameters.get("to", 1.03)) - 1) * progress
            w, h = image.size
            scaled = image.resize((max(1, int(w * amount)), max(1, int(h * amount))), Image.Resampling.LANCZOS)
            canvas = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            canvas.alpha_composite(scaled, ((w - scaled.width) // 2, (h - scaled.height) // 2))
            return canvas
        if event.representation == "texture_loop":
            speed = float(event.parameters.get("speed_px_per_second", 50))
            offset = round((frame / max(1, duration)) * speed)
            moved = Image.new("RGBA", image.size, (0, 0, 0, 0))
            moved.alpha_composite(image, (0, offset))
            moved.alpha_composite(image, (0, offset - image.height))
            return moved
        # local_deformation is represented by a replacement state when available;
        # otherwise keep the approved artwork static rather than inventing motion.
        if event.representation == "local_deformation" and len(images) > 1:
            return images[min(len(images) - 1, int(progress * len(images)))].copy()
        return image


class RemotionHybridExporter:
    """Writes a production Remotion project for the same hybrid packages."""

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)

    def create_project(self, scenes: list[HybridScenePackage], support_bed_path: str = "") -> Path:
        public = ensure_dir(self.root / "public")
        src = ensure_dir(self.root / "src")
        data_scenes = []
        frame_cursor = 0
        for scene in scenes:
            layers = []
            for layer in scene.layers:
                paths = [layer.path, *layer.pose_paths]
                copied = []
                for path in paths:
                    source = Path(path)
                    if not source.exists():
                        continue
                    name = f"{scene.scene_id}_{layer.layer_id}_{source.name}".replace(" ", "_")
                    destination = public / name
                    shutil.copy2(source, destination)
                    copied.append(name)
                if not copied:
                    continue
                layers.append({**layer.model_dump(mode="json"), "path": copied[0], "pose_paths": copied[1:]})
            data_scenes.append(
                {
                    **scene.model_dump(mode="json", exclude={"layers"}),
                    "layers": layers,
                    "from_frame": frame_cursor,
                }
            )
            frame_cursor += scene.duration_frames
        support_bed_name = ""
        if support_bed_path and Path(support_bed_path).exists():
            support_bed_name = "support_bed" + Path(support_bed_path).suffix
            shutil.copy2(support_bed_path, public / support_bed_name)
        # Copy per-scene narration audio.
        for scene_data, scene in zip(data_scenes, scenes):
            if scene.voice_path and Path(scene.voice_path).exists():
                voice_name = f"{scene.scene_id}_voice{Path(scene.voice_path).suffix}"
                shutil.copy2(scene.voice_path, public / voice_name)
                scene_data["voice_path"] = voice_name
            else:
                scene_data["voice_path"] = ""
        payload = {
            "width": scenes[0].canvas[0] if scenes else 1080,
            "height": scenes[0].canvas[1] if scenes else 1920,
            "fps": scenes[0].fps if scenes else 30,
            "duration_frames": frame_cursor,
            "scenes": data_scenes,
            "support_bed": support_bed_name,
        }
        save_json(public / "data.json", payload)
        (src / "index.tsx").write_text(self._typescript(), encoding="utf-8")
        save_json(
            self.root / "package.json",
            {
                "name": "scientific-motion-studio-v10-render",
                "version": "10.0.0",
                "private": True,
                "dependencies": {
                    "@remotion/cli": "4.0.489",
                    "remotion": "4.0.489",
                    "react": "19.0.0",
                    "react-dom": "19.0.0",
                },
                "devDependencies": {"typescript": "5.6.3", "@types/react": "19.0.0", "@types/react-dom": "19.0.0"},
            },
        )
        save_json(
            self.root / "tsconfig.json",
            {
                "compilerOptions": {
                    "target": "ES2020",
                    "module": "ESNext",
                    "moduleResolution": "Bundler",
                    "jsx": "react-jsx",
                    "strict": True,
                    "resolveJsonModule": True,
                },
                "include": ["src"],
            },
        )
        return self.root

    @staticmethod
    def _typescript() -> str:
        return r"""import React from 'react';
import {AbsoluteFill, Composition, Html5Audio, Img, OffthreadVideo, Sequence, interpolate, registerRoot, staticFile, useCurrentFrame, useVideoConfig} from 'remotion';
import data from '../public/data.json';

const eventFor=(scene:any,id:string)=>(scene.animation.events||[]).find((e:any)=>e.target_layer===id);
const progress=(event:any,frame:number)=>event?interpolate(frame,[event.start_frame,event.end_frame],[0,1],{extrapolateLeft:'clamp',extrapolateRight:'clamp'}):0;
const SceneView=({scene}:any)=>{const frame=useCurrentFrame();const {width,height}=useVideoConfig();return <AbsoluteFill style={{background:scene.background,overflow:'hidden'}}>{scene.layers.map((layer:any)=>{const event=eventFor(scene,layer.layer_id);const p=progress(event,frame);let source=layer.path;let transform='';let opacity=layer.opacity??1;if(event?.representation==='replacement_pose'&&layer.pose_paths?.length){source=layer.pose_paths[Math.min(layer.pose_paths.length-1,Math.floor(p*layer.pose_paths.length))]||source;}if(event?.representation==='translate'){const a=event.parameters?.from||[0,0],b=event.parameters?.to||[0,0];transform=`translate(${a[0]+(b[0]-a[0])*p}px,${a[1]+(b[1]-a[1])*p}px)`;}if(event?.representation==='rotate'){transform=`rotate(${(event.parameters?.degrees||5)*p}deg)`;}if(event?.representation==='opacity'||event?.representation==='mask_reveal'){opacity*=p;}return layer.kind==='video_clip'?<OffthreadVideo key={layer.layer_id} src={staticFile(source)} muted style={{position:'absolute',inset:0,width,height,objectFit:'fill',zIndex:layer.z_index}}/>:<Img key={layer.layer_id} src={staticFile(source)} style={{position:'absolute',inset:0,width,height,objectFit:'fill',zIndex:layer.z_index,opacity,transform,transformOrigin:'center'}}/>})}{scene.voice_path?<Html5Audio src={staticFile(scene.voice_path)} volume={1}/>:null}</AbsoluteFill>};
const Film=()=> <AbsoluteFill>{(data as any).support_bed?<Html5Audio src={staticFile((data as any).support_bed)} volume={0.20}/>:null}{(data.scenes as any[]).map((scene:any)=><Sequence key={scene.scene_id} from={scene.from_frame} durationInFrames={scene.duration_frames}><SceneView scene={scene}/></Sequence>)}</AbsoluteFill>;
const Root=()=> <Composition id="ScientificMotionV10" component={Film} durationInFrames={(data as any).duration_frames} fps={(data as any).fps} width={(data as any).width} height={(data as any).height}/>;
registerRoot(Root);"""


In [ ]:
%%writefile scistudio_v10/job_runtime.py
"""Persistent, resumable, crash-safe stage runtime.

Safety properties:

* **Atomic manifests** — every save goes through the atomic JSON writer; a
  killed process can never leave a truncated manifest.
* **Schema version** — the manifest carries an explicit schema version.
* **Interruption recovery** — a stage found in ``running`` on startup was
  interrupted; it is marked ``interrupted`` and re-executed.
* **Config-change invalidation** — when the effective configuration hash
  changes, previously completed stages are marked ``stale`` and re-run.
* **Output checksums** — completed stages record the SHA-256 of their output
  file; resume only trusts a completed stage whose artifact still exists and
  matches its checksum (no false "completed" state).
* **Retry budget** — each stage has a bounded attempt count.
* **Concurrency** — a pid-stamped lock file rejects a second live worker on
  the same job directory; locks from dead processes are reclaimed.
* **No tracebacks in manifests** — errors are stored as redacted
  type + message; full tracebacks go only to the debug log when enabled.
"""

from __future__ import annotations

import os
import traceback
from datetime import datetime, UTC
from pathlib import Path
from typing import Any, Callable

from .errors import JobConcurrencyError, JobStateError, StageRetryExhaustedError
from .schemas import JobManifest, StageRecord
from .security import redacted_exception_text
from .utils import ensure_dir, hash_value, load_json, save_json, sha256_file

MANIFEST_SCHEMA_VERSION = "10.1"

_VALID_TRANSITIONS: dict[str, set[str]] = {
    "pending": {"running", "skipped"},
    "running": {"completed", "failed", "interrupted"},
    "completed": {"stale", "running"},
    "failed": {"running"},
    "interrupted": {"running"},
    "stale": {"running"},
    "skipped": {"running"},
}


def _now() -> str:
    return datetime.now(UTC).isoformat()


def _pid_alive(pid: int) -> bool:
    if pid <= 0:
        return False
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    return True


class ResumableJobRuntime:
    """Idempotent stage runtime with crash recovery and integrity checks."""

    def __init__(
        self,
        run_dir: str | Path,
        *,
        job_id: str,
        topic: str,
        config: dict[str, Any],
        max_stage_attempts: int = 5,
        debug_tracebacks: bool = False,
        on_stage_event: Any | None = None,
        cancellation_token: Any | None = None,
    ):
        # on_stage_event(stage_id, event, record_dict) is invoked on
        # "cached", "started", "completed" and "failed"; it must never raise
        # into the pipeline. cancellation_token exposes a truthy `.cancelled`
        # checked BEFORE each stage — a set token stops after the current
        # safe stage (never mid-provider-request).
        self.on_stage_event = on_stage_event
        self.cancellation_token = cancellation_token
        self.run_dir = ensure_dir(run_dir)
        self.manifest_path = self.run_dir / "job_manifest.json"
        self.lock_path = self.run_dir / "job.lock"
        self.max_stage_attempts = int(max_stage_attempts)
        self.debug_tracebacks = bool(debug_tracebacks)
        self._acquire_lock()

        config_hash = hash_value(config, 24)
        existing = load_json(self.manifest_path)
        if isinstance(existing, dict):
            self.manifest = JobManifest.model_validate(existing)
            self.manifest.schema_version = MANIFEST_SCHEMA_VERSION
            self._recover_interrupted_stages()
            if self.manifest.config_hash and self.manifest.config_hash != config_hash:
                self._invalidate_for_config_change(config_hash)
            self.manifest.config_hash = config_hash
            self._save()
        else:
            self.manifest = JobManifest(
                schema_version=MANIFEST_SCHEMA_VERSION,
                job_id=job_id,
                topic=topic,
                run_dir=str(self.run_dir),
                config_hash=config_hash,
            )
            self._save()

    # -- locking -----------------------------------------------------------
    def _acquire_lock(self) -> None:
        my_pid = os.getpid()
        for _ in range(2):
            try:
                fd = os.open(self.lock_path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                with os.fdopen(fd, "w", encoding="utf-8") as handle:
                    handle.write(f"{my_pid}\n{_now()}\n")
                return
            except FileExistsError:
                try:
                    holder = int(self.lock_path.read_text(encoding="utf-8").splitlines()[0])
                except (OSError, ValueError, IndexError):
                    holder = -1
                if holder == my_pid:
                    return  # re-entrant within the same process
                if not _pid_alive(holder):
                    # Stale lock from a dead worker: reclaim it.
                    try:
                        self.lock_path.unlink()
                    except OSError:
                        pass
                    continue
                raise JobConcurrencyError(
                    f"Job directory {self.run_dir} is locked by live pid {holder}; "
                    "refusing concurrent execution of the same job"
                ) from None
        raise JobConcurrencyError(f"Could not acquire job lock at {self.lock_path}")

    def release_lock(self) -> None:
        try:
            holder = int(self.lock_path.read_text(encoding="utf-8").splitlines()[0])
            if holder == os.getpid():
                self.lock_path.unlink()
        except (OSError, ValueError, IndexError):
            pass

    # -- recovery / invalidation -------------------------------------------
    def _recover_interrupted_stages(self) -> None:
        for record in self.manifest.stages.values():
            if record.status == "running":
                record.status = "interrupted"
                record.error_type = "Interrupted"
                record.error_message = "Stage was left running by a previous worker; it will re-run."

    def _invalidate_for_config_change(self, new_hash: str) -> None:
        for record in self.manifest.stages.values():
            if record.status == "completed":
                record.status = "stale"
        self.manifest.warnings.append(
            f"Configuration changed (hash {self.manifest.config_hash} -> {new_hash}); "
            "completed stages were invalidated."
        )

    # -- persistence ---------------------------------------------------------
    def _save(self) -> None:
        self.manifest.updated_at = _now()
        save_json(self.manifest_path, self.manifest)

    def _transition(self, record: StageRecord, new_status: str) -> None:
        allowed = _VALID_TRANSITIONS.get(record.status, set())
        if new_status not in allowed:
            raise JobStateError(
                f"Invalid stage transition {record.status!r} -> {new_status!r} for stage {record.stage_id}"
            )
        record.status = new_status  # type: ignore[assignment]

    # -- execution -------------------------------------------------------------
    def execute(
        self,
        stage_id: str,
        input_payload: Any,
        fn: Callable[[], Any],
        *,
        force: bool = False,
    ) -> Any:
        if self.cancellation_token is not None and getattr(self.cancellation_token, "cancelled", False):
            from .errors import JobCancelledError

            self._emit_stage_event(stage_id, "cancelled", None)
            raise JobCancelledError(
                f"Run cancelled before stage {stage_id}; the job is resumable with the same job id."
            )
        input_hash = hash_value(input_payload, 32)
        current = self.manifest.stages.get(stage_id)
        if current and current.status == "completed" and current.input_hash == input_hash and not force:
            output_path = Path(current.output_path) if current.output_path else None
            if output_path and output_path.exists():
                if not current.output_sha256 or sha256_file(output_path) == current.output_sha256:
                    self._emit_stage_event(stage_id, "cached", current)
                    return load_json(output_path)
                # Artifact was modified/corrupted after completion: re-run.
                self._transition(current, "stale")
                self._save()

        record = current or StageRecord(stage_id=stage_id)
        if record.attempt >= self.max_stage_attempts and not force:
            raise StageRetryExhaustedError(
                f"Stage {stage_id} exceeded {self.max_stage_attempts} attempts; "
                "pass force=True after fixing the underlying failure"
            )
        self._transition(record, "running")
        record.attempt += 1
        record.input_hash = input_hash
        record.started_at = _now()
        record.error_type = ""
        record.error_message = ""
        self.manifest.stages[stage_id] = record
        self.manifest.status = "running"
        self._save()
        self._emit_stage_event(stage_id, "started", record)
        try:
            result = fn()
            serializable = result.model_dump(mode="json") if hasattr(result, "model_dump") else result
            output = self.run_dir / "stage_outputs" / f"{stage_id}.json"
            ensure_dir(output.parent)
            save_json(output, serializable)
            self._transition(record, "completed")
            record.completed_at = _now()
            record.output_path = str(output)
            record.output_sha256 = sha256_file(output)
            self._save()
            self._emit_stage_event(stage_id, "completed", record)
            return serializable
        except Exception as exc:
            self._transition(record, "failed")
            record.completed_at = _now()
            record.error_type = type(exc).__name__
            record.error_message = redacted_exception_text(exc, 1000)
            if self.debug_tracebacks:
                debug_path = self.run_dir / "debug" / f"{stage_id}.traceback.txt"
                ensure_dir(debug_path.parent)
                debug_path.write_text(traceback.format_exc(limit=20), encoding="utf-8")
                record.metadata["traceback_file"] = str(debug_path)
            self.manifest.status = "failed"
            self._save()
            self._emit_stage_event(stage_id, "failed", record)
            self.release_lock()
            raise

    def _emit_stage_event(self, stage_id: str, event: str, record: StageRecord | None) -> None:
        """Deliver a stage event to the optional observer; observer failures
        are swallowed so UI callbacks can never break the pipeline."""
        if self.on_stage_event is None:
            return
        try:
            payload = record.model_dump(mode="json") if record is not None else {}
            self.on_stage_event(stage_id, event, payload)
        except Exception:
            pass

    # -- terminal states ---------------------------------------------------------
    def mark_completed(self) -> None:
        self.manifest.status = "completed"
        self._save()
        self.release_lock()

    def stage_completed(self, stage_id: str) -> bool:
        record = self.manifest.stages.get(stage_id)
        if not (record and record.status == "completed"):
            return False
        if record.output_path and not Path(record.output_path).exists():
            return False
        return True


In [ ]:
%%writefile scistudio_v10/llm.py
"""Provider router for reasoning (OpenAI GPT) and image generation (BFL FLUX
Kontext), with execution-mode-aware provider locking.

Production boundary
-------------------
* ``production`` mode: OpenAI GPT is the **only** text/vision provider and
  BFL FLUX Kontext the **only** image provider. When they are unavailable or
  exhausted, calls raise :class:`ProviderUnavailableError` — the pipeline
  fails clearly instead of silently degrading to Gemini, OpenRouter, local
  models, OpenAI Images or deterministic fallbacks.
* ``development`` / ``test`` modes: alternative providers stay available for
  experimentation, and deterministic fallbacks are permitted — but fallback
  results are cached in a separate short-lived namespace and marked as
  fallbacks; they never masquerade as live provider output.

Reliability
-----------
Every remote call runs under a bounded retry policy (exponential backoff with
jitter; 408/409/429/5xx and connection errors retry, auth/validation errors
do not). Malformed JSON from OpenAI triggers a bounded repair round-trip.
Request IDs, token usage and latency are captured into the event log when an
event logger is attached. Secrets are registered for global redaction and
never logged.
"""

from __future__ import annotations

import base64
import os
import time
from pathlib import Path
from typing import Any

from .errors import ProviderError, ProviderUnavailableError
from .http_safety import call_with_retries
from .security import redacted_exception_text, register_secret
from .utils import ensure_dir, extract_json, hash_value, load_json, save_json, sha256_file

CACHE_SCHEMA_VERSION = 5
FALLBACK_NAMESPACE = "_fallback"
FALLBACK_MARKER = "__scistudio_fallback__"

_PRODUCTION_TEXT_PROVIDERS = ("openai",)
_KNOWN_TEXT_PROVIDERS = ("openai", "gemini", "openrouter", "local")
_KNOWN_VISION_PROVIDERS = ("openai", "gemini", "openrouter")


class LLMRouter:
    """Lazy, execution-mode-aware provider router.

    Local checkpoints are never loaded when a remote provider succeeds,
    preventing multi-GB downloads in remote-provider runs.
    """

    def __init__(
        self,
        config: dict[str, Any],
        secrets: dict[str, Any],
        cache_root: str | Path,
        *,
        event_logger: Any | None = None,
    ):
        self.config = config
        self.secrets = secrets
        self.cache_root = ensure_dir(cache_root)
        self.events = event_logger
        self._local = None
        self._local_tokenizer = None
        self._gemini_client = None
        self._openai_client = None
        self._openrouter_client = None
        self._bfl_client = None
        for name in ("OPENAI_API_KEY", "BFL_API_KEY", "BFL_KEY", "GEMINI_API_KEY", "OPENROUTER_API_KEY", "HF_TOKEN"):
            register_secret(self._secret(name))

    # -- mode & provider selection ----------------------------------------
    @property
    def execution_mode(self) -> str:
        return str(self.config.get("execution_mode", "development")).lower()

    @property
    def is_production(self) -> bool:
        return self.execution_mode == "production"

    @property
    def provider_order(self) -> list[str]:
        order = self.config.get("provider_order", ["openai"])
        if isinstance(order, str):
            order = [item.strip() for item in order.split(",") if item.strip()]
        order = [str(item).lower() for item in order]
        if self.is_production:
            # Hard runtime lock, independent of config validation.
            order = [item for item in order if item in _PRODUCTION_TEXT_PROVIDERS] or ["openai"]
        return list(order)

    @property
    def vision_provider_order(self) -> list[str]:
        order = self.config.get("vision_provider_order")
        if order is None:
            order = self.provider_order
        if isinstance(order, str):
            order = [item.strip() for item in order.split(",") if item.strip()]
        order = [str(item).lower() for item in order]
        if self.is_production:
            order = [item for item in order if item in _PRODUCTION_TEXT_PROVIDERS] or ["openai"]
        return [item for item in order if item in _KNOWN_VISION_PROVIDERS]

    def _secret(self, *names: str) -> str:
        for name in names:
            value = self.secrets.get(name) or os.environ.get(name)
            if value:
                return value
        return ""

    def available(self, provider: str) -> bool:
        provider = provider.lower()
        if self.is_production and provider not in _PRODUCTION_TEXT_PROVIDERS:
            return False
        if provider == "gemini":
            return bool(self._secret("GEMINI_API_KEY"))
        if provider == "openai":
            return bool(self._secret("OPENAI_API_KEY"))
        if provider == "openrouter":
            return bool(self._secret("OPENROUTER_API_KEY"))
        if provider == "local":
            return bool(self.config.get("enable_local_fallback", False))
        return False

    def available_bfl(self) -> bool:
        return bool(self._secret("BFL_API_KEY", "BFL_KEY"))

    # -- observability ------------------------------------------------------
    def _emit(self, event: str, **payload: Any) -> None:
        if self.events is not None:
            try:
                self.events.emit(event, **payload)
            except Exception:
                pass

    def _retry_kwargs(self) -> dict[str, Any]:
        retry_cfg = self.config.get("retry") or {}
        return {
            "max_attempts": int(retry_cfg.get("max_attempts", 4)),
            "base_delay": float(retry_cfg.get("base_delay_s", 1.0)),
            "max_delay": float(retry_cfg.get("max_delay_s", 30.0)),
            "jitter": float(retry_cfg.get("jitter", 0.25)),
        }

    # -- fallback cache (short-lived, clearly marked) ------------------------
    def _fallback_cache_path(self, namespace: str, key: str) -> Path:
        return self.cache_root / FALLBACK_NAMESPACE / namespace / f"{key}.json"

    def _load_fresh_fallback(self, namespace: str, key: str) -> tuple[bool, Any]:
        path = self._fallback_cache_path(namespace, key)
        envelope = load_json(path)
        if not isinstance(envelope, dict) or not envelope.get(FALLBACK_MARKER):
            return False, None
        ttl = float(self.config.get("fallback_cache_ttl_s", 3600.0))
        if ttl > 0 and (time.time() - float(envelope.get("cached_at", 0))) > ttl:
            return False, None
        return True, envelope.get("value")

    def _store_fallback(self, namespace: str, key: str, value: Any, reasons: list[str]) -> None:
        save_json(
            self._fallback_cache_path(namespace, key),
            {
                FALLBACK_MARKER: True,
                "reason": [
                    redacted_exception_text(RuntimeError(r), 300) if not isinstance(r, str) else r for r in reasons[-3:]
                ],
                "cached_at": time.time(),
                "value": value,
            },
        )

    # -- text JSON generation -------------------------------------------------
    def generate_json(
        self,
        *,
        system: str,
        prompt: str,
        namespace: str,
        fallback: Any,
        json_schema: dict[str, Any] | None = None,
        force: bool = False,
        temperature: float | None = None,
    ) -> Any:
        key = hash_value(
            {
                "order": self.provider_order,
                "models": {
                    "openai": self.config.get("openai_model", "gpt-5-mini"),
                    "gemini": self.config.get("gemini_model", ""),
                    "openrouter": self.config.get("openrouter_model", ""),
                    "local": self.config.get("local_model", ""),
                },
                "system": system,
                "prompt": prompt,
                "schema": json_schema,
                "temperature": temperature,
                "effort": self.config.get("openai_reasoning_effort", "low"),
                "max_output_tokens": self.config.get("openai_max_output_tokens", 8000),
                "cache_schema": CACHE_SCHEMA_VERSION,
            },
            32,
        )
        cache_path = self.cache_root / namespace / f"{key}.json"
        if cache_path.exists() and not force:
            cached = load_json(cache_path, None)
            if cached is not None:
                return cached
        if not force:
            fresh, value = self._load_fresh_fallback(namespace, key)
            if fresh:
                return value

        errors: list[str] = []
        prompt_hash = hash_value(prompt, 24)
        for provider in self.provider_order:
            if not self.available(provider):
                continue
            started = time.perf_counter()
            try:
                output = self._call_text_provider(provider, system, prompt, json_schema, temperature)
                parsed = extract_json(output, fallback=None)
                if parsed is None and provider == "openai":
                    parsed = self._openai_json_repair(system, prompt, output, temperature)
                if parsed is not None:
                    save_json(cache_path, parsed)
                    self._emit(
                        "llm.completed",
                        provider=provider,
                        namespace=namespace,
                        prompt_hash=prompt_hash,
                        latency_s=round(time.perf_counter() - started, 3),
                        request_id=getattr(self, "_last_request_id", ""),
                        usage=getattr(self, "_last_usage", {}),
                        status="ok",
                    )
                    return parsed
                errors.append(f"{provider}: returned non-JSON output")
            except Exception as exc:
                errors.append(f"{provider}: {redacted_exception_text(exc, 400)}")
                self._emit(
                    "llm.failed",
                    provider=provider,
                    namespace=namespace,
                    prompt_hash=prompt_hash,
                    latency_s=round(time.perf_counter() - started, 3),
                    error_class=type(exc).__name__,
                )

        if self.is_production:
            raise ProviderUnavailableError(
                "Production reasoning provider (OpenAI) unavailable or exhausted; "
                "refusing silent fallback. Errors: " + (" | ".join(errors) or "no provider configured"),
                provider="openai",
            )
        value = fallback() if callable(fallback) else fallback
        self._store_fallback(namespace, key, value, errors or ["no provider configured"])
        if errors:
            self._emit("llm.fallback", namespace=namespace, prompt_hash=prompt_hash, reasons=errors[-3:])
        return value

    def _call_text_provider(
        self, provider: str, system: str, prompt: str, schema: dict[str, Any] | None, temperature: float | None
    ) -> str:
        retry_kwargs = self._retry_kwargs()
        if provider == "openai":
            return call_with_retries(lambda: self._openai_json(system, prompt, temperature), **retry_kwargs)
        if provider == "gemini":
            return call_with_retries(lambda: self._gemini_json(system, prompt, schema, temperature), **retry_kwargs)
        if provider == "openrouter":
            return call_with_retries(lambda: self._openrouter_json(system, prompt, temperature), **retry_kwargs)
        if provider == "local":
            return self._local_json(system, prompt, temperature)
        raise ProviderError(f"Unknown text provider {provider!r}", provider=provider)

    def _openai_json_repair(self, system: str, prompt: str, bad_output: str, temperature: float | None) -> Any:
        """Bounded repair loop for malformed OpenAI JSON output."""
        attempts = int(self.config.get("openai_json_repair_attempts", 1))
        for _ in range(max(0, attempts)):
            repair_prompt = (
                prompt
                + "\n\nYour previous output was not valid JSON. Previous output (truncated):\n"
                + str(bad_output)[:2000]
                + "\nReturn ONLY the corrected, complete JSON object."
            )
            try:
                output = call_with_retries(
                    lambda rp=repair_prompt: self._openai_json(system, rp, temperature),
                    **self._retry_kwargs(),
                )
            except Exception:
                return None
            parsed = extract_json(output, fallback=None)
            if parsed is not None:
                return parsed
            bad_output = output
        return None

    # -- vision critique -------------------------------------------------------
    def critique_image(
        self,
        *,
        image_path: str | Path,
        prompt: str,
        namespace: str = "vision_critic",
        fallback: Any = None,
        force: bool = False,
    ) -> Any:
        image_path = Path(image_path)
        # Cache key uses the image *content* hash — never filename + size.
        content_hash = sha256_file(image_path) if image_path.exists() else ""
        key = hash_value(
            {
                "image_sha256": content_hash,
                "prompt": prompt,
                "order": self.vision_provider_order,
                "models": {
                    "openai": self.config.get("openai_vision_model", self.config.get("openai_model", "gpt-5-mini")),
                    "gemini": self.config.get("gemini_vision_model", ""),
                    "openrouter": self.config.get("openrouter_vision_model", ""),
                },
                "cache_schema": CACHE_SCHEMA_VERSION,
            },
            32,
        )
        cache_path = self.cache_root / namespace / f"{key}.json"
        if cache_path.exists() and not force:
            cached = load_json(cache_path, None)
            if cached is not None:
                return cached

        errors: list[str] = []
        retry_kwargs = self._retry_kwargs()
        for provider in self.vision_provider_order:
            if not self.available(provider):
                continue
            try:
                if provider == "gemini":
                    output = call_with_retries(lambda: self._gemini_vision(image_path, prompt), **retry_kwargs)
                elif provider == "openrouter":
                    output = call_with_retries(lambda: self._openrouter_vision(image_path, prompt), **retry_kwargs)
                else:
                    output = call_with_retries(lambda: self._openai_vision(image_path, prompt), **retry_kwargs)
                parsed = extract_json(output, fallback=None)
                if parsed is not None:
                    save_json(cache_path, parsed)
                    return parsed
                errors.append(f"{provider}: non-JSON vision response")
            except Exception as exc:
                errors.append(f"{provider}: {redacted_exception_text(exc, 400)}")

        if self.is_production:
            raise ProviderUnavailableError(
                "Production vision provider (OpenAI) unavailable or exhausted; "
                "refusing unreviewed approval. Errors: " + (" | ".join(errors) or "no provider configured"),
                provider="openai",
            )
        if errors:
            self._emit("llm.vision_fallback", namespace=namespace, reasons=errors[-2:])
        return fallback() if callable(fallback) else fallback

    # -- image generation -------------------------------------------------------
    def generate_reference_image(
        self,
        *,
        prompt: str,
        output_path: str | Path,
        init_image: str | Path | None = None,
        force: bool = False,
    ) -> Path | None:
        """Generate a raster image.

        Production: BFL FLUX Kontext exclusively; failure raises.
        Development/test: BFL first, then explicitly configured alternates.
        """
        output_path = Path(output_path)
        ensure_dir(output_path.parent)
        min_bytes = int(self.config.get("min_valid_image_bytes", 1024))
        if output_path.exists() and output_path.stat().st_size > min_bytes and not force:
            return output_path

        if self.is_production:
            if str(self.config.get("image_provider", "bfl")) != "bfl":
                raise ProviderUnavailableError("Production image provider must be BFL FLUX Kontext", provider="bfl")
            if not self.available_bfl():
                raise ProviderUnavailableError(
                    "BFL_API_KEY is not configured; production image generation "
                    "cannot proceed (no fallback is authorized)",
                    provider="bfl",
                )
            return self._bfl_flux_image(prompt, output_path, init_image=init_image, force=force)

        errors: list[str] = []
        if self.config.get("image_provider", "bfl") == "bfl" and self.available_bfl():
            try:
                return self._bfl_flux_image(prompt, output_path, init_image=init_image, force=force)
            except Exception as exc:
                errors.append(f"bfl: {redacted_exception_text(exc, 300)}")
        alternate = self._development_alternate_image(prompt, output_path, errors)
        if alternate is not None:
            return alternate
        if errors:
            self._emit("image.unavailable", reasons=errors[-3:])
        return None

    def _development_alternate_image(self, prompt: str, output_path: Path, errors: list[str]) -> Path | None:
        """Non-production alternates; each requires explicit configuration."""
        gemini_model = self.config.get("gemini_image_model")
        if gemini_model and self.available("gemini"):
            try:
                from google.genai import types

                response = self._gemini().models.generate_images(
                    model=gemini_model,
                    prompt=prompt,
                    config=types.GenerateImagesConfig(number_of_images=1),
                )
                generated = getattr(response, "generated_images", None) or []
                if generated:
                    image = getattr(generated[0], "image", generated[0])
                    data = getattr(image, "image_bytes", None) or getattr(image, "bytes", None)
                    if data:
                        from .utils import atomic_write_bytes

                        atomic_write_bytes(output_path, data)
                        return output_path
            except Exception as exc:
                errors.append(f"gemini image: {redacted_exception_text(exc, 300)}")
        openai_model = self.config.get("openai_image_model")
        if openai_model and self.available("openai"):
            try:
                image_kwargs = {
                    "model": openai_model,
                    "prompt": prompt,
                    "size": self.config.get("openai_image_size", "1024x1024"),
                }
                if "dall-e" in str(openai_model).lower():
                    image_kwargs["response_format"] = "b64_json"
                response = self._openai().images.generate(**image_kwargs)
                item = response.data[0]
                encoded = getattr(item, "b64_json", None)
                if encoded:
                    from .utils import atomic_write_bytes

                    atomic_write_bytes(output_path, base64.b64decode(encoded))
                    return output_path
            except Exception as exc:
                errors.append(f"openai image: {redacted_exception_text(exc, 300)}")
        local_model = self.config.get("local_image_model")
        if local_model and self.config.get("enable_local_image_generation", False):
            local = self._flux_image(prompt, output_path, local_model)
            if local is not None:
                return local
            errors.append("local image: pipeline unavailable or failed")
        return None

    def _bfl(self):
        if self._bfl_client is None:
            from .bfl_client import BFLClient

            self._bfl_client = BFLClient(
                self.config,
                self._secret("BFL_API_KEY", "BFL_KEY"),
                self.cache_root,
                event_logger=self.events,
            )
        return self._bfl_client

    def _bfl_flux_image(
        self,
        prompt: str,
        output_path: Path,
        *,
        init_image: str | Path | None = None,
        force: bool = False,
    ) -> Path | None:
        """BFL FLUX Kontext generation via the hardened client.

        In production a failure raises; in development it returns ``None`` so
        explicitly configured alternates can be tried.
        """
        # Per-call overrides (aspect/seed/format) may have been set on config
        # by the studio; the client reads them at construction, so rebuild when
        # the relevant knobs changed.
        self._bfl_client = None
        try:
            return self._bfl().generate(prompt, output_path, init_image=init_image, force=force)
        except Exception:
            if self.is_production:
                raise
            self._emit("bfl.failed_development", prompt_hash=hash_value(prompt, 24))
            return None

    def _flux_image(self, prompt: str, output_path: Path, model: str) -> Path | None:
        """Optional local FLUX pipeline (development only; needs a large GPU)."""
        try:
            import torch

            dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
            steps = int(self.config.get("flux_steps", self.config.get("local_image_steps", 28)))
            guidance = float(self.config.get("flux_guidance", 2.5))
            token = self.secrets.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or None
            init_path = self.config.get("flux_init_image")
            is_kontext = "kontext" in str(model).lower()
            if is_kontext and init_path and Path(init_path).exists():
                from diffusers import FluxKontextPipeline
                from diffusers.utils import load_image

                pipe = FluxKontextPipeline.from_pretrained(model, torch_dtype=dtype, token=token)
                if torch.cuda.is_available():
                    pipe.enable_model_cpu_offload()
                image = pipe(
                    image=load_image(str(init_path)),
                    prompt=prompt,
                    guidance_scale=guidance,
                    num_inference_steps=steps,
                ).images[0]
            else:
                from diffusers import FluxPipeline

                base = "black-forest-labs/FLUX.1-dev" if is_kontext else model
                pipe = FluxPipeline.from_pretrained(base, torch_dtype=dtype, token=token)
                if torch.cuda.is_available():
                    pipe.enable_model_cpu_offload()
                image = pipe(
                    prompt=prompt,
                    guidance_scale=guidance,
                    num_inference_steps=steps,
                    height=int(self.config.get("local_image_height", 1024)),
                    width=int(self.config.get("local_image_width", 1024)),
                ).images[0]
            image.save(output_path)
            del pipe
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return output_path
        except Exception as exc:
            self._emit("flux_local.failed", error_class=type(exc).__name__)
            return None

    # -- provider clients -------------------------------------------------------
    def _gemini(self):
        if self._gemini_client is None:
            from google import genai

            api_key = self._secret("GEMINI_API_KEY")
            self._gemini_client = genai.Client(api_key=api_key)
        return self._gemini_client

    def _gemini_json(self, system: str, prompt: str, schema: dict[str, Any] | None, temperature: float | None):
        from google.genai import types

        model = self.config.get("gemini_model", "gemini-2.5-flash")
        kwargs: dict[str, Any] = {
            "system_instruction": system,
            "temperature": self.config.get("temperature", 0.75) if temperature is None else temperature,
            "response_mime_type": "application/json",
        }
        if schema:
            kwargs["response_json_schema"] = schema
        response = self._gemini().models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(**kwargs),
        )
        return response.text

    def _gemini_vision(self, image_path: Path, prompt: str):
        from google.genai import types

        model = self.config.get("gemini_vision_model", self.config.get("gemini_model", "gemini-2.5-flash"))
        mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
        response = self._gemini().models.generate_content(
            model=model,
            contents=[types.Part.from_bytes(data=image_path.read_bytes(), mime_type=mime), prompt],
            config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.25),
        )
        return response.text

    def _openai(self):
        if self._openai_client is None:
            from openai import OpenAI

            api_key = self._secret("OPENAI_API_KEY")
            # Timeouts are enforced client-side; retries are handled by the
            # router's classified retry policy, so the SDK's own retry is off.
            self._openai_client = OpenAI(
                api_key=api_key,
                timeout=float(self.config.get("openai_read_timeout_s", 180.0)),
                max_retries=0,
            )
        return self._openai_client

    @staticmethod
    def _reasoning_kwargs(model: str, config: dict) -> dict:
        # gpt-5 / o-series are reasoning models: cap effort (default low) so
        # reasoning tokens don't consume the whole budget and leave empty
        # output. Non-reasoning models ignore this.
        low = model.lower()
        if low.startswith(("gpt-5", "o1", "o3", "o4")) or "gpt-5" in low:
            return {"reasoning": {"effort": config.get("openai_reasoning_effort", "low")}}
        return {}

    def _capture_response_meta(self, response) -> None:
        self._last_request_id = str(getattr(response, "id", "") or "")
        usage = getattr(response, "usage", None)
        self._last_usage = {}
        if usage is not None:
            for field in ("input_tokens", "output_tokens", "total_tokens"):
                value = getattr(usage, field, None)
                if value is not None:
                    self._last_usage[field] = value

    @staticmethod
    def _responses_text(response) -> str:
        # output_text can be empty on reasoning models; fall back to walking items.
        text = getattr(response, "output_text", None)
        if text:
            return text
        parts: list[str] = []
        for item in getattr(response, "output", None) or []:
            for content in getattr(item, "content", None) or []:
                value = getattr(content, "text", None)
                if isinstance(value, str) and value:
                    parts.append(value)
        return "".join(parts)

    def _openai_json(self, system: str, prompt: str, temperature: float | None):
        model = self.config.get("openai_model", "gpt-5-mini")
        response = self._openai().responses.create(
            model=model,
            input=[
                {"role": "system", "content": [{"type": "input_text", "text": system}]},
                {
                    "role": "user",
                    "content": [{"type": "input_text", "text": prompt + "\nReturn a single valid JSON object only."}],
                },
            ],
            text={"format": {"type": "json_object"}},
            max_output_tokens=int(self.config.get("openai_max_output_tokens", 8000)),
            **self._reasoning_kwargs(model, self.config),
        )
        self._capture_response_meta(response)
        text = self._responses_text(response)
        if not text:
            raise ProviderError(
                f"OpenAI returned empty output (request_id={self._last_request_id})",
                provider="openai",
                retryable=True,
                request_id=self._last_request_id,
            )
        return text

    def _openai_vision(self, image_path: Path, prompt: str):
        model = self.config.get("openai_vision_model", self.config.get("openai_model", "gpt-5-mini"))
        mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
        data_url = f"data:{mime};base64,{base64.b64encode(image_path.read_bytes()).decode('ascii')}"
        response = self._openai().responses.create(
            model=model,
            input=[
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": prompt + "\nReturn a single valid JSON object only."},
                        {"type": "input_image", "image_url": data_url},
                    ],
                }
            ],
            text={"format": {"type": "json_object"}},
            max_output_tokens=int(self.config.get("openai_max_output_tokens", 8000)),
            **self._reasoning_kwargs(model, self.config),
        )
        self._capture_response_meta(response)
        return self._responses_text(response)

    def _openrouter(self):
        # OpenRouter is OpenAI-API compatible; reuse the OpenAI SDK with its base_url.
        if getattr(self, "_openrouter_client", None) is None:
            from openai import OpenAI

            self._openrouter_client = OpenAI(
                api_key=self._secret("OPENROUTER_API_KEY"),
                base_url=self.config.get("openrouter_base_url", "https://openrouter.ai/api/v1"),
                default_headers={
                    "HTTP-Referer": self.config.get("openrouter_referer", "https://scientific-motion-studio.local"),
                    "X-Title": "Scientific Motion Studio V10",
                },
                max_retries=0,
            )
        return self._openrouter_client

    def _openrouter_json(self, system: str, prompt: str, temperature: float | None):
        model = self.config.get("openrouter_model", "meta-llama/llama-3.3-70b-instruct:free")
        response = self._openrouter().chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt + "\nReturn valid JSON only, no Markdown."},
            ],
            temperature=self.config.get("temperature", 0.7) if temperature is None else temperature,
            response_format={"type": "json_object"},
        )
        return response.choices[0].message.content

    def _openrouter_vision(self, image_path: Path, prompt: str):
        model = self.config.get("openrouter_vision_model", "meta-llama/llama-3.2-11b-vision-instruct:free")
        mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
        data_url = f"data:{mime};base64,{base64.b64encode(image_path.read_bytes()).decode('ascii')}"
        response = self._openrouter().chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt + "\nReturn valid JSON only."},
                        {"type": "image_url", "image_url": {"url": data_url}},
                    ],
                }
            ],
        )
        return response.choices[0].message.content

    def _load_local(self):
        if self._local is not None:
            return self._local_tokenizer, self._local
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        model_name = self.config.get("local_model", "Qwen/Qwen2.5-3B-Instruct")
        token = self.secrets.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or None
        quantization_config = None
        if torch.cuda.is_available() and self.config.get("local_4bit", True):
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=token,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype="auto",
            quantization_config=quantization_config,
            low_cpu_mem_usage=True,
        )
        self._local_tokenizer, self._local = tokenizer, model
        return tokenizer, model

    def _local_json(self, system: str, prompt: str, temperature: float | None):
        import torch

        tokenizer, model = self._load_local()
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt + "\nReturn valid JSON only, without Markdown."},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        device = next(model.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=int(self.config.get("local_max_new_tokens", 2600)),
                do_sample=True,
                temperature=float(self.config.get("temperature", 0.7) if temperature is None else temperature),
                top_p=0.92,
                repetition_penalty=1.04,
            )
        output = generated[0, inputs["input_ids"].shape[1] :]
        return tokenizer.decode(output, skip_special_tokens=True)


In [ ]:
%%writefile scistudio_v10/notebook_runner.py
"""Unified notebook run controller.

Every UI surface (ipywidgets Control Center, Streamlit, headless notebook
cells) dispatches through :class:`NotebookRunController`, so there is exactly
one implementation of validation, the paid-run safety gate, preflight,
execution and artifact packaging.

Safety invariants:
* No action performs paid provider calls unless it is a LIVE action AND the
  request carries ``confirm_paid_run=True`` AND the confirmation phrase
  ``RUN LIVE`` AND preflight reports zero FAIL checks.
* Secrets never enter configs, reports or logs — resolved values live only in
  process memory and are registered for global redaction.
"""

from __future__ import annotations

import os
import re
import threading
import time
import zipfile
from datetime import datetime, UTC
from enum import Enum
from pathlib import Path
from typing import Any, Callable

from pydantic import BaseModel, Field, ValidationError

from .config_models import ExecutionMode, StudioConfig
from .errors import (
    ConfigurationError,
    PreflightError,
    ProviderAuthenticationError,
    ReferenceVideoError,
    RenderDependencyError,
    classify_provider_error,
)
from .preflight import FAIL, VIDEO_SUFFIXES, run_preflight, save_preflight_report
from .security import redact_secrets, redacted_exception_text, register_secret
from .utils import ensure_dir, ensure_within, hash_value, load_json, save_json, slugify

PAID_CONFIRMATION_PHRASE = "RUN LIVE"

CANONICAL_STAGES = [
    "01_style_reference",
    "02_research",
    "03_script",
    "04_storyboard",
    "05_audio",
    "06_art_bible",
    "07_continuity_canon",
    "08_scene_architectures",
    "09_shot_state_*",
    "10_reference_retrieval",
    "10_reference_plans",
    "11_drawing_briefs",
    "12_candidate_tournaments",
    "13_flux_studio",
    "14_semantic",
    "15_overlays",
    "16_animation",
    "17_temporal",
    "18_hybrid_packages",
    "19_remotion",
    "publishing",
]


def stage_progress_fraction(stage_id: str) -> float:
    """Best-effort 0..1 position of a stage id in the canonical pipeline."""
    normalized = re.sub(r"^(09_shot_state)_.+$", r"\1_*", stage_id)
    try:
        index = CANONICAL_STAGES.index(normalized)
    except ValueError:
        return 0.0
    return (index + 1) / len(CANONICAL_STAGES)


def generate_job_id(topic: str, reference_video: str) -> str:
    """Readable, directory-safe, stable job id for a topic+reference pair.

    Matches the pipeline's own default so UI previews are accurate.
    """
    return slugify(topic, 40) + "-" + hash_value({"topic": topic, "reference": str(Path(reference_video))}, 10)


class PipelineAction(str, Enum):
    environment_check = "environment_check"
    config_validation = "config_validation"
    quick_smoke = "quick_smoke"
    full_offline_tests = "full_offline_tests"
    provider_live_smoke = "provider_live_smoke"
    plan_only = "plan_only"
    development_dry_run = "development_dry_run"
    production_plan = "production_plan"
    live_generation = "live_generation"
    live_generation_preview = "live_generation_preview"
    full_production_render = "full_production_render"
    resume_job = "resume_job"
    force_rerun = "force_rerun"
    package_artifacts = "package_artifacts"


LIVE_ACTIONS = frozenset(
    {
        PipelineAction.provider_live_smoke,
        PipelineAction.live_generation,
        PipelineAction.live_generation_preview,
        PipelineAction.full_production_render,
    }
)
PIPELINE_ACTIONS = frozenset(
    {
        PipelineAction.plan_only,
        PipelineAction.development_dry_run,
        PipelineAction.production_plan,
        PipelineAction.live_generation,
        PipelineAction.live_generation_preview,
        PipelineAction.full_production_render,
        PipelineAction.resume_job,
        PipelineAction.force_rerun,
    }
)

ACTION_DESCRIPTIONS: dict[PipelineAction, str] = {
    PipelineAction.environment_check: "Check runtime, tools and dependencies. No API calls.",
    PipelineAction.config_validation: "Validate the configuration and provider locks. No API calls.",
    PipelineAction.quick_smoke: "Offline mini-pipeline with a synthetic reference video. No API calls.",
    PipelineAction.full_offline_tests: "Run the full 200+ offline test suite and security sweep. No API calls.",
    PipelineAction.provider_live_smoke: "One tiny OpenAI JSON call + one small BFL image. PAID — needs confirmation.",
    PipelineAction.plan_only: "Research → script → storyboard → briefs. No paid image generation.",
    PipelineAction.development_dry_run: "Plan-only in development mode with deterministic fallbacks (clearly marked).",
    PipelineAction.production_plan: "Plan-only under production provider locks (OpenAI reasoning only).",
    PipelineAction.live_generation: "Full FLUX Kontext generation without final render. PAID — needs confirmation.",
    PipelineAction.live_generation_preview: "Live generation plus a fast deterministic preview render. PAID.",
    PipelineAction.full_production_render: "Live generation plus the full production render. PAID.",
    PipelineAction.resume_job: "Continue an existing job from its manifest; completed stages are reused.",
    PipelineAction.force_rerun: "Re-run stages ignoring caches (scope selectable). May repeat paid calls when live.",
    PipelineAction.package_artifacts: "ZIP an existing run directory for download. No API calls.",
}


class PipelineRunRequest(BaseModel):
    """One request object for every Control Center action."""

    action: PipelineAction
    topic: str = ""
    reference_video: str = ""
    config: dict[str, Any] = Field(default_factory=dict)
    plan_only: bool = True
    render_video: bool = False
    force: bool = False
    force_scope: str = "all"  # all | failed_and_downstream | selected_and_downstream
    force_stage: str = ""
    job_id: str | None = None
    confirm_paid_run: bool = False
    paid_confirmation_phrase: str = ""
    debug: bool = False


class SecretsManager:
    """Resolves secrets from Colab Secrets → environment → manual input.

    Values never leave process memory; every resolved value is registered for
    global redaction. ``status()`` reports availability without exposure.
    """

    SUPPORTED = ("OPENAI_API_KEY", "BFL_API_KEY", "UPLOAD_POST_TOKEN")

    def __init__(self) -> None:
        self._manual: dict[str, str] = {}

    @staticmethod
    def _from_colab(name: str) -> str:
        try:
            from google.colab import userdata  # type: ignore[import-not-found]

            return str(userdata.get(name) or "")
        except Exception:
            return ""

    def set_manual(self, name: str, value: str) -> None:
        if value:
            self._manual[name] = value
            register_secret(value)

    def clear_manual(self) -> None:
        self._manual.clear()

    def get(self, name: str) -> str:
        value = self._manual.get(name) or self._from_colab(name) or os.environ.get(name, "")
        if value:
            register_secret(value)
        return value

    def available(self, name: str) -> bool:
        return bool(self.get(name))

    def status(self, required: dict[str, bool] | None = None) -> dict[str, str]:
        required = required or {}
        result = {}
        for name in self.SUPPORTED:
            if name in required and not required[name]:
                result[name] = "Not required for selected mode"
            else:
                result[name] = "Configured" if self.available(name) else "Missing"
        return result

    def as_secrets_dict(self) -> dict[str, str]:
        return {name: self.get(name) for name in self.SUPPORTED if self.get(name)}


class CancellationToken:
    """Cooperative cancellation: the run stops after the current safe stage."""

    def __init__(self) -> None:
        self._event = threading.Event()

    def cancel(self) -> None:
        self._event.set()

    @property
    def cancelled(self) -> bool:
        return self._event.is_set()


CONFIG_PRESETS: dict[str, dict[str, Any]] = {
    "Safe Development": {
        "execution_mode": "development",
        "research_search": {"enabled": False},
        "llm": {"provider_order": [], "vision_provider_order": []},
        "audio": {"build_in_plan_mode": False},
        "temporal": {"enabled": False},
        "render": {"backend": "pil"},
    },
    "Offline Smoke Test": {
        "execution_mode": "test",
        "research_search": {"enabled": False},
        "llm": {"provider_order": [], "vision_provider_order": []},
        "audio": {"enabled": False},
        "temporal": {"enabled": False},
        "render": {"backend": "pil"},
    },
    "Production Plan-Only": {
        "execution_mode": "production",
        "research_search": {"enabled": True},
        "llm": {
            "provider_order": ["openai"],
            "vision_provider_order": ["openai"],
            "image_provider": "bfl",
            "bfl_model": "flux-kontext-pro",
        },
    },
    "Production Live Generation": {
        "execution_mode": "production",
        "research_search": {"enabled": True},
        "llm": {
            "provider_order": ["openai"],
            "vision_provider_order": ["openai"],
            "image_provider": "bfl",
            "bfl_model": "flux-kontext-pro",
            "bfl_aspect_ratio": "9:16",
        },
        "candidate_tournament": {"candidate_count": 4},
    },
    "Production Full Render": {
        "execution_mode": "production",
        "research_search": {"enabled": True},
        "llm": {
            "provider_order": ["openai"],
            "vision_provider_order": ["openai"],
            "image_provider": "bfl",
            "bfl_model": "flux-kontext-pro",
            "bfl_aspect_ratio": "9:16",
        },
        "candidate_tournament": {"candidate_count": 4},
        "render": {"backend": "remotion", "install_dependencies": True, "crf": 18},
    },
    "Low-Cost Live Smoke": {
        "execution_mode": "production",
        "research_search": {"enabled": False},
        "llm": {
            "provider_order": ["openai"],
            "vision_provider_order": ["openai"],
            "image_provider": "bfl",
            "bfl_model": "flux-kontext-pro",
            "bfl_aspect_ratio": "1:1",
            "bfl_output_format": "jpeg",
            "openai_max_output_tokens": 512,
        },
        "candidate_tournament": {"candidate_count": 1},
    },
    "Resume Existing Job": {
        "execution_mode": "development",
        "research_search": {"enabled": False},
        "llm": {"provider_order": [], "vision_provider_order": []},
    },
}


def workspace_layout(workspace: str | Path) -> dict[str, Path]:
    """Create and return the standard workspace directory layout."""
    base = ensure_dir(workspace)
    return {
        "base": base,
        "configs": ensure_dir(base / "configs"),
        "jobs": ensure_dir(base / "jobs"),
        "reports": ensure_dir(base / "reports"),
        "smoke_tests": ensure_dir(base / "smoke_tests"),
        "exports": ensure_dir(base / "exports"),
        "logs": ensure_dir(base / "logs"),
    }


def security_source_sweep(package_dir: str | Path) -> dict[str, Any]:
    """Repo sweep shared by the notebook cell and the Control Center."""
    package_dir = Path(package_dir)
    findings: list[str] = []
    # Pattern literals are built with `+` so this module's own source never
    # contains the raw markers it scans for (self-match prevention that
    # survives code formatters).
    anthropic_pattern = "ANTHROPIC" + "_API_KEY|import " + "anthropic|from " + "anthropic"
    stale_marker = "scistudio" + "_v8"
    shell_label = "unguarded shell" + "=True"
    files = sorted(package_dir.glob("*.py"))
    for path in files:
        text = path.read_text(encoding="utf-8")
        if re.search(anthropic_pattern, text):
            findings.append(f"{path.name}: Anthropic runtime reference")
        if stale_marker in text:
            findings.append(f"{path.name}: stale {stale_marker} reference")
        for match in re.finditer(r"(?<!\w)shell\s*=\s*True", text):
            line_no = text[: match.start()].count("\n") + 1
            if "noqa: S602" not in text.splitlines()[line_no - 1]:
                findings.append(f"{path.name}:{line_no}: {shell_label}")
        if re.search(r"[\"']/content/", text):
            findings.append(f"{path.name}: hard-coded /content path")
    return {"files_scanned": len(files), "findings": findings, "ok": not findings}


class NotebookRunController:
    """Single dispatch point for every Control Center action."""

    def __init__(self, workspace: str | Path = "./scientific_motion_studio_v10", secrets: SecretsManager | None = None):
        self.paths = workspace_layout(workspace)
        self.secrets = secrets or SecretsManager()

    # -- helpers -----------------------------------------------------------
    def required_secrets(self, action: PipelineAction) -> dict[str, bool]:
        live = action in LIVE_ACTIONS
        return {
            "OPENAI_API_KEY": live,
            "BFL_API_KEY": live,
            "UPLOAD_POST_TOKEN": False,
        }

    def validate_reference_video(self, reference_video: str) -> Path:
        if not reference_video:
            raise ReferenceVideoError("No reference video selected.")
        path = Path(reference_video)
        if not path.exists() or not path.is_file():
            raise ReferenceVideoError(f"Reference video not found: {path}")
        if path.suffix.lower() not in VIDEO_SUFFIXES:
            raise ReferenceVideoError(
                f"Unsupported reference format {path.suffix!r}; allowed: {', '.join(VIDEO_SUFFIXES)}"
            )
        if path.stat().st_size == 0:
            raise ReferenceVideoError(f"Reference video is empty: {path}")
        return path

    def validate_config(self, config: dict[str, Any]) -> StudioConfig:
        try:
            return StudioConfig.from_dict(config)
        except ValidationError as exc:
            fields = [
                {"field": ".".join(str(part) for part in err["loc"]), "message": err["msg"]} for err in exc.errors()
            ]
            error = ConfigurationError(
                "Configuration invalid: " + "; ".join(f"{f['field']}: {f['message']}" for f in fields[:8])
            )
            error.field_errors = fields  # type: ignore[attr-defined]
            raise error from exc

    def validate_request(self, request: PipelineRunRequest) -> StudioConfig:
        """Full pre-run validation, including the paid-run safety gate."""
        settings = self.validate_config(request.config)
        if request.action in PIPELINE_ACTIONS:
            self.validate_reference_video(request.reference_video)
            if not request.topic.strip():
                raise ConfigurationError("Topic must not be empty.")
        if request.action in LIVE_ACTIONS:
            if not request.confirm_paid_run:
                raise PreflightError(
                    "Paid action blocked: tick 'I understand this action may create paid API requests.'"
                )
            if request.paid_confirmation_phrase.strip() != PAID_CONFIRMATION_PHRASE:
                raise PreflightError(f"Paid action blocked: type the confirmation phrase {PAID_CONFIRMATION_PHRASE!r}.")
            for name, required in self.required_secrets(request.action).items():
                if required and not self.secrets.available(name):
                    raise ProviderAuthenticationError(
                        f"{name} is not configured; provide it before a paid run.",
                        provider=name.split("_")[0].lower(),
                    )
            if request.action in {
                PipelineAction.live_generation,
                PipelineAction.live_generation_preview,
                PipelineAction.full_production_render,
            }:
                if settings.execution_mode != ExecutionMode.production:
                    raise ConfigurationError(
                        "Live generation requires execution_mode='production' (provider locks must be active)."
                    )
        if request.action == PipelineAction.full_production_render:
            backend = settings.render.backend
            if backend == "remotion":
                import shutil as _shutil

                missing = [t for t in ("node", "npm", "npx") if not _shutil.which(t)]
                if missing:
                    raise RenderDependencyError(
                        f"Remotion render needs {', '.join(missing)} on PATH; "
                        "install Node.js or switch render.backend to 'pil'."
                    )
        return settings

    def request_summary(self, request: PipelineRunRequest) -> dict[str, Any]:
        """Human-reviewable summary shown before any paid run (request counts,
        never currency estimates)."""
        config = request.config or {}
        candidates = int((config.get("candidate_tournament") or {}).get("candidate_count", 4))
        revisions = int((config.get("flux_studio") or {}).get("maximum_director_revisions", 3))
        scenes = "unknown until storyboard exists"
        job_id = request.job_id or (
            generate_job_id(request.topic, request.reference_video) if request.topic and request.reference_video else ""
        )
        manifest = load_json(self.paths["jobs"] / job_id / "job_manifest.json") if job_id else None
        if isinstance(manifest, dict):
            scene_stages = [s for s in manifest.get("stages", {}) if s.startswith("09_shot_state_")]
            if scene_stages:
                scenes = len(scene_stages)
        image_requests = (
            "unknown (≈ scenes × candidates + revisions + variants)"
            if not isinstance(scenes, int)
            else scenes * candidates + scenes * revisions + scenes
        )
        return redact_secrets(
            {
                "action": request.action.value,
                "job_id": job_id,
                "scenes": scenes,
                "candidate_count": candidates,
                "max_director_revisions": revisions,
                "potential_image_generations": image_requests,
                "temporal_backends": (config.get("temporal") or {}).get(
                    "backend_preference", ["sketch-controlled-video", "deterministic-compositor"]
                ),
                "render_backend": (config.get("render") or {}).get("backend", "remotion"),
                "publishing_enabled": bool((config.get("publishing") or {}).get("enabled", False)),
            }
        )

    # -- actions -----------------------------------------------------------
    def run_preflight(self, request: PipelineRunRequest) -> dict[str, Any]:
        live = request.action in LIVE_ACTIONS
        needs_reference = request.action in PIPELINE_ACTIONS
        report = run_preflight(
            config=request.config or None,
            reference_video=request.reference_video if needs_reference else None,
            live=live,
            required_secrets=self.required_secrets(request.action)
            if live
            else {name: False for name in SecretsManager.SUPPORTED},
            secret_status=self.secrets.available,
            workspace=self.paths["base"],
            job_id=request.job_id,
        )
        save_preflight_report(report, self.paths["reports"] / "preflight_report.json")
        return report

    def run_quick_smoke(self, request: PipelineRunRequest | None = None) -> dict[str, Any]:
        """Offline smoke: synthetic video → dev plan-only run → resume check
        → redaction check. Requires no API keys."""
        import subprocess

        from .pipeline import ScientificMotionStudioV10

        root = ensure_dir(self.paths["smoke_tests"] / datetime.now(UTC).strftime("%Y%m%dT%H%M%S"))
        reference = root / "reference.mp4"
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-v",
                "error",
                "-f",
                "lavfi",
                "-i",
                "color=c=gray:s=180x320:r=12",
                "-t",
                "1",
                "-c:v",
                "libx264",
                "-pix_fmt",
                "yuv420p",
                str(reference),
            ],
            check=True,
        )
        config = {**CONFIG_PRESETS["Safe Development"], "workspace": str(root / "studio")}
        settings = self.validate_config(config)  # config parse + provider-lock validation
        assert settings.execution_mode == ExecutionMode.development
        events: list[tuple[str, str]] = []
        studio = ScientificMotionStudioV10(config)
        result = studio.run(
            "Quick smoke: what if rivers ran backward?",
            reference,
            plan_only=True,
            job_id="quick-smoke",
            progress_callback=lambda s, e, r: events.append((s, e)),
        )
        manifest = load_json(Path(result["job_manifest"]))
        artifacts_ok = all(Path(result[key]).exists() for key in ("job_manifest", "drawing_briefs", "shot_states"))
        # Resume: second run must reuse cached stages.
        events2: list[tuple[str, str]] = []
        ScientificMotionStudioV10(config).run(
            "Quick smoke: what if rivers ran backward?",
            reference,
            plan_only=True,
            job_id="quick-smoke",
            progress_callback=lambda s, e, r: events2.append((s, e)),
        )
        resumed = any(e == "cached" for _s, e in events2)
        register_secret("smoke-test-secret-value-123")
        redaction_ok = "smoke-test-secret-value-123" not in redact_secrets(
            "error with smoke-test-secret-value-123 embedded"
        )
        report = {
            "passed": bool(
                manifest and manifest.get("status") == "completed" and artifacts_ok and resumed and redaction_ok
            ),
            "manifest_status": manifest.get("status") if manifest else "missing",
            "artifacts_ok": artifacts_ok,
            "resume_used_cache": resumed,
            "redaction_ok": redaction_ok,
            "stage_events": len(events),
            "run_dir": result["run_dir"],
        }
        save_json(self.paths["reports"] / "quick_smoke_report.json", report)
        return report

    def run_offline_tests(self, request: PipelineRunRequest | None = None) -> dict[str, Any]:
        from .tests_final import run_all_tests

        report = run_all_tests(self.paths["reports"] / "offline_test_scratch")
        package_dir = Path(__file__).parent
        sweep = security_source_sweep(package_dir)
        combined = {
            "passed": bool(report["passed"] and sweep["ok"]),
            "test_count": report["count"],
            "regression_count": report["regression_count"],
            "final_count": report["final_count"],
            "security_sweep": sweep,
        }
        save_json(self.paths["reports"] / "offline_test_report.json", {**combined, "tests": report["tests"]})
        return combined

    def run_provider_smoke(self, request: PipelineRunRequest) -> dict[str, Any]:
        """Minimal paid smoke per provider; gated by the paid-run rules."""
        if not request.confirm_paid_run or request.paid_confirmation_phrase.strip() != PAID_CONFIRMATION_PHRASE:
            raise PreflightError(f"Live smoke blocked: confirm the checkbox and type {PAID_CONFIRMATION_PHRASE!r}.")
        results: dict[str, Any] = {"openai": {"status": "SKIPPED"}, "bfl": {"status": "SKIPPED"}}
        smoke_root = ensure_dir(self.paths["smoke_tests"] / "provider_live")
        if self.secrets.available("OPENAI_API_KEY"):
            from .llm import LLMRouter

            try:
                router = LLMRouter(
                    {
                        "execution_mode": "production",
                        "provider_order": ["openai"],
                        "openai_max_output_tokens": 512,
                        "retry": {"max_attempts": 2, "base_delay_s": 1.0},
                    },
                    self.secrets.as_secrets_dict(),
                    smoke_root / "openai_cache",
                )
                value = router.generate_json(
                    system="You return strict JSON.",
                    prompt='Return exactly {"status": "ok", "provider": "openai"}.',
                    namespace="live_smoke",
                    fallback=None,
                    force=True,
                )
                ok = isinstance(value, dict) and value.get("status") == "ok"
                results["openai"] = {
                    "status": "PASS" if ok else "FAIL",
                    "request_id": getattr(router, "_last_request_id", ""),
                }
            except Exception as exc:
                classified = classify_provider_error(exc)
                results["openai"] = {"status": "FAIL", "error": redacted_exception_text(classified, 300)}
        if self.secrets.available("BFL_API_KEY"):
            from .bfl_client import BFLClient

            try:
                client = BFLClient(
                    {
                        "bfl_model": "flux-kontext-pro",
                        "bfl_aspect_ratio": "1:1",
                        "bfl_output_format": "jpeg",
                        "bfl_timeout": 120,
                        "retry": {"max_attempts": 2},
                    },
                    self.secrets.get("BFL_API_KEY"),
                    smoke_root / "bfl_cache",
                )
                out = client.generate(
                    "Minimal test: one small blue circle on white, flat vector style.", smoke_root / "bfl_smoke.jpg"
                )
                results["bfl"] = {"status": "PASS" if out.exists() else "FAIL", "artifact": str(out)}
            except Exception as exc:
                classified = classify_provider_error(exc)
                results["bfl"] = {"status": "FAIL", "error": redacted_exception_text(classified, 300)}
        save_json(self.paths["reports"] / "provider_live_smoke.json", redact_secrets(results))
        return results

    def _apply_force_scope(self, request: PipelineRunRequest, settings: StudioConfig) -> bool:
        """Prepare a scoped force-rerun by marking manifest stages stale.

        Returns the ``force`` flag to pass to the pipeline (True only for the
        'all' scope)."""
        if request.action != PipelineAction.force_rerun:
            return request.force
        if request.force_scope == "all":
            return True
        job_id = request.job_id or generate_job_id(request.topic, request.reference_video)
        manifest_path = self.paths["jobs"] / job_id / "job_manifest.json"
        manifest = load_json(manifest_path)
        if not isinstance(manifest, dict):
            return True  # fresh job: nothing to scope
        stages = manifest.get("stages", {})
        ordered = sorted(stages.keys())
        if request.force_scope == "failed_and_downstream":
            failed = [s for s in ordered if stages[s].get("status") == "failed"]
            anchor = failed[0] if failed else None
        else:  # selected_and_downstream
            anchor = request.force_stage or None
            if anchor is not None and anchor not in stages:
                raise ConfigurationError(f"Unknown stage for force scope: {anchor!r}")
        if anchor is None:
            return False
        for stage_id in ordered:
            if stage_id >= anchor and stages[stage_id].get("status") == "completed":
                stages[stage_id]["status"] = "stale"
        save_json(manifest_path, manifest)
        return False

    def run_pipeline(
        self,
        request: PipelineRunRequest,
        *,
        progress_callback: Callable[[str, str, dict], None] | None = None,
        cancellation_token: CancellationToken | None = None,
    ) -> dict[str, Any]:
        """Validate, preflight-gate and execute a pipeline action."""
        settings = self.validate_request(request)
        if request.action in LIVE_ACTIONS:
            preflight = self.run_preflight(request)
            if preflight["summary"][FAIL]:
                failed = [c["check"] for c in preflight["checks"] if c["status"] == FAIL]
                raise PreflightError("Paid action blocked by failed preflight checks: " + ", ".join(failed))
        plan_only, render_video = request.plan_only, request.render_video
        action = request.action
        if action in {PipelineAction.plan_only, PipelineAction.development_dry_run, PipelineAction.production_plan}:
            plan_only, render_video = True, False
        elif action == PipelineAction.live_generation:
            plan_only, render_video = False, False
        elif action in {PipelineAction.live_generation_preview, PipelineAction.full_production_render}:
            plan_only, render_video = False, True
        force = self._apply_force_scope(request, settings)
        if action == PipelineAction.resume_job:
            force = False  # never force on resume unless the user picked Force Rerun

        job_id = request.job_id or generate_job_id(request.topic, request.reference_video)
        run_dir = ensure_dir(self.paths["jobs"] / job_id)
        save_json(run_dir / "run_request.json", redact_secrets(request.model_dump(mode="json")))
        config = dict(request.config)
        config["workspace"] = str(self.paths["base"])
        save_json(run_dir / "resolved_config.json", redact_secrets(config))

        from .pipeline import ScientificMotionStudioV10

        try:
            studio = ScientificMotionStudioV10(config, secrets=self.secrets.as_secrets_dict())
            result = studio.run(
                request.topic,
                request.reference_video,
                plan_only=plan_only,
                render_video=render_video,
                force=force,
                job_id=job_id,
                progress_callback=progress_callback,
                cancellation_token=cancellation_token,
            )
            return result
        except Exception as exc:
            classified = classify_provider_error(exc)
            save_json(
                run_dir / "error_report.json",
                {
                    "error_type": type(classified).__name__,
                    "message": redacted_exception_text(classified, 800),
                    "stage_id": getattr(classified, "stage_id", ""),
                    "timestamp": datetime.now(UTC).isoformat(),
                    "action": request.action.value,
                },
            )
            raise classified from exc

    def resume_pipeline(self, request: PipelineRunRequest, **kwargs: Any) -> dict[str, Any]:
        resumed = request.model_copy(update={"action": PipelineAction.resume_job, "force": False})
        return self.run_pipeline(resumed, **kwargs)

    def job_status(self, job_id: str) -> dict[str, Any] | None:
        manifest = load_json(self.paths["jobs"] / job_id / "job_manifest.json")
        return redact_secrets(manifest) if isinstance(manifest, dict) else None

    def package_artifacts(self, job_id: str) -> Path:
        """ZIP a run directory into workspace/exports (path-traversal safe)."""
        run_dir = ensure_within(self.paths["jobs"], self.paths["jobs"] / job_id)
        if not run_dir.exists():
            raise ConfigurationError(f"No run directory for job {job_id!r}.")
        target = self.paths["exports"] / f"{slugify(job_id, 60)}.zip"
        with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(run_dir.rglob("*")):
                if path.is_file() and path.name != "job.lock":
                    archive.write(path, path.relative_to(run_dir.parent))
        return target

    # -- dispatch -----------------------------------------------------------
    def dispatch(self, request: PipelineRunRequest, **kwargs: Any) -> Any:
        """Route any action through one entry point (used by all UIs)."""
        handlers: dict[PipelineAction, Callable[..., Any]] = {
            PipelineAction.environment_check: lambda: self.run_preflight(request),
            PipelineAction.config_validation: lambda: {
                "valid": True,
                "config": redact_secrets(self.validate_config(request.config).as_runtime_dict()),
            },
            PipelineAction.quick_smoke: lambda: self.run_quick_smoke(request),
            PipelineAction.full_offline_tests: lambda: self.run_offline_tests(request),
            PipelineAction.provider_live_smoke: lambda: self.run_provider_smoke(request),
            PipelineAction.package_artifacts: lambda: {
                "archive": str(
                    self.package_artifacts(request.job_id or generate_job_id(request.topic, request.reference_video))
                )
            },
        }
        if request.action in handlers:
            return handlers[request.action]()
        if request.action in PIPELINE_ACTIONS:
            return self.run_pipeline(request, **kwargs)
        raise ConfigurationError(f"Unknown action {request.action!r}")


class RunMonitorState:
    """Thread-safe aggregation of stage events for progress displays."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.started_at = time.time()
        self.current_stage = ""
        self.completed: list[str] = []
        self.cached: list[str] = []
        self.failed: list[str] = []
        self.retries = 0
        self.events: list[dict[str, Any]] = []

    def callback(self, stage_id: str, event: str, record: dict[str, Any]) -> None:
        with self._lock:
            self.events.append({"stage": stage_id, "event": event, "attempt": record.get("attempt", 0)})
            if event == "started":
                self.current_stage = stage_id
                if int(record.get("attempt", 1)) > 1:
                    self.retries += 1
            elif event == "completed":
                self.completed.append(stage_id)
            elif event == "cached":
                self.cached.append(stage_id)
            elif event == "failed":
                self.failed.append(stage_id)

    def snapshot(self) -> dict[str, Any]:
        with self._lock:
            return {
                "elapsed_s": round(time.time() - self.started_at, 1),
                "current_stage": self.current_stage,
                "progress": stage_progress_fraction(self.current_stage),
                "completed": list(self.completed),
                "cached": list(self.cached),
                "failed": list(self.failed),
                "retries": self.retries,
                "event_count": len(self.events),
            }


In [ ]:
%%writefile scistudio_v10/notebook_ui.py
"""Notebook-native Control Center (ipywidgets).

``launch_control_center()`` renders a tabbed production console inside
Jupyter/Colab. Every button dispatches through the shared
:class:`NotebookRunController`, so notebook, Streamlit and headless usage all
run identical logic. When ``ipywidgets`` is unavailable the function returns
a :class:`HeadlessControlCenter` with the same controller and printed
instructions instead of crashing — safe for plain ``python -c`` imports and
CI.

Safety: rendering the Control Center performs **no network calls**. Paid
actions stay disabled until the confirmation checkbox is ticked and the
phrase ``RUN LIVE`` is typed, and they still pass the controller's gate.
"""

from __future__ import annotations

import json
import threading
from pathlib import Path
from typing import Any

from .config_models import BFL_ALLOWED_MODELS
from .notebook_runner import (
    ACTION_DESCRIPTIONS,
    CONFIG_PRESETS,
    LIVE_ACTIONS,
    PAID_CONFIRMATION_PHRASE,
    CancellationToken,
    NotebookRunController,
    PipelineAction,
    PipelineRunRequest,
    RunMonitorState,
    generate_job_id,
)
from .preflight import VIDEO_SUFFIXES
from .security import redact_secrets, redacted_exception_text
from .utils import ffprobe_duration, load_json

STATUS_ICONS = {
    "PASS": "✓ PASS",
    "WARNING": "⚠ WARNING",
    "FAIL": "✕ FAIL",
    "SKIPPED": "○ SKIPPED",
    "RUNNING": "● RUNNING",
}

_CSS = """<style>
.scs-card {display:inline-block;border:1px solid #b8c1c6;border-radius:8px;
  padding:8px 14px;margin:4px;min-width:130px;font-family:sans-serif;font-size:12px;}
.scs-card b {display:block;font-size:11px;color:#475157;text-transform:uppercase;}
.scs-warn {border-left:4px solid #d8483e;background:#fff6f5;padding:6px 10px;
  margin:6px 0;font-family:sans-serif;font-size:13px;}
.scs-ok {border-left:4px solid #2e77a6;background:#f4f9fc;padding:6px 10px;
  margin:6px 0;font-family:sans-serif;font-size:13px;}
</style>"""


def _widgets_available() -> bool:
    try:
        import ipywidgets  # noqa: F401
        import IPython.display  # noqa: F401

        return True
    except ImportError:
        return False


class HeadlessControlCenter:
    """Fallback returned when ipywidgets is missing: same controller,
    programmatic access, printed guidance — never a crash."""

    def __init__(self, controller: NotebookRunController):
        self.controller = controller
        self.headless = True

    def instructions(self) -> str:
        return (
            "ipywidgets is not installed, so the visual Control Center is unavailable.\n"
            "Install it with: pip install ipywidgets\n"
            "Headless equivalent:\n"
            "  from scistudio_v10.notebook_runner import NotebookRunController, "
            "PipelineRunRequest, PipelineAction\n"
            "  controller = NotebookRunController(workspace)\n"
            "  controller.dispatch(PipelineRunRequest(action=PipelineAction.environment_check))"
        )


def launch_control_center(
    workspace: str | Path = "./scientific_motion_studio_v10",
    *,
    controller: NotebookRunController | None = None,
):
    """Render the Control Center; returns the ControlCenter object.

    Safe under ``Run all``: only widgets are constructed — no provider calls.
    """
    controller = controller or NotebookRunController(workspace)
    if not _widgets_available():
        center = HeadlessControlCenter(controller)
        print(center.instructions())
        return center
    center = ControlCenter(controller)
    center.display()
    return center


class ControlCenter:
    """The interactive widget application."""

    def __init__(self, controller: NotebookRunController):
        import ipywidgets as w

        self.w = w
        self.controller = controller
        self.state: dict[str, Any] = {
            "config": dict(CONFIG_PRESETS["Safe Development"]),
            "last_result": None,
            "preflight": None,
        }
        self.monitor = RunMonitorState()
        self.cancellation: CancellationToken | None = None
        self._run_thread: threading.Thread | None = None
        self._build()

    # ------------------------------------------------------------------ UI
    def _build(self) -> None:
        w = self.w
        self.cards = w.HTML()
        self.out_project = w.Output()
        self.out_execution = w.Output()
        self.out_secrets = w.Output()
        self.out_preflight = w.Output()
        self.out_monitor = w.Output()
        self.out_results = w.Output()
        self.out_config = w.Output()

        # -- Tab 1: Project
        self.topic = w.Textarea(
            description="Topic",
            placeholder="Scientific question, e.g. What happens if it rains nonstop for one year?",
            layout=w.Layout(width="95%", height="60px"),
        )
        self.reference = w.Text(
            description="Reference", placeholder="path/to/reference.mp4", layout=w.Layout(width="70%")
        )
        self.job_id = w.Text(
            description="Job ID", placeholder="(auto from topic + reference)", layout=w.Layout(width="70%")
        )
        self.upload = w.FileUpload(accept=",".join(VIDEO_SUFFIXES), multiple=False, description="Upload video")
        upload_button = w.Button(description="Save uploaded file", icon="save")
        use_existing = w.Button(description="Use path above", icon="check")
        drive_button = w.Button(description="Mount Google Drive", icon="cloud")
        upload_button.on_click(self._on_save_upload)
        use_existing.on_click(lambda _b: self._describe_reference())
        drive_button.on_click(self._on_mount_drive)
        tab_project = w.VBox(
            [
                self.topic,
                self.reference,
                self.job_id,
                w.HBox([self.upload, upload_button, use_existing, drive_button]),
                self.out_project,
            ]
        )

        # -- Tab 2: Execution
        self.action = w.Dropdown(
            description="Action",
            options=[(a.value.replace("_", " ").title(), a) for a in PipelineAction],
            value=PipelineAction.environment_check,
            layout=w.Layout(width="60%"),
        )
        self.action_help = w.HTML()
        self.force_scope = w.Dropdown(
            description="Force scope", options=["all", "failed_and_downstream", "selected_and_downstream"], value="all"
        )
        self.force_stage = w.Text(description="Force stage", placeholder="e.g. 13_flux_studio")
        self.primary = w.Button(
            description="Run selected action", button_style="primary", icon="play", layout=w.Layout(width="240px")
        )
        self.stop_button = w.Button(description="Stop after current safe stage", icon="stop", disabled=True)
        self.action.observe(self._on_action_change, names="value")
        self.primary.on_click(self._on_run)
        self.stop_button.on_click(self._on_stop)
        tab_execution = w.VBox(
            [
                self.action,
                self.action_help,
                w.HBox([self.force_scope, self.force_stage]),
                w.HBox([self.primary, self.stop_button]),
                self.out_execution,
            ]
        )

        # -- Tab 3: Providers & Secrets
        self.secret_inputs = {
            name: w.Password(description=name, placeholder="paste value (memory only)", layout=w.Layout(width="60%"))
            for name in ("OPENAI_API_KEY", "BFL_API_KEY", "UPLOAD_POST_TOKEN")
        }
        refresh_secrets = w.Button(description="Refresh Secret Status", icon="refresh")
        load_colab = w.Button(description="Load from Colab Secrets", icon="download")
        clear_manual = w.Button(description="Clear Manual Secrets", icon="trash")
        apply_manual = w.Button(description="Apply manual values", icon="key")
        refresh_secrets.on_click(lambda _b: self._render_secret_status())
        load_colab.on_click(lambda _b: self._render_secret_status())
        clear_manual.on_click(self._on_clear_secrets)
        apply_manual.on_click(self._on_apply_secrets)
        tab_secrets = w.VBox(
            [
                w.HTML(
                    "<div class='scs-ok'>Secrets are resolved from Colab Secrets → environment "
                    "variables → the fields below. Values stay in memory, are registered for "
                    "redaction, and are never written to configs, logs or notebook output.</div>"
                ),
                *self.secret_inputs.values(),
                w.HBox([apply_manual, refresh_secrets, load_colab, clear_manual]),
                self.out_secrets,
            ]
        )

        # -- Tab 4: Model & Generation
        self.f_openai_model = w.Text(description="OpenAI model", value="gpt-5-mini")
        self.f_openai_vision = w.Text(description="Vision model", placeholder="(defaults to OpenAI model)")
        self.f_effort = w.Dropdown(description="Effort", options=["minimal", "low", "medium", "high"], value="low")
        self.f_max_tokens = w.BoundedIntText(description="Max tokens", value=8000, min=256, max=200000)
        self.f_retries = w.BoundedIntText(description="Retries", value=4, min=1, max=10)
        self.f_retry_delay = w.BoundedFloatText(description="Retry delay", value=1.0, min=0.05, max=30.0)
        self.f_bfl_model = w.Dropdown(
            description="BFL model", options=sorted(BFL_ALLOWED_MODELS), value="flux-kontext-pro"
        )
        self.f_aspect = w.Dropdown(description="Aspect", options=["9:16", "16:9", "1:1", "4:5", "3:4"], value="9:16")
        self.f_format = w.Dropdown(description="Format", options=["png", "jpeg"], value="png")
        self.f_upsampling = w.Checkbox(description="Prompt upsampling", value=False, indent=False)
        self.f_safety = w.BoundedIntText(description="Safety tol.", value=2, min=0, max=6)
        self.f_candidates = w.BoundedIntText(description="Candidates", value=4, min=1, max=12)
        self.f_seed_stride = w.BoundedIntText(description="Seed stride", value=9973, min=1, max=10**6)
        self.f_revisions = w.BoundedIntText(description="Max revisions", value=3, min=1, max=10)
        self.f_vision_director = w.Checkbox(description="Require vision director", value=True, indent=False)
        self.f_research = w.Checkbox(description="Research search", value=True, indent=False)
        self.f_audio = w.Checkbox(description="Audio enabled", value=True, indent=False)
        self.f_audio_plan = w.Checkbox(description="Build audio in plan mode", value=False, indent=False)
        tab_model = w.VBox(
            [
                w.HTML("<b>OpenAI (reasoning + vision)</b>"),
                self.f_openai_model,
                self.f_openai_vision,
                self.f_effort,
                self.f_max_tokens,
                w.HBox([self.f_retries, self.f_retry_delay]),
                w.HTML("<b>BFL FLUX Kontext (image generation)</b>"),
                self.f_bfl_model,
                w.HBox([self.f_aspect, self.f_format]),
                w.HBox([self.f_upsampling, self.f_safety]),
                w.HTML("<b>Generation policy</b>"),
                w.HBox([self.f_candidates, self.f_seed_stride]),
                w.HBox([self.f_revisions, self.f_vision_director]),
                w.HBox([self.f_research, self.f_audio, self.f_audio_plan]),
            ]
        )

        # -- Tab 5: Temporal & Rendering
        self.f_temporal = w.Checkbox(description="Temporal enabled", value=True, indent=False)
        self.f_temporal_artic = w.Checkbox(description="Use for articulated motion", value=False, indent=False)
        self.f_backend_pref = w.Text(
            description="Backend pref",
            value="sketch-controlled-video, deterministic-compositor",
            layout=w.Layout(width="70%"),
        )
        self.f_temporal_argv = w.Text(
            description="Command argv",
            placeholder='JSON list, e.g. ["/usr/bin/model", "--in", "{start}"]',
            layout=w.Layout(width="70%"),
        )
        self.f_temporal_timeout = w.BoundedFloatText(description="Cmd timeout", value=1800.0, min=1.0, max=86400.0)
        self.f_allow_shell = w.Checkbox(description="Allow shell (unsafe)", value=False, indent=False)
        self.f_shell_override = w.Checkbox(description="Security override for unsafe shell", value=False, indent=False)
        self.f_render_backend = w.Dropdown(
            description="Render", value="pil", options=["remotion", "preview", "pil", "deterministic"]
        )
        self.f_npm_install = w.Checkbox(description="Install npm deps", value=True, indent=False)
        self.f_crf = w.BoundedIntText(description="CRF", value=18, min=0, max=51)
        self.f_render_timeout = w.BoundedIntText(description="Render timeout", value=2400, min=60, max=24000)
        self.f_publish = w.Checkbox(description="Publishing enabled", value=False, indent=False)
        self.f_publish_provider = w.Dropdown(
            description="Publisher", options=["local-archive", "upload-post"], value="local-archive"
        )
        self.f_archive_dir = w.Text(description="Archive dir", placeholder="(default: run dir/published)")
        self.f_endpoint = w.Text(description="Endpoint", placeholder="https://…")
        self.f_allowed_hosts = w.Text(description="Allowed hosts", placeholder="api.example.com, *.example.com")
        self.render_warnings = w.HTML()
        for widget in (self.f_allow_shell, self.f_publish, self.f_render_backend):
            widget.observe(lambda _c: self._render_render_warnings(), names="value")
        tab_temporal = w.VBox(
            [
                self.f_temporal,
                self.f_temporal_artic,
                self.f_backend_pref,
                self.f_temporal_argv,
                w.HBox([self.f_temporal_timeout, self.f_allow_shell, self.f_shell_override]),
                w.HTML("<b>Rendering</b>"),
                w.HBox([self.f_render_backend, self.f_npm_install]),
                w.HBox([self.f_crf, self.f_render_timeout]),
                w.HTML("<b>Publishing</b>"),
                w.HBox([self.f_publish, self.f_publish_provider]),
                self.f_archive_dir,
                self.f_endpoint,
                self.f_allowed_hosts,
                self.render_warnings,
            ]
        )

        # -- Tab 6: Advanced Configuration
        self.preset = w.Dropdown(description="Preset", options=list(CONFIG_PRESETS), value="Safe Development")
        self.mode = w.Dropdown(description="Mode", options=["development", "test", "production"], value="development")
        apply_preset = w.Button(description="Reset to preset", icon="undo")
        form_to_json = w.Button(description="Generate JSON from form", icon="arrow-down")
        json_to_form = w.Button(description="Load form from JSON", icon="arrow-up")
        validate_button = w.Button(description="Validate config", icon="check", button_style="info")
        export_button = w.Button(description="Export config JSON", icon="save")
        import_button = w.Button(description="Import config JSON file", icon="folder-open")
        diff_button = w.Button(description="Diff vs preset defaults", icon="exchange")
        self.json_editor = w.Textarea(
            layout=w.Layout(width="95%", height="220px"), placeholder="Advanced JSON config editor"
        )
        self.import_path = w.Text(description="Import path", placeholder="workspace/configs/config.json")
        apply_preset.on_click(self._on_apply_preset)
        form_to_json.on_click(lambda _b: self._form_to_json(confirm=True))
        json_to_form.on_click(lambda _b: self._json_to_form())
        validate_button.on_click(lambda _b: self._validate_config_clicked())
        export_button.on_click(self._on_export_config)
        import_button.on_click(self._on_import_config)
        diff_button.on_click(self._on_diff_config)
        tab_config = w.VBox(
            [
                w.HBox([self.preset, self.mode, apply_preset]),
                w.HBox([form_to_json, json_to_form, validate_button, diff_button]),
                self.json_editor,
                w.HBox([self.import_path, import_button, export_button]),
                self.out_config,
            ]
        )

        # -- Tab 7: Preflight
        preflight_button = w.Button(description="Run preflight", button_style="info", icon="search")
        rerun_button = w.Button(description="Rerun preflight", icon="refresh")
        install_button = w.Button(description="Install missing Python packages", icon="wrench")
        download_button = w.Button(description="Write preflight_report.json", icon="download")
        preflight_button.on_click(lambda _b: self._run_preflight_clicked())
        rerun_button.on_click(lambda _b: self._run_preflight_clicked())
        install_button.on_click(self._on_install_missing)
        download_button.on_click(self._on_download_preflight)
        tab_preflight = w.VBox(
            [w.HBox([preflight_button, rerun_button, install_button, download_button]), self.out_preflight]
        )

        # -- Tab 8: Run Monitor
        self.progress = w.FloatProgress(
            value=0.0, min=0.0, max=1.0, description="Progress", layout=w.Layout(width="70%")
        )
        self.monitor_html = w.HTML("<i>No run yet. Choose an action in the Execution tab.</i>")
        tab_monitor = w.VBox([self.progress, self.monitor_html, self.out_monitor])

        # -- Tab 9: Results
        results_refresh = w.Button(description="Refresh results", icon="refresh")
        zip_button = w.Button(description="ZIP run directory", icon="file-archive")
        compare_a = w.Text(description="Compare A", placeholder="job id")
        compare_b = w.Text(description="Compare B", placeholder="job id")
        compare_button = w.Button(description="Compare manifests", icon="exchange")
        results_refresh.on_click(lambda _b: self._render_results())
        zip_button.on_click(self._on_zip)
        compare_button.on_click(lambda _b: self._compare_manifests(compare_a.value, compare_b.value))
        tab_results = w.VBox(
            [w.HBox([results_refresh, zip_button]), w.HBox([compare_a, compare_b, compare_button]), self.out_results]
        )

        self.tabs = w.Tab(
            children=[
                tab_project,
                tab_execution,
                tab_secrets,
                tab_model,
                tab_temporal,
                tab_config,
                tab_preflight,
                tab_monitor,
                tab_results,
            ]
        )
        for index, title in enumerate(
            [
                "Project",
                "Execution",
                "Providers & Secrets",
                "Model & Generation",
                "Temporal & Rendering",
                "Advanced Config",
                "Preflight & Tests",
                "Run Monitor",
                "Results & Artifacts",
            ]
        ):
            self.tabs.set_title(index, title)

        # Paid-run gate widgets (shown above tabs, next to primary action)
        self.paid_checkbox = w.Checkbox(
            description="I understand this action may create paid API requests.",
            value=False,
            indent=False,
            layout=w.Layout(width="60%"),
        )
        self.paid_phrase = w.Text(
            description="Type", placeholder=PAID_CONFIRMATION_PHRASE, layout=w.Layout(width="40%")
        )
        self.paid_box = w.HBox([self.paid_checkbox, self.paid_phrase])
        self.paid_box.layout.display = "none"
        self.paid_checkbox.observe(lambda _c: self._update_primary_enabled(), names="value")
        self.paid_phrase.observe(lambda _c: self._update_primary_enabled(), names="value")

        self._on_action_change(None)
        self._render_cards()
        self._render_secret_status()
        self._render_render_warnings()
        self._form_to_json(confirm=False)

    def display(self) -> None:
        from IPython.display import HTML, display

        display(HTML(_CSS))
        display(self.cards)
        display(self.paid_box)
        display(self.tabs)

    # ------------------------------------------------------------- helpers
    def _card(self, title: str, value: str) -> str:
        return f"<span class='scs-card'><b>{title}</b>{value}</span>"

    def _render_cards(self) -> None:
        import shutil as _shutil

        env = "✓" if _shutil.which("ffmpeg") else "✕ ffmpeg missing"
        secrets = self.controller.secrets.status()
        configured = sum(1 for v in secrets.values() if v == "Configured")
        reference = Path(self.reference.value).name if self.reference.value else "—"
        job = self.job_id.value or (
            generate_job_id(self.topic.value, self.reference.value)
            if self.topic.value and self.reference.value
            else "—"
        )
        last = self.state.get("last_result")
        preflight = self.state.get("preflight")
        self.cards.value = _CSS + "".join(
            [
                self._card("Environment", env),
                self._card("Configuration", self.mode.value),
                self._card("Secrets", f"{configured}/3 configured"),
                self._card("Reference", reference),
                self._card("Current Job", job),
                self._card("Last Result", (last or {}).get("mode", "—") if isinstance(last, dict) else "—"),
                self._card(
                    "Preflight",
                    f"{preflight['summary']['FAIL']} fail / {preflight['summary']['WARNING']} warn"
                    if preflight
                    else "not run",
                ),
            ]
        )

    def _describe_reference(self) -> None:
        with self.out_project:
            self.out_project.clear_output()
            try:
                path = self.controller.validate_reference_video(self.reference.value)
                duration = ffprobe_duration(path)
                print(
                    f"{STATUS_ICONS['PASS']}  {path.name} | {path.suffix} | "
                    f"{path.stat().st_size / 1024:.0f} KiB | {duration:.1f}s"
                )
                try:
                    from IPython.display import Video, display

                    display(Video(str(path), embed=False, width=200))
                except Exception:
                    print("(inline preview unavailable in this frontend)")
            except Exception as exc:
                print(f"{STATUS_ICONS['FAIL']}  {redacted_exception_text(exc, 300)}")
        self._render_cards()

    def _on_save_upload(self, _button) -> None:
        with self.out_project:
            self.out_project.clear_output()
            value = self.upload.value
            items = list(value.values()) if isinstance(value, dict) else list(value)
            if not items:
                print("No file selected in the upload widget.")
                return
            item = items[0]
            name = item.get("name") if isinstance(item, dict) else getattr(item, "name", "upload.mp4")
            content = item.get("content") if isinstance(item, dict) else getattr(item, "content", b"")
            if isinstance(content, memoryview):
                content = content.tobytes()
            if isinstance(content, dict):
                content = content.get("content", b"")
            target = self.controller.paths["base"] / "uploads" / str(name)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(bytes(content))
            self.reference.value = str(target)
            print(f"Saved upload to {target}")
        self._describe_reference()

    def _on_mount_drive(self, _button) -> None:
        with self.out_project:
            self.out_project.clear_output()
            try:
                from google.colab import drive  # type: ignore[import-not-found]

                drive.mount("/gdrive")
                print("Google Drive mounted at /gdrive.")
            except ImportError:
                print("Google Drive mounting is only available inside Google Colab.")
            except Exception as exc:
                print(f"Drive mount failed: {redacted_exception_text(exc, 200)}")

    # -------------------------------------------------------------- config
    def _config_from_form(self) -> dict[str, Any]:
        argv: list[str] = []
        if self.f_temporal_argv.value.strip():
            try:
                parsed = json.loads(self.f_temporal_argv.value)
                if isinstance(parsed, list):
                    argv = [str(x) for x in parsed]
            except json.JSONDecodeError:
                argv = [self.f_temporal_argv.value]
        config: dict[str, Any] = {
            "workspace": str(self.controller.paths["base"]),
            "execution_mode": self.mode.value,
            "research_search": {"enabled": self.f_research.value},
            "llm": {
                "provider_order": ["openai"]
                if self.mode.value == "production"
                else (["openai"] if self.controller.secrets.available("OPENAI_API_KEY") else []),
                "vision_provider_order": ["openai"]
                if self.mode.value == "production"
                else (["openai"] if self.controller.secrets.available("OPENAI_API_KEY") else []),
                "openai_model": self.f_openai_model.value,
                "openai_vision_model": self.f_openai_vision.value,
                "openai_reasoning_effort": self.f_effort.value,
                "openai_max_output_tokens": int(self.f_max_tokens.value),
                "image_provider": "bfl",
                "bfl_model": self.f_bfl_model.value,
                "bfl_aspect_ratio": self.f_aspect.value,
                "bfl_output_format": self.f_format.value,
                "bfl_prompt_upsampling": bool(self.f_upsampling.value),
                "bfl_safety_tolerance": int(self.f_safety.value),
                "retry": {"max_attempts": int(self.f_retries.value), "base_delay_s": float(self.f_retry_delay.value)},
            },
            "candidate_tournament": {
                "candidate_count": int(self.f_candidates.value),
                "seed_stride": int(self.f_seed_stride.value),
            },
            "flux_studio": {
                "maximum_director_revisions": int(self.f_revisions.value),
                "require_vision_director": bool(self.f_vision_director.value),
            },
            "audio": {"enabled": bool(self.f_audio.value), "build_in_plan_mode": bool(self.f_audio_plan.value)},
            "temporal": {
                "enabled": bool(self.f_temporal.value),
                "use_for_articulated": bool(self.f_temporal_artic.value),
                "backend_preference": [x.strip() for x in self.f_backend_pref.value.split(",") if x.strip()],
                "sketch_backend": {
                    "argv": argv,
                    "timeout_s": float(self.f_temporal_timeout.value),
                    "allow_shell": bool(self.f_allow_shell.value),
                    "security_override_unsafe_shell": bool(self.f_shell_override.value),
                },
            },
            "render": {
                "backend": self.f_render_backend.value,
                "install_dependencies": bool(self.f_npm_install.value),
                "crf": int(self.f_crf.value),
                "render_timeout": int(self.f_render_timeout.value),
            },
            "publishing": {
                "enabled": bool(self.f_publish.value),
                "provider": self.f_publish_provider.value,
                "archive_dir": self.f_archive_dir.value,
                "endpoint": self.f_endpoint.value,
                "allowed_hosts": [x.strip() for x in self.f_allowed_hosts.value.split(",") if x.strip()],
            },
        }
        return config

    def _apply_config_to_form(self, config: dict[str, Any]) -> None:
        llm = config.get("llm") or {}
        self.mode.value = str(config.get("execution_mode", "development"))
        self.f_openai_model.value = str(llm.get("openai_model", "gpt-5-mini"))
        self.f_openai_vision.value = str(llm.get("openai_vision_model", ""))
        self.f_effort.value = str(llm.get("openai_reasoning_effort", "low"))
        self.f_max_tokens.value = int(llm.get("openai_max_output_tokens", 8000))
        retry = llm.get("retry") or {}
        self.f_retries.value = int(retry.get("max_attempts", 4))
        self.f_retry_delay.value = float(retry.get("base_delay_s", 1.0))
        if str(llm.get("bfl_model", "flux-kontext-pro")) in BFL_ALLOWED_MODELS:
            self.f_bfl_model.value = str(llm.get("bfl_model", "flux-kontext-pro"))
        aspect = str(llm.get("bfl_aspect_ratio", "9:16"))
        if aspect not in list(self.f_aspect.options):
            self.f_aspect.options = [*self.f_aspect.options, aspect]
        self.f_aspect.value = aspect
        self.f_format.value = str(llm.get("bfl_output_format", "png"))
        self.f_upsampling.value = bool(llm.get("bfl_prompt_upsampling", False))
        self.f_safety.value = int(llm.get("bfl_safety_tolerance", 2))
        tournament = config.get("candidate_tournament") or {}
        self.f_candidates.value = int(tournament.get("candidate_count", 4))
        self.f_seed_stride.value = int(tournament.get("seed_stride", 9973))
        flux = config.get("flux_studio") or {}
        self.f_revisions.value = int(flux.get("maximum_director_revisions", 3))
        self.f_vision_director.value = bool(flux.get("require_vision_director", True))
        self.f_research.value = bool((config.get("research_search") or {}).get("enabled", True))
        audio = config.get("audio") or {}
        self.f_audio.value = bool(audio.get("enabled", True))
        self.f_audio_plan.value = bool(audio.get("build_in_plan_mode", False))
        temporal = config.get("temporal") or {}
        self.f_temporal.value = bool(temporal.get("enabled", True))
        self.f_temporal_artic.value = bool(temporal.get("use_for_articulated", False))
        self.f_backend_pref.value = ", ".join(
            temporal.get("backend_preference", []) or ["sketch-controlled-video", "deterministic-compositor"]
        )
        sketch = temporal.get("sketch_backend") or {}
        self.f_temporal_argv.value = json.dumps(sketch.get("argv", [])) if sketch.get("argv") else ""
        self.f_temporal_timeout.value = float(sketch.get("timeout_s", 1800.0))
        self.f_allow_shell.value = bool(sketch.get("allow_shell", False))
        self.f_shell_override.value = bool(sketch.get("security_override_unsafe_shell", False))
        render = config.get("render") or {}
        self.f_render_backend.value = str(render.get("backend", "pil"))
        self.f_npm_install.value = bool(render.get("install_dependencies", True))
        self.f_crf.value = int(render.get("crf", 18))
        self.f_render_timeout.value = int(render.get("render_timeout", 2400))
        publishing = config.get("publishing") or {}
        self.f_publish.value = bool(publishing.get("enabled", False))
        self.f_publish_provider.value = str(publishing.get("provider", "local-archive"))
        self.f_archive_dir.value = str(publishing.get("archive_dir", ""))
        self.f_endpoint.value = str(publishing.get("endpoint", ""))
        self.f_allowed_hosts.value = ", ".join(publishing.get("allowed_hosts", []) or [])
        self._render_render_warnings()

    def _form_to_json(self, confirm: bool) -> None:
        config = self._config_from_form()
        if confirm and self.json_editor.value.strip():
            try:
                current = json.loads(self.json_editor.value)
            except json.JSONDecodeError:
                current = None
            if current is not None and current != config:
                with self.out_config:
                    self.out_config.clear_output()
                    print(
                        "JSON editor content differs from the form. Press the button again "
                        "within this session to overwrite it."
                    )
                if self.state.get("pending_overwrite") != True:  # noqa: E712
                    self.state["pending_overwrite"] = True
                    return
        self.state["pending_overwrite"] = False
        self.state["config"] = config
        self.json_editor.value = json.dumps(config, indent=2)
        self._render_cards()

    def _json_to_form(self) -> None:
        with self.out_config:
            self.out_config.clear_output()
            try:
                config = json.loads(self.json_editor.value or "{}")
                self.controller.validate_config(config)
                self._apply_config_to_form(config)
                self.state["config"] = config
                print(f"{STATUS_ICONS['PASS']}  JSON loaded into the form.")
            except Exception as exc:
                self._print_config_error(exc)
        self._render_cards()

    def _print_config_error(self, exc: Exception) -> None:
        print(f"{STATUS_ICONS['FAIL']}  {redacted_exception_text(exc, 300)}")
        for item in getattr(exc, "field_errors", []) or []:
            print(f"   • {item['field']}: {item['message']}")

    def _validate_config_clicked(self) -> None:
        with self.out_config:
            self.out_config.clear_output()
            try:
                config = json.loads(self.json_editor.value or "{}")
                settings = self.controller.validate_config(config)
                print(f"{STATUS_ICONS['PASS']}  Valid ({settings.execution_mode.value} mode).")
                print(json.dumps(redact_secrets(settings.as_runtime_dict()), indent=2)[:2500])
            except Exception as exc:
                self._print_config_error(exc)

    def _on_apply_preset(self, _button) -> None:
        preset = dict(CONFIG_PRESETS[self.preset.value])
        self._apply_config_to_form(preset)
        self.state["config"] = preset
        self.json_editor.value = json.dumps(preset, indent=2)
        with self.out_config:
            self.out_config.clear_output()
            print(f"Preset {self.preset.value!r} applied to form and JSON editor.")
        self._render_cards()

    def _on_export_config(self, _button) -> None:
        with self.out_config:
            self.out_config.clear_output()
            config = redact_secrets(json.loads(self.json_editor.value or "{}"))
            target = self.controller.paths["configs"] / "control_center_config.json"
            target.write_text(json.dumps(config, indent=2), encoding="utf-8")
            print(f"Exported (redacted) config to {target}")

    def _on_import_config(self, _button) -> None:
        with self.out_config:
            self.out_config.clear_output()
            path = Path(self.import_path.value)
            if not path.is_file():
                print(f"{STATUS_ICONS['FAIL']}  File not found: {path}")
                return
            try:
                config = json.loads(path.read_text(encoding="utf-8"))
                self.controller.validate_config(config)
                self.json_editor.value = json.dumps(config, indent=2)
                self._apply_config_to_form(config)
                self.state["config"] = config
                print(f"{STATUS_ICONS['PASS']}  Imported {path}")
            except Exception as exc:
                self._print_config_error(exc)

    def _on_diff_config(self, _button) -> None:
        with self.out_config:
            self.out_config.clear_output()
            base = CONFIG_PRESETS[self.preset.value]
            try:
                current = json.loads(self.json_editor.value or "{}")
            except json.JSONDecodeError as exc:
                print(f"{STATUS_ICONS['FAIL']}  JSON invalid: {exc}")
                return

            def walk(prefix: str, a: Any, b: Any) -> None:
                if isinstance(a, dict) or isinstance(b, dict):
                    keys = sorted(set((a or {}).keys()) | set((b or {}).keys()))
                    for key in keys:
                        walk(f"{prefix}.{key}" if prefix else str(key), (a or {}).get(key), (b or {}).get(key))
                elif a != b:
                    print(f"  {prefix}: preset={a!r} → current={b!r}")

            print(f"Differences vs preset {self.preset.value!r}:")
            walk("", base, current)

    # ------------------------------------------------------------ execution
    def _on_action_change(self, _change) -> None:
        action = self.action.value
        self.action_help.value = (
            f"<div class='{'scs-warn' if action in LIVE_ACTIONS else 'scs-ok'}'>{ACTION_DESCRIPTIONS[action]}</div>"
        )
        self.paid_box.layout.display = "" if action in LIVE_ACTIONS else "none"
        self._update_primary_enabled()

    def _update_primary_enabled(self) -> None:
        action = self.action.value
        if action in LIVE_ACTIONS:
            gate = self.paid_checkbox.value and self.paid_phrase.value.strip() == PAID_CONFIRMATION_PHRASE
            self.primary.disabled = not gate
            self.primary.tooltip = "" if gate else "Tick the paid-run checkbox and type the confirmation phrase."
        else:
            self.primary.disabled = self._run_thread is not None and self._run_thread.is_alive()
            self.primary.tooltip = ""

    def _current_request(self) -> PipelineRunRequest:
        try:
            config = json.loads(self.json_editor.value or "{}") or self._config_from_form()
        except json.JSONDecodeError:
            config = self._config_from_form()
        return PipelineRunRequest(
            action=self.action.value,
            topic=self.topic.value,
            reference_video=self.reference.value,
            config=config,
            job_id=self.job_id.value or None,
            force_scope=self.force_scope.value,
            force_stage=self.force_stage.value,
            confirm_paid_run=self.paid_checkbox.value,
            paid_confirmation_phrase=self.paid_phrase.value,
        )

    def _on_run(self, _button) -> None:
        request = self._current_request()
        with self.out_execution:
            self.out_execution.clear_output()
            if request.action in LIVE_ACTIONS:
                print("Request summary (review before the paid run):")
                print(json.dumps(self.controller.request_summary(request), indent=2))
            print(f"{STATUS_ICONS['RUNNING']}  {request.action.value} …")
        if request.action in self._non_pipeline_actions():
            self._run_sync(request)
        else:
            self._run_async(request)

    @staticmethod
    def _non_pipeline_actions() -> set[PipelineAction]:
        return {
            PipelineAction.environment_check,
            PipelineAction.config_validation,
            PipelineAction.quick_smoke,
            PipelineAction.full_offline_tests,
            PipelineAction.provider_live_smoke,
            PipelineAction.package_artifacts,
        }

    def _run_sync(self, request: PipelineRunRequest) -> None:
        with self.out_execution:
            try:
                result = self.controller.dispatch(request)
                if request.action == PipelineAction.environment_check:
                    self.state["preflight"] = result
                    self._render_preflight(result)
                print(f"{STATUS_ICONS['PASS']}  done")
                print(json.dumps(redact_secrets(result), indent=2, default=str)[:4000])
            except Exception as exc:
                print(f"{STATUS_ICONS['FAIL']}  {redacted_exception_text(exc, 400)}")
                for item in getattr(exc, "field_errors", []) or []:
                    print(f"   • {item['field']}: {item['message']}")
        self._render_cards()

    def _run_async(self, request: PipelineRunRequest) -> None:
        self.monitor = RunMonitorState()
        self.cancellation = CancellationToken()
        self.stop_button.disabled = False
        self.primary.disabled = True

        def worker() -> None:
            try:
                result = self.controller.dispatch(
                    request, progress_callback=self._progress_event, cancellation_token=self.cancellation
                )
                self.state["last_result"] = result
                with self.out_monitor:
                    print(f"{STATUS_ICONS['PASS']}  completed: mode={result.get('mode')}")
                self._render_results()
            except Exception as exc:
                with self.out_monitor:
                    print(f"{STATUS_ICONS['FAIL']}  {type(exc).__name__}: {redacted_exception_text(exc, 400)}")
                    snapshot = self.monitor.snapshot()
                    if snapshot["failed"]:
                        print(f"   failed stage: {snapshot['failed'][-1]}")
                    print("   The job is resumable: choose 'Resume Job' with the same Job ID.")
            finally:
                self.stop_button.disabled = True
                self._update_primary_enabled()
                self._render_cards()

        self._run_thread = threading.Thread(target=worker, daemon=True)
        self._run_thread.start()

    def _progress_event(self, stage_id: str, event: str, record: dict[str, Any]) -> None:
        self.monitor.callback(stage_id, event, record)
        snapshot = self.monitor.snapshot()
        self.progress.value = snapshot["progress"]
        self.monitor_html.value = (
            f"<b>Stage:</b> {snapshot['current_stage'] or '—'} ({event}) | "
            f"<b>elapsed:</b> {snapshot['elapsed_s']}s | "
            f"<b>completed:</b> {len(snapshot['completed'])} | "
            f"<b>cached:</b> {len(snapshot['cached'])} | "
            f"<b>retries:</b> {snapshot['retries']} | "
            f"<b>failed:</b> {len(snapshot['failed'])}"
        )

    def _on_stop(self, _button) -> None:
        if self.cancellation is not None:
            self.cancellation.cancel()
            with self.out_monitor:
                print("Stop requested — the run will halt after the current safe stage.")

    # ------------------------------------------------------------ preflight
    def _run_preflight_clicked(self) -> None:
        request = self._current_request()
        report = self.controller.run_preflight(request)
        self.state["preflight"] = report
        self._render_preflight(report)
        self._render_cards()

    def _render_preflight(self, report: dict[str, Any]) -> None:
        with self.out_preflight:
            self.out_preflight.clear_output()
            summary = report["summary"]
            print(
                f"PASS {summary['PASS']} | WARNING {summary['WARNING']} | "
                f"FAIL {summary['FAIL']} | SKIPPED {summary['SKIPPED']}"
            )
            for check in report["checks"]:
                icon = STATUS_ICONS.get(check["status"], check["status"])
                line = f"{icon:12s} {check['check']}: {check['detail']}"
                if check["recommendation"]:
                    line += f"  → {check['recommendation']}"
                print(line)

    def _on_install_missing(self, _button) -> None:
        import subprocess
        import sys as _sys

        with self.out_preflight:
            report = self.state.get("preflight")
            if not report:
                print("Run preflight first.")
                return
            missing = [
                c["check"].split(":", 1)[1]
                for c in report["checks"]
                if c["check"].startswith(("required_package:", "optional_package:"))
                and c["status"] in ("FAIL", "WARNING")
            ]
            if not missing:
                print("No missing Python packages.")
                return
            print("Installing:", ", ".join(missing))
            subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", *missing], check=False)
            print("Done — rerun preflight to confirm.")

    def _on_download_preflight(self, _button) -> None:
        with self.out_preflight:
            path = self.controller.paths["reports"] / "preflight_report.json"
            print(f"Preflight report path: {path} (exists={path.exists()})")

    # -------------------------------------------------------------- results
    def _render_results(self) -> None:
        with self.out_results:
            self.out_results.clear_output()
            result = self.state.get("last_result")
            job = self.job_id.value or (result or {}).get("run_dir", "")
            if not result and not job:
                print("Empty state: no run yet. Run an action first, or enter a Job ID and refresh.")
                return
            if not result and self.job_id.value:
                manifest = self.controller.job_status(self.job_id.value)
                if manifest is None:
                    print(f"No manifest for job {self.job_id.value!r}.")
                    return
                result = {
                    "run_dir": str(self.controller.paths["jobs"] / self.job_id.value),
                    "mode": manifest.get("status"),
                }
            print(f"mode: {result.get('mode')} | run_dir: {result.get('run_dir')}")
            run_dir = Path(result.get("run_dir", ""))
            manifest = load_json(run_dir / "job_manifest.json")
            if isinstance(manifest, dict):
                stages = manifest.get("stages", {})
                resumed = [s for s, r in stages.items() if r.get("status") == "completed"]
                print(
                    f"manifest status: {manifest.get('status')} | stages: {len(stages)} "
                    f"| completed/cached-eligible: {len(resumed)}"
                )
            for key in (
                "job_manifest",
                "reference_board",
                "drawing_briefs",
                "shot_states",
                "candidate_plans",
                "beauty_frames",
                "semantic_contracts",
                "animation_plans",
                "temporal_results",
                "video",
            ):
                value = result.get(key)
                if value:
                    exists = Path(value).exists()
                    print(f"  {key}: {value} ({'exists' if exists else 'missing'})")
            provenance = load_json(run_dir / "provenance_manifest.json")
            if provenance:
                print("provenance providers:", provenance.get("providers"))
            warnings = result.get("warnings") or []
            for warning in warnings:
                print(f"  ⚠ {warning}")
            video = result.get("video")
            if video and Path(video).exists():
                try:
                    from IPython.display import Video, display

                    display(Video(str(video), embed=False, width=240))
                except Exception:
                    print("(video preview unavailable in this frontend)")

    def _on_zip(self, _button) -> None:
        with self.out_results:
            try:
                job = self.job_id.value or generate_job_id(self.topic.value, self.reference.value)
                archive = self.controller.package_artifacts(job)
                print(f"{STATUS_ICONS['PASS']}  ZIP written: {archive}")
            except Exception as exc:
                print(f"{STATUS_ICONS['FAIL']}  {redacted_exception_text(exc, 300)}")

    def _compare_manifests(self, job_a: str, job_b: str) -> None:
        with self.out_results:
            self.out_results.clear_output()
            manifest_a = self.controller.job_status(job_a)
            manifest_b = self.controller.job_status(job_b)
            if not manifest_a or not manifest_b:
                print("Both job ids must have manifests.")
                return
            stages = sorted(set(manifest_a.get("stages", {})) | set(manifest_b.get("stages", {})))
            print(f"{'stage':32s} {job_a[:14]:14s} {job_b[:14]:14s}")
            for stage in stages:
                status_a = manifest_a.get("stages", {}).get(stage, {}).get("status", "—")
                status_b = manifest_b.get("stages", {}).get(stage, {}).get("status", "—")
                marker = "" if status_a == status_b else "  ← differs"
                print(f"{stage:32s} {status_a:14s} {status_b:14s}{marker}")

    # -------------------------------------------------------------- secrets
    def _render_secret_status(self) -> None:
        with self.out_secrets:
            self.out_secrets.clear_output()
            required = self.controller.required_secrets(self.action.value)
            for name, status in self.controller.secrets.status(required).items():
                icon = {
                    "Configured": STATUS_ICONS["PASS"],
                    "Missing": STATUS_ICONS["FAIL"],
                    "Not required for selected mode": STATUS_ICONS["SKIPPED"],
                }[status]
                print(f"{icon:12s} {name}: {status}")
        self._render_cards()

    def _on_apply_secrets(self, _button) -> None:
        for name, widget in self.secret_inputs.items():
            if widget.value:
                self.controller.secrets.set_manual(name, widget.value)
                widget.value = ""
        self._render_secret_status()

    def _on_clear_secrets(self, _button) -> None:
        self.controller.secrets.clear_manual()
        self._render_secret_status()

    # ------------------------------------------------------------- warnings
    def _render_render_warnings(self) -> None:
        import shutil as _shutil

        warnings = []
        if self.f_allow_shell.value:
            warnings.append(
                "allow_shell=True is an unsafe opt-in; production rejects it without the security override."
            )
        if self.f_publish.value:
            warnings.append("Publishing is enabled — the final video will leave this runtime.")
        if self.f_render_backend.value == "remotion" and not all(_shutil.which(t) for t in ("node", "npm", "npx")):
            warnings.append(
                "Remotion selected but node/npm/npx is missing — install Node.js or switch to the 'pil' backend."
            )
        if self.mode.value == "production" and self.action.value in LIVE_ACTIONS:
            warnings.append("Production live run selected: paid provider calls after confirmation.")
        self.render_warnings.value = (
            "".join(f"<div class='scs-warn'>⚠ {text}</div>" for text in warnings)
            or "<div class='scs-ok'>No configuration warnings.</div>"
        )


In [ ]:
%%writefile scistudio_v10/observability.py
"""Structured observability: JSONL event log + provider-call records.

Every record is redacted before it is written. Prompts are never stored in
events — only prompt hashes; full prompts may be stored by artifact writers
under an explicit debug setting.
"""

from __future__ import annotations

import json
import threading
import time
from contextlib import contextmanager
from datetime import datetime, UTC
from pathlib import Path
from typing import Any, Iterator

from .security import redact_secrets
from .utils import ensure_dir


class EventLogger:
    """Append-only JSONL event log for stage latency, retries and failures.

    Appends are serialized with a process-local lock; the log is intended for
    a single worker process per job directory (enforced by the job lock).
    """

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)
        self.path = self.root / "events.jsonl"
        self._lock = threading.Lock()

    def emit(self, event: str, **payload: Any) -> None:
        record = {
            "timestamp": datetime.now(UTC).isoformat(),
            "event": event,
            **redact_secrets(payload),
        }
        line = json.dumps(record, ensure_ascii=False, default=str) + "\n"
        with self._lock:
            with self.path.open("a", encoding="utf-8") as handle:
                handle.write(line)

    def record_provider_call(
        self,
        *,
        job_id: str = "",
        stage_id: str = "",
        provider: str = "",
        model: str = "",
        request_id: str = "",
        attempt: int = 1,
        input_hash: str = "",
        output_hash: str = "",
        latency_s: float = 0.0,
        cache: str = "miss",
        status: str = "ok",
        error_class: str = "",
        **extra: Any,
    ) -> None:
        """Standardized provider-call observability record."""
        self.emit(
            "provider.call",
            job_id=job_id,
            stage_id=stage_id,
            provider=provider,
            model=model,
            request_id=request_id,
            attempt=attempt,
            input_hash=input_hash,
            output_hash=output_hash,
            latency_s=round(float(latency_s), 4),
            cache=cache,
            status=status,
            error_class=error_class,
            **extra,
        )

    @contextmanager
    def timed(self, event: str, **payload: Any) -> Iterator[None]:
        start = time.perf_counter()
        self.emit(event + ".started", **payload)
        try:
            yield
        except Exception as exc:
            self.emit(
                event + ".failed",
                duration_s=round(time.perf_counter() - start, 4),
                error_type=type(exc).__name__,
                **payload,
            )
            raise
        else:
            self.emit(event + ".completed", duration_s=round(time.perf_counter() - start, 4), **payload)


In [ ]:
%%writefile scistudio_v10/overlay.py
from __future__ import annotations

import html
from pathlib import Path
from typing import Any

from .schemas import SceneRequest
from .style_canon import build_hard_coded_canon
from .utils import ensure_dir


class ScientificOverlayBuilder:
    """Builds only scientific UI/labels as SVG.

    Hero illustration pixels remain raster/authored. SVG is intentionally limited
    to typography, experiment panels, arrows, metrics and machine-readable masks.
    """

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)
        self.canon = build_hard_coded_canon()

    def build(self, scene: SceneRequest, scene_index: int, total_scenes: int) -> str:
        width = int(self.config.get("width", 1080))
        height = int(self.config.get("height", 1920))
        margin = int(width * 0.055)
        ink = self.canon.color.ink
        paper = self.canon.color.paper
        red = self.canon.color.warning_red
        blue = self.canon.color.blue_primary
        experiment = f"EXPERIMENT {scene_index:03d}"
        stage = (scene.time_stage or f"STAGE {scene_index}/{total_scenes}").upper()
        headline = self._wrap(scene.headline.upper(), 23)
        metric, critical = self._metric(scene)
        body = [
            f'<rect x="{margin}" y="{margin}" width="300" height="92" fill="{paper}" stroke="{ink}" stroke-width="3"/>',
            f'<text x="{margin + 18}" y="{margin + 32}" font-family="Arial Narrow, Arial, sans-serif" font-size="24" font-weight="900" fill="{ink}">{html.escape(experiment)}</text>',
            f'<line x1="{margin + 18}" y1="{margin + 44}" x2="{margin + 282}" y2="{margin + 44}" stroke="{ink}" stroke-width="2"/>',
            f'<text x="{margin + 18}" y="{margin + 70}" font-family="Arial Narrow, Arial, sans-serif" font-size="18" font-weight="700" fill="{ink}">SIMULATION STATUS: ACTIVE</text>',
            f'<rect x="{width - margin - 290}" y="{margin}" width="290" height="92" fill="{paper}" stroke="{ink}" stroke-width="3"/>',
            f'<text x="{width - margin - 272}" y="{margin + 30}" font-family="Arial Narrow, Arial, sans-serif" font-size="18" font-weight="700" fill="{ink}">{html.escape(stage)}</text>',
            f'<text x="{width - margin - 272}" y="{margin + 67}" font-family="Arial Narrow, Arial, sans-serif" font-size="27" font-weight="900" fill="{red if critical else ink}">{html.escape(metric)}</text>',
        ]
        y = 245
        for line in headline:
            body.append(
                f'<text x="{margin}" y="{y}" font-family="Arial Narrow, Arial, sans-serif" font-size="68" '
                f'font-weight="900" letter-spacing="-1.5" fill="{ink}">{html.escape(line)}</text>'
            )
            y += 70
        body.extend(
            [
                f'<line x1="{margin}" y1="{y + 12}" x2="{margin + 300}" y2="{y + 12}" stroke="{blue}" stroke-width="10"/>',
                f'<text x="{width - margin}" y="{height - margin}" text-anchor="end" font-family="Arial Narrow, Arial, sans-serif" font-size="18" font-weight="900" fill="{ink}">SCIENTIFIC MOTION STUDIO</text>',
            ]
        )
        svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}">{"".join(body)}</svg>'
        output = self.root / f"{scene.scene_id}_overlay.svg"
        output.write_text(svg, encoding="utf-8")
        return str(output)

    @staticmethod
    def _wrap(text: str, max_chars: int) -> list[str]:
        words = text.split()
        lines: list[str] = []
        current: list[str] = []
        for word in words:
            trial = " ".join([*current, word])
            if current and len(trial) > max_chars:
                lines.append(" ".join(current))
                current = [word]
            else:
                current.append(word)
        if current:
            lines.append(" ".join(current))
        return lines[:3] or ["EXPERIMENT"]

    @staticmethod
    def _metric(scene: SceneRequest) -> tuple[str, bool]:
        text = (scene.desired_change + " " + scene.scientific_claim).lower()
        critical = any(k in text for k in ("critical", "collapse", "failure", "danger", "fatal", "uninhabitable"))
        if critical:
            return "CRITICAL", True
        if scene.time_stage:
            return scene.time_stage.upper()[:18], False
        return "ACTIVE", False


In [ ]:
%%writefile scistudio_v10/pipeline.py
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path
from typing import Any

from .animation_director import AnimationDirector
from .asset_registry import AssetRegistry
from .audio import AudioEngine
from .cache_store import ArtifactCache
from .candidate_tournament import CandidateTournament
from .character_registry import CharacterRegistry
from .director import NarrativeDirector
from .drawing_brief import DrawingBriefCompiler
from .flux_studio import FluxKontextStudio
from .hybrid_package import HybridPackageBuilder
from .hybrid_render import PILHybridRenderer, RemotionHybridExporter
from .job_runtime import ResumableJobRuntime
from .llm import LLMRouter
from .overlay import ScientificOverlayBuilder
from .observability import EventLogger
from .publisher import LocalArchivePublisher, UploadPostPublisher
from .provider_registry import ProviderRegistry
from .reference_director import ReferenceDirector
from .reference_selector import DynamicReferenceSelector
from .research import PublicResearchSearch, ResearchEngine
from .scene_architect import SceneIllustrationArchitect
from .schemas import (
    ArtDirectionBible,
    AssetQuery,
    AssetRecord,
    CanonAnchor,
    CharacterProfile,
    CharacterView,
    ContinuityCanon,
    DrawingBrief,
    PipelineResult,
    ResearchPack,
    ScriptPackage,
    ShotState,
    Storyboard,
    TemporalRequest,
)
from .semantic import SemanticMaskExtractor, SemanticSeparationPlanner
from .shot_state import ShotStatePlanner
from .sketch_control import ControlSketchBuilder
from .studio_director import ExecutiveArtDirector
from .style_reference import StyleReferenceExtractor
from .temporal_backends import (
    DeterministicCompositorBackend,
    SketchControlledVideoBackend,
    TemporalBackendRouter,
)
from .utils import ensure_dir, hash_value, save_json, slugify


class ScientificMotionStudioV10:
    """Production-oriented agentic scientific animation studio.

    V10 combines:
    - a persistent, resumable job runtime and provider shell;
    - retrieval-based visual continuity and a searchable asset registry;
    - multi-candidate FLUX tournaments before director-led revisions;
    - explicit first/last shot states and character/environment canon;
    - hybrid authored beauty art, scientific overlays and optional
      sketch-controlled temporal video backends.

    Hero art is never replaced with procedural SVG or clip-art. When a required
    high-complexity temporal backend is unavailable, the pipeline stops rather
    than degrading the scene into PowerPoint-like motion.
    """

    def __init__(
        self,
        config: dict[str, Any] | Any,
        secrets: dict[str, Any] | None = None,
        *,
        llm_router: Any | None = None,
        image_generator: Any | None = None,
        mask_generator: Any | None = None,
        temporal_backends: list[Any] | None = None,
    ):
        from .config_models import ExecutionMode, StudioConfig
        from .errors import ProviderLockViolationError

        raw_config = config if isinstance(config, dict) else None
        self.settings = StudioConfig.from_dict(config)
        self.execution_mode = self.settings.execution_mode.value
        # Preserve the user's raw dict shape (modules read nested dicts), but
        # guarantee the validated execution mode is propagated everywhere.
        self.config = dict(raw_config) if raw_config is not None else self.settings.as_runtime_dict()
        self.config["execution_mode"] = self.execution_mode
        llm_config = dict(self.config.get("llm") or {})
        llm_config["execution_mode"] = self.execution_mode
        self.config["llm"] = llm_config

        if self.settings.execution_mode == ExecutionMode.production and any(
            x is not None for x in (llm_router, image_generator, mask_generator, temporal_backends)
        ):
            raise ProviderLockViolationError(
                "Injected provider implementations (llm_router/image_generator/"
                "mask_generator/temporal_backends) are not allowed in production "
                "mode; deterministic or fake providers are test/development only."
            )

        self.secrets = secrets or {}
        self.workspace = ensure_dir(self.config.get("workspace", "./scientific_motion_v10"))
        self.cache = ArtifactCache(self.workspace / "cache" / "artifacts")
        self.llm = llm_router or LLMRouter(llm_config, self.secrets, self.workspace / "cache" / "llm")
        self.image_generator = image_generator
        self.mask_generator = mask_generator
        self.temporal_backends = temporal_backends
        self.providers = ProviderRegistry.from_config(self.config, self.workspace / "state" / "providers")

    def run(
        self,
        topic: str,
        reference_video: str | Path,
        *,
        plan_only: bool = True,
        render_video: bool = False,
        force: bool = False,
        job_id: str | None = None,
        progress_callback: Any | None = None,
        cancellation_token: Any | None = None,
    ) -> dict[str, Any]:
        """Execute the pipeline; the job lock is always released on exit.

        ``progress_callback(stage_id, event, record)`` (optional) receives
        "cached"/"started"/"completed"/"failed" stage events and must not
        raise. ``cancellation_token`` (optional) is any object with a truthy
        ``cancelled`` attribute; setting it stops the run after the current
        safe stage with :class:`JobCancelledError` — the job stays resumable.
        Both parameters are optional, preserving the original signature.
        """
        reference_video = str(Path(reference_video))
        job_id = (
            job_id
            or self.config.get("job_id")
            or (slugify(topic, 40) + "-" + hash_value({"topic": topic, "reference": reference_video}, 10))
        )
        run_dir = ensure_dir(self.workspace / "jobs" / job_id)
        limits = self.config.get("limits") or {}
        runtime = ResumableJobRuntime(
            run_dir,
            job_id=job_id,
            topic=topic,
            config=self.config,
            max_stage_attempts=int(limits.get("max_stage_attempts", 5)),
            debug_tracebacks=bool(self.config.get("debug_tracebacks", False)),
            on_stage_event=progress_callback,
            cancellation_token=cancellation_token,
        )
        try:
            return self._run_stages(
                topic,
                reference_video,
                run_dir,
                runtime,
                job_id,
                plan_only=plan_only,
                render_video=render_video,
                force=force,
            )
        finally:
            runtime.release_lock()

    def _run_stages(
        self,
        topic: str,
        reference_video: str,
        run_dir: Path,
        runtime: ResumableJobRuntime,
        job_id: str,
        *,
        plan_only: bool,
        render_video: bool,
        force: bool,
    ) -> dict[str, Any]:
        events = EventLogger(run_dir / "observability")
        self.llm.events = events
        events.emit(
            "job.started",
            job_id=job_id,
            topic=topic,
            plan_only=plan_only,
            render_video=render_video,
            execution_mode=self.execution_mode,
        )
        warnings: list[str] = []

        # ------------------------------------------------------------------
        # Foundation stages: resumable and input-hash aware.
        # ------------------------------------------------------------------
        style_ref = StyleReferenceExtractor(run_dir / "01_style_reference", self.config.get("style_reference", {}))
        reference_profile = runtime.execute(
            "01_style_reference",
            {"reference_video": reference_video, "config": self.config.get("style_reference", {})},
            lambda: style_ref.extract(reference_video, force=force),
            force=force,
        )
        reference_board = reference_profile["board_path"]

        search = PublicResearchSearch(self.config.get("research_search", {}), run_dir / "02_research" / "search")
        sources = (
            search.search(topic, force=force) if self.config.get("research_search", {}).get("enabled", True) else []
        )
        research = ResearchPack.model_validate(
            runtime.execute(
                "02_research",
                {"topic": topic, "sources": sources},
                lambda: ResearchEngine(self.llm, run_dir / "02_research").build(topic, sources, force=force),
                force=force,
            )
        )
        narrative = NarrativeDirector(self.llm, self.config.get("narrative", {}), run_dir / "03_narrative")
        script = ScriptPackage.model_validate(
            runtime.execute(
                "03_script",
                research.model_dump(mode="json"),
                lambda: narrative.script(research, force=force),
                force=force,
            )
        )
        storyboard = Storyboard.model_validate(
            runtime.execute(
                "04_storyboard",
                {"script": script.model_dump(mode="json"), "research_hash": research.research_hash},
                lambda: narrative.storyboard(script, research, force=force),
                force=force,
            )
        )

        audio_manifest: dict[str, Any] = {
            "voice": {},
            "word_timing": {},
            "support_bed": "",
            "duration_s": storyboard.estimated_duration_s,
        }
        audio_enabled = bool(self.config.get("audio", {}).get("enabled", True))
        if audio_enabled and (not plan_only or self.config.get("audio", {}).get("build_in_plan_mode", False)):
            audio = AudioEngine(self.config.get("audio", {}), run_dir / "05_audio")
            script, storyboard, audio_manifest = audio.build(script, storyboard)

        executive = ExecutiveArtDirector(self.llm, self.config.get("art_director", {}), run_dir / "06_art_direction")
        bible = ArtDirectionBible.model_validate(
            runtime.execute(
                "06_art_bible",
                {
                    "topic": topic,
                    "script": script.script_hash,
                    "storyboard": storyboard.storyboard_hash,
                    "reference": reference_board,
                },
                lambda: executive.art_bible(topic, research, script, storyboard, reference_board, force=force),
                force=force,
            )
        )
        continuity = ContinuityCanon.model_validate(
            runtime.execute(
                "07_continuity_canon",
                {"bible": bible.model_dump(mode="json"), "storyboard": storyboard.model_dump(mode="json")},
                lambda: executive.continuity_canon(bible, storyboard, reference_board, force=force),
                force=force,
            )
        )
        architect = SceneIllustrationArchitect(
            self.llm, self.config.get("scene_architect", {}), run_dir / "08_scene_architecture"
        )
        architectures_raw = runtime.execute(
            "08_scene_architectures",
            {"storyboard": storyboard.model_dump(mode="json"), "bible": bible.model_dump(mode="json")},
            lambda: architect.plan_all(storyboard, bible, continuity, force=force),
            force=force,
        )
        from .schemas import SceneIllustrationArchitecture

        architectures = [SceneIllustrationArchitecture.model_validate(x) for x in architectures_raw]

        shot_planner = ShotStatePlanner(self.llm, self.config.get("shot_state", {}), run_dir / "09_shot_states")
        shot_states: list[ShotState] = []
        for scene, architecture in zip(storyboard.scenes, architectures):
            state = ShotState.model_validate(
                runtime.execute(
                    f"09_shot_state_{scene.scene_id}",
                    {"scene": scene.model_dump(mode="json"), "architecture": architecture.model_dump(mode="json")},
                    lambda s=scene, a=architecture: shot_planner.plan(s, a, force=force),
                    force=force,
                )
            )
            shot_states.append(state)
        save_json(run_dir / "09_shot_states" / "shot_states.json", shot_states)

        # ------------------------------------------------------------------
        # Searchable continuity databases.
        # ------------------------------------------------------------------
        registry = AssetRegistry(run_dir / "state" / "asset_registry.sqlite")
        characters = CharacterRegistry(run_dir / "state" / "character_registry")
        style_hash = bible.locked_canon.style_hash or hash_value(bible.locked_canon.model_dump(mode="json"), 20)
        registry.upsert(
            AssetRecord(
                asset_id="reference-video-board",
                path=reference_board,
                asset_type="style_anchor",
                role_tags=["style", "reference_video"],
                style_hash=style_hash,
                quality_score=1.0,
            )
        )
        for recurring in continuity.recurring_subjects:
            if not characters.get(recurring.subject_id):
                characters.upsert(
                    CharacterProfile(
                        subject_id=recurring.subject_id,
                        display_name=recurring.description or recurring.subject_id,
                        immutable_traits=recurring.immutable_traits,
                        style_hash=style_hash,
                    )
                )

        selector = DynamicReferenceSelector(registry, run_dir / "10_reference_retrieval")
        refs = ReferenceDirector(self.llm, self.config.get("references", {}), run_dir / "10_reference_plans")
        brief_compiler = DrawingBriefCompiler(self.config.get("drawing", {}), run_dir / "11_drawing_briefs")
        master_brief = brief_compiler.master_anchor(topic, bible, reference_board)
        reference_packs = []
        beauty_briefs: list[DrawingBrief] = []
        selections = []
        candidate_plans = []

        if plan_only:
            for index, (scene, architecture, state) in enumerate(zip(storyboard.scenes, architectures, shot_states)):
                query = self._asset_query(scene.scene_id, index, architecture, style_hash)
                selection = selector.select(query)
                selections.append(selection)
                pack = refs.plan(architecture, bible, continuity, previous_approved_scene="", force=force)
                selected_paths = [x.asset.path for x in selection.selected]
                pack.subject_anchor_paths = selected_paths
                pack.environment_anchor_paths = []
                pack.board_path = selection.board_path or reference_board
                brief = brief_compiler.beauty_frame(
                    architecture,
                    bible,
                    continuity,
                    pack,
                    scene.narration,
                    scene.headline,
                    index,
                    style_source_path=reference_board,
                    init_strategy="reference_board",
                )
                brief.request_metadata.update(
                    {
                        "shot_state": state.model_dump(mode="json"),
                        "reference_asset_ids": [x.asset.asset_id for x in selection.selected],
                        "candidate_count": int(self.config.get("candidate_tournament", {}).get("candidate_count", 4)),
                    }
                )
                reference_packs.append(pack)
                beauty_briefs.append(brief)
                candidate_plans.append(
                    {
                        "scene_id": scene.scene_id,
                        "candidate_seeds": [
                            (brief.seed or 0) + i * 9973
                            for i in range(
                                max(2, int(self.config.get("candidate_tournament", {}).get("candidate_count", 4)))
                            )
                        ],
                        "ranking_dimensions": [
                            "style_consistency",
                            "subject_consistency",
                            "composition_fitness",
                            "motion_readiness",
                            "causal_clarity",
                        ],
                        "shot_state": state.model_dump(mode="json"),
                    }
                )
            save_json(run_dir / "10_reference_retrieval" / "selections.json", selections)
            save_json(run_dir / "10_reference_plans" / "reference_packs.json", reference_packs)
            save_json(run_dir / "11_drawing_briefs" / "beauty_briefs.json", beauty_briefs)
            save_json(run_dir / "12_candidate_tournaments" / "candidate_plans.json", candidate_plans)
            result = PipelineResult(
                topic=topic,
                mode="plan_only",
                run_dir=str(run_dir),
                reference_board=reference_board,
                art_direction_bible=str(run_dir / "06_art_direction" / "art_direction_bible.json"),
                continuity_canon=str(run_dir / "06_art_direction" / "continuity_canon.json"),
                architectures=str(run_dir / "08_scene_architecture" / "scene_architectures.json"),
                drawing_briefs=str(run_dir / "11_drawing_briefs" / "beauty_briefs.json"),
                warnings=warnings,
                job_manifest=str(runtime.manifest_path),
                asset_registry=str(registry.path),
                character_registry=str(characters.path),
                shot_states=str(run_dir / "09_shot_states" / "shot_states.json"),
                reference_selections=str(run_dir / "10_reference_retrieval" / "selections.json"),
                candidate_plans=str(run_dir / "12_candidate_tournaments" / "candidate_plans.json"),
                provider_registry=self.providers.snapshot(),
            )
            self._write_provenance(
                run_dir,
                runtime,
                job_id,
                topic,
                {
                    "reference_board": reference_board,
                    "script": str(run_dir / "stage_outputs" / "03_script.json"),
                    "storyboard": str(run_dir / "stage_outputs" / "04_storyboard.json"),
                    "art_direction_bible": result.art_direction_bible,
                    "drawing_briefs": result.drawing_briefs,
                    "shot_states": result.shot_states,
                    "candidate_plans": result.candidate_plans,
                },
                warnings,
            )
            save_json(run_dir / "result.json", result)
            runtime.mark_completed()
            registry.close()
            events.emit("job.completed", job_id=job_id, mode="plan_only")
            return result.model_dump(mode="json")

        # ------------------------------------------------------------------
        # Live studio: tournament -> revision -> semantic separation -> motion.
        # ------------------------------------------------------------------
        flux = FluxKontextStudio(
            self.llm,
            brief_compiler,
            self.config.get("flux_studio", {}),
            run_dir / "13_flux_studio",
            image_generator=self.image_generator,
        )
        master_anchor = flux.create_master_anchor(master_brief, force=force)
        continuity.anchors.append(
            CanonAnchor(
                anchor_id="master-style-anchor",
                role="master_style_anchor",
                path=master_anchor,
                notes="Primary generated visual canon for all scene generations.",
            )
        )
        registry.upsert(
            AssetRecord(
                asset_id="master-style-anchor",
                path=master_anchor,
                asset_type="style_anchor",
                role_tags=["style", "master"],
                style_hash=style_hash,
                quality_score=1.0,
            )
        )
        save_json(run_dir / "06_art_direction" / "continuity_canon_live.json", continuity)

        semantic_planner = SemanticSeparationPlanner(self.config.get("semantic", {}), run_dir / "14_semantic")
        mask_extractor = SemanticMaskExtractor(
            self.llm,
            self.config.get("semantic", {}),
            run_dir / "14_semantic" / "masks",
            mask_generator=self.mask_generator,
        )
        overlay_builder = ScientificOverlayBuilder(
            {"width": storyboard.width, "height": storyboard.height, **self.config.get("overlay", {})},
            run_dir / "15_overlays",
        )
        animation_director = AnimationDirector(self.llm, self.config.get("animation", {}), run_dir / "16_animation")
        package_builder = HybridPackageBuilder(
            {"width": storyboard.width, "height": storyboard.height, **self.config.get("hybrid", {})},
            run_dir / "18_hybrid_packages",
        )
        sketch_builder = ControlSketchBuilder(run_dir / "17_temporal" / "control_sketches")
        temporal_backend_list = self.temporal_backends or [
            SketchControlledVideoBackend(
                self.config.get("temporal", {}).get("sketch_backend", {}),
                run_dir / "17_temporal" / "sketch_backend",
                execution_mode=self.execution_mode,
            ),
            DeterministicCompositorBackend(run_dir / "17_temporal" / "deterministic"),
        ]
        temporal_router = TemporalBackendRouter(temporal_backend_list, run_dir / "17_temporal" / "results")
        tournament = CandidateTournament(
            self.llm, self.config.get("candidate_tournament", {}), run_dir / "12_candidate_tournaments"
        )

        beauty_frames = []
        semantic_contracts = []
        animation_plans = []
        hybrid_scenes = []
        temporal_results = []
        previous_approved = ""
        for index, (scene, architecture, state) in enumerate(zip(storyboard.scenes, architectures, shot_states)):
            query = self._asset_query(scene.scene_id, index, architecture, style_hash)
            selection = selector.select(query)
            selections.append(selection)
            pack = refs.plan(architecture, bible, continuity, previous_approved_scene=previous_approved, force=force)
            pack.style_anchor_ids = ["reference-video-board", "master-style-anchor", *pack.style_anchor_ids]
            selected_paths = [x.asset.path for x in selection.selected]
            pack.subject_anchor_paths = selected_paths
            pack.board_path = selection.board_path or refs.build_board(pack, continuity)
            style_source = previous_approved or master_anchor
            brief = brief_compiler.beauty_frame(
                architecture,
                bible,
                continuity,
                pack,
                scene.narration,
                scene.headline,
                index,
                style_source_path=style_source,
                init_strategy="previous_approved_scene" if previous_approved else "master_style_anchor",
            )
            brief.request_metadata.update(
                {
                    "shot_state": state.model_dump(mode="json"),
                    "reference_asset_ids": [x.asset.asset_id for x in selection.selected],
                }
            )
            reference_packs.append(pack)
            beauty_briefs.append(brief)

            tour = tournament.run(brief, architecture, state, lambda b: flux.generate(b, force=force), force=force)
            beauty = flux.direct_scene(
                brief, architecture, bible, continuity, initial_path=tour.winner_path, force=force
            )
            if not beauty.approved:
                raise RuntimeError(f"Scene {scene.scene_id} was not explicitly approved by the art director")
            beauty_frames.append(beauty)
            previous_approved = beauty.image_path
            continuity = refs.add_approved_anchor(continuity, scene.scene_id, beauty.image_path)
            registry.upsert(
                AssetRecord(
                    asset_id=f"approved-{scene.scene_id}",
                    path=beauty.image_path,
                    asset_type="approved_scene",
                    scene_id=scene.scene_id,
                    subject_ids=[f.figure_id for f in architecture.figure_construction],
                    environment_id=self._environment_id(architecture),
                    camera_view=architecture.perspective.view,
                    perspective=architecture.perspective.lens_language,
                    chronology_index=index,
                    role_tags=["approved", "continuity"],
                    style_hash=style_hash,
                    quality_score=1.0,
                )
            )
            for figure in architecture.figure_construction:
                characters.add_view(
                    figure.figure_id,
                    CharacterView(
                        view_id=f"{scene.scene_id}-{figure.body_orientation}",
                        angle=self._angle(figure.body_orientation),
                        path=beauty.image_path,
                        notes=f"Approved whole-scene view; crop/isolation can be derived later for {figure.figure_id}.",
                    ),
                )
                registry.upsert(
                    AssetRecord(
                        asset_id=f"{figure.figure_id}-{scene.scene_id}",
                        path=beauty.image_path,
                        asset_type="subject_view",
                        scene_id=scene.scene_id,
                        subject_ids=[figure.figure_id],
                        camera_view=architecture.perspective.view,
                        perspective=architecture.perspective.lens_language,
                        chronology_index=index,
                        role_tags=["character", "approved"],
                        style_hash=style_hash,
                        quality_score=0.95,
                    )
                )
            save_json(run_dir / "06_art_direction" / "continuity_canon_live.json", continuity)

            pose_variants = flux.create_pose_variants(beauty, brief, architecture, force=force)
            contract = semantic_planner.plan(architecture, beauty, pose_variants)
            contract = mask_extractor.extract_all(contract, architecture, force=force)
            semantic_contracts.append(contract)
            overlay_path = overlay_builder.build(scene, index + 1, len(storyboard.scenes))
            timing = {"words": audio_manifest.get("word_timing", {}).get(scene.beat_id, [])}
            animation = animation_director.plan(
                scene, architecture, contract, fps=storyboard.fps, audio_timing=timing, force=force
            )
            animation_plans.append(animation)

            temporal_clip = ""
            temporal_backend = ""
            use_temporal = self._should_use_temporal(state)
            if use_temporal:
                end_path = next(iter(pose_variants.values()), beauty.image_path)
                first_sketch, last_sketch = sketch_builder.build_pair(
                    scene.scene_id, beauty.image_path, end_path, state
                )
                request = TemporalRequest(
                    scene_id=scene.scene_id,
                    backend_preference=self.config.get("temporal", {}).get(
                        "backend_preference", ["sketch-controlled-video", "deterministic-compositor"]
                    ),
                    beauty_start=beauty.image_path,
                    beauty_end=end_path,
                    control_sketch_start=first_sketch.output_path,
                    control_sketch_end=last_sketch.output_path,
                    preserve_masks=[x.mask_path for x in contract.layers if x.locked and x.mask_path],
                    motion_prompt="; ".join(state.motion_bridge),
                    duration_frames=animation.duration_frames,
                    fps=animation.fps,
                    complexity=state.temporal_complexity,
                    output_path=str(run_dir / "17_temporal" / "clips" / f"{scene.scene_id}.mp4"),
                )
                result = temporal_router.generate(request)
                temporal_results.append(result)
                temporal_clip = result.output_path
                temporal_backend = result.backend_id

            voice_path = audio_manifest.get("voice", {}).get(scene.beat_id, "")
            hybrid_scenes.append(
                package_builder.build(
                    scene,
                    contract,
                    animation,
                    overlay_path,
                    voice_path,
                    temporal_clip_path=temporal_clip,
                    temporal_backend=temporal_backend,
                )
            )

        save_json(run_dir / "10_reference_retrieval" / "selections.json", selections)
        save_json(run_dir / "10_reference_plans" / "reference_packs.json", reference_packs)
        save_json(run_dir / "11_drawing_briefs" / "beauty_briefs.json", beauty_briefs)
        save_json(run_dir / "13_flux_studio" / "beauty_frames.json", beauty_frames)
        save_json(run_dir / "14_semantic" / "semantic_contracts.json", semantic_contracts)
        save_json(run_dir / "16_animation" / "animation_plans.json", animation_plans)
        save_json(run_dir / "17_temporal" / "temporal_results.json", temporal_results)

        remotion = RemotionHybridExporter(self.config.get("render", {}), run_dir / "19_remotion")
        remotion.create_project(hybrid_scenes, audio_manifest.get("support_bed", ""))
        video_path = ""
        mode = "live_generation"
        if render_video:
            output = run_dir / "scientific_motion_v10.mp4"
            backend = str(self.config.get("render", {}).get("backend", "remotion")).lower()
            if backend in {"pil", "deterministic", "preview"}:
                PILHybridRenderer(self.config.get("render", {}), run_dir / "19_preview_render").render(
                    hybrid_scenes, output
                )
            else:
                self._render_remotion(run_dir / "19_remotion", output)
            video_path = str(output)
            mode = "rendered"

        publish_manifest = {}
        publish_config = self.config.get("publishing", {})
        if video_path and publish_config.get("enabled", False):
            provider = str(publish_config.get("provider", "local-archive"))
            metadata = {"title": script.title or topic, "topic": topic, "job_id": job_id}
            if provider == "local-archive":
                publisher = LocalArchivePublisher(publish_config.get("archive_dir", run_dir / "published"))
            elif provider == "upload-post":
                publisher = UploadPostPublisher(
                    str(publish_config.get("endpoint", "")),
                    str(self.secrets.get("UPLOAD_POST_TOKEN", publish_config.get("token", ""))),
                    allowed_hosts=list(publish_config.get("allowed_hosts") or []),
                )
            else:
                raise RuntimeError(f"Unknown publishing provider: {provider}")
            publish_manifest = publisher.publish(video_path, metadata)
            save_json(run_dir / "publishing_manifest.json", publish_manifest)

        result = PipelineResult(
            topic=topic,
            mode=mode,
            run_dir=str(run_dir),
            reference_board=reference_board,
            art_direction_bible=str(run_dir / "06_art_direction" / "art_direction_bible.json"),
            continuity_canon=str(run_dir / "06_art_direction" / "continuity_canon_live.json"),
            architectures=str(run_dir / "08_scene_architecture" / "scene_architectures.json"),
            drawing_briefs=str(run_dir / "11_drawing_briefs" / "beauty_briefs.json"),
            beauty_frames=str(run_dir / "13_flux_studio" / "beauty_frames.json"),
            semantic_contracts=str(run_dir / "14_semantic" / "semantic_contracts.json"),
            animation_plans=str(run_dir / "16_animation" / "animation_plans.json"),
            video=video_path,
            warnings=warnings,
            job_manifest=str(runtime.manifest_path),
            asset_registry=str(registry.path),
            character_registry=str(characters.path),
            shot_states=str(run_dir / "09_shot_states" / "shot_states.json"),
            reference_selections=str(run_dir / "10_reference_retrieval" / "selections.json"),
            candidate_tournaments=str(run_dir / "12_candidate_tournaments"),
            temporal_results=str(run_dir / "17_temporal" / "temporal_results.json"),
            provider_registry=self.providers.snapshot(),
            publishing_manifest=publish_manifest,
        )
        self._write_provenance(
            run_dir,
            runtime,
            job_id,
            topic,
            {
                "reference_board": reference_board,
                "script": str(run_dir / "stage_outputs" / "03_script.json"),
                "storyboard": str(run_dir / "stage_outputs" / "04_storyboard.json"),
                "art_direction_bible": result.art_direction_bible,
                "continuity_canon": result.continuity_canon,
                "drawing_briefs": result.drawing_briefs,
                "beauty_frames": result.beauty_frames,
                "semantic_contracts": result.semantic_contracts,
                "animation_plans": result.animation_plans,
                "temporal_results": result.temporal_results,
                "video": video_path,
            },
            warnings,
        )
        save_json(run_dir / "result.json", result)
        runtime.mark_completed()
        registry.close()
        events.emit("job.completed", job_id=job_id, mode=mode, video=video_path)
        return result.model_dump(mode="json")

    def _write_provenance(
        self,
        run_dir: Path,
        runtime: ResumableJobRuntime,
        job_id: str,
        topic: str,
        artifacts: dict[str, Any],
        warnings: list[str],
    ) -> None:
        from .provenance import build_provenance_manifest, write_provenance_manifest

        manifest = build_provenance_manifest(
            job_id=job_id,
            topic=topic,
            execution_mode=self.execution_mode,
            config_hash=runtime.manifest.config_hash,
            artifacts=artifacts,
            providers={
                "reasoning": "openai-gpt",
                "vision_review": "openai-gpt",
                "image_generation": "bfl-flux-kontext",
                "execution_mode": self.execution_mode,
            },
            warnings=warnings,
        )
        write_provenance_manifest(run_dir, manifest)

    def _asset_query(self, scene_id: str, index: int, architecture: Any, style_hash: str) -> AssetQuery:
        return AssetQuery(
            scene_id=scene_id,
            subject_ids=[x.figure_id for x in architecture.figure_construction],
            environment_id=self._environment_id(architecture),
            camera_view=architecture.perspective.view,
            perspective=architecture.perspective.lens_language,
            desired_types=["style_anchor", "approved_scene", "subject_view", "environment", "material"],
            required_roles=["style", "continuity", "approved"],
            style_hash=style_hash,
            chronology_index=index,
            maximum_results=int(self.config.get("references", {}).get("maximum_retrieved_assets", 8)),
        )

    @staticmethod
    def _environment_id(architecture: Any) -> str:
        contents = " ".join(p.contents for p in architecture.depth_planes if p.depth in {"background", "midground"})
        return slugify(contents or architecture.visual_thesis, 40)

    @staticmethod
    def _angle(body_orientation: str) -> str:
        text = body_orientation.lower()
        if "rear" in text or "back" in text:
            return "rear"
        if "side" in text or "profile" in text:
            return "side"
        if "three" in text:
            return "three_quarter"
        return "front"

    def _should_use_temporal(self, state: ShotState) -> bool:
        config = self.config.get("temporal", {})
        if not config.get("enabled", True):
            return False
        if state.temporal_complexity in {"organic", "deformation"}:
            return True
        if state.temporal_complexity == "articulated":
            return bool(config.get("use_for_articulated", False))
        return False

    def _render_remotion(self, project: Path, output: Path) -> None:
        if not shutil.which("npm") or not shutil.which("npx"):
            raise RuntimeError("npm/npx are required for production Remotion rendering")
        render_config = self.config.get("render", {})
        timeout = float(render_config.get("render_timeout", 2400))
        if render_config.get("install_dependencies", True):
            subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=project, check=True, timeout=timeout)
        subprocess.run(
            [
                "npx",
                "remotion",
                "render",
                "src/index.tsx",
                "ScientificMotionV10",
                str(output),
                "--codec",
                "h264",
                "--pixel-format",
                "yuv420p",
            ],
            cwd=project,
            check=True,
            timeout=timeout,
        )


In [ ]:
%%writefile scistudio_v10/preflight.py
"""Structured preflight checker.

Every check yields PASS / WARNING / FAIL / SKIPPED with a detail line and an
actionable recommendation. The report is JSON-serializable, redacted, and can
be persisted as ``preflight_report.json``. Live-only checks (network, API
keys) are SKIPPED for offline actions instead of failing them.
"""

from __future__ import annotations

import importlib
import os
import platform
import shutil
import sys
from pathlib import Path
from typing import Any, Callable

from .config_models import StudioConfig
from .security import redact_secrets
from .utils import load_json, save_json

PASS = "PASS"
WARNING = "WARNING"
FAIL = "FAIL"
SKIPPED = "SKIPPED"

VIDEO_SUFFIXES = (".mp4", ".mov", ".mkv", ".webm")
_MIN_FREE_DISK_BYTES = 2 * 1024**3
_REQUIRED_PACKAGES = ("pydantic", "PIL", "requests")
_OPTIONAL_PACKAGES = ("openai", "fastapi", "ipywidgets", "cairosvg")


def _check(name: str, status: str, detail: str = "", recommendation: str = "") -> dict[str, str]:
    return {"check": name, "status": status, "detail": detail, "recommendation": recommendation}


def run_preflight(
    *,
    config: dict[str, Any] | StudioConfig | None = None,
    reference_video: str | Path | None = None,
    live: bool = False,
    render_backend: str | None = None,
    required_secrets: dict[str, bool] | None = None,
    secret_status: Callable[[str], bool] | None = None,
    workspace: str | Path | None = None,
    job_id: str | None = None,
    which: Callable[[str], str | None] = shutil.which,
) -> dict[str, Any]:
    """Run all preflight checks and return a structured report.

    ``required_secrets`` maps secret names to whether the selected action
    needs them; ``secret_status(name)`` reports availability without exposing
    values. ``which`` is injectable for tests.
    """
    checks: list[dict[str, str]] = []

    # --- runtime ---------------------------------------------------------
    py = sys.version_info
    checks.append(
        _check(
            "python_version",
            PASS if py >= (3, 11) else FAIL,
            f"Python {py.major}.{py.minor}.{py.micro}",
            "" if py >= (3, 11) else "Use a Python 3.11+ runtime.",
        )
    )
    checks.append(_check("operating_system", PASS, platform.platform()))

    cpu = os.cpu_count() or 0
    checks.append(
        _check(
            "cpu",
            PASS if cpu >= 2 else WARNING,
            f"{cpu} logical CPUs",
            "" if cpu >= 2 else "Pipelines are slow below 2 CPUs.",
        )
    )
    try:
        total_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
        ram_gb = total_ram / 1024**3
        checks.append(
            _check(
                "ram",
                PASS if ram_gb >= 4 else WARNING,
                f"{ram_gb:.1f} GiB",
                "" if ram_gb >= 4 else "4+ GiB RAM recommended.",
            )
        )
    except (ValueError, OSError, AttributeError):
        checks.append(_check("ram", SKIPPED, "unavailable on this platform"))
    gpu = which("nvidia-smi")
    checks.append(
        _check(
            "gpu", PASS if gpu else SKIPPED, "NVIDIA GPU tooling present" if gpu else "no GPU detected (not required)"
        )
    )

    # --- workspace -------------------------------------------------------
    settings: StudioConfig | None = None
    config_error = ""
    if config is not None:
        try:
            settings = StudioConfig.from_dict(config)
        except Exception as exc:
            config_error = str(redact_secrets(str(exc)))[:600]
    ws = Path(workspace or (settings.workspace if settings else "./scientific_motion_studio_v10"))
    try:
        ws.mkdir(parents=True, exist_ok=True)
        probe = ws / ".preflight_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        checks.append(_check("workspace_writable", PASS, str(ws)))
    except OSError as exc:
        checks.append(_check("workspace_writable", FAIL, f"{ws}: {exc}", "Choose a writable workspace directory."))
    try:
        usage = shutil.disk_usage(str(ws if ws.exists() else Path.cwd()))
        free_gb = usage.free / 1024**3
        status = PASS if usage.free >= _MIN_FREE_DISK_BYTES else WARNING
        checks.append(
            _check(
                "free_disk",
                status,
                f"{free_gb:.1f} GiB free",
                "" if status == PASS else "Less than 2 GiB free; renders may fail.",
            )
        )
    except OSError:
        checks.append(_check("free_disk", SKIPPED, "disk usage unavailable"))

    # --- package & configuration -----------------------------------------
    try:
        importlib.import_module("scistudio_v10")
        checks.append(_check("package_import", PASS, "scistudio_v10 importable"))
    except Exception as exc:  # pragma: no cover - import of self normally succeeds
        checks.append(_check("package_import", FAIL, type(exc).__name__, "Re-run the package generation cells."))
    if config is None:
        checks.append(_check("config_validation", SKIPPED, "no configuration supplied"))
        checks.append(_check("provider_lock", SKIPPED, "no configuration supplied"))
    elif settings is None:
        checks.append(_check("config_validation", FAIL, config_error, "Fix the configuration fields listed above."))
        checks.append(
            _check(
                "provider_lock",
                FAIL,
                "configuration invalid",
                "Provider lock can only be evaluated on a valid configuration.",
            )
        )
    else:
        checks.append(_check("config_validation", PASS, f"mode={settings.execution_mode.value}"))
        if settings.execution_mode.value == "production":
            checks.append(_check("provider_lock", PASS, "OpenAI-only reasoning/vision, BFL-only images enforced"))
        else:
            checks.append(
                _check("provider_lock", PASS, f"{settings.execution_mode.value} mode (locks apply in production)")
            )

    for module_name in _REQUIRED_PACKAGES:
        try:
            importlib.import_module(module_name)
            checks.append(_check(f"required_package:{module_name}", PASS, "installed"))
        except ImportError:
            checks.append(_check(f"required_package:{module_name}", FAIL, "missing", f"pip install {module_name}"))
    for module_name in _OPTIONAL_PACKAGES:
        try:
            importlib.import_module(module_name)
            checks.append(_check(f"optional_package:{module_name}", PASS, "installed"))
        except ImportError:
            checks.append(
                _check(
                    f"optional_package:{module_name}",
                    WARNING,
                    "missing",
                    f"pip install {module_name} (only needed for related features)",
                )
            )

    # --- secrets ----------------------------------------------------------
    for name, required in (required_secrets or {}).items():
        available = bool(secret_status(name)) if secret_status else bool(os.environ.get(name))
        if not required:
            checks.append(_check(f"secret:{name}", SKIPPED, "not required for selected mode"))
        elif available:
            checks.append(_check(f"secret:{name}", PASS, "configured"))
        else:
            checks.append(
                _check(
                    f"secret:{name}",
                    FAIL,
                    "missing",
                    f"Provide {name} via Colab Secrets, environment variable or the Secrets tab.",
                )
            )

    # --- reference video --------------------------------------------------
    if reference_video is None:
        checks.append(_check("reference_video", SKIPPED, "not required for selected action"))
    else:
        ref = Path(reference_video)
        if not ref.exists() or not ref.is_file():
            checks.append(
                _check(
                    "reference_video",
                    FAIL,
                    f"not found: {ref}",
                    "Upload a reference video or point to an existing file.",
                )
            )
        elif ref.suffix.lower() not in VIDEO_SUFFIXES:
            checks.append(
                _check(
                    "reference_video",
                    FAIL,
                    f"unsupported extension {ref.suffix!r}",
                    f"Use one of: {', '.join(VIDEO_SUFFIXES)}",
                )
            )
        elif ref.stat().st_size == 0:
            checks.append(_check("reference_video", FAIL, "file is empty", "Re-upload the reference video."))
        else:
            checks.append(_check("reference_video", PASS, f"{ref.name} ({ref.stat().st_size / 1024:.0f} KiB)"))

    # --- external tools ---------------------------------------------------
    for tool in ("ffmpeg", "ffprobe"):
        found = which(tool)
        checks.append(
            _check(
                tool,
                PASS if found else FAIL,
                found or "not on PATH",
                "" if found else "Install FFmpeg (apt-get install -y ffmpeg).",
            )
        )
    backend = (render_backend or (settings.render.backend if settings else "remotion")).lower()
    node_tools = {tool: which(tool) for tool in ("node", "npm", "npx")}
    if backend == "remotion":
        for tool, found in node_tools.items():
            checks.append(
                _check(
                    tool,
                    PASS if found else FAIL,
                    found or "not on PATH",
                    "" if found else "Install Node.js 18+ or switch render backend to 'pil'.",
                )
            )
        checks.append(
            _check(
                "remotion_prerequisites",
                PASS if all(node_tools.values()) else FAIL,
                "node/npm/npx " + ("available" if all(node_tools.values()) else "incomplete"),
                "" if all(node_tools.values()) else "Remotion rendering needs Node.js tooling.",
            )
        )
    else:
        for tool, found in node_tools.items():
            checks.append(_check(tool, PASS if found else SKIPPED, found or f"not needed for backend {backend!r}"))
        checks.append(_check("remotion_prerequisites", SKIPPED, f"render backend is {backend!r}"))

    # --- network (live only) ----------------------------------------------
    if not live:
        checks.append(_check("network", SKIPPED, "offline action; no network required"))
    else:
        try:
            import socket

            with socket.create_connection(("api.bfl.ai", 443), timeout=5):
                pass
            checks.append(_check("network", PASS, "outbound https reachable"))
        except OSError as exc:
            checks.append(
                _check(
                    "network",
                    FAIL,
                    f"outbound https failed: {type(exc).__name__}",
                    "Live runs need outbound network access.",
                )
            )

    # --- temporal / publishing safety --------------------------------------
    if settings is not None:
        sketch = settings.temporal.sketch_backend
        if sketch.allow_shell:
            checks.append(
                _check(
                    "temporal_shell",
                    WARNING,
                    "allow_shell is enabled (unsafe opt-in path)",
                    "Prefer the argv template; shell execution is rejected in production without an explicit override.",
                )
            )
        else:
            checks.append(_check("temporal_shell", PASS, "argv template mode (safe)"))
        if settings.publishing.enabled:
            hosts = settings.publishing.allowed_hosts
            checks.append(
                _check(
                    "publishing_safety",
                    PASS if (settings.publishing.provider == "local-archive" or hosts) else FAIL,
                    f"provider={settings.publishing.provider}",
                    ""
                    if (settings.publishing.provider == "local-archive" or hosts)
                    else "upload-post publishing requires allowed_hosts.",
                )
            )
        else:
            checks.append(_check("publishing_safety", PASS, "publishing disabled"))

    # --- existing job ------------------------------------------------------
    if job_id:
        job_dir = ws / "jobs" / job_id
        manifest = load_json(job_dir / "job_manifest.json")
        if isinstance(manifest, dict):
            stages = manifest.get("stages", {})
            done = sum(1 for s in stages.values() if s.get("status") == "completed")
            checks.append(
                _check(
                    "existing_job_manifest",
                    PASS,
                    f"{job_id}: {manifest.get('status')} — {done}/{len(stages)} stages completed",
                )
            )
        else:
            checks.append(_check("existing_job_manifest", SKIPPED, f"no manifest for {job_id} (fresh job)"))
        lock = job_dir / "job.lock"
        if lock.exists():
            checks.append(
                _check(
                    "existing_job_lock",
                    WARNING,
                    "lock file present",
                    "A crashed worker's stale lock is reclaimed automatically; a live worker blocks the run.",
                )
            )
        else:
            checks.append(_check("existing_job_lock", PASS, "no lock"))

    summary = {status: sum(1 for c in checks if c["status"] == status) for status in (PASS, WARNING, FAIL, SKIPPED)}
    return {
        "checks": [redact_secrets(c) for c in checks],
        "summary": summary,
        "ok": summary[FAIL] == 0,
    }


def save_preflight_report(report: dict[str, Any], destination: str | Path) -> Path:
    return save_json(Path(destination), report)


In [ ]:
%%writefile scistudio_v10/provenance.py
"""Artifact provenance manifest.

Connects every stage of a run — topic → research → script → storyboard →
architecture → references → prompts → FLUX outputs → approvals → semantic
layers → animation → render — with content hashes so any published video can
be traced back to the exact inputs that produced it.
"""

from __future__ import annotations

from datetime import datetime, UTC
from pathlib import Path
from typing import Any

from .utils import hash_value, save_json, sha256_file

PROVENANCE_SCHEMA_VERSION = "1.0"


def _artifact_entry(path: str | Path | None) -> dict[str, Any]:
    if not path:
        return {"path": "", "exists": False, "sha256": ""}
    path = Path(path)
    if not path.exists() or not path.is_file():
        return {"path": str(path), "exists": False, "sha256": ""}
    return {"path": str(path), "exists": True, "sha256": sha256_file(path)}


def build_provenance_manifest(
    *,
    job_id: str,
    topic: str,
    execution_mode: str,
    config_hash: str,
    artifacts: dict[str, str | Path | None],
    providers: dict[str, str],
    warnings: list[str] | None = None,
) -> dict[str, Any]:
    """Assemble the provenance manifest for one pipeline run.

    ``artifacts`` maps stage names (script, storyboard, beauty_frames, video,
    ...) to file paths; each entry is recorded with existence and SHA-256.
    """
    entries = {name: _artifact_entry(path) for name, path in artifacts.items()}
    manifest = {
        "schema_version": PROVENANCE_SCHEMA_VERSION,
        "job_id": job_id,
        "topic": topic,
        "execution_mode": execution_mode,
        "config_hash": config_hash,
        "created_at": datetime.now(UTC).isoformat(),
        "providers": providers,
        "artifacts": entries,
        "warnings": list(warnings or []),
    }
    manifest["provenance_hash"] = hash_value({k: v for k, v in manifest.items() if k != "provenance_hash"}, 32)
    return manifest


def write_provenance_manifest(run_dir: str | Path, manifest: dict[str, Any]) -> Path:
    path = Path(run_dir) / "provenance_manifest.json"
    save_json(path, manifest)
    return path


In [ ]:
%%writefile scistudio_v10/provider_registry.py
from __future__ import annotations

from pathlib import Path
from typing import Any

from .schemas import ProviderSpec
from .utils import ensure_dir, save_json


class ProviderRegistry:
    """Central registry for production providers.

    Providers are resolved by capability and priority rather than hard-coded
    conditionals spread across the pipeline. Implementations can be callables,
    objects, local commands or remote adapters.
    """

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)
        self.specs: dict[str, ProviderSpec] = {}
        self.implementations: dict[str, Any] = {}

    def register(self, spec: ProviderSpec, implementation: Any = None) -> None:
        self.specs[spec.provider_id] = spec
        if implementation is not None:
            self.implementations[spec.provider_id] = implementation
        self.snapshot()

    def resolve(self, provider_type: str, *, require: str | None = None) -> tuple[ProviderSpec, Any]:
        options = [s for s in self.specs.values() if s.enabled and s.provider_type == provider_type]
        if require:
            options = [s for s in options if any(c.name == require and c.available for c in s.capabilities)]
        if not options:
            raise RuntimeError(f"No enabled provider for type={provider_type!r}, capability={require!r}")
        spec = sorted(options, key=lambda x: (x.priority, x.provider_id))[0]
        return spec, self.implementations.get(spec.provider_id)

    def available(self, provider_type: str) -> list[ProviderSpec]:
        return sorted(
            [s for s in self.specs.values() if s.enabled and s.provider_type == provider_type],
            key=lambda x: (x.priority, x.provider_id),
        )

    def snapshot(self) -> str:
        path = self.root / "provider_registry.json"
        save_json(path, [s.model_dump(mode="json") for s in self.specs.values()])
        return str(path)

    @classmethod
    def from_config(cls, config: dict[str, Any], root: str | Path) -> ProviderRegistry:
        registry = cls(root)
        for raw in config.get("providers", []):
            registry.register(ProviderSpec.model_validate(raw))
        return registry


In [ ]:
%%writefile scistudio_v10/publisher.py
"""Publishing adapters.

``UploadPostPublisher`` only ever posts to an https endpoint whose host is in
the explicitly configured allowlist — a bearer token is never sent to an
arbitrary or plain-http destination.
"""

from __future__ import annotations

import json
import urllib.request
from pathlib import Path
from typing import Any, Protocol

from .http_safety import validate_url
from .security import redacted_exception_text, register_secret
from .utils import atomic_copy, ensure_dir, save_json


class Publisher(Protocol):
    publisher_id: str

    def publish(self, video_path: str, metadata: dict[str, Any]) -> dict[str, Any]: ...


class LocalArchivePublisher:
    publisher_id = "local-archive"

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)

    def publish(self, video_path: str, metadata: dict[str, Any]) -> dict[str, Any]:
        source = Path(video_path)
        if not source.exists():
            raise FileNotFoundError(source)
        destination = self.root / source.name
        atomic_copy(source, destination)
        manifest = {
            "publisher": self.publisher_id,
            "video": str(destination),
            "metadata": metadata,
            "status": "archived",
        }
        save_json(destination.with_suffix(".publish.json"), manifest)
        return manifest


class UploadPostPublisher:
    """Minimal optional Upload-Post compatible adapter.

    The endpoint must be https and its host must appear in *allowed_hosts*;
    both come from configuration. Not invoked unless publishing is enabled.
    """

    publisher_id = "upload-post"

    def __init__(self, endpoint: str, token: str, allowed_hosts: list[str] | None = None):
        self.endpoint = endpoint
        self.token = token
        self.allowed_hosts = list(allowed_hosts or [])
        register_secret(token)

    def publish(self, video_path: str, metadata: dict[str, Any]) -> dict[str, Any]:
        if not self.endpoint or not self.token:
            raise RuntimeError("Upload-Post endpoint/token are required")
        validate_url(self.endpoint, allowed_hosts=self.allowed_hosts, purpose="publishing endpoint")
        payload = json.dumps({"video_path": video_path, "metadata": metadata}).encode("utf-8")
        request = urllib.request.Request(
            self.endpoint,
            data=payload,
            headers={"Authorization": f"Bearer {self.token}", "Content-Type": "application/json"},
            method="POST",
        )
        try:
            with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 - https+allowlist enforced above
                return json.loads(response.read().decode("utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Upload-Post publish failed: {redacted_exception_text(exc, 300)}") from exc


In [ ]:
%%writefile scistudio_v10/reference_director.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw, ImageOps

from .llm import LLMRouter
from .schemas import (
    ArtDirectionBible,
    CanonAnchor,
    ContinuityCanon,
    ReferencePack,
    ReferenceRequirement,
    SceneIllustrationArchitecture,
)
from .utils import ensure_dir, load_json, save_json


class ReferenceDirector:
    SYSTEM = """You are the Reference Director for a scientific illustration studio.
Return JSON only. Request only references that materially improve observation: style, anatomy, pose,
environment, material, lighting or continuity. References are evidence and construction aids—not final art.
Never request copying a composition or another artist's exact drawing."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def plan(
        self,
        architecture: SceneIllustrationArchitecture,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        previous_approved_scene: str = "",
        *,
        force: bool = False,
    ) -> ReferencePack:
        path = self.root / f"{architecture.scene_id}.json"
        if path.exists() and not force:
            return ReferencePack.model_validate(load_json(path))
        fallback = self._fallback(architecture, bible, continuity, previous_approved_scene)
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""Scene architecture:
{json.dumps(architecture.model_dump(mode="json"), ensure_ascii=False)}
Continuity subjects:
{json.dumps([s.model_dump(mode="json") for s in continuity.recurring_subjects], ensure_ascii=False)}

Return requirements with reference_id, purpose, query, priority, use_rule and source_preference.
Always include the locked style anchor and continuity reference. Add pose/anatomy references for human action,
environment references for perspective/materials, and material references for complex fluids, terrain or damage.
Keep the list compact and purposeful.""",
            namespace=f"v10_reference_plan_{architecture.scene_id}",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        try:
            pack = ReferencePack.model_validate(raw)
        except Exception:
            pack = fallback
        pack.scene_id = architecture.scene_id
        pack.style_anchor_ids = self._unique(["reference-video-board", *pack.style_anchor_ids])
        pack.previous_approved_scene = previous_approved_scene
        save_json(path, pack)
        return pack

    def build_board(
        self,
        pack: ReferencePack,
        continuity: ContinuityCanon,
        *,
        extra_paths: list[str] | None = None,
    ) -> str:
        output = self.root / "boards" / f"{pack.scene_id}_anchor_board.png"
        ensure_dir(output.parent)
        paths: list[tuple[str, str]] = []
        anchor_map = {a.anchor_id: a for a in continuity.anchors if a.approved and Path(a.path).exists()}
        for anchor_id in pack.style_anchor_ids:
            anchor = anchor_map.get(anchor_id)
            if anchor:
                paths.append((anchor.role, anchor.path))
        if pack.previous_approved_scene and Path(pack.previous_approved_scene).exists():
            paths.append(("previous approved scene", pack.previous_approved_scene))
        for path in pack.subject_anchor_paths + pack.environment_anchor_paths + list(extra_paths or []):
            if path and Path(path).exists():
                paths.append(("continuity anchor", path))
        # Avoid a single image being duplicated across all cells.
        dedup: list[tuple[str, str]] = []
        seen: set[str] = set()
        for label, path in paths:
            key = str(Path(path).resolve())
            if key not in seen:
                seen.add(key)
                dedup.append((label, path))
        if not dedup:
            raise RuntimeError("No usable visual anchors are available for the reference board")
        self._board(dedup[:4], output)
        pack.board_path = str(output)
        save_json(self.root / f"{pack.scene_id}.json", pack)
        return str(output)

    def add_approved_anchor(
        self,
        continuity: ContinuityCanon,
        scene_id: str,
        image_path: str,
        role: str = "approved_scene",
    ) -> ContinuityCanon:
        anchor_id = f"approved-{scene_id}"
        continuity.anchors = [a for a in continuity.anchors if a.anchor_id != anchor_id]
        continuity.anchors.append(
            CanonAnchor(
                anchor_id=anchor_id,
                role=role,  # type: ignore[arg-type]
                path=image_path,
                scene_id=scene_id,
                notes="Approved visual canon for downstream continuity.",
            )
        )
        return continuity

    def _fallback(
        self,
        architecture: SceneIllustrationArchitecture,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        previous: str,
    ) -> ReferencePack:
        requirements = [
            ReferenceRequirement(
                reference_id="style-canon",
                purpose="style",
                query="locked Experiment Ledger Editorial Ink visual language from the user-supplied reference board",
                priority=1,
                use_rule="preserve line, palette, paper field, UI and motion grammar; do not copy source content",
                source_preference=["user reference"],
            )
        ]
        for figure in architecture.figure_construction:
            requirements.extend(
                [
                    ReferenceRequirement(
                        reference_id=f"{figure.figure_id}-pose",
                        purpose="pose",
                        query=f"adult human {figure.body_orientation}; {figure.weight_distribution}; {figure.gesture_line}",
                        priority=1,
                    ),
                    ReferenceRequirement(
                        reference_id=f"{figure.figure_id}-anatomy",
                        purpose="anatomy",
                        query=f"adult anatomy and hand construction for {figure.body_orientation}",
                        priority=1,
                    ),
                ]
            )
        for material in architecture.material_marks:
            requirements.append(
                ReferenceRequirement(
                    reference_id=f"material-{self._safe(material.material)}",
                    purpose="material",
                    query=f"observational reference for {material.material}: {', '.join(material.visual_cues)}",
                    priority=2,
                )
            )
        requirements.append(
            ReferenceRequirement(
                reference_id="environment-perspective",
                purpose="environment",
                query=f"environment and perspective reference for {architecture.visual_thesis}",
                priority=2,
            )
        )
        anchors = ["reference-video-board"]
        if previous:
            anchors.append(f"approved-{architecture.scene_id}-previous")
        return ReferencePack(
            scene_id=architecture.scene_id,
            style_anchor_ids=anchors,
            requirements=requirements,
            previous_approved_scene=previous,
        )

    @staticmethod
    def _board(items: list[tuple[str, str]], output: Path) -> None:
        w, h = 1536, 1536
        board = Image.new("RGB", (w, h), "#FAFAF7")
        draw = ImageDraw.Draw(board)
        cells = [(0, 0, w // 2, h // 2), (w // 2, 0, w, h // 2), (0, h // 2, w // 2, h), (w // 2, h // 2, w, h)]
        for i, (label, path) in enumerate(items[:4]):
            x0, y0, x1, y1 = cells[i]
            source = Image.open(path).convert("RGB")
            image = ImageOps.contain(source, (x1 - x0 - 36, y1 - y0 - 70))
            x = x0 + (x1 - x0 - image.width) // 2
            y = y0 + 18
            board.paste(image, (x, y))
            draw.rectangle([x0 + 8, y0 + 8, x1 - 8, y1 - 8], outline="#475157", width=3)
            draw.rectangle([x0 + 8, y1 - 52, x1 - 8, y1 - 8], fill="#FAFAF7", outline="#475157", width=2)
            draw.text((x0 + 22, y1 - 42), label.upper()[:48], fill="#20282D")
        draw.text(
            (24, h - 24), "VISUAL LANGUAGE ONLY — DO NOT COPY CONTENT OR COMPOSITION", fill="#D8483E", anchor="ls"
        )
        output.parent.mkdir(parents=True, exist_ok=True)
        board.save(output)

    @staticmethod
    def _safe(value: str) -> str:
        return "-".join("".join(ch.lower() if ch.isalnum() else " " for ch in value).split())

    @staticmethod
    def _unique(items: list[str]) -> list[str]:
        out: list[str] = []
        for item in items:
            if item and item not in out:
                out.append(item)
        return out


In [ ]:
%%writefile scistudio_v10/reference_selector.py
from __future__ import annotations

from pathlib import Path

from PIL import Image, ImageDraw, ImageOps

from .asset_registry import AssetRegistry
from .schemas import AssetQuery, AssetRecord, RankedAsset, ReferenceSelection
from .utils import ensure_dir, save_json


class DynamicReferenceSelector:
    """ViMax-inspired retrieval selector using subject, environment, camera and chronology."""

    def __init__(self, registry: AssetRegistry, root: str | Path):
        self.registry = registry
        self.root = ensure_dir(root)

    def select(self, query: AssetQuery) -> ReferenceSelection:
        ranked: list[RankedAsset] = []
        for asset in self.registry.list(approved_only=True):
            score, reasons = self._score(asset, query)
            if score > 0:
                ranked.append(RankedAsset(asset=asset, score=score, reasons=reasons))
        ranked.sort(key=lambda x: (-x.score, x.asset.asset_id))
        chosen = self._diversify(ranked, query.maximum_results)
        selection = ReferenceSelection(scene_id=query.scene_id, query=query, selected=chosen)
        selection.board_path = self.build_board(selection)
        save_json(self.root / f"{query.scene_id}.json", selection)
        return selection

    @staticmethod
    def _score(asset: AssetRecord, query: AssetQuery) -> tuple[float, list[str]]:
        score = 0.0
        reasons: list[str] = []
        if query.desired_types and asset.asset_type in query.desired_types:
            score += 2.0
            reasons.append("desired asset type")
        shared = set(query.subject_ids) & set(asset.subject_ids)
        if shared:
            score += 4.0 + 0.5 * len(shared)
            reasons.append("subject identity")
        if query.environment_id and asset.environment_id == query.environment_id:
            score += 3.0
            reasons.append("environment identity")
        if query.camera_view and asset.camera_view == query.camera_view:
            score += 2.5
            reasons.append("camera match")
        if query.perspective and asset.perspective == query.perspective:
            score += 1.5
            reasons.append("perspective match")
        if query.style_hash and asset.style_hash == query.style_hash:
            score += 3.0
            reasons.append("style fingerprint")
        if query.required_roles and set(query.required_roles) & set(asset.role_tags):
            score += 1.5
            reasons.append("role match")
        if query.chronology_index >= 0 and asset.chronology_index >= 0:
            delta = query.chronology_index - asset.chronology_index
            if 0 <= delta <= 2:
                score += 2.0 - 0.4 * delta
                reasons.append("recent approved continuity")
            elif delta < 0:
                return -1000.0, ["future continuity excluded"]
        score += min(1.0, max(0.0, asset.quality_score))
        return score, reasons

    @staticmethod
    def _diversify(ranked: list[RankedAsset], limit: int) -> list[RankedAsset]:
        selected: list[RankedAsset] = []
        type_counts: dict[str, int] = {}
        for item in ranked:
            t = item.asset.asset_type
            cap = 3 if t in {"approved_scene", "subject_view"} else 2
            if type_counts.get(t, 0) >= cap:
                continue
            selected.append(item)
            type_counts[t] = type_counts.get(t, 0) + 1
            if len(selected) >= limit:
                break
        return selected

    def build_board(self, selection: ReferenceSelection) -> str:
        output = self.root / "boards" / f"{selection.scene_id}.png"
        ensure_dir(output.parent)
        items = [x for x in selection.selected if Path(x.asset.path).exists()][:8]
        if not items:
            return ""
        w, h = 1600, 1200
        board = Image.new("RGB", (w, h), "#FAFAF7")
        draw = ImageDraw.Draw(board)
        cols, rows = 4, 2
        cw, ch = w // cols, h // rows
        for i, item in enumerate(items):
            x0 = (i % cols) * cw
            y0 = (i // cols) * ch
            image = Image.open(item.asset.path).convert("RGB")
            image = ImageOps.contain(image, (cw - 30, ch - 80))
            board.paste(image, (x0 + (cw - image.width) // 2, y0 + 15))
            draw.rectangle((x0 + 5, y0 + 5, x0 + cw - 5, y0 + ch - 5), outline="#475157", width=2)
            label = f"{item.asset.asset_type} | {item.score:.1f} | {item.asset.asset_id}"[:54]
            draw.rectangle((x0 + 8, y0 + ch - 54, x0 + cw - 8, y0 + ch - 8), fill="#FAFAF7", outline="#475157")
            draw.text((x0 + 15, y0 + ch - 43), label, fill="#20282D")
        board.save(output)
        return str(output)


In [ ]:
%%writefile scistudio_v10/research.py
"""Public research search (Wikipedia, Crossref, OpenAlex, arXiv) and the
LLM-backed research pack builder."""

from __future__ import annotations

import html
import json
import logging
import re
from pathlib import Path
from typing import Any

import requests

from .llm import LLMRouter
from .schemas import Fact, ResearchPack, SourceDoc
from .security import redacted_exception_text
from .utils import ensure_dir, hash_value, load_json, save_json

logger = logging.getLogger(__name__)

# Public metadata APIs return small JSON/XML documents; cap defensively.
_MAX_RESEARCH_RESPONSE_BYTES = 4 * 1024 * 1024


class PublicResearchSearch:
    """Federated public-source search with per-provider failure isolation."""

    def __init__(self, config: dict[str, Any], cache_root: str | Path):
        self.config = config
        self.cache_root = ensure_dir(cache_root)
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "ScientificMotionStudio/10.1 research@example.invalid"})
        self.timeout = int(config.get("timeout", 25))

    def _get_json(self, url: str, params: dict[str, Any]) -> Any:
        response = self.session.get(url, params=params, timeout=self.timeout)
        response.raise_for_status()
        if len(response.content) > _MAX_RESEARCH_RESPONSE_BYTES:
            raise ValueError(f"research response from {url} exceeds size cap")
        return response.json()

    def search(self, topic: str, force: bool = False) -> list[SourceDoc]:
        if not self.config.get("enabled", True):
            return []
        cache_path = self.cache_root / f"search-{hash_value(topic)}.json"
        if cache_path.exists() and not force:
            return [SourceDoc.model_validate(x) for x in load_json(cache_path, [])]
        sources: list[SourceDoc] = []
        for fn in (self._wikipedia, self._crossref, self._openalex, self._arxiv):
            try:
                sources.extend(fn(topic))
            except (requests.RequestException, ValueError, KeyError) as exc:
                logger.warning("[research] %s skipped: %s", fn.__name__, redacted_exception_text(exc, 200))
        dedup: dict[str, SourceDoc] = {}
        for item in sources:
            key = (item.url or item.title).strip().lower()
            if key and key not in dedup:
                dedup[key] = item
        ranked = list(dedup.values())
        words = set(re.findall(r"[a-z0-9]+", topic.lower()))
        for item in ranked:
            text_words = set(re.findall(r"[a-z0-9]+", f"{item.title} {item.snippet}".lower()))
            overlap = len(words & text_words) / max(1, len(words))
            item.relevance_score = min(1.0, overlap)
            item.final_score = 0.6 * item.authority_score + 0.4 * item.relevance_score
        ranked.sort(key=lambda x: x.final_score, reverse=True)
        ranked = ranked[: int(self.config.get("max_sources", 24))]
        for index, item in enumerate(ranked, 1):
            item.source_id = f"S{index:02d}"
        save_json(cache_path, ranked)
        return ranked

    def _wikipedia(self, topic: str) -> list[SourceDoc]:
        endpoint = "https://en.wikipedia.org/w/api.php"
        payload = self._get_json(
            endpoint,
            {
                "action": "query",
                "generator": "search",
                "gsrsearch": topic,
                "gsrlimit": 5,
                "prop": "extracts|info",
                "exintro": 1,
                "explaintext": 1,
                "inprop": "url",
                "format": "json",
            },
        )
        output = []
        for page in payload.get("query", {}).get("pages", {}).values():
            output.append(
                SourceDoc(
                    provider="wikipedia",
                    title=page.get("title", "Wikipedia"),
                    url=page.get("fullurl", ""),
                    snippet=page.get("extract", "")[:1600],
                    authority_score=0.72,
                )
            )
        return output

    def _crossref(self, topic: str) -> list[SourceDoc]:
        payload = self._get_json(
            "https://api.crossref.org/works",
            {"query": topic, "rows": 6, "select": "DOI,title,author,published,URL,abstract,publisher,type"},
        )
        output = []
        for item in payload.get("message", {}).get("items", []):
            title = " ".join(item.get("title") or ["Untitled"])
            authors = item.get("author") or []
            author = ", ".join(" ".join(filter(None, [a.get("given", ""), a.get("family", "")])) for a in authors[:3])
            abstract = re.sub(r"<[^>]+>", " ", item.get("abstract", ""))
            output.append(
                SourceDoc(
                    provider="crossref",
                    title=title,
                    url=item.get("URL", ""),
                    author=author,
                    snippet=html.unescape(abstract)[:1800],
                    authority_score=0.86,
                    metadata={"doi": item.get("DOI", ""), "publisher": item.get("publisher", "")},
                )
            )
        return output

    @staticmethod
    def _search_safe(topic: str) -> str:
        """Strip punctuation that some scholarly APIs reject with HTTP 400
        (OpenAlex fails on '?' and similar); keep words and hyphens."""
        cleaned = re.sub(r"[^\w\s-]", " ", topic)
        return re.sub(r"\s+", " ", cleaned).strip() or topic

    def _openalex(self, topic: str) -> list[SourceDoc]:
        payload = self._get_json(
            "https://api.openalex.org/works",
            {"search": self._search_safe(topic), "per-page": 6, "mailto": "research@example.invalid"},
        )
        output = []
        for item in payload.get("results", []):
            inverted = item.get("abstract_inverted_index") or {}
            words = sorted(
                ((pos, word) for word, positions in inverted.items() for pos in positions), key=lambda x: x[0]
            )
            abstract = " ".join(word for _, word in words)
            source = ((item.get("primary_location") or {}).get("source") or {}).get("display_name", "")
            output.append(
                SourceDoc(
                    provider="openalex",
                    title=item.get("display_name", "Untitled"),
                    url=item.get("doi") or item.get("id", ""),
                    snippet=abstract[:1800],
                    authority_score=0.88,
                    published_at=str(item.get("publication_year", "")),
                    metadata={"cited_by_count": item.get("cited_by_count", 0), "venue": source},
                )
            )
        return output

    def _arxiv(self, topic: str) -> list[SourceDoc]:
        import xml.etree.ElementTree as ET

        response = self.session.get(
            "https://export.arxiv.org/api/query",
            params={
                "search_query": f"all:{topic}",
                "start": 0,
                "max_results": 5,
                "sortBy": "relevance",
                "sortOrder": "descending",
            },
            timeout=self.timeout,
        )
        response.raise_for_status()
        if len(response.content) > _MAX_RESEARCH_RESPONSE_BYTES:
            raise ValueError("arxiv response exceeds size cap")
        root = ET.fromstring(response.content)
        ns = {"a": "http://www.w3.org/2005/Atom"}
        output = []
        for entry in root.findall("a:entry", ns):
            output.append(
                SourceDoc(
                    provider="arxiv",
                    title=" ".join((entry.findtext("a:title", default="", namespaces=ns)).split()),
                    url=entry.findtext("a:id", default="", namespaces=ns),
                    snippet=" ".join((entry.findtext("a:summary", default="", namespaces=ns)).split())[:1800],
                    author=", ".join(
                        a.findtext("a:name", default="", namespaces=ns) for a in entry.findall("a:author", ns)[:3]
                    ),
                    published_at=entry.findtext("a:published", default="", namespaces=ns),
                    authority_score=0.82,
                )
            )
        return output


class ResearchEngine:
    SYSTEM = """You are a scientific research editor for a short-form motion graphics studio.
Preserve uncertainty. Do not invent citations. Return rich JSON, but you may use nested objects when useful.
The downstream engine is permissive: prioritize factual quality over matching a brittle schema."""

    def __init__(self, llm: LLMRouter, cache_root: str | Path):
        self.llm = llm
        self.cache_root = ensure_dir(cache_root)

    def build(self, topic: str, sources: list[SourceDoc], force: bool = False) -> ResearchPack:
        cache_path = self.cache_root / f"research-{hash_value([topic, [s.url for s in sources]])}.json"
        if cache_path.exists() and not force:
            return ResearchPack.model_validate(load_json(cache_path))
        source_payload = [s.model_dump(mode="json") for s in sources]
        fallback = self._fallback(topic, sources)
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""Build an evidence pack for this topic: {topic}

Sources:
{json.dumps(source_payload, ensure_ascii=False, indent=2)}

Return an object with topic, summary, hooks, facts, comparisons, visual_ideas, limitations.
Each fact should include claim, source_ids, confidence, numeric_values, comparison, visual_hint.
Structured numeric_values are allowed. Never cite a source ID that is not in the supplied list.""",
            namespace="research",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        if not isinstance(raw, dict):
            raw = fallback.model_dump(mode="json")
        raw["topic"] = topic
        raw["sources"] = source_payload
        raw["raw_llm_output"] = raw.copy()
        pack = ResearchPack.model_validate(raw)
        valid_ids = {s.source_id for s in sources}
        for index, fact in enumerate(pack.facts, 1):
            fact.fact_id = fact.fact_id or f"F{index:02d}"
            fact.source_ids = [sid for sid in fact.source_ids if sid in valid_ids]
            fact.confidence = max(0.0, min(1.0, float(fact.confidence or 0.0)))
        supported = sum(1 for f in pack.facts if f.source_ids)
        pack.validation_score = round(supported / max(1, len(pack.facts)), 3)
        pack.research_hash = hash_value(pack.model_dump(exclude={"research_hash", "raw_llm_output"}))
        save_json(cache_path, pack)
        return pack

    def _fallback(self, topic: str, sources: list[SourceDoc]) -> ResearchPack:
        facts = []
        for index, source in enumerate(sources[:8], 1):
            claim = source.snippet.strip().split(". ")[0].strip()
            if claim:
                facts.append(
                    Fact(
                        fact_id=f"F{index:02d}",
                        claim=claim[:500],
                        source_ids=[source.source_id],
                        confidence=0.55,
                        visual_hint=source.title,
                    )
                )
        return ResearchPack(
            topic=topic,
            summary=f"Evidence pack assembled from {len(sources)} public sources for {topic}.",
            hooks=[f"What changes first if {topic.lower()}?"],
            facts=facts,
            comparisons=[],
            visual_ideas=[topic],
            limitations=["Fallback extraction used; review claims before publication."],
            sources=sources,
        )


In [ ]:
%%writefile scistudio_v10/scene_architect.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import (
    ArtDirectionBible,
    ContinuityCanon,
    DepthPlane,
    FigureConstruction,
    MaterialMarkPlan,
    MotionSeam,
    PerspectivePlan,
    SceneIllustrationArchitecture,
    SceneRequest,
    Storyboard,
)
from .utils import ensure_dir, load_json, save_json


class SceneIllustrationArchitect:
    SYSTEM = """You are the Lead Scene Illustration Architect for an institutional science-animation studio.
Return JSON only. The hard-coded house style and continuity canon are immutable.
Design one whole authored illustration before any semantic separation. Think like an illustrator:
composition, visual route, perspective, gesture, anatomy, overlap, material marks, value structure,
negative space, contour architecture and motion seams. Never design by assembling isolated icons.
Never request procedural SVG hero art, primitive shapes, mascot anatomy or generic AI cinematic art.
Semantic separation occurs only after the beauty frame is approved."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def plan_all(
        self,
        storyboard: Storyboard,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        *,
        force: bool = False,
    ) -> list[SceneIllustrationArchitecture]:
        results = []
        previous_summary = ""
        for index, scene in enumerate(storyboard.scenes):
            architecture = self.plan(scene, storyboard, bible, continuity, previous_summary, index=index, force=force)
            results.append(architecture)
            previous_summary = architecture.visual_thesis + " | " + "; ".join(architecture.color_script)
        save_json(self.root / "scene_architectures.json", results)
        return results

    def plan(
        self,
        scene: SceneRequest,
        storyboard: Storyboard,
        bible: ArtDirectionBible,
        continuity: ContinuityCanon,
        previous_scene_summary: str,
        *,
        index: int,
        force: bool = False,
    ) -> SceneIllustrationArchitecture:
        path = self.root / f"{scene.scene_id}.json"
        if path.exists() and not force:
            return SceneIllustrationArchitecture.model_validate(load_json(path))
        fallback = self._fallback(scene, storyboard, bible, index)
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""LOCKED ART DIRECTION:
{json.dumps(bible.model_dump(mode="json"), ensure_ascii=False)}

LOCKED CONTINUITY:
{json.dumps(continuity.model_dump(mode="json"), ensure_ascii=False)}

CURRENT SCENE:
{json.dumps(scene.model_dump(mode="json"), ensure_ascii=False)}
Previous scene visual summary: {previous_scene_summary or "none"}
Canvas: {storyboard.width}x{storyboard.height}, vertical.

Return a SceneIllustrationArchitecture with:
- visual_thesis and a 3-6 step composition_route describing the eye path;
- coherent perspective and 3-5 depth planes;
- focal_subject, secondary_subjects, negative_space and contour_architecture;
- figure_construction for every human, including adult proportion, body orientation, weight distribution,
  gesture line, joint logic, hand construction and clothing logic;
- material_marks for water, soil, concrete, metal, vegetation, atmosphere or other visible materials;
- color_script and scientific_annotations;
- motion_seams chosen after imagining a finished beauty frame;
- animation_representation and required_pose_variants;
- director_notes and prohibited_visual_shortcuts.

Hard rules:
1. beauty_frame_first=true; semantic_split_after_approval=true; procedural_hero_allowed=false.
2. The image must read as one authored environment, never a collage of isolated objects.
3. Interior lines explain form, material, force or perspective; no decorative pseudo-detail.
4. For human action, prefer replacement poses/local redraws over circle-joint puppet rigs.
5. Preserve the house line, palette, experiment UI and adult scientific tone.
6. Motion seams must be minimal and selected only for the stated desired_change.
""",
            namespace=f"v10_scene_architecture_{scene.scene_id}",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        try:
            architecture = SceneIllustrationArchitecture.model_validate(raw)
        except Exception:
            architecture = fallback
        architecture.scene_id = scene.scene_id
        architecture.beat_id = scene.beat_id
        architecture.narrative_claim = scene.scientific_claim or scene.narration
        architecture.canvas = (storyboard.width, storyboard.height)
        architecture.beauty_frame_first = True
        architecture.semantic_split_after_approval = True
        architecture.procedural_hero_allowed = False
        architecture.prohibited_visual_shortcuts = self._merge(
            architecture.prohibited_visual_shortcuts,
            bible.locked_canon.forbidden,
        )
        save_json(path, architecture)
        return architecture

    def _fallback(
        self,
        scene: SceneRequest,
        storyboard: Storyboard,
        bible: ArtDirectionBible,
        index: int,
    ) -> SceneIllustrationArchitecture:
        event = scene.visual_event or scene.desired_change or scene.narration
        lower = event.lower()
        figures: list[FigureConstruction] = []
        if any(token in lower for token in ("person", "human", "people", "hand", "body", "worker", "scientist")):
            figures.append(
                FigureConstruction(
                    figure_id="primary-human",
                    body_orientation="three-quarter profile aligned with the action",
                    weight_distribution="weight follows gravity and the physical action; feet and pelvis support the gesture",
                    gesture_line="one continuous directional sweep from head through rib cage, pelvis and action limb",
                    joint_logic=[
                        "shoulder mass connects arm to rib cage",
                        "elbow direction follows the forearm plane",
                        "wrist remains aligned with load",
                        "overlap hides seams in the resting pose",
                    ],
                )
            )
        materials: list[MaterialMarkPlan] = []
        candidates = [
            (
                "water",
                ("water", "rain", "ocean", "flood"),
                ["layered contour rhythm", "directional surface lines", "selective foam breaks"],
            ),
            (
                "atmosphere",
                ("wind", "air", "cloud", "atmosphere"),
                ["sparse flow traces", "soft value mass", "directional density"],
            ),
            (
                "concrete",
                ("city", "building", "wall", "infrastructure"),
                ["perspective-aligned edges", "irregular surface breaks", "window rhythm"],
            ),
            ("soil", ("soil", "ground", "slope", "land"), ["stratified contour", "grain marks", "compression cracks"]),
            (
                "vegetation",
                ("tree", "plant", "crop", "ecosystem"),
                ["branch hierarchy", "leaf masses", "growth-direction marks"],
            ),
        ]
        for material, tokens, cues in candidates:
            if any(t in lower for t in tokens):
                materials.append(
                    MaterialMarkPlan(
                        material=material,
                        visual_cues=cues,
                        line_marks=cues,
                        value_behavior="2-4 restrained value bands",
                    )
                )
        if not materials:
            materials.append(
                MaterialMarkPlan(
                    material="primary physical system",
                    visual_cues=["observed silhouette", "material-specific seams", "controlled value breaks"],
                    line_marks=["form-following interior lines"],
                )
            )

        motion_seams = []
        desired = (scene.desired_change or event).strip()
        if desired and desired.lower() not in {"hold", "none", "static"}:
            method = "replacement_pose" if figures else "local_deformation"
            if any(t in lower for t in ("rain", "wind", "water", "cloud", "smoke")):
                method = "texture_loop"
            motion_seams.append(
                MotionSeam(
                    seam_id="primary-causal-change",
                    subject=desired,
                    method=method,
                    region="the smallest local region that visibly carries the causal change",
                    resting_overlap_rule="The approved beauty frame remains visually continuous; the seam is hidden by overlap, value match or material edge.",
                    required_variants=["initial", "peak", "settled"]
                    if method == "replacement_pose"
                    else ["initial", "changed"],
                )
            )

        return SceneIllustrationArchitecture(
            scene_id=scene.scene_id,
            beat_id=scene.beat_id,
            narrative_claim=scene.scientific_claim or scene.narration,
            visual_thesis=f"Show {event} as one coherent scientific environment, with the causal change dominating the eye path.",
            canvas=(storyboard.width, storyboard.height),
            composition_route=[
                "experiment/status panel establishes context",
                "primary silhouette establishes the physical system",
                "directional contour or material flow leads to the causal change",
                "single metric or annotation confirms the consequence",
            ],
            perspective=PerspectivePlan(
                camera_height="slightly above or at subject center according to system scale",
                view="three-quarter editorial cutaway when depth is relevant; orthographic only for explicit diagrams",
                horizon_y=0.48,
                vanishing_points=[(0.18, 0.48), (0.82, 0.48)],
            ),
            depth_planes=[
                DepthPlane(
                    plane_id="background",
                    depth="background",
                    contents="paper field and restrained contextual environment",
                    line_weight_role="light structural",
                ),
                DepthPlane(
                    plane_id="system", depth="midground", contents=event, line_weight_role="primary authored contour"
                ),
                DepthPlane(
                    plane_id="causal-change",
                    depth="foreground",
                    contents=desired,
                    line_weight_role="selective emphasis",
                    movement_role="primary",
                ),
                DepthPlane(
                    plane_id="scientific-ui",
                    depth="overlay",
                    contents="experiment identifier, status, one metric and concise labels",
                    line_weight_role="technical",
                ),
            ],
            focal_subject=event,
            secondary_subjects=["contextual environment", "single scientific metric"],
            figure_construction=figures,
            material_marks=materials,
            negative_space="Reserve clear paper around the focal silhouette and keep the reading path unobstructed.",
            contour_architecture=[
                "one dominant silhouette with selective breaks at overlaps",
                "lighter structural lines describe perspective and material",
                "texture lines stop before becoming noise",
                "no uniform sticker outline around every internal region",
            ],
            color_script=[
                "paper and charcoal establish the instrument-like field",
                "blue-gray encodes the physical system",
                "one blue accent encodes direction or flow",
                "red appears only for critical threshold; yellow only for energy/light",
            ],
            scientific_annotations=[
                scene.time_stage or "causal stage",
                "one measured variable or condition",
                "one direct label for the changed region",
            ],
            motion_seams=motion_seams,
            animation_representation=[m.method for m in motion_seams] or ["hold"],
            required_pose_variants=[v for m in motion_seams for v in m.required_variants],
            director_notes=[
                "Create the full beauty composition first with Flux Kontext Pro.",
                "Do not isolate subjects before the scene reads as a complete authored illustration.",
                "Use the approved previous frame as a continuity reference, not a composition template.",
            ],
            prohibited_visual_shortcuts=list(bible.locked_canon.forbidden),
        )

    @staticmethod
    def _merge(a: list[str], b: list[str]) -> list[str]:
        out: list[str] = []
        for item in [*a, *b]:
            text = str(item).strip()
            if text and text not in out:
                out.append(text)
        return out


In [ ]:
%%writefile scistudio_v10/schemas.py
from __future__ import annotations

import json
from datetime import datetime, UTC
from typing import Any, Literal

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator


class OpenModel(BaseModel):
    """Base model for LLM-facing schemas.

    LLM output is structurally free; this base applies a *systemic* coercion
    before validation so shape drift never crashes a paid pipeline run:

    - a field declared ``list[...]`` that receives a dict becomes the dict's
      values (a dict of named notes is a list of notes); a scalar becomes a
      one-element list;
    - a field declared ``str`` (or ``str | None``) that receives a list is
      joined; a dict is serialized to compact JSON;
    - fields declared ``Any`` are never touched.
    """

    model_config = ConfigDict(extra="allow", arbitrary_types_allowed=True, validate_assignment=False)

    @staticmethod
    def _extract_number(value: Any) -> float | None:
        """Best-effort numeric value from a drifted LLM payload.

        A dict like ``{"label": "high", "score": 0.8}`` yields ``0.8``; a
        string like ``"about 0.6 (medium)"`` yields ``0.6``. ``bool`` is never
        treated as a number. Returns ``None`` when nothing numeric is found.
        """
        import re

        if isinstance(value, bool):
            return None
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            match = re.search(r"-?\d+(?:\.\d+)?", value)
            return float(match.group()) if match else None
        if isinstance(value, dict):
            for key in (
                "score",
                "value",
                "amount",
                "seconds",
                "count",
                "number",
                "confidence",
                "probability",
                "weight",
            ):
                if key in value:
                    inner = OpenModel._extract_number(value[key])
                    if inner is not None:
                        return inner
            for item in value.values():
                inner = OpenModel._extract_number(item)
                if inner is not None:
                    return inner
        if isinstance(value, (list, tuple)):
            for item in value:
                inner = OpenModel._extract_number(item)
                if inner is not None:
                    return inner
        return None

    @model_validator(mode="before")
    @classmethod
    def _coerce_open_types(cls, data: Any) -> Any:
        """Systemically reconcile LLM shape drift with declared field types
        before validation, so a paid pipeline run never crashes on a dict/str
        where a list/str/number was declared. ``Any`` fields are untouched."""
        if not isinstance(data, dict):
            return data
        import types
        import typing

        coerced = dict(data)
        for name, field in cls.model_fields.items():
            if name not in coerced:
                continue
            value = coerced[name]
            annotation = field.annotation
            if annotation is Any or value is None:
                continue
            origin = typing.get_origin(annotation)
            args = set(typing.get_args(annotation)) if origin in (typing.Union, types.UnionType) else set()
            wants_list = origin is list or annotation is list
            wants_str = annotation is str or args == {str, type(None)}
            wants_float = annotation is float or args == {float, type(None)}
            wants_int = annotation is int or args == {int, type(None)}
            if wants_list:
                if isinstance(value, dict):
                    coerced[name] = list(value.values())
                elif isinstance(value, (str, int, float)) and not isinstance(value, bool):
                    coerced[name] = [value]
            elif wants_str:
                if isinstance(value, (list, tuple)):
                    coerced[name] = "; ".join(str(item) for item in value)
                elif isinstance(value, dict):
                    coerced[name] = json.dumps(value, ensure_ascii=False)
            elif (wants_float or wants_int) and isinstance(value, (dict, list, tuple, str)):
                number = OpenModel._extract_number(value)
                if number is not None:
                    coerced[name] = int(round(number)) if wants_int else number
                elif not field.is_required():
                    coerced[name] = field.get_default(call_default_factory=True)
        return coerced


class SourceDoc(OpenModel):
    source_id: str = ""
    provider: str = "unknown"
    title: str = "Untitled source"
    url: str = ""
    snippet: str = ""
    author: str = ""
    published_at: str = ""
    authority_score: float = 0.5
    relevance_score: float = 0.0
    final_score: float = 0.0
    metadata: dict[str, Any] = Field(default_factory=dict)


class Fact(OpenModel):
    fact_id: str = ""
    claim: str = ""
    source_ids: list[str] = Field(default_factory=list)
    confidence: float = 0.6
    numeric_values: list[Any] = Field(default_factory=list)
    comparison: Any = ""
    visual_hint: Any = ""

    @field_validator("source_ids", mode="before")
    @classmethod
    def normalize_source_ids(cls, value: Any) -> list[str]:
        if value is None:
            return []
        values = value if isinstance(value, (list, tuple, set)) else [value]
        result: list[str] = []
        for item in values:
            if isinstance(item, dict):
                item = item.get("source_id", item.get("id", item.get("value", "")))
            text = str(item).strip()
            if text and text not in result:
                result.append(text)
        return result

    @field_validator("numeric_values", mode="before")
    @classmethod
    def normalize_numeric_values(cls, value: Any) -> list[Any]:
        if value is None:
            return []
        return list(value) if isinstance(value, (list, tuple, set)) else [value]


class ResearchPack(OpenModel):
    topic: str
    summary: str = ""
    hooks: list[Any] = Field(default_factory=list)
    facts: list[Fact] = Field(default_factory=list)
    comparisons: list[Any] = Field(default_factory=list)
    visual_ideas: list[Any] = Field(default_factory=list)
    limitations: list[Any] = Field(default_factory=list)
    sources: list[SourceDoc] = Field(default_factory=list)
    validation_score: float = 0.0
    research_hash: str = ""
    raw_llm_output: Any = None


class Beat(OpenModel):
    beat_id: str = ""
    purpose: str = "information_gain"
    duration_s: float = 5.0
    spoken_line: str = ""
    visual_event: str = ""
    evidence_refs: list[str] = Field(default_factory=list)
    emotion: str = "curiosity"
    retention_function: str = "information_gain"
    emphasis_words: list[str] = Field(default_factory=list)
    pause_after_ms: int = 80
    sfx: str = "none"
    raw: dict[str, Any] = Field(default_factory=dict)

    @field_validator("duration_s", mode="before")
    @classmethod
    def duration_to_float(cls, value: Any) -> float:
        if isinstance(value, dict):
            value = value.get("seconds", value.get("value", 5.0))
        try:
            return max(0.25, float(value))
        except Exception:
            return 5.0


class ScriptPackage(OpenModel):
    topic: str
    title: str = ""
    hook: str = ""
    beats: list[Beat] = Field(default_factory=list)
    closing: str = ""
    total_words: int = 0
    estimated_duration_s: float = 0.0
    script_hash: str = ""
    raw_llm_output: Any = None


class SceneRequest(OpenModel):
    scene_id: str = ""
    beat_id: str = ""
    duration_s: float = 5.0
    narration: str = ""
    headline: str = ""
    visual_event: str = ""
    scientific_claim: str = ""
    time_stage: str = ""
    attention_goal: str = ""
    desired_change: str = ""
    transition: str = "cut"
    raw_llm_output: Any = None

    @field_validator("duration_s", mode="before")
    @classmethod
    def duration_to_float(cls, value: Any) -> float:
        if isinstance(value, dict):
            value = value.get("seconds", value.get("value", 5.0))
        try:
            return max(0.25, float(value))
        except Exception:
            return 5.0


class Storyboard(OpenModel):
    topic: str
    width: int = 1080
    height: int = 1920
    fps: int = 30
    scenes: list[SceneRequest] = Field(default_factory=list)
    estimated_duration_s: float = 0.0
    storyboard_hash: str = ""
    raw_llm_output: Any = None


class LineLanguage(OpenModel):
    silhouette_px: tuple[float, float] = (3.2, 4.4)
    structural_px: tuple[float, float] = (1.4, 2.7)
    texture_px: tuple[float, float] = (0.8, 1.5)
    behavior: list[str] = Field(default_factory=list)
    endings: str = "selectively tapered and occasionally broken"
    contour_rule: str = "contours follow observed form and overlap, never primitive geometry"


class ColorLanguage(OpenModel):
    paper: str = "#FAFAF7"
    ink: str = "#20282D"
    slate: str = "#475157"
    mist: str = "#EBEFF0"
    steel: str = "#A8BAC2"
    blue_primary: str = "#2E77A6"
    blue_secondary: str = "#4190C3"
    warning_red: str = "#D8483E"
    sun_yellow: str = "#F3BD38"
    maximum_dominant_hues: int = 4
    value_bands: int = 4
    rule: str = "restrained scientific palette; accents encode causal meaning"


class TypographyLanguage(OpenModel):
    family: str = "condensed grotesk sans serif"
    fallback_stack: list[str] = Field(
        default_factory=lambda: ["Arial Narrow", "Roboto Condensed", "Arial", "sans-serif"]
    )
    headline_case: str = "uppercase"
    headline_weight: int = 900
    label_weight: int = 700
    alignment: str = "left or optically centered according to composition"
    rule: str = "typography behaves like an experimental instrument panel, not a presentation template"


class CompositionLanguage(OpenModel):
    scene_first: bool = True
    depth_planes: tuple[int, int] = (3, 5)
    visual_mass_fraction: tuple[float, float] = (0.38, 0.62)
    asymmetry_required: bool = True
    integrated_environment_required: bool = True
    negative_space_rule: str = "breathing room is authored around the focal route, never leftover blank space"
    perspective_rule: str = "one coherent perspective system per shot"


class MotionLanguage(OpenModel):
    default_state: Literal["hold"] = "hold"
    camera_locked_by_default: bool = True
    maximum_primary_motion_groups: int = 1
    maximum_secondary_motion_groups: int = 2
    default_fade_allowed: bool = False
    default_zoom_allowed: bool = False
    decorative_motion_allowed: bool = False
    preferred_representations: list[str] = Field(
        default_factory=lambda: [
            "replacement_pose_sequence",
            "layered_texture_loop",
            "local_deformation",
            "semantic_layer_transform",
            "scientific_overlay",
        ]
    )


class HardCodedStyleCanon(OpenModel):
    canon_id: Literal["experiment-ledger-editorial-ink-v1"] = "experiment-ledger-editorial-ink-v1"
    display_name: str = "Experiment Ledger Editorial Ink"
    reference_origin: str = "user-supplied experiment-style scientific motion reference"
    medium: str = "authored digital editorial ink illustration with restrained flat material planes"
    audience_age: str = "adult general science audience"
    line: LineLanguage = Field(default_factory=LineLanguage)
    color: ColorLanguage = Field(default_factory=ColorLanguage)
    typography: TypographyLanguage = Field(default_factory=TypographyLanguage)
    composition: CompositionLanguage = Field(default_factory=CompositionLanguage)
    motion: MotionLanguage = Field(default_factory=MotionLanguage)
    recurring_ui: list[str] = Field(
        default_factory=lambda: [
            "experiment identifier panel",
            "simulation status panel",
            "single metric panel",
            "stage label",
        ]
    )
    visual_traits: list[str] = Field(
        default_factory=lambda: [
            "mature scientific editorial illustration",
            "fluid authored contours",
            "observational anatomy and material construction",
            "integrated causal environments",
            "paper-like light background",
            "charcoal and blue-gray structural palette",
            "sparse red warning and yellow energy accents",
            "condensed uppercase scientific labels",
            "controlled texture and line imperfections",
        ]
    )
    forbidden: list[str] = Field(
        default_factory=lambda: [
            "child mascot anatomy",
            "oversized round head",
            "capsule limbs",
            "mitten hands",
            "rounded-rectangle buildings as hero art",
            "Microsoft Word shape assembly",
            "Paint-like polygon collage",
            "isolated icon sticker collection",
            "uniform black sticker outline",
            "random micro-details",
            "generic AI glossy rendering",
            "different art style per shot",
            "procedural SVG hero illustration",
            "prompt-only style reset",
            "PowerPoint entrance motion",
        ]
    )
    style_hash: str = ""

    @model_validator(mode="after")
    def lock_identity(self):
        # The LLM may enrich topic-specific direction elsewhere, but it cannot
        # rename or mutate the studio's hard-coded visual identity.
        self.canon_id = "experiment-ledger-editorial-ink-v1"
        self.display_name = "Experiment Ledger Editorial Ink"
        return self


class ArtDirectionBible(OpenModel):
    schema_version: str = "10.0"
    topic: str
    canon_id: str = "experiment-ledger-editorial-ink-v1"
    locked_canon: HardCodedStyleCanon = Field(default_factory=HardCodedStyleCanon)
    visual_thesis: str = ""
    topic_specific_motifs: list[str] = Field(default_factory=list)
    recurring_symbols: list[str] = Field(default_factory=list)
    scientific_readability_rules: list[str] = Field(default_factory=list)
    anti_ai_rules: list[str] = Field(default_factory=list)
    continuity_priorities: list[str] = Field(default_factory=list)
    reference_board_path: str = ""
    raw_llm_output: Any = None

    @model_validator(mode="after")
    def enforce_locked_canon(self):
        self.canon_id = "experiment-ledger-editorial-ink-v1"
        self.locked_canon = HardCodedStyleCanon()
        return self


class CanonAnchor(OpenModel):
    anchor_id: str
    role: Literal[
        "reference_video_board", "master_style_anchor", "approved_scene", "subject_sheet", "environment_sheet"
    ]
    path: str
    scene_id: str = ""
    notes: str = ""
    approved: bool = True


class RecurringSubjectCanon(OpenModel):
    subject_id: str
    description: str
    immutable_traits: list[str] = Field(default_factory=list)
    allowed_variations: list[str] = Field(default_factory=list)
    reference_anchor_ids: list[str] = Field(default_factory=list)


class ContinuityCanon(OpenModel):
    schema_version: str = "10.0"
    canon_id: str = "experiment-ledger-editorial-ink-v1"
    palette_lock: dict[str, Any] = Field(default_factory=dict)
    line_lock: dict[str, Any] = Field(default_factory=dict)
    typography_lock: dict[str, Any] = Field(default_factory=dict)
    ui_lock: list[str] = Field(default_factory=list)
    recurring_subjects: list[RecurringSubjectCanon] = Field(default_factory=list)
    anchors: list[CanonAnchor] = Field(default_factory=list)
    continuity_rules: list[str] = Field(default_factory=list)
    prohibited_drift: list[str] = Field(default_factory=list)


class DepthPlane(OpenModel):
    plane_id: str
    depth: Literal["foreground", "midground", "background", "atmosphere", "overlay"]
    contents: str
    value_range: str = ""
    line_weight_role: str = ""
    movement_role: str = "hold"


class PerspectivePlan(OpenModel):
    camera_height: str = "eye level"
    view: str = "three-quarter editorial view"
    lens_language: str = "moderate perspective without photographic distortion"
    horizon_y: float = 0.50
    vanishing_points: list[tuple[float, float]] = Field(default_factory=list)
    scale_logic: str = "consistent within scene"


class FigureConstruction(OpenModel):
    figure_id: str
    role: str = "human subject"
    proportion_heads: float = 7.2
    body_orientation: str = "three-quarter"
    weight_distribution: str = "credible weight-bearing stance"
    gesture_line: str = ""
    joint_logic: list[str] = Field(default_factory=list)
    hand_construction: str = "palm mass, knuckle plane, finger direction and thumb lock"
    clothing_logic: str = "folds follow tension, gravity and joint compression"
    forbidden_shortcuts: list[str] = Field(
        default_factory=lambda: [
            "circle-joint puppet",
            "tube limbs",
            "oval fist",
            "front-facing torso with side-facing action",
        ]
    )


class MaterialMarkPlan(OpenModel):
    material: str
    visual_cues: list[str] = Field(default_factory=list)
    line_marks: list[str] = Field(default_factory=list)
    value_behavior: str = ""
    motion_behavior: str = ""


class MotionSeam(OpenModel):
    seam_id: str
    subject: str
    method: Literal[
        "replacement_pose", "semantic_mask", "layer_transform", "local_deformation", "texture_loop", "overlay_only"
    ]
    region: str
    resting_overlap_rule: str
    required_variants: list[str] = Field(default_factory=list)


class SceneIllustrationArchitecture(OpenModel):
    schema_version: str = "10.0"
    scene_id: str
    beat_id: str = ""
    narrative_claim: str = ""
    visual_thesis: str = ""
    canvas: tuple[int, int] = (1080, 1920)
    composition_route: list[str] = Field(default_factory=list)
    perspective: PerspectivePlan = Field(default_factory=PerspectivePlan)
    depth_planes: list[DepthPlane] = Field(default_factory=list)
    focal_subject: str = ""
    secondary_subjects: list[str] = Field(default_factory=list)
    figure_construction: list[FigureConstruction] = Field(default_factory=list)
    material_marks: list[MaterialMarkPlan] = Field(default_factory=list)
    negative_space: str = ""
    contour_architecture: list[str] = Field(default_factory=list)
    color_script: list[str] = Field(default_factory=list)
    scientific_annotations: list[str] = Field(default_factory=list)
    motion_seams: list[MotionSeam] = Field(default_factory=list)
    animation_representation: list[str] = Field(default_factory=list)
    required_pose_variants: list[str] = Field(default_factory=list)
    director_notes: list[str] = Field(default_factory=list)
    prohibited_visual_shortcuts: list[str] = Field(default_factory=list)
    beauty_frame_first: Literal[True] = True
    semantic_split_after_approval: Literal[True] = True
    procedural_hero_allowed: Literal[False] = False
    raw_llm_output: Any = None


class ReferenceRequirement(OpenModel):
    reference_id: str
    purpose: Literal["style", "pose", "anatomy", "environment", "material", "lighting", "continuity"]
    query: str
    priority: int = 1
    use_rule: str = "observe structure only; do not copy composition"
    source_preference: list[str] = Field(
        default_factory=lambda: ["user reference", "licensed stock", "generated reference"]
    )


class ReferencePack(OpenModel):
    scene_id: str
    style_anchor_ids: list[str] = Field(default_factory=list)
    requirements: list[ReferenceRequirement] = Field(default_factory=list)
    board_path: str = ""
    previous_approved_scene: str = ""
    subject_anchor_paths: list[str] = Field(default_factory=list)
    environment_anchor_paths: list[str] = Field(default_factory=list)


class FluxStyleFingerprint(OpenModel):
    schema_version: str = "10.0"
    canon_id: str = "experiment-ledger-editorial-ink-v1"
    medium_sentence: str = ""
    contour_sentence: str = ""
    anatomy_sentence: str = ""
    palette_sentence: str = ""
    composition_sentence: str = ""
    texture_sentence: str = ""
    anti_ai_sentence: str = ""
    immutable_prompt: str = ""
    fingerprint_hash: str = ""


class FluxPromptBlock(OpenModel):
    block_id: str
    role: Literal[
        "image_type",
        "subject_action",
        "environment",
        "composition",
        "style",
        "anatomy_material",
        "lighting_color",
        "continuity",
        "edit_scope",
        "animation_preparation",
        "negative_constraints",
    ]
    text: str
    immutable: bool = False
    priority: int = 1


class FluxPromptStack(OpenModel):
    schema_version: str = "10.0"
    purpose: Literal["master_anchor", "beauty_frame", "revision", "pose_variant", "layer_isolation"]
    blocks: list[FluxPromptBlock] = Field(default_factory=list)
    compiled_prompt: str = ""
    immutable_style_hash: str = ""
    scene_delta_hash: str = ""
    word_count: int = 0
    init_strategy: Literal[
        "reference_board", "master_style_anchor", "previous_approved_scene", "approved_beauty_frame", "current_revision"
    ] = "master_style_anchor"


class FluxPromptDiagnostics(OpenModel):
    valid: bool = True
    errors: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    word_count: int = 0
    immutable_style_hash: str = ""
    has_explicit_subject: bool = False
    has_explicit_preservation: bool = False
    has_local_edit_scope: bool = False
    vague_language_found: list[str] = Field(default_factory=list)
    contradictory_language_found: list[str] = Field(default_factory=list)


class FluxGenerationPolicy(OpenModel):
    schema_version: str = "10.0"
    model: str = "flux-kontext-pro"
    aspect_ratio: str = "9:16"
    prompt_upsampling: bool = False
    safety_tolerance: int = 2
    output_format: Literal["png", "jpeg"] = "png"
    seed_base: int = 240921
    max_initial_prompt_words: int = 380
    min_initial_prompt_words: int = 80
    max_edit_prompt_words: int = 180
    max_adjustments_per_pass: int = 2
    one_reference_only: Literal[True] = True


class FluxRevisionPass(OpenModel):
    pass_id: str
    target_regions: list[str] = Field(default_factory=list)
    adjustments: list[ConcreteAdjustment] = Field(default_factory=list)
    instruction: str = ""
    preserve: list[str] = Field(default_factory=list)
    priority: Literal["critical", "high", "medium", "low"] = "high"


class FluxConsistencyState(OpenModel):
    schema_version: str = "10.0"
    canon_id: str = "experiment-ledger-editorial-ink-v1"
    immutable_style_hash: str = ""
    master_anchor_path: str = ""
    previous_approved_scene: str = ""
    seed_family_base: int = 240921
    approved_scene_paths: list[str] = Field(default_factory=list)
    recurring_subject_descriptors: list[str] = Field(default_factory=list)
    drift_watchlist: list[str] = Field(default_factory=list)


class DrawingBrief(OpenModel):
    schema_version: str = "10.0"
    brief_id: str
    scene_id: str
    canon_id: str = "experiment-ledger-editorial-ink-v1"
    purpose: Literal["master_anchor", "beauty_frame", "revision", "pose_variant", "layer_isolation"] = "beauty_frame"
    positive_prompt: str
    negative_prompt: str
    kontext_instruction: str
    anchor_board_path: str = ""
    init_image_path: str = ""
    output_path: str = ""
    aspect_ratio: str = "9:16"
    seed: int | None = None
    preserve: list[str] = Field(default_factory=list)
    change: list[str] = Field(default_factory=list)
    semantic_requirements: list[str] = Field(default_factory=list)
    motion_requirements: list[str] = Field(default_factory=list)
    prompt_stack: FluxPromptStack | None = None
    compiled_prompt: str = ""
    prompt_diagnostics: FluxPromptDiagnostics | None = None
    style_fingerprint_hash: str = ""
    init_strategy: str = "master_style_anchor"
    prompt_upsampling: bool = False
    safety_tolerance: int = 2
    output_format: Literal["png", "jpeg"] = "png"
    request_metadata: dict[str, Any] = Field(default_factory=dict)

    @model_validator(mode="after")
    def enforce_non_procedural(self):
        joined = (self.positive_prompt + " " + self.kontext_instruction).lower()
        if "procedural svg hero" in joined or "assemble from primitive" in joined:
            raise ValueError("DrawingBrief may not request procedural hero art")
        return self


class ConcreteAdjustment(OpenModel):
    adjustment_id: str
    target_region: str
    problem: str
    instruction: str
    preserve: list[str] = Field(default_factory=list)
    priority: Literal["critical", "high", "medium", "low"] = "high"


class DirectorChangeOrder(OpenModel):
    schema_version: str = "10.0"
    scene_id: str
    revision_number: int = 1
    status: Literal["approve", "revise", "requires_human_or_vision_director"] = "revise"
    diagnosis: str = ""
    adjustments: list[ConcreteAdjustment] = Field(default_factory=list)
    continuity_corrections: list[str] = Field(default_factory=list)
    animation_readiness_corrections: list[str] = Field(default_factory=list)
    immutable_preserve_list: list[str] = Field(default_factory=list)
    revised_kontext_instruction: str = ""

    @model_validator(mode="after")
    def concrete_when_revising(self):
        if self.status == "revise" and not self.adjustments:
            raise ValueError("A revision order must contain concrete adjustments")
        return self


class RevisionRecord(OpenModel):
    scene_id: str
    revision_number: int
    draft_path: str
    change_order: DirectorChangeOrder
    output_path: str = ""
    created_at: str = Field(default_factory=lambda: datetime.now(UTC).isoformat())


class BeautyFrame(OpenModel):
    scene_id: str
    image_path: str
    approved: bool = False
    approval_source: str = ""
    style_anchor_ids: list[str] = Field(default_factory=list)
    revision_records: list[RevisionRecord] = Field(default_factory=list)
    prompt_hash: str = ""
    provider: str = "flux-kontext-pro"
    seed: int | None = None


class SemanticLayer(OpenModel):
    layer_id: str
    description: str
    source_region: str
    extraction_method: Literal["external_mask", "kontext_isolation", "artist_layer", "full_frame", "scientific_overlay"]
    z_index: int = 0
    locked: bool = True
    mask_path: str = ""
    image_path: str = ""
    pose_variant_paths: list[str] = Field(default_factory=list)


class SemanticLayerContract(OpenModel):
    scene_id: str
    beauty_frame_path: str
    layers: list[SemanticLayer] = Field(default_factory=list)
    separation_occurs_after_approval: Literal[True] = True
    preserve_original_beauty: Literal[True] = True
    extraction_notes: list[str] = Field(default_factory=list)


class MotionEvent(OpenModel):
    event_id: str
    reason_id: str
    target_layer: str
    representation: Literal[
        "hold",
        "translate",
        "rotate",
        "scale",
        "opacity",
        "replacement_pose",
        "texture_loop",
        "local_deformation",
        "mask_reveal",
        "overlay_draw",
    ]
    start_frame: int
    end_frame: int
    easing: str = "linear"
    parameters: dict[str, Any] = Field(default_factory=dict)
    secondary: bool = False

    @model_validator(mode="after")
    def validate_event(self):
        if not self.reason_id.strip():
            raise ValueError("MotionEvent requires reason_id")
        if self.end_frame <= self.start_frame:
            raise ValueError("MotionEvent end_frame must be after start_frame")
        return self


class AnimationPlan(OpenModel):
    scene_id: str
    fps: int = 30
    duration_frames: int
    camera_locked: bool = True
    events: list[MotionEvent] = Field(default_factory=list)
    audio_sync: dict[str, Any] = Field(default_factory=dict)
    hold_regions: list[str] = Field(default_factory=list)
    maximum_simultaneous_groups: int = 3


class HybridLayer(OpenModel):
    layer_id: str
    kind: Literal["raster", "svg_overlay", "pose_sequence", "mask", "video_clip"]
    path: str
    z_index: int = 0
    bbox: tuple[float, float, float, float] = (0.0, 0.0, 1.0, 1.0)
    opacity: float = 1.0
    blend_mode: str = "normal"
    mask_path: str = ""
    pose_paths: list[str] = Field(default_factory=list)


class HybridScenePackage(OpenModel):
    scene_id: str
    duration_frames: int
    fps: int = 30
    canvas: tuple[int, int] = (1080, 1920)
    narration: str = ""
    headline: str = ""
    background: str = "#FAFAF7"
    layers: list[HybridLayer] = Field(default_factory=list)
    animation: AnimationPlan
    voice_path: str = ""
    transition: str = "cut"
    temporal_clip_path: str = ""
    temporal_backend: str = ""


class PipelineResult(OpenModel):
    schema_version: str = "10.0"
    topic: str
    mode: Literal["plan_only", "live_generation", "rendered"]
    run_dir: str
    reference_board: str = ""
    art_direction_bible: str = ""
    continuity_canon: str = ""
    architectures: str = ""
    drawing_briefs: str = ""
    beauty_frames: str = ""
    semantic_contracts: str = ""
    animation_plans: str = ""
    video: str = ""
    warnings: list[str] = Field(default_factory=list)


def format_numeric_value(value: Any) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return f"{value:g}" if isinstance(value, float) else str(value)
    if isinstance(value, dict):
        amount = value.get("value", value.get("amount", value.get("number", "")))
        unit = value.get("unit", value.get("units", ""))
        context = value.get("context", value.get("label", value.get("description", "")))
        main = " ".join(str(x).strip() for x in (amount, unit) if str(x).strip())
        return f"{main} ({context})" if main and str(context).strip() else main or json.dumps(value, ensure_ascii=False)
    return str(value)


# ---------------------------------------------------------------------------
# V10 production runtime, continuity retrieval, candidate tournament and
# temporal-control contracts.
# ---------------------------------------------------------------------------


class ProviderCapability(OpenModel):
    name: str
    available: bool = True
    supports_streaming: bool = False
    supports_images: bool = False
    supports_video: bool = False
    supports_audio: bool = False
    metadata: dict[str, Any] = Field(default_factory=dict)


class ProviderSpec(OpenModel):
    provider_id: str
    provider_type: Literal[
        "llm",
        "vision",
        "image",
        "tts",
        "transcription",
        "segmentation",
        "temporal_video",
        "stock",
        "publisher",
        "renderer",
    ]
    implementation: str
    priority: int = 100
    enabled: bool = True
    capabilities: list[ProviderCapability] = Field(default_factory=list)
    config: dict[str, Any] = Field(default_factory=dict)


class StageRecord(OpenModel):
    stage_id: str
    status: Literal["pending", "running", "completed", "failed", "skipped", "interrupted", "stale"] = "pending"
    attempt: int = 0
    input_hash: str = ""
    output_path: str = ""
    output_sha256: str = ""
    started_at: str = ""
    completed_at: str = ""
    error_type: str = ""
    error_message: str = ""
    metadata: dict[str, Any] = Field(default_factory=dict)


class JobManifest(OpenModel):
    schema_version: str = "10.1"
    job_id: str
    topic: str
    run_dir: str
    status: Literal["created", "running", "completed", "failed", "cancelled"] = "created"
    config_hash: str = ""
    created_at: str = Field(default_factory=lambda: datetime.now(UTC).isoformat())
    updated_at: str = Field(default_factory=lambda: datetime.now(UTC).isoformat())
    stages: dict[str, StageRecord] = Field(default_factory=dict)
    warnings: list[str] = Field(default_factory=list)


class AssetRecord(OpenModel):
    asset_id: str
    path: str
    asset_type: Literal[
        "style_anchor",
        "approved_scene",
        "subject_view",
        "environment",
        "material",
        "pose",
        "beauty_frame",
        "control_sketch",
        "mask",
        "video_clip",
    ]
    approved: bool = True
    scene_id: str = ""
    subject_ids: list[str] = Field(default_factory=list)
    environment_id: str = ""
    camera_view: str = ""
    perspective: str = ""
    chronology_index: int = -1
    role_tags: list[str] = Field(default_factory=list)
    style_hash: str = ""
    identity_hash: str = ""
    composition_hash: str = ""
    quality_score: float = 0.0
    created_at: str = Field(default_factory=lambda: datetime.now(UTC).isoformat())
    metadata: dict[str, Any] = Field(default_factory=dict)


class AssetQuery(OpenModel):
    scene_id: str
    subject_ids: list[str] = Field(default_factory=list)
    environment_id: str = ""
    camera_view: str = ""
    perspective: str = ""
    desired_types: list[str] = Field(default_factory=list)
    required_roles: list[str] = Field(default_factory=list)
    style_hash: str = ""
    chronology_index: int = -1
    maximum_results: int = 8


class RankedAsset(OpenModel):
    asset: AssetRecord
    score: float
    reasons: list[str] = Field(default_factory=list)


class ReferenceSelection(OpenModel):
    scene_id: str
    query: AssetQuery
    selected: list[RankedAsset] = Field(default_factory=list)
    board_path: str = ""
    selection_notes: list[str] = Field(default_factory=list)


class CharacterView(OpenModel):
    view_id: str
    angle: Literal["front", "three_quarter", "side", "rear", "expression", "action", "detail"]
    path: str
    approved: bool = True
    notes: str = ""


class CharacterProfile(OpenModel):
    subject_id: str
    display_name: str
    immutable_traits: list[str] = Field(default_factory=list)
    proportion_rules: list[str] = Field(default_factory=list)
    wardrobe_rules: list[str] = Field(default_factory=list)
    palette_roles: dict[str, str] = Field(default_factory=dict)
    views: list[CharacterView] = Field(default_factory=list)
    style_hash: str = ""


class ShotState(OpenModel):
    scene_id: str
    first_frame_description: str
    last_frame_description: str
    invariant_elements: list[str] = Field(default_factory=list)
    changed_elements: list[str] = Field(default_factory=list)
    motion_bridge: list[str] = Field(default_factory=list)
    camera_motion: str = "locked"
    preserve_regions: list[str] = Field(default_factory=list)
    control_sketch_required: bool = False
    control_sketch_regions: list[str] = Field(default_factory=list)
    temporal_complexity: Literal["hold", "simple", "articulated", "deformation", "organic"] = "simple"


class CandidateFrame(OpenModel):
    candidate_id: str
    scene_id: str
    image_path: str
    seed: int | None = None
    prompt_hash: str = ""
    provider: str = "flux-kontext-pro"
    metrics: dict[str, float] = Field(default_factory=dict)


class CandidateScore(OpenModel):
    candidate_id: str
    total_score: float
    style_consistency: float = 0.0
    subject_consistency: float = 0.0
    composition_fitness: float = 0.0
    motion_readiness: float = 0.0
    causal_clarity: float = 0.0
    penalties: list[str] = Field(default_factory=list)
    rationale: str = ""


class CandidateTournamentResult(OpenModel):
    scene_id: str
    candidates: list[CandidateFrame] = Field(default_factory=list)
    scores: list[CandidateScore] = Field(default_factory=list)
    winner_id: str = ""
    winner_path: str = ""
    comparison_board: str = ""
    ranking_source: str = ""


class ControlSketchSpec(OpenModel):
    sketch_id: str
    scene_id: str
    frame_role: Literal["first", "last", "intermediate"]
    output_path: str
    regions: list[str] = Field(default_factory=list)
    line_color: str = "#FFFFFF"
    background_color: str = "#000000"
    instructions: list[str] = Field(default_factory=list)


class TemporalRequest(OpenModel):
    scene_id: str
    backend_preference: list[str] = Field(default_factory=list)
    beauty_start: str
    beauty_end: str = ""
    control_sketch_start: str = ""
    control_sketch_end: str = ""
    preserve_masks: list[str] = Field(default_factory=list)
    motion_prompt: str = ""
    duration_frames: int = 1
    fps: int = 30
    complexity: Literal["hold", "simple", "articulated", "deformation", "organic"] = "simple"
    output_path: str = ""


class TemporalResult(OpenModel):
    scene_id: str
    backend_id: str
    output_path: str
    success: bool = True
    deterministic: bool = False
    metadata: dict[str, Any] = Field(default_factory=dict)
    warnings: list[str] = Field(default_factory=list)


In [ ]:
%%writefile scistudio_v10/script_critic.py
"""Retention-first script quality gate."""

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import ResearchPack, ScriptPackage
from .utils import ensure_dir, save_json


STOPWORDS = {
    "the",
    "a",
    "an",
    "and",
    "or",
    "but",
    "if",
    "then",
    "of",
    "to",
    "in",
    "on",
    "at",
    "for",
    "is",
    "are",
    "was",
    "were",
    "it",
    "its",
    "this",
    "that",
    "these",
    "those",
    "as",
    "by",
    "with",
    "from",
    "into",
    "would",
    "will",
    "can",
    "could",
    "so",
    "you",
    "your",
    "we",
    "our",
    "they",
    "their",
    "he",
    "she",
    "his",
    "her",
    "not",
    "no",
    "yes",
    "here",
    "there",
    "what",
}


class ScriptCritic:
    """Retention-first script quality gate — the narration counterpart to the
    visual critic. It is deterministic first (hook, information gain, pacing,
    curiosity, payoff), then optionally sharpened by an LLM. Weak scripts are
    rewritten instead of silently narrated.
    """

    SYSTEM = """You are a ruthless retention editor for science shorts (Kurzgesagt-level).
You judge whether a script hooks in the first 3 seconds, never stalls, escalates stakes,
and pays off. You return strict JSON only and propose concrete beat-level rewrites."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    # -- deterministic metrics ------------------------------------------------
    @staticmethod
    def _content_words(text: str) -> list[str]:
        return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS and len(w) > 2]

    def _metrics(self, script: ScriptPackage) -> dict[str, float]:
        beats = script.beats or []
        lines = [b.spoken_line for b in beats]
        first = lines[0] if lines else ""
        # Hook: question, number, or tension word in the first beat, and not too long.
        hook_signals = bool(re.search(r"\?|\d", first)) or bool(
            re.search(r"\b(imagine|what|suddenly|never|stop|vanish|collapse|instant|impossible)\b", first.lower())
        )
        hook_len_ok = 3 <= len(first.split()) <= 22
        hook = float(0.6 * hook_signals + 0.4 * hook_len_ok)
        # Information gain: fraction of beats that introduce new content words.
        seen: set[str] = set()
        novel = 0
        for line in lines:
            words = set(self._content_words(line))
            if len(words - seen) >= max(2, int(0.4 * max(1, len(words)))):
                novel += 1
            seen |= words
        info_gain = novel / max(1, len(lines))
        # Pacing: total duration within target window and no over-long single beat.
        lo = float(self.config.get("target_duration_min", 40.0))
        hi = float(self.config.get("target_duration_max", 60.0))
        dur = script.estimated_duration_s or sum(b.duration_s for b in beats)
        pacing = 1.0 if lo <= dur <= hi else max(0.0, 1.0 - abs(dur - (lo + hi) / 2) / (hi))
        longest = max((len(line.split()) for line in lines), default=0)
        no_stall = 1.0 if longest <= int(self.config.get("max_beat_words", 26)) else 0.6
        # Escalation: later beats raise stakes via consequence language OR a
        # time-stage progression (instant -> seconds -> minutes -> hours -> days),
        # which is itself escalation for a what-if chain.
        esc_terms = (
            "then",
            "next",
            "worse",
            "more",
            "every",
            "entire",
            "cascade",
            "chain",
            "finally",
            "result",
            "faster",
            "spread",
        )
        time_terms = ("instant", "moment", "second", "minute", "hour", "day", "week", "month", "year")
        second_half = lines[len(lines) // 2 :]
        esc_hits = sum(
            any(t in line.lower() for t in esc_terms) or any(t in line.lower() for t in time_terms)
            for line in second_half
        )
        escalation = min(1.0, esc_hits / max(1, len(second_half)) + 0.2)
        # Payoff: closing resolves / states the lesson.
        closing = (script.closing or (lines[-1] if lines else "")).lower()
        payoff = float(
            bool(re.search(r"\b(lesson|because|connected|why|matters|truth|reason|result|is that)\b", closing))
        )
        beat_count_ok = 1.0 if 6 <= len(beats) <= 11 else 0.6

        # Relevance: the opening must actually establish the topic's subject (so
        # the video is about something), without forcing every line to repeat the
        # subject noun. Drift is measured separately by `coherence` below.
        subject_words = set(self._content_words(script.topic))
        head_words = set(self._content_words(" ".join([script.hook or ""] + lines[:2])))
        relevance = 1.0 if (not subject_words or (subject_words & head_words)) else 0.4

        # Coherence: penalise beats that read like pasted paper abstracts or drift
        # to unrelated astronomy. This is what catches the "narasi ke planet lain".
        OFF_TOPIC = (
            "we present",
            "simulations of",
            "orbital period",
            "spin axis",
            "exoplanet",
            "super-earth",
            "super earth",
            "quasi-satellite",
            "quasi satellite",
            "aquaplanet",
            "tidally locked",
            "near-earth",
            "solar system bodies",
            "have found a special place",
            "in this paper",
            "we investigate",
            "we study",
            "this study",
            "et al",
        )
        offtopic_beats = sum(1 for line in lines if any(marker in line.lower() for marker in OFF_TOPIC))
        coherence = 1.0 - offtopic_beats / max(1, len(lines))

        return {
            "hook_strength": round(hook, 3),
            "relevance": round(relevance, 3),
            "coherence": round(coherence, 3),
            "information_gain": round(info_gain, 3),
            "pacing": round(pacing, 3),
            "no_stall": round(no_stall, 3),
            "escalation": round(escalation, 3),
            "payoff": round(payoff, 3),
            "structure": round(beat_count_ok, 3),
            "estimated_duration_s": round(dur, 2),
            "beat_count": len(beats),
        }

    # -- review ---------------------------------------------------------------
    def review(self, script: ScriptPackage, research: ResearchPack, force: bool = False) -> dict[str, Any]:
        metrics = self._metrics(script)
        keys = [
            "hook_strength",
            "relevance",
            "coherence",
            "information_gain",
            "pacing",
            "no_stall",
            "escalation",
            "payoff",
            "structure",
        ]
        structural_min = min(metrics[k] for k in keys)
        threshold = float(self.config.get("acceptance_threshold", 0.6))
        vision_or_text_llm = self.llm.available("gemini") or self.llm.available("openai") or self.llm.available("local")

        report: dict[str, Any] = {
            "metrics": metrics,
            "scores": {k: metrics[k] for k in keys},
            "failures": [],
            "repair_plan": [],
        }
        for k in keys:
            if metrics[k] < threshold:
                report["failures"].append(f"{k}={metrics[k]:.2f} < {threshold:.2f}")

        if not vision_or_text_llm:
            report["passed"] = structural_min >= threshold
            report["mode"] = "structural-only (no LLM)"
            save_json(self.root / f"{script.script_hash or 'script'}-quality.json", report)
            return report

        prompt = f"""Score this science-short script for retention and propose rewrites.
Deterministic metrics already computed: {json.dumps(metrics, ensure_ascii=False)}
Script: {json.dumps(script.model_dump(mode="json", exclude={"raw_llm_output"}), ensure_ascii=False)}

Return JSON: {{"passed": bool, "scores": {{"hook_strength":0-1,"information_gain":0-1,"curiosity":0-1,
"escalation":0-1,"payoff":0-1,"clarity":0-1,"naturalness":0-1}}, "failures": [..],
"repair_plan": [{{"beat_id":"B0x","rewrite":"stronger spoken line"}}]}}
Be strict: a first line that does not create an open loop in 3 seconds fails hook_strength."""
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=prompt,
            namespace="script_critic",
            fallback={
                "passed": structural_min >= threshold,
                "scores": report["scores"],
                "failures": report["failures"],
                "repair_plan": [],
            },
            force=force,
        )
        if not isinstance(raw, dict):
            raw = {
                "passed": structural_min >= threshold,
                "scores": report["scores"],
                "failures": report["failures"],
                "repair_plan": [],
            }
        raw.setdefault("metrics", metrics)
        scores = {**{k: metrics[k] for k in keys}, **(raw.get("scores") or {})}
        llm_min = min([float(v) for v in scores.values()] or [0.0])
        raw["passed"] = bool(raw.get("passed", True) and structural_min >= threshold and llm_min >= threshold)
        raw["scores"] = scores
        save_json(self.root / f"{script.script_hash or 'script'}-quality.json", raw)
        return raw

    # -- refine ---------------------------------------------------------------
    def refine(
        self, script: ScriptPackage, research: ResearchPack, director, force: bool = False
    ) -> tuple[ScriptPackage, dict[str, Any]]:
        max_rounds = int(self.config.get("max_rounds", 2))
        strict = bool(self.config.get("strict", False))
        report = self.review(script, research, force=force)
        for _ in range(max_rounds):
            if report.get("passed"):
                break
            repairs = report.get("repair_plan") or report.get("failures")
            if self.llm.available("gemini") or self.llm.available("openai") or self.llm.available("local"):
                prompt = f"""Rewrite this script to fix these problems while keeping the facts and evidence refs.
Problems: {json.dumps(repairs, ensure_ascii=False)}
Current script: {json.dumps(script.model_dump(mode="json", exclude={"raw_llm_output"}), ensure_ascii=False)}
Return the same JSON shape (topic, title, hook, beats[], closing). Keep 7-10 beats, {self.config.get("target_duration_min", 40)}-{self.config.get("target_duration_max", 60)}s."""
                raw = self.llm.generate_json(
                    system=director.SCRIPT_SYSTEM,
                    prompt=prompt,
                    namespace="script_refine",
                    fallback=script.model_dump(mode="json"),
                    force=True,
                )
                script = director._normalize_script(raw, script, script.topic)
            else:
                break
            report = self.review(script, research, force=True)
        if strict and not report.get("passed"):
            report["note"] = "script did not reach retention threshold after refinement"
        return script, report


In [ ]:
%%writefile scistudio_v10/security.py
"""Secret redaction utilities.

Redaction happens in three layers:

1. **Exact-value scrubbing** — every secret the process actually loaded is
   registered with :func:`register_secret`; any occurrence of the exact value
   in any string is replaced. This catches secrets embedded in URLs,
   tracebacks and provider error bodies regardless of format.
2. **Pattern scrubbing** — realistic secret shapes (OpenAI ``sk-`` keys,
   bearer tokens, ``x-key`` headers, UUID-style keys after a key-like word,
   long random tokens after ``key/token/secret/password`` assignments).
3. **Key-name scrubbing** — dictionary entries whose key names look
   secret-bearing are replaced wholesale.

``redact_secrets`` is safe to call on nested dict/list structures.
"""

from __future__ import annotations

import re
import threading
from typing import Any

REDACTED = "[REDACTED]"

_SECRET_KEY_MARKERS = ("key", "token", "secret", "password", "credential", "authorization")

_SECRET_PATTERNS = [
    # OpenAI-style keys (sk-..., sk-proj-...): long, no spaces.
    re.compile(r"\bsk-[A-Za-z0-9_-]{16,}\b"),
    # Bearer tokens.
    re.compile(r"(?i)\bbearer\s+[A-Za-z0-9._~+/=-]{8,}"),
    # x-key / api-key style headers: `x-key: value` or `"x-key": "value"`.
    re.compile(r"(?i)(['\"]?x-?key['\"]?\s*[:=]\s*['\"]?)([^\s,'\"}]+)"),
    # key/token/secret/password assignments followed by the value.
    re.compile(
        r"(?i)\b((?:api[_-]?key|access[_-]?key|secret[_-]?key|auth[_-]?token|refresh[_-]?token|token|secret|password|passwd|authorization)\s*[:=]\s*['\"]?)([^\s,'\"}]{6,})"
    ),
    # UUID-shaped values directly after a key-like word (BFL keys are UUIDs).
    re.compile(r"(?i)\b(key\S{0,12}\s*[:=]\s*['\"]?)([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})"),
]

_registry_lock = threading.Lock()
_registered_secrets: set[str] = set()


def register_secret(value: str | None) -> None:
    """Register a live secret value for exact-match scrubbing everywhere."""
    if value and isinstance(value, str) and len(value) >= 6:
        with _registry_lock:
            _registered_secrets.add(value)


def clear_registered_secrets() -> None:
    """Testing hook: forget all registered secret values."""
    with _registry_lock:
        _registered_secrets.clear()


def _scrub_text(text: str) -> str:
    with _registry_lock:
        known = list(_registered_secrets)
    for secret in known:
        if secret in text:
            text = text.replace(secret, REDACTED)

    def _sub(match: re.Match) -> str:
        if match.lastindex and match.lastindex >= 2:
            return match.group(1) + REDACTED
        return REDACTED

    for pattern in _SECRET_PATTERNS:
        text = pattern.sub(_sub, text)
    return text


def redact_secrets(value: Any) -> Any:
    """Recursively redact secrets from strings, dicts and lists.

    Secret-named dictionary keys are redacted wholesale only when their value
    is a string — numeric metadata such as ``input_tokens``/``total_tokens``
    usage counts is never a credential and must survive for observability.
    """
    if isinstance(value, dict):
        return {
            key: (
                REDACTED
                if isinstance(item, str) and any(marker in str(key).lower() for marker in _SECRET_KEY_MARKERS)
                else redact_secrets(item)
            )
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [redact_secrets(item) for item in value]
    if isinstance(value, str):
        return _scrub_text(value)
    return value


def redacted_exception_text(exc: BaseException, limit: int = 2000) -> str:
    """A single-line, redacted, length-capped rendering of an exception."""
    text = f"{type(exc).__name__}: {exc}"
    return _scrub_text(text)[:limit]


In [ ]:
%%writefile scistudio_v10/semantic.py
from __future__ import annotations

from pathlib import Path
from typing import Any, Callable

from PIL import Image

from .llm import LLMRouter
from .schemas import (
    BeautyFrame,
    SceneIllustrationArchitecture,
    SemanticLayer,
    SemanticLayerContract,
)
from .utils import ensure_dir, save_json


class SemanticSeparationPlanner:
    """Plans the minimum semantic separation after beauty approval.

    The approved beauty image remains the source of visible pixels. Generated or
    external segmentation is used primarily to make masks, so animation control
    does not redraw the entire artwork or degrade its style.
    """

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)

    def plan(
        self,
        architecture: SceneIllustrationArchitecture,
        beauty: BeautyFrame,
        pose_variants: dict[str, str] | None = None,
    ) -> SemanticLayerContract:
        if not beauty.approved and self.config.get("require_approved_beauty", True):
            raise RuntimeError("Semantic separation must occur after explicit beauty-frame approval")
        pose_variants = pose_variants or {}
        layers = [
            SemanticLayer(
                layer_id="beauty-base",
                description="Approved full-scene beauty frame",
                source_region="entire canvas",
                extraction_method="full_frame",
                z_index=0,
                locked=True,
                image_path=beauty.image_path,
            )
        ]
        for index, seam in enumerate(architecture.motion_seams, 1):
            related_poses = [path for key, path in pose_variants.items() if key.startswith(seam.seam_id + "-")]
            layers.append(
                SemanticLayer(
                    layer_id=seam.seam_id,
                    description=seam.subject,
                    source_region=seam.region,
                    extraction_method="external_mask" if self.config.get("mask_provider") else "kontext_isolation",
                    z_index=10 + index,
                    locked=False,
                    pose_variant_paths=related_poses,
                )
            )
        layers.append(
            SemanticLayer(
                layer_id="scientific-overlay",
                description="Experiment UI, labels, arrows, metrics and scientific annotations",
                source_region="overlay safe zones",
                extraction_method="scientific_overlay",
                z_index=100,
                locked=True,
            )
        )
        contract = SemanticLayerContract(
            scene_id=architecture.scene_id,
            beauty_frame_path=beauty.image_path,
            layers=layers,
            extraction_notes=[
                "The approved beauty frame remains the visible pixel source.",
                "Masks isolate only the smallest moving regions.",
                "No whole-scene vectorization or primitive reconstruction is permitted.",
                "Replacement poses inherit the approved scene as their Kontext init image.",
            ],
        )
        save_json(self.root / "contracts" / f"{architecture.scene_id}.json", contract)
        return contract


class SemanticMaskExtractor:
    def __init__(
        self,
        llm: LLMRouter,
        config: dict[str, Any],
        root: str | Path,
        mask_generator: Callable[[str, str, Path], Path | None] | None = None,
    ):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)
        self.mask_generator = mask_generator

    def extract_all(
        self,
        contract: SemanticLayerContract,
        architecture: SceneIllustrationArchitecture,
        *,
        force: bool = False,
    ) -> SemanticLayerContract:
        seam_map = {m.seam_id: m for m in architecture.motion_seams}
        for layer in contract.layers:
            if layer.extraction_method not in {"kontext_isolation", "external_mask"}:
                continue
            target = seam_map.get(layer.layer_id)
            if target is None:
                continue
            output = self.root / architecture.scene_id / f"{layer.layer_id}_mask.png"
            ensure_dir(output.parent)
            if output.exists() and output.stat().st_size > 256 and not force:
                layer.mask_path = str(output)
                continue
            if self.mask_generator is not None:
                result = self.mask_generator(contract.beauty_frame_path, target.region, output)
            else:
                result = self._kontext_mask(contract.beauty_frame_path, target.region, output, force=force)
            if result is None:
                raise RuntimeError(f"Could not create semantic mask for {architecture.scene_id}/{layer.layer_id}")
            self._normalize_mask(result, output)
            layer.mask_path = str(output)
        save_json(self.root / "contracts" / f"{architecture.scene_id}_extracted.json", contract)
        return contract

    def _kontext_mask(self, beauty_path: str, region: str, output: Path, *, force: bool) -> Path | None:
        raw = output.with_name(output.stem + "_raw.png")
        prompt = f"""Using the supplied approved scientific illustration, create an exact aligned binary segmentation mask.
The target region is: {region}.
Output the target region as pure white (#FFFFFF), every other pixel as pure black (#000000).
Keep exactly the same canvas, camera, scale, silhouette and position as the input. No text, shading, gray edges,
new objects, style transfer or composition change. This is a machine mask, not visible artwork."""
        result = self.llm.generate_reference_image(
            prompt=prompt,
            output_path=raw,
            init_image=beauty_path,
            force=force,
        )
        return result

    @staticmethod
    def _normalize_mask(source: str | Path, output: Path) -> None:
        image = Image.open(source).convert("L")
        # Hard binary mask. Because visible pixels come from the original beauty
        # frame, this normalization cannot degrade the authored line style.
        image = image.point(lambda p: 255 if p >= 128 else 0, mode="1").convert("L")
        image.save(output)


In [ ]:
%%writefile scistudio_v10/service_api.py
"""FastAPI service shell.

Endpoints:

* ``POST /jobs`` — validate the request, start generation in a background
  thread, return ``202`` with a job id. (Long-running generation belongs in a
  real worker/queue for multi-instance deployments; this in-process thread
  model is single-worker only and documented as such — distributed job
  execution is **not** implemented.)
* ``GET /jobs/{job_id}`` — job status from the persisted job manifest.

Errors are structured and redacted; raw exception strings are never exposed.
"""

# NOTE: no `from __future__ import annotations` here — FastAPI must resolve
# the locally-defined request/response models from real (non-string) annotations.
import threading
import uuid
from pathlib import Path
from typing import Any

from .security import redacted_exception_text


def create_app(studio_factory):
    """Build the FastAPI app. FastAPI is imported lazily."""
    try:
        from fastapi import FastAPI, HTTPException
        from fastapi.responses import JSONResponse
        from pydantic import BaseModel, Field
    except ImportError as exc:
        raise RuntimeError("Install fastapi and uvicorn to enable the REST service") from exc

    class GenerateRequest(BaseModel):
        topic: str = Field(min_length=3, max_length=500)
        reference_video: str = Field(min_length=1, max_length=4096)
        plan_only: bool = True
        render_video: bool = False
        job_id: str | None = Field(default=None, max_length=120)

    class JobAccepted(BaseModel):
        job_id: str
        status: str = "accepted"
        detail: str = (
            "Generation started in a background thread. Poll "
            "GET /jobs/{job_id} for status. Deploy a dedicated "
            "worker/queue for multi-instance production use."
        )

    class JobStatus(BaseModel):
        job_id: str
        status: str
        mode: str = ""
        video: str = ""
        error_type: str = ""
        error_message: str = ""

    app = FastAPI(
        title="Scientific Motion Studio V10",
        description="Single-worker generation API. Long-running generation "
        "requires a worker/queue in multi-instance deployments; "
        "distributed job execution is not implemented here.",
    )
    jobs: dict[str, dict[str, Any]] = {}
    jobs_lock = threading.Lock()

    def _run_job(job_id: str, payload: GenerateRequest) -> None:
        try:
            studio = studio_factory()
            result = studio.run(
                payload.topic,
                payload.reference_video,
                plan_only=payload.plan_only,
                render_video=payload.render_video,
                job_id=job_id,
            )
            with jobs_lock:
                jobs[job_id] = {"status": "completed", "result": result}
        except Exception as exc:
            with jobs_lock:
                jobs[job_id] = {
                    "status": "failed",
                    "error_type": type(exc).__name__,
                    "error_message": redacted_exception_text(exc, 500),
                }

    @app.post("/jobs", status_code=202, response_model=JobAccepted)
    def create_job(payload: GenerateRequest) -> JobAccepted:
        reference = Path(payload.reference_video)
        if not reference.exists() or not reference.is_file():
            raise HTTPException(
                status_code=400,
                detail={
                    "error": "invalid_reference_video",
                    "message": "reference_video must be an existing file path on the server",
                },
            )
        job_id = payload.job_id or f"api-{uuid.uuid4().hex[:12]}"
        with jobs_lock:
            if job_id in jobs and jobs[job_id].get("status") in {"accepted", "running"}:
                raise HTTPException(
                    status_code=409,
                    detail={
                        "error": "job_exists",
                        "message": f"job {job_id} is already running",
                    },
                )
            jobs[job_id] = {"status": "running"}
        worker = threading.Thread(target=_run_job, args=(job_id, payload), daemon=True)
        worker.start()
        return JobAccepted(job_id=job_id)

    @app.get("/jobs/{job_id}", response_model=JobStatus)
    def get_job(job_id: str) -> JobStatus:
        with jobs_lock:
            state = jobs.get(job_id)
        if state is None:
            raise HTTPException(
                status_code=404,
                detail={
                    "error": "unknown_job",
                    "message": f"no job with id {job_id}",
                },
            )
        result = state.get("result") or {}
        return JobStatus(
            job_id=job_id,
            status=str(state.get("status", "unknown")),
            mode=str(result.get("mode", "")),
            video=str(result.get("video", "")),
            error_type=str(state.get("error_type", "")),
            error_message=str(state.get("error_message", "")),
        )

    @app.exception_handler(Exception)
    def _unhandled(request, exc):  # noqa: ANN001 - FastAPI signature
        return JSONResponse(
            status_code=500,
            content={
                "error": "internal_error",
                "error_type": type(exc).__name__,
                "message": redacted_exception_text(exc, 300),
            },
        )

    return app


In [ ]:
%%writefile scistudio_v10/shot_state.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .schemas import SceneIllustrationArchitecture, SceneRequest, ShotState
from .utils import ensure_dir, load_json, save_json


class ShotStatePlanner:
    SYSTEM = """You are a shot-state director. Return JSON only. Define a precise first frame, last frame,
invariants, changed elements and the motion bridge. The last frame must be reachable from the first without
restyling or recomposing unaffected regions. Prefer locked camera and local causal change."""

    def __init__(self, llm: Any, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def plan(
        self, scene: SceneRequest, architecture: SceneIllustrationArchitecture, *, force: bool = False
    ) -> ShotState:
        path = self.root / f"{scene.scene_id}.json"
        if path.exists() and not force:
            return ShotState.model_validate(load_json(path))
        fallback = self._fallback(scene, architecture)
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"Scene: {json.dumps(scene.model_dump(mode='json'), ensure_ascii=False)}\nArchitecture: {json.dumps(architecture.model_dump(mode='json'), ensure_ascii=False)}",
            namespace=f"v10_shot_state_{scene.scene_id}",
            fallback=fallback.model_dump(mode="json"),
            force=force,
        )
        try:
            state = ShotState.model_validate(raw)
        except Exception:
            state = fallback
        state.scene_id = scene.scene_id
        save_json(path, state)
        return state

    @staticmethod
    def _fallback(scene: SceneRequest, architecture: SceneIllustrationArchitecture) -> ShotState:
        changed = [s.region for s in architecture.motion_seams]
        complexity = (
            "hold"
            if not changed
            else (
                "organic"
                if any(s.method in {"local_deformation", "texture_loop"} for s in architecture.motion_seams)
                else "articulated"
            )
        )
        return ShotState(
            scene_id=scene.scene_id,
            first_frame_description=f"Established approved composition before {scene.desired_change or scene.visual_event}.",
            last_frame_description=f"The same composition after the causal change: {scene.desired_change or scene.visual_event}.",
            invariant_elements=["camera", "palette", "line language", "unaffected environment", "subject identity"],
            changed_elements=changed,
            motion_bridge=[f"locally transform {x}" for x in changed] or ["deliberate hold"],
            preserve_regions=[p.contents for p in architecture.depth_planes if p.movement_role == "hold"],
            control_sketch_required=complexity in {"articulated", "deformation", "organic"},
            control_sketch_regions=changed,
            temporal_complexity=complexity,
        )


In [ ]:
%%writefile scistudio_v10/sketch_control.py
from __future__ import annotations

from pathlib import Path

from PIL import Image, ImageFilter, ImageOps

from .schemas import ControlSketchSpec, ShotState
from .utils import ensure_dir, save_json


class ControlSketchBuilder:
    """Creates invisible motion-control sketches from approved beauty frames.

    These sketches are controls for a temporal model, never visible final art.
    """

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)

    def build_pair(
        self, scene_id: str, beauty_start: str, beauty_end: str, state: ShotState
    ) -> tuple[ControlSketchSpec, ControlSketchSpec]:
        start = self._edge_sketch(beauty_start, self.root / f"{scene_id}_first.png")
        end = self._edge_sketch(beauty_end or beauty_start, self.root / f"{scene_id}_last.png")
        a = ControlSketchSpec(
            sketch_id=f"{scene_id}-first",
            scene_id=scene_id,
            frame_role="first",
            output_path=start,
            regions=state.control_sketch_regions,
            instructions=state.motion_bridge,
        )
        b = ControlSketchSpec(
            sketch_id=f"{scene_id}-last",
            scene_id=scene_id,
            frame_role="last",
            output_path=end,
            regions=state.control_sketch_regions,
            instructions=state.motion_bridge,
        )
        save_json(
            self.root / f"{scene_id}.json", {"first": a.model_dump(mode="json"), "last": b.model_dump(mode="json")}
        )
        return a, b

    @staticmethod
    def _edge_sketch(source: str, output: str | Path) -> str:
        image = Image.open(source).convert("L").filter(ImageFilter.FIND_EDGES)
        image = image.point(lambda x: 255 if x > 28 else 0)
        image = ImageOps.invert(image)
        output = Path(output)
        ensure_dir(output.parent)
        image.save(output)
        return str(output)


In [ ]:
%%writefile scistudio_v10/studio_director.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import (
    ArtDirectionBible,
    CanonAnchor,
    ContinuityCanon,
    RecurringSubjectCanon,
    ResearchPack,
    ScriptPackage,
    Storyboard,
)
from .style_canon import base_bible, build_hard_coded_canon
from .utils import ensure_dir, load_json, save_json


class ExecutiveArtDirector:
    """The LLM is the head of the visual studio, not a post-hoc scorekeeper.

    The studio's house style is hard-coded. The LLM may direct how that style
    expresses a topic, but it cannot replace the canon with a generic or childish
    vocabulary. This keeps style consistent without forcing beauty art into SVG.
    """

    SYSTEM = """You are the Executive Art Director of an institutional scientific animation studio.
The studio's locked visual canon is supplied below. You do not invent a new style for each scene.
Your job is to translate the topic into one coherent visual thesis, recurring motifs, scientific
readability rules, anti-AI drawing instructions and continuity priorities. Return JSON only.
Never recommend procedural SVG hero art, icon collage, mascot anatomy, primitive-shape construction,
or prompt-only generation without visual anchors."""

    CONTINUITY_SYSTEM = """You are the Continuity Director for one scientific animation.
Return JSON only. Build a production canon that makes every shot look authored by one studio:
fixed palette, line behavior, typography, experiment UI, recurring subject construction, environment
vocabulary and explicit drift prohibitions. Use approved frames as visual anchors. Do not redesign the style."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def art_bible(
        self,
        topic: str,
        research: ResearchPack,
        script: ScriptPackage,
        storyboard: Storyboard,
        reference_board_path: str,
        *,
        force: bool = False,
    ) -> ArtDirectionBible:
        path = self.root / "art_direction_bible.json"
        if path.exists() and not force:
            return ArtDirectionBible.model_validate(load_json(path))
        fallback = base_bible(topic, reference_board_path)
        canon = build_hard_coded_canon()
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""LOCKED HOUSE CANON — DO NOT CHANGE:
{json.dumps(canon.model_dump(mode="json"), ensure_ascii=False, indent=2)}

Topic: {topic}
Research summary: {research.summary[:1800]}
Script: {json.dumps(script.model_dump(mode="json"), ensure_ascii=False)}
Storyboard causal events: {json.dumps([s.model_dump(mode="json") for s in storyboard.scenes], ensure_ascii=False)}
Reference style board: {reference_board_path}

Return only topic-specific additions:
visual_thesis, topic_specific_motifs, recurring_symbols, scientific_readability_rules,
anti_ai_rules and continuity_priorities. Do not return a replacement canon.""",
            namespace="v10_art_bible",
            fallback={
                "visual_thesis": fallback.visual_thesis,
                "topic_specific_motifs": fallback.topic_specific_motifs,
                "recurring_symbols": fallback.recurring_symbols,
                "scientific_readability_rules": fallback.scientific_readability_rules,
                "anti_ai_rules": fallback.anti_ai_rules,
                "continuity_priorities": fallback.continuity_priorities,
            },
            force=force,
        )
        data = raw if isinstance(raw, dict) else {}
        bible = ArtDirectionBible(
            topic=topic,
            locked_canon=canon,
            visual_thesis=str(data.get("visual_thesis", fallback.visual_thesis)),
            topic_specific_motifs=self._list(data.get("topic_specific_motifs", [])),
            recurring_symbols=self._merge(fallback.recurring_symbols, self._list(data.get("recurring_symbols", []))),
            scientific_readability_rules=self._merge(
                fallback.scientific_readability_rules, self._list(data.get("scientific_readability_rules", []))
            ),
            anti_ai_rules=self._merge(fallback.anti_ai_rules, self._list(data.get("anti_ai_rules", []))),
            continuity_priorities=self._merge(
                fallback.continuity_priorities, self._list(data.get("continuity_priorities", []))
            ),
            reference_board_path=reference_board_path,
            raw_llm_output=raw,
        )
        save_json(path, bible)
        return bible

    def continuity_canon(
        self,
        bible: ArtDirectionBible,
        storyboard: Storyboard,
        reference_board_path: str,
        *,
        force: bool = False,
    ) -> ContinuityCanon:
        path = self.root / "continuity_canon.json"
        if path.exists() and not force:
            return ContinuityCanon.model_validate(load_json(path))
        locked = bible.locked_canon
        fallback_subjects = self._fallback_subjects(storyboard)
        raw = self.llm.generate_json(
            system=self.CONTINUITY_SYSTEM,
            prompt=f"""Locked canon:
{json.dumps(locked.model_dump(mode="json"), ensure_ascii=False)}
Visual thesis: {bible.visual_thesis}
Topic motifs: {json.dumps(bible.topic_specific_motifs, ensure_ascii=False)}
Scenes: {json.dumps([s.model_dump(mode="json") for s in storyboard.scenes], ensure_ascii=False)}

Return recurring_subjects, continuity_rules and prohibited_drift.
Recurring subjects must define immutable_traits and allowed_variations.
Do not alter palette, line system, typography or UI.""",
            namespace="v10_continuity",
            fallback={
                "recurring_subjects": [s.model_dump(mode="json") for s in fallback_subjects],
                "continuity_rules": [
                    "Every new scene uses the master style anchor and at least one approved prior scene when available.",
                    "Repeated subjects inherit silhouette, proportions, palette role and material marks from their canon sheet.",
                    "The experiment UI remains optically identical in position, spacing, line weight and typography.",
                    "The current scene may change content but may not reset the rendering language.",
                ],
                "prohibited_drift": [
                    "head-to-body ratio changes",
                    "line weight reset",
                    "new unrelated palette",
                    "different background treatment",
                    "mascot facial simplification",
                    "random outline thickness",
                    "scene-by-scene art-style improvisation",
                    "generic AI cinematic lighting",
                ],
            },
            force=force,
        )
        data = raw if isinstance(raw, dict) else {}
        subjects: list[RecurringSubjectCanon] = []
        for i, item in enumerate(data.get("recurring_subjects", [])):
            try:
                subjects.append(RecurringSubjectCanon.model_validate(item))
            except Exception:
                if i < len(fallback_subjects):
                    subjects.append(fallback_subjects[i])
        if not subjects:
            subjects = fallback_subjects
        canon = ContinuityCanon(
            palette_lock=locked.color.model_dump(mode="json"),
            line_lock=locked.line.model_dump(mode="json"),
            typography_lock=locked.typography.model_dump(mode="json"),
            ui_lock=list(locked.recurring_ui),
            recurring_subjects=subjects,
            anchors=[
                CanonAnchor(
                    anchor_id="reference-video-board",
                    role="reference_video_board",
                    path=reference_board_path,
                    notes="Visual-language reference only; never copy content or composition.",
                )
            ],
            continuity_rules=self._list(data.get("continuity_rules", [])),
            prohibited_drift=self._list(data.get("prohibited_drift", [])),
        )
        save_json(path, canon)
        return canon

    @staticmethod
    def _fallback_subjects(storyboard: Storyboard) -> list[RecurringSubjectCanon]:
        text = " ".join((s.visual_event + " " + s.scientific_claim).lower() for s in storyboard.scenes)
        subjects: list[RecurringSubjectCanon] = []
        if any(word in text for word in ("earth", "planet", "globe", "world")):
            subjects.append(
                RecurringSubjectCanon(
                    subject_id="earth-system",
                    description="Recurring scientific Earth/world system",
                    immutable_traits=[
                        "same continent abstraction",
                        "same charcoal-blue palette",
                        "same contour hierarchy",
                    ],
                    allowed_variations=["rotation", "damage state", "day-night state", "scale within composition"],
                )
            )
        if any(word in text for word in ("city", "building", "infrastructure")):
            subjects.append(
                RecurringSubjectCanon(
                    subject_id="city-system",
                    description="Recurring urban environment family",
                    immutable_traits=[
                        "same window rhythm",
                        "same structural line language",
                        "same material value bands",
                    ],
                    allowed_variations=["damage", "water level", "perspective crop", "density"],
                )
            )
        if any(word in text for word in ("human", "person", "people", "body", "hand")):
            subjects.append(
                RecurringSubjectCanon(
                    subject_id="human-figure-family",
                    description="Adult human figure construction used throughout the video",
                    immutable_traits=[
                        "adult 7-7.5 head proportion",
                        "same facial construction",
                        "same hand anatomy",
                        "same line pressure",
                    ],
                    allowed_variations=["pose", "expression", "wardrobe within fixed palette", "view angle"],
                )
            )
        if not subjects:
            subjects.append(
                RecurringSubjectCanon(
                    subject_id="primary-system",
                    description="The recurring physical system of the hypothetical",
                    immutable_traits=["same construction logic", "same material marks", "same palette role"],
                    allowed_variations=["causal state", "scale", "view angle"],
                )
            )
        return subjects

    @staticmethod
    def _list(value: Any) -> list[str]:
        if value is None:
            return []
        if isinstance(value, list):
            return [str(x).strip() for x in value if str(x).strip()]
        return [str(value).strip()] if str(value).strip() else []

    @staticmethod
    def _merge(a: list[str], b: list[str]) -> list[str]:
        out: list[str] = []
        for item in [*a, *b]:
            if item not in out:
                out.append(item)
        return out


In [ ]:
%%writefile scistudio_v10/style_canon.py
from __future__ import annotations

from .schemas import ArtDirectionBible, HardCodedStyleCanon
from .utils import hash_value


REFERENCE_OBSERVATIONS = {
    "source": "user-supplied scientific experiment motion reference",
    "visual_grammar": [
        "off-white instrument-paper field",
        "charcoal and blue-gray authored illustration",
        "condensed uppercase experiment typography",
        "small measurement/status panels at the edges",
        "one large causal phenomenon per shot",
        "clear phase changes rather than decorative entrances",
        "restrained red warning and yellow energy accents",
    ],
    "palette_observed": {
        "paper": "#FAFAFA",
        "ink": "#20282D",
        "dark_slate": "#474F54",
        "mist": "#EBEFF0",
        "steel": "#A8BAC2",
        "force_blue": "#2E6C96",
        "water_blue": "#3F90C5",
        "warning_red": "#D8483E",
        "energy_yellow": "#F3BD38",
    },
    "motion_grammar": [
        "locked or nearly locked camera",
        "local causal movement",
        "state change within a persistent visual world",
        "simple but strong directional arrows only when explanatory",
        "minimal transitions between causal stages",
    ],
}


def build_hard_coded_canon() -> HardCodedStyleCanon:
    canon = HardCodedStyleCanon()
    canon.style_hash = hash_value(canon.model_dump(exclude={"style_hash"}), length=20)
    return canon


def base_bible(topic: str, reference_board_path: str = "") -> ArtDirectionBible:
    canon = build_hard_coded_canon()
    return ArtDirectionBible(
        topic=topic,
        locked_canon=canon,
        visual_thesis=(
            "Explain the causal chain as a single authored scientific world: mature editorial ink, "
            "coherent perspective, fluid observational contours, restrained experiment UI, and no icon collage."
        ),
        topic_specific_motifs=[],
        recurring_symbols=[
            "experiment identifier",
            "simulation status",
            "single measured variable",
            "causal stage label",
        ],
        scientific_readability_rules=[
            "Every visual exaggeration must clarify a real causal relationship.",
            "Labels attach to physical subjects rather than floating as presentation bullets.",
            "Magnitude, direction and sequence must be encoded consistently.",
            "The audience must identify the physical system before reading any label.",
        ],
        anti_ai_rules=[
            "Avoid perfectly mirrored silhouettes and uniformly polished contours.",
            "Interior marks must describe anatomy, material, force or depth—not random decoration.",
            "Do not invent tiny nonsensical details or pseudo-text.",
            "Do not change line pressure, palette logic or facial construction between scenes.",
            "Use one integrated composition, not isolated assets pasted together.",
        ],
        continuity_priorities=[
            "line language",
            "palette",
            "paper field",
            "UI geometry",
            "perspective language",
            "recurring subject construction",
            "environment material vocabulary",
        ],
        reference_board_path=reference_board_path,
    )


In [ ]:
%%writefile scistudio_v10/style_reference.py
from __future__ import annotations

import math
import shutil
import subprocess
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw, ImageOps

from .utils import ensure_dir, save_json


class StyleReferenceExtractor:
    """Extracts a compact visual board from a user-provided reference video.

    The board is used as a visual-language anchor only. Drawing briefs explicitly
    prohibit copying its content, composition, labels or exact objects.
    """

    def __init__(self, root: str | Path, config: dict[str, Any] | None = None):
        self.root = ensure_dir(root)
        self.config = config or {}

    def extract(self, video_path: str | Path, *, force: bool = False) -> dict[str, Any]:
        video_path = Path(video_path)
        if not video_path.exists():
            raise FileNotFoundError(video_path)
        board = self.root / "reference_style_board.png"
        profile_path = self.root / "reference_style_profile.json"
        if board.exists() and profile_path.exists() and not force:
            import json

            return json.loads(profile_path.read_text(encoding="utf-8"))

        frames_dir = ensure_dir(self.root / "frames")
        for old in frames_dir.glob("*.png"):
            old.unlink()
        duration = self._duration(video_path)
        sample_count = int(self.config.get("sample_count", 8))
        times = [duration * (i + 0.65) / sample_count for i in range(sample_count)]
        images: list[Image.Image] = []
        frame_paths: list[str] = []
        for index, second in enumerate(times):
            out = frames_dir / f"style_{index:02d}.png"
            self._frame(video_path, second, out)
            if out.exists():
                im = Image.open(out).convert("RGB")
                images.append(im)
                frame_paths.append(str(out))
        if not images:
            raise RuntimeError("Could not extract reference frames")

        self._contact_sheet(images, board)
        palette = self._palette(images)
        profile = {
            "video_path": str(video_path),
            "duration_s": duration,
            "frame_paths": frame_paths,
            "board_path": str(board),
            "palette": palette,
            "usage_rule": "Use visual language only. Never copy source composition, text, subjects or sequence.",
        }
        save_json(profile_path, profile)
        return profile

    @staticmethod
    def _duration(path: Path) -> float:
        if not shutil.which("ffprobe"):
            return 10.0
        command = [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            str(path),
        ]
        try:
            return max(1.0, float(subprocess.check_output(command, text=True).strip()))
        except Exception:
            return 10.0

    @staticmethod
    def _frame(video: Path, second: float, output: Path) -> None:
        if not shutil.which("ffmpeg"):
            return
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-v",
                "error",
                "-ss",
                f"{second:.3f}",
                "-i",
                str(video),
                "-frames:v",
                "1",
                "-vf",
                "scale=640:-2",
                str(output),
            ],
            check=False,
        )

    @staticmethod
    def _contact_sheet(images: list[Image.Image], output: Path) -> None:
        cols = 2
        cell_w, cell_h = 680, 400
        rows = math.ceil(len(images) / cols)
        sheet = Image.new("RGB", (cols * cell_w, rows * cell_h), "#F2F2EF")
        draw = ImageDraw.Draw(sheet)
        for index, source in enumerate(images):
            card = Image.new("RGB", (cell_w - 18, cell_h - 18), "#FAFAF7")
            image = ImageOps.contain(source, (cell_w - 38, cell_h - 52))
            card.paste(image, ((card.width - image.width) // 2, 12))
            d = ImageDraw.Draw(card)
            d.text((14, card.height - 26), f"STYLE FRAME {index + 1:02d}", fill="#20282D")
            x = (index % cols) * cell_w + 9
            y = (index // cols) * cell_h + 9
            sheet.paste(card, (x, y))
            draw.rectangle([x, y, x + card.width, y + card.height], outline="#475157", width=2)
        output.parent.mkdir(parents=True, exist_ok=True)
        sheet.save(output)

    @staticmethod
    def _palette(images: list[Image.Image], colors: int = 10) -> list[str]:
        strip = Image.new("RGB", (256, max(1, 128 * len(images))), "white")
        for index, image in enumerate(images):
            reduced = ImageOps.fit(image.convert("RGB"), (256, 128))
            strip.paste(reduced, (0, index * 128))
        quantized = strip.quantize(colors=colors, method=Image.Quantize.MEDIANCUT)
        palette = quantized.getpalette() or []
        counts = quantized.getcolors() or []
        counts.sort(reverse=True)
        output: list[str] = []
        for _, idx in counts:
            rgb = palette[idx * 3 : idx * 3 + 3]
            if len(rgb) == 3:
                output.append("#" + "".join(f"{int(v):02X}" for v in rgb))
        return output


In [ ]:
%%writefile scistudio_v10/temporal_backends.py
"""Temporal video backends.

``SketchControlledVideoBackend`` runs an external command described as an
**argv token list** with ``{placeholder}`` substitution applied per token.
Tokens are passed to ``subprocess.run`` as a list — never joined through a
shell — so LLM-derived values such as the motion prompt cannot inject
commands. Legacy single-string ``command`` configuration is parsed with
``shlex.split`` into the same argv form.

Shell execution is supported only for advanced users via
``unsafe_shell_command`` + ``allow_shell=true``; it is disabled by default,
explicitly marked unsafe, and **rejected in production mode** unless
``security_override_unsafe_shell=true`` is also supplied.
"""

from __future__ import annotations

import shlex
import shutil
import subprocess
from pathlib import Path
from typing import Any, Protocol

from .errors import UnsafeCommandError
from .schemas import TemporalRequest, TemporalResult
from .utils import ensure_dir, save_json

_ALLOWED_PLACEHOLDERS = frozenset(
    {
        "start",
        "end",
        "sketch_start",
        "sketch_end",
        "output",
        "prompt",
        "fps",
        "frames",
    }
)


class TemporalBackend(Protocol):
    backend_id: str

    def available(self) -> bool: ...
    def supports(self, request: TemporalRequest) -> bool: ...
    def generate(self, request: TemporalRequest) -> TemporalResult: ...


class DeterministicCompositorBackend:
    """Still-frame hold compositor built on FFmpeg (always available)."""

    backend_id = "deterministic-compositor"

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)

    def available(self) -> bool:
        return True

    def supports(self, request: TemporalRequest) -> bool:
        return request.complexity in {"hold", "simple", "articulated"}

    def generate(self, request: TemporalRequest) -> TemporalResult:
        output = Path(request.output_path or self.root / f"{request.scene_id}.mp4")
        ensure_dir(output.parent)
        duration = max(1 / request.fps, request.duration_frames / request.fps)
        command = [
            "ffmpeg",
            "-y",
            "-loop",
            "1",
            "-i",
            request.beauty_start,
            "-t",
            f"{duration:.4f}",
            "-r",
            str(request.fps),
            "-vf",
            "format=yuv420p",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            str(output),
        ]
        subprocess.run(command, check=True, capture_output=True)
        return TemporalResult(
            scene_id=request.scene_id,
            backend_id=self.backend_id,
            output_path=str(output),
            deterministic=True,
            metadata={"command": "ffmpeg still-frame compositor"},
        )


class _SafeFormatMap(dict):
    """Formatting map that rejects unknown placeholders instead of KeyError-ing
    into silent empty substitutions."""

    def __missing__(self, key: str) -> str:
        raise UnsafeCommandError(
            f"Unknown placeholder {{{key}}} in temporal command template; allowed: {sorted(_ALLOWED_PLACEHOLDERS)}"
        )


class SketchControlledVideoBackend:
    """Adapter for SketchVideo-like one/two-keyframe sketch conditioning.

    It intentionally fails rather than silently replacing organic motion with
    PowerPoint-like motion.
    """

    backend_id = "sketch-controlled-video"

    def __init__(self, config: dict[str, Any], root: str | Path, execution_mode: str = "development"):
        self.config = config
        self.root = ensure_dir(root)
        self.execution_mode = str(execution_mode).lower()

    def _argv_template(self) -> list[str]:
        argv = self.config.get("argv") or []
        if argv:
            return [str(token) for token in argv]
        legacy = str(self.config.get("command", "") or "")
        if legacy:
            return shlex.split(legacy)
        return []

    def available(self) -> bool:
        if self.config.get("allow_shell") and self.config.get("unsafe_shell_command"):
            return bool(shutil.which(shlex.split(str(self.config["unsafe_shell_command"]))[0]))
        template = self._argv_template()
        return bool(template and shutil.which(template[0]))

    def supports(self, request: TemporalRequest) -> bool:
        return request.complexity in {"articulated", "deformation", "organic"} and bool(request.control_sketch_start)

    def _values(self, request: TemporalRequest, output: Path) -> dict[str, Any]:
        return {
            "start": request.beauty_start,
            "end": request.beauty_end or request.beauty_start,
            "sketch_start": request.control_sketch_start,
            "sketch_end": request.control_sketch_end,
            "output": str(output),
            "prompt": request.motion_prompt,
            "fps": request.fps,
            "frames": request.duration_frames,
        }

    def generate(self, request: TemporalRequest) -> TemporalResult:
        if not self.available():
            raise RuntimeError("Sketch-controlled temporal backend is unavailable")
        output = Path(request.output_path or self.root / f"{request.scene_id}.mp4")
        ensure_dir(output.parent)
        values = self._values(request, output)
        timeout = float(self.config.get("timeout_s", 1800.0))

        if self.config.get("allow_shell") and self.config.get("unsafe_shell_command"):
            self._run_unsafe_shell(values, timeout)
        else:
            argv = [token.format_map(_SafeFormatMap(values)) for token in self._argv_template()]
            subprocess.run(argv, check=True, timeout=timeout)

        if not output.exists() or output.stat().st_size == 0:
            raise RuntimeError(f"Temporal backend produced no output for scene {request.scene_id}")
        return TemporalResult(
            scene_id=request.scene_id,
            backend_id=self.backend_id,
            output_path=str(output),
            metadata={"external_command": True},
        )

    def _run_unsafe_shell(self, values: dict[str, Any], timeout: float) -> None:
        """UNSAFE opt-in shell path. Disabled by default; rejected in production
        without an explicit security override."""
        if self.execution_mode == "production" and not self.config.get("security_override_unsafe_shell"):
            raise UnsafeCommandError(
                "Shell-based temporal command execution is not allowed in "
                "production mode without security_override_unsafe_shell=true"
            )
        command = str(self.config["unsafe_shell_command"]).format_map(
            _SafeFormatMap({key: shlex.quote(str(value)) for key, value in values.items()})
        )
        subprocess.run(command, shell=True, check=True, timeout=timeout)  # noqa: S602 - explicit opt-in, quoted values


class TemporalBackendRouter:
    """Chooses the first available backend that supports the request; fails
    loudly when high-complexity motion has no capable backend."""

    def __init__(self, backends: list[TemporalBackend], root: str | Path):
        self.backends = backends
        self.root = ensure_dir(root)

    def select(self, request: TemporalRequest) -> TemporalBackend:
        preferred = request.backend_preference or [
            "sketch-controlled-video",
            "deterministic-compositor",
        ]
        by_id = {backend.backend_id: backend for backend in self.backends}
        for backend_id in preferred:
            backend = by_id.get(backend_id)
            if backend and backend.available() and backend.supports(request):
                return backend
        for backend in self.backends:
            if backend.available() and backend.supports(request):
                return backend
        raise RuntimeError(f"No temporal backend can satisfy scene {request.scene_id} complexity={request.complexity}")

    def generate(self, request: TemporalRequest) -> TemporalResult:
        backend = self.select(request)
        result = backend.generate(request)
        save_json(self.root / f"{request.scene_id}.json", result)
        return result


In [ ]:
%%writefile scistudio_v10/tests.py
from __future__ import annotations

import json
import tempfile
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw

from .animation_director import AnimationDirector
from .drawing_brief import DrawingBriefCompiler
from .flux_studio import FluxKontextStudio
from .llm import LLMRouter
from .hybrid_package import HybridPackageBuilder
from .hybrid_render import PILHybridRenderer, RemotionHybridExporter
from .schemas import (
    ArtDirectionBible,
    BeautyFrame,
    CanonAnchor,
    ConcreteAdjustment,
    ContinuityCanon,
    DepthPlane,
    DirectorChangeOrder,
    DrawingBrief,
    MotionSeam,
    ReferencePack,
    SceneIllustrationArchitecture,
    SceneRequest,
)
from .semantic import SemanticMaskExtractor, SemanticSeparationPlanner
from .style_canon import base_bible, build_hard_coded_canon
from .utils import load_json


class FakeLLM:
    def generate_json(self, **kwargs):
        return kwargs.get("fallback")

    def critique_image(self, **kwargs):
        scene_id = (
            str(kwargs.get("namespace", "SC")).split("_")[-2] if "_" in str(kwargs.get("namespace", "")) else "SC"
        )
        return DirectorChangeOrder(scene_id=scene_id, status="approve", revision_number=1).model_dump(mode="json")

    def generate_reference_image(self, *, prompt, output_path, init_image=None, force=False):
        output = Path(output_path)
        output.parent.mkdir(parents=True, exist_ok=True)
        if init_image and Path(init_image).exists():
            Image.open(init_image).convert("RGB").save(output)
        else:
            Image.new("RGB", (180, 320), "#FAFAF7").save(output)
        return output


class ResultCollector:
    def __init__(self):
        self.items: list[dict[str, Any]] = []

    def check(self, name: str, condition: bool, detail: str = ""):
        self.items.append({"name": name, "passed": bool(condition), "detail": detail})
        if not condition:
            raise AssertionError(f"{name}: {detail}")


def run_regression_tests(root: str | Path | None = None) -> dict[str, Any]:
    base = Path(root) if root else Path(tempfile.mkdtemp(prefix="scistudio_v10_tests_"))
    base.mkdir(parents=True, exist_ok=True)
    r = ResultCollector()

    canon = build_hard_coded_canon()
    r.check("canon identity locked", canon.canon_id == "experiment-ledger-editorial-ink-v1")
    r.check("canon forbids procedural hero", "procedural SVG hero illustration" in canon.forbidden)
    r.check("canon has stable palette", canon.color.paper == "#FAFAF7" and canon.color.blue_primary == "#2E77A6")
    r.check(
        "canon motion defaults hold", canon.motion.default_state == "hold" and not canon.motion.default_zoom_allowed
    )

    bible = base_bible("Test topic", "reference.png")
    r.check("bible embeds locked canon", bible.locked_canon.canon_id == canon.canon_id)
    tampered = ArtDirectionBible(
        topic="x",
        canon_id="other",
        locked_canon={"canon_id": "experiment-ledger-editorial-ink-v1", "display_name": "tampered"},
    )
    r.check("LLM cannot rename canon", tampered.locked_canon.display_name == "Experiment Ledger Editorial Ink")

    continuity = ContinuityCanon(
        palette_lock=canon.color.model_dump(mode="json"),
        line_lock=canon.line.model_dump(mode="json"),
        typography_lock=canon.typography.model_dump(mode="json"),
        ui_lock=canon.recurring_ui,
        continuity_rules=["same line and palette"],
        prohibited_drift=["style reset"],
    )
    architecture = SceneIllustrationArchitecture(
        scene_id="SC01",
        beat_id="B01",
        narrative_claim="Rain accumulates",
        visual_thesis="One integrated city under persistent rain",
        depth_planes=[
            DepthPlane(plane_id="bg", depth="background", contents="sky"),
            DepthPlane(plane_id="city", depth="midground", contents="city"),
            DepthPlane(plane_id="rain", depth="foreground", contents="rain", movement_role="primary"),
        ],
        motion_seams=[
            MotionSeam(
                seam_id="rain-field",
                subject="continuous rain",
                method="texture_loop",
                region="rain strokes and water veil",
                resting_overlap_rule="masked by atmospheric layer",
                required_variants=["initial", "changed"],
            )
        ],
        animation_representation=["texture_loop"],
        required_pose_variants=["initial", "changed"],
    )
    r.check("scene is beauty-first", architecture.beauty_frame_first and architecture.semantic_split_after_approval)
    r.check("scene forbids procedural hero", architecture.procedural_hero_allowed is False)

    brief_compiler = DrawingBriefCompiler({"seed_base": 1234, "aspect_ratio": "9:16"}, base / "briefs")
    pack = ReferencePack(scene_id="SC01", board_path=str(base / "board.png"))
    Image.new("RGB", (180, 320), "#FAFAF7").save(pack.board_path)
    scene = SceneRequest(
        scene_id="SC01",
        beat_id="B01",
        duration_s=1.0,
        narration="Rain continues.",
        headline="NONSTOP RAIN",
        visual_event="city in rain",
        desired_change="rain never stops",
    )
    brief = brief_compiler.beauty_frame(architecture, bible, continuity, pack, scene.narration, scene.headline, 0)
    prompt_low = brief.positive_prompt.lower()
    r.check("brief is scene-first", "one integrated" in prompt_low and "isolated icon" in brief.negative_prompt.lower())
    r.check("brief hard-codes line system", "variable pressure" in prompt_low and "purposeful breaks" in prompt_low)
    r.check("brief forbids office-shape look", "microsoft word shape assembly" in brief.negative_prompt.lower())
    r.check("brief uses stable seed policy", brief.seed == 1234)
    try:
        DrawingBrief(
            brief_id="bad",
            scene_id="x",
            positive_prompt="procedural SVG hero",
            negative_prompt="",
            kontext_instruction="assemble from primitive",
        )
        bad_rejected = False
    except Exception:
        bad_rejected = True
    r.check("procedural hero brief rejected", bad_rejected)

    try:
        DirectorChangeOrder(scene_id="SC", status="revise", adjustments=[])
        vague_rejected = False
    except Exception:
        vague_rejected = True
    r.check("vague revision order rejected", vague_rejected)
    concrete = DirectorChangeOrder(
        scene_id="SC",
        status="revise",
        adjustments=[
            ConcreteAdjustment(
                adjustment_id="A1",
                target_region="hand",
                problem="oval hand",
                instruction="construct palm and knuckle plane",
            )
        ],
    )
    r.check("concrete director revision accepted", concrete.adjustments[0].target_region == "hand")

    approved_path = base / "approved.png"
    im = Image.new("RGB", (180, 320), "#FAFAF7")
    d = ImageDraw.Draw(im)
    d.rectangle([40, 60, 140, 260], fill="#475157")
    im.save(approved_path)
    beauty = BeautyFrame(
        scene_id="SC01", image_path=str(approved_path), approved=True, approval_source="vision-llm-art-director"
    )
    pose_dir = base / "poses"
    pose_dir.mkdir()
    pose = im.copy()
    ImageDraw.Draw(pose).rectangle([75, 80, 150, 240], fill="#2E77A6")
    pose_path = pose_dir / "changed.png"
    pose.save(pose_path)
    semantic_planner = SemanticSeparationPlanner({}, base / "semantic")
    contract = semantic_planner.plan(
        architecture, beauty, {"rain-field-initial": str(approved_path), "rain-field-changed": str(pose_path)}
    )
    r.check("semantic split after approval", contract.separation_occurs_after_approval)
    r.check("beauty pixels preserved", contract.preserve_original_beauty)

    def mask_gen(beauty_path, region, output):
        mask = Image.new("L", (180, 320), 0)
        ImageDraw.Draw(mask).rectangle([40, 60, 140, 260], fill=255)
        mask.save(output)
        return output

    extractor = SemanticMaskExtractor(FakeLLM(), {}, base / "semantic" / "masks", mask_generator=mask_gen)
    contract = extractor.extract_all(contract, architecture, force=True)
    moving = next(x for x in contract.layers if x.layer_id == "rain-field")
    r.check("mask extracted without redrawing beauty", Path(moving.mask_path).exists())

    animation = AnimationDirector(FakeLLM(), {}, base / "animation").plan(scene, architecture, contract, fps=10)
    r.check("beauty base never animated", all(e.target_layer != "beauty-base" for e in animation.events))
    r.check("no default camera motion", animation.camera_locked)
    r.check("motion has causal reason", all(e.reason_id for e in animation.events))

    overlay = base / "overlay.svg"
    overlay.write_text(
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 180 320"><text x="5" y="20">TEST</text></svg>'
    )
    package = HybridPackageBuilder({"width": 180, "height": 320}, base / "hybrid").build(
        scene, contract, animation, str(overlay)
    )
    base_layer = next(x for x in package.layers if x.layer_id == "beauty-base")
    r.check("hybrid package keeps raster beauty", base_layer.kind == "raster" and Path(base_layer.path).exists())
    r.check("SVG limited to overlay", any(x.kind == "svg_overlay" for x in package.layers))

    remotion = RemotionHybridExporter({}, base / "remotion").create_project([package])
    ts = (Path(remotion) / "src/index.tsx").read_text()
    r.check("remotion supports raster layers", "<Img" in ts)
    r.check("remotion supports replacement poses", "replacement_pose" in ts)

    video = base / "preview.mp4"
    PILHybridRenderer({"crf": 30}, base / "render").render([package], video)
    r.check("deterministic H264 preview renders", video.exists() and video.stat().st_size > 1000)

    # Option 3: production FLUX prompting system.
    r.check("prompt stack compiled", brief.prompt_stack is not None and bool(brief.compiled_prompt))
    r.check("prompt diagnostics pass", bool(brief.prompt_diagnostics and brief.prompt_diagnostics.valid))
    r.check(
        "prompt is natural language not JSON", "{" not in brief.compiled_prompt and "}" not in brief.compiled_prompt
    )
    r.check("prompt length is directed not bloated", 80 <= brief.prompt_diagnostics.word_count <= 380)
    first_55 = " ".join(brief.compiled_prompt.lower().split()[:55])
    r.check("style stated before scene detail", "scientific editorial" in first_55 and "illustration" in first_55)
    r.check("Kontext prompt upsampling disabled", brief.prompt_upsampling is False)
    r.check(
        "single input strategy recorded",
        brief.init_strategy in {"reference_board", "master_style_anchor", "previous_approved_scene"},
    )
    r.check(
        "style fingerprint is immutable",
        bool(brief.style_fingerprint_hash) and brief.prompt_stack.immutable_style_hash == brief.style_fingerprint_hash,
    )

    architecture_2 = architecture.model_copy(deep=True)
    architecture_2.scene_id = "SC02"
    architecture_2.visual_thesis = "The same city after drains overflow"
    pack_2 = pack.model_copy(deep=True)
    pack_2.scene_id = "SC02"
    pack_2.previous_approved_scene = str(approved_path)
    brief_2 = brief_compiler.beauty_frame(
        architecture_2,
        bible,
        continuity,
        pack_2,
        "Water rises.",
        "DRAINAGE OVERFLOW",
        1,
        style_source_path=str(approved_path),
        init_strategy="previous_approved_scene",
    )
    r.check("style hash identical across scenes", brief_2.style_fingerprint_hash == brief.style_fingerprint_hash)
    r.check(
        "scene delta changes without style reset",
        brief_2.prompt_stack.scene_delta_hash != brief.prompt_stack.scene_delta_hash,
    )
    r.check(
        "previous approved scene drives continuity",
        brief_2.init_image_path == str(approved_path) and brief_2.init_strategy == "previous_approved_scene",
    )

    change_order = DirectorChangeOrder(
        scene_id="SC01",
        revision_number=1,
        status="revise",
        adjustments=[
            ConcreteAdjustment(
                adjustment_id="A_HAND",
                target_region="right hand",
                problem="oval hand",
                instruction="construct a palm mass, four knuckle plane and locked thumb aligned with the wrist",
                preserve=["head", "torso", "background"],
                priority="critical",
            ),
            ConcreteAdjustment(
                adjustment_id="A_WALL",
                target_region="wall surface",
                problem="flat rectangle",
                instruction="add perspective convergence and irregular concrete edge wear without changing its position",
                preserve=["character", "impact point"],
                priority="high",
            ),
        ],
        immutable_preserve_list=["camera", "palette", "line rhythm"],
    )
    revision_briefs = brief_compiler.revision_passes(brief, change_order, str(approved_path), 1)
    r.check("director changes become sequential local passes", len(revision_briefs) == 2)
    r.check("revision chain uses prior pass", revision_briefs[1].init_image_path == revision_briefs[0].output_path)
    r.check(
        "revision explicitly scopes target", all(b.prompt_diagnostics.has_local_edit_scope for b in revision_briefs)
    )
    r.check(
        "revision explicitly preserves unaffected image",
        all(b.prompt_diagnostics.has_explicit_preservation for b in revision_briefs),
    )
    r.check(
        "revision avoids vague quality language",
        all(not b.prompt_diagnostics.vague_language_found for b in revision_briefs),
    )
    r.check("revision prompts stay concise", all(b.prompt_diagnostics.word_count <= 180 for b in revision_briefs))

    pose_brief = brief_compiler.pose_variant(
        brief,
        str(approved_path),
        "impact",
        "rotate only the forearm and construct the compressed fist at the wall contact point",
    )
    r.check("pose edit uses approved frame", pose_brief.init_strategy == "approved_beauty_frame")
    r.check(
        "pose edit is local and preservation-first",
        pose_brief.prompt_diagnostics.has_local_edit_scope and pose_brief.prompt_diagnostics.has_explicit_preservation,
    )

    # The Flux studio sends only the compiled prompt and records a reproducible request manifest.
    generated_root = base / "flux_request_test"

    def generator_fixture(drawing_brief):
        out = Path(drawing_brief.output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        Image.new("RGB", (180, 320), "#FAFAF7").save(out)
        return out

    flux_fixture = FluxKontextStudio(FakeLLM(), brief_compiler, {}, generated_root, image_generator=generator_fixture)
    generated = flux_fixture.generate(brief, force=True)
    request_manifest = load_json(generated_root / "requests" / f"{brief.brief_id}.json")
    r.check("Flux request manifest exists", generated is not None and bool(request_manifest))
    r.check("request contains one compiled prompt", request_manifest["prompt"] == brief.compiled_prompt)
    r.check(
        "request locks prompt parameters",
        request_manifest["prompt_upsampling"] is False and request_manifest["output_format"] == "png",
    )

    # The vision art director reviews MASTER / PREVIOUS / CURRENT on one comparison board.
    continuity_with_images = continuity.model_copy(deep=True)
    continuity_with_images.anchors = [
        CanonAnchor(anchor_id="master", role="master_style_anchor", path=str(approved_path)),
        CanonAnchor(anchor_id="previous", role="approved_scene", path=str(pose_path), scene_id="SC00"),
    ]
    compare_path = flux_fixture._comparison_board(str(approved_path), "SC01", continuity_with_images, 1)
    compare_image = Image.open(compare_path)
    r.check("director comparison board generated", compare_image.size == (1536, 1024))

    # Verify the real BFL payload contract without making a network call.
    # The fixture serves a real PNG with image headers: the hardened client
    # validates Content-Type and verifies the payload with Pillow, so an
    # unrealistic body would (correctly) be rejected.
    import io
    from unittest.mock import patch

    png_buffer = io.BytesIO()
    Image.new("RGB", (32, 32), "#2E77A6").save(png_buffer, format="PNG")
    real_png_bytes = png_buffer.getvalue()

    class FakeResponse:
        def __init__(self, data=None, content=b"", headers=None, status_code=200):
            self._data = data or {}
            self.content = content
            self.headers = headers or {}
            self.status_code = status_code

        def raise_for_status(self):
            return None

        def json(self):
            return self._data

    post_calls = []

    def fake_post(url, headers=None, json=None, timeout=None, **kwargs):
        post_calls.append({"url": url, "json": json})
        return FakeResponse({"id": "REQ", "polling_url": "https://poll.local/result"})

    def fake_get(url, headers=None, params=None, timeout=None, **kwargs):
        if "poll.local" in url:
            return FakeResponse({"status": "Ready", "result": {"sample": "https://delivery.local/sample"}})
        return FakeResponse(content=real_png_bytes, headers={"Content-Type": "image/png"})

    bfl_router = LLMRouter(
        {
            "bfl_model": "flux-kontext-pro",
            "bfl_aspect_ratio": "9:16",
            "bfl_seed": 99,
            "bfl_prompt_upsampling": False,
            "bfl_safety_tolerance": 2,
            "bfl_output_format": "png",
            "bfl_allowed_url_hosts": ["api.bfl.ai", "poll.local", "delivery.local"],
        },
        {"BFL_API_KEY": "TEST-BFL-KEY"},
        base / "bfl_cache",
    )
    bfl_out = base / "bfl_mock.png"
    with patch("requests.post", side_effect=fake_post), patch("requests.get", side_effect=fake_get):
        result_path = bfl_router._bfl_flux_image(
            "A directed test prompt", bfl_out, init_image=approved_path, force=True
        )
    payload = post_calls[0]["json"]
    r.check("BFL mock request succeeds", result_path == bfl_out and bfl_out.exists())
    r.check("BFL downloaded image is a valid PNG", bfl_out.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n")
    r.check("BFL payload uses Kontext input image", "input_image" in payload and payload["aspect_ratio"] == "9:16")
    r.check(
        "BFL payload locks deterministic prompt policy",
        payload["prompt_upsampling"] is False and payload["safety_tolerance"] == 2 and payload["seed"] == 99,
    )
    r.check(
        "BFL safe manifest excludes base64",
        "input_image" not in load_json(next((base / "bfl_cache" / "bfl_images").glob("*.request.json")))["payload"],
    )

    report = {"passed": all(x["passed"] for x in r.items), "count": len(r.items), "tests": r.items, "root": str(base)}
    (base / "regression_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


In [ ]:
%%writefile scistudio_v10/tests_final.py
"""Finalization test suite: unit, mocked-integration and offline end-to-end
tests added on top of the retained V9/V10 regression suites.

No test in this module performs a live network call or spends API credits;
all provider traffic is mocked. Run everything with
``run_final_validation_tests()`` or the aggregate ``run_all_tests()``.
"""

from __future__ import annotations

import io
import json
import tempfile
import time
from pathlib import Path
from typing import Any
from unittest.mock import patch

from PIL import Image, ImageDraw

from .bfl_client import BFLClient, BFLError
from .cache_store import ArtifactCache
from .config_models import ExecutionMode, StudioConfig
from .errors import (
    DownloadPolicyError,
    JobConcurrencyError,
    ProviderLockViolationError,
    ProviderUnavailableError,
    StageRetryExhaustedError,
    UnsafeCommandError,
)
from .http_safety import (
    backoff_delays,
    call_with_retries,
    classify_exception,
    download_image,
    is_retryable_status,
    validate_url,
)
from .job_runtime import ResumableJobRuntime
from .llm import FALLBACK_MARKER, LLMRouter
from .schemas import TemporalRequest
from .security import (
    clear_registered_secrets,
    redact_secrets,
    redacted_exception_text,
    register_secret,
)
from .temporal_backends import SketchControlledVideoBackend
from .utils import ensure_within, extract_json, load_json, save_json


class Collector:
    def __init__(self) -> None:
        self.items: list[dict[str, Any]] = []

    def check(self, name: str, condition: bool, detail: str = "") -> None:
        self.items.append({"name": name, "passed": bool(condition), "detail": detail})
        if not condition:
            raise AssertionError(f"{name}: {detail}")


def _png_bytes(color: str = "#2E77A6", size: tuple[int, int] = (32, 32)) -> bytes:
    buffer = io.BytesIO()
    Image.new("RGB", size, color).save(buffer, format="PNG")
    return buffer.getvalue()


def _image_file(path: Path, color: str = "#475157") -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    image = Image.new("RGB", (180, 320), "#FAFAF7")
    ImageDraw.Draw(image).rectangle((25, 40, 155, 280), fill=color)
    image.save(path)
    return path


class FakeHTTPResponse:
    def __init__(self, data=None, content=b"", headers=None, status_code=200):
        self._data = data or {}
        self.content = content
        self.headers = headers or {}
        self.status_code = status_code

    def raise_for_status(self):
        if self.status_code >= 400:
            error = RuntimeError(f"HTTP {self.status_code}")
            error.response = self  # type: ignore[attr-defined]
            raise error

    def json(self):
        return self._data


# ---------------------------------------------------------------------------
# Unit tests
# ---------------------------------------------------------------------------


def _test_config_validation(c: Collector, base: Path) -> None:
    # Production locks reject unauthorized fallbacks.
    for bad in (
        {"execution_mode": "production", "llm": {"provider_order": ["openai", "gemini"]}},
        {"execution_mode": "production", "llm": {"provider_order": ["openrouter"]}},
        {
            "execution_mode": "production",
            "llm": {"provider_order": ["openai"], "vision_provider_order": ["openai", "gemini"]},
        },
        {"execution_mode": "production", "llm": {"provider_order": ["openai"], "enable_local_fallback": True}},
        {"execution_mode": "production", "llm": {"provider_order": ["openai"], "image_provider": "openai"}},
        {"execution_mode": "production", "llm": {"provider_order": ["openai"], "gemini_image_model": "imagen-3"}},
    ):
        try:
            StudioConfig.from_dict(bad)
            rejected = False
        except (ProviderLockViolationError, Exception):
            rejected = True
        c.check(f"production lock rejects {json.dumps(bad['llm'])[:60]}", rejected)

    good = StudioConfig.from_dict(
        {
            "execution_mode": "production",
            "llm": {"provider_order": ["openai"], "vision_provider_order": ["openai"]},
        }
    )
    c.check("valid production config accepted", good.execution_mode == ExecutionMode.production)
    c.check("mode propagates into llm config", good.as_runtime_dict()["llm"]["execution_mode"] == "production")

    dev = StudioConfig.from_dict({})
    c.check("development is the default mode", dev.execution_mode == ExecutionMode.development)

    for invalid in (
        {"llm": {"bfl_model": "sdxl"}},
        {"llm": {"bfl_base_url": "http://api.bfl.ai/v1"}},
        {"llm": {"bfl_aspect_ratio": "1:9"}},
        {"llm": {"bfl_output_format": "webp"}},
        {"llm": {"openai_reasoning_effort": "extreme"}},
        {"render": {"backend": "imovie"}},
        {"workspace": "   "},
    ):
        try:
            StudioConfig.from_dict(invalid)
            rejected = False
        except Exception:
            rejected = True
        c.check(f"config rejects {json.dumps(invalid)[:60]}", rejected)

    prod_shell = {
        "execution_mode": "production",
        "llm": {"provider_order": ["openai"], "vision_provider_order": ["openai"]},
        "temporal": {"sketch_backend": {"allow_shell": True, "unsafe_shell_command": "run {output}"}},
    }
    try:
        StudioConfig.from_dict(prod_shell)
        rejected = False
    except ProviderLockViolationError:
        rejected = True
    c.check("production rejects shell temporal command without override", rejected)


def _test_secret_redaction(c: Collector) -> None:
    clear_registered_secrets()
    openai_key = "sk-proj-Abc123XYZ456pqrstu789"
    text = f"failed with api_key={openai_key} while calling"
    c.check("sk- key pattern redacted", openai_key not in redact_secrets(text))
    c.check("bearer token redacted", "Bearer abcdef123456" not in redact_secrets("auth: Bearer abcdef123456 sent"))
    c.check(
        "x-key header redacted", "1d2e3f4a" not in redact_secrets('"x-key": "1d2e3f4a-9999-4bbb-8ccc-121212121212"')
    )
    uuid_key = "0f0e0d0c-1111-4222-8333-444455556666"
    c.check("uuid after key context redacted", uuid_key not in redact_secrets(f"BFL key={uuid_key} rejected"))
    nested = redact_secrets({"config": {"openai_api_key": "raw-value", "model": "gpt-5-mini"}})
    c.check(
        "secret-named dict keys redacted",
        nested["config"]["openai_api_key"] == "[REDACTED]" and nested["config"]["model"] == "gpt-5-mini",
    )
    usage = redact_secrets({"usage": {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15}})
    c.check(
        "numeric token-usage metadata survives redaction",
        usage["usage"] == {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15},
    )
    register_secret("super-secret-raw-token-9876")
    c.check(
        "registered exact secret scrubbed anywhere",
        "super-secret-raw-token-9876"
        not in redacted_exception_text(RuntimeError("url?tok=super-secret-raw-token-9876&x=1")),
    )
    clear_registered_secrets()


def _test_url_and_retry_policy(c: Collector) -> None:
    try:
        validate_url("http://api.bfl.ai/v1/x", allowed_hosts=["api.bfl.ai"])
        ok = False
    except DownloadPolicyError:
        ok = True
    c.check("plain http rejected", ok)
    try:
        validate_url("https://evil.example/x", allowed_hosts=["api.bfl.ai", "*.bfl.ai"])
        ok = False
    except DownloadPolicyError:
        ok = True
    c.check("non-allowlisted host rejected", ok)
    c.check(
        "wildcard host accepted",
        validate_url("https://delivery-eu1.bfl.ai/img", allowed_hosts=["*.bfl.ai"])
        == "https://delivery-eu1.bfl.ai/img",
    )
    try:
        validate_url("https://api.bfl.ai/x", allowed_hosts=[])
        ok = False
    except DownloadPolicyError:
        ok = True
    c.check("empty allowlist refuses outbound", ok)

    c.check("429 is retryable", is_retryable_status(429))
    c.check("408 is retryable", is_retryable_status(408))
    c.check("409 is retryable", is_retryable_status(409))
    c.check("503 is retryable", is_retryable_status(503))
    c.check("401 is not retryable", not is_retryable_status(401))
    c.check("422 is not retryable", not is_retryable_status(422))

    class E401(Exception):
        status_code = 401

    class E429(Exception):
        status_code = 429

    c.check("exception with 401 not retryable", not classify_exception(E401()))
    c.check("exception with 429 retryable", classify_exception(E429()))
    c.check("timeout-class exception retryable", classify_exception(TimeoutError("t")))

    delays = backoff_delays(5, base_delay=1.0, max_delay=8.0, jitter=0.25)
    c.check(
        "backoff grows exponentially and caps", len(delays) == 4 and delays[0] < delays[2] and max(delays) <= 8.0 * 1.25
    )

    calls = {"n": 0}

    def flaky():
        calls["n"] += 1
        if calls["n"] < 3:
            raise E429()
        return "ok"

    result = call_with_retries(flaky, max_attempts=4, base_delay=0.01, sleep=lambda _s: None)
    c.check("retry loop recovers after 429s", result == "ok" and calls["n"] == 3)

    calls["n"] = 0

    def auth_fail():
        calls["n"] += 1
        raise E401()

    try:
        call_with_retries(auth_fail, max_attempts=4, base_delay=0.01, sleep=lambda _s: None)
        ok = False
    except E401:
        ok = True
    c.check("401 fails fast without retries", ok and calls["n"] == 1)


def _test_atomic_and_cache(c: Collector, base: Path) -> None:
    target = base / "atomic" / "value.json"
    save_json(target, {"a": 1})
    c.check(
        "atomic save leaves no temp files", load_json(target) == {"a": 1} and not list(target.parent.glob(".*.part"))
    )

    corrupt = base / "atomic" / "broken.json"
    corrupt.parent.mkdir(parents=True, exist_ok=True)
    corrupt.write_text('{"truncated": ', encoding="utf-8")
    c.check(
        "corrupt json returns default and quarantines",
        load_json(corrupt, {"fallback": True}) == {"fallback": True}
        and not corrupt.exists()
        and any(p.name.startswith("broken.json.corrupt") for p in corrupt.parent.iterdir()),
    )

    cache = ArtifactCache(base / "cache")
    key1 = cache.key("stage", {"model": "gpt-5-mini", "prompt": "p"})
    key2 = cache.key("stage", {"model": "gpt-5.2", "prompt": "p"})
    c.check("cache key changes with model", key1 != key2)
    c.check("fallback cache key is namespaced", cache.fallback_key("stage", {"x": 1}).startswith("fallback-"))
    cache.put_json(key1, {"v": 1})
    bad_path = cache.json_path(key1)
    bad_path.write_text("NOT JSON", encoding="utf-8")
    c.check("corrupt cache entry treated as miss", cache.get_json(key1) is None)

    try:
        ensure_within(base, base / ".." / "outside.txt")
        ok = False
    except ValueError:
        ok = True
    c.check("path traversal outside base rejected", ok)
    c.check(
        "path inside base accepted", str(ensure_within(base, base / "sub" / "file.txt")).startswith(str(base.resolve()))
    )

    c.check("extract_json parses fenced JSON", extract_json('text ```json\n{"k": 1}\n``` more') == {"k": 1})
    c.check(
        "extract_json handles nested braces", extract_json('prefix {"a": {"b": [1, 2]}} suffix') == {"a": {"b": [1, 2]}}
    )


def _test_safe_subprocess(c: Collector, base: Path) -> None:
    marker = base / "temporal_out" / "pwned.txt"
    injection = f"'; touch {marker}; echo '"
    # /bin/true accepts and ignores its arguments: the injection payload is
    # passed as an inert argv token and never reaches a shell.
    backend = SketchControlledVideoBackend(
        {"argv": ["/bin/true", "prompt={prompt}", "out={output}"]},
        base / "temporal",
        execution_mode="development",
    )
    start = _image_file(base / "temporal" / "start.png")
    request = TemporalRequest(
        scene_id="INJ",
        beauty_start=str(start),
        control_sketch_start=str(start),
        motion_prompt=injection,
        duration_frames=5,
        fps=5,
        complexity="organic",
        output_path=str(base / "temporal" / "clip.mp4"),
    )
    try:
        backend.generate(request)  # echo produces no output file -> RuntimeError
    except RuntimeError:
        pass
    c.check("argv template blocks shell injection", not marker.exists())

    backend_bad = SketchControlledVideoBackend(
        {"argv": ["/bin/true", "{not_allowed}"]},
        base / "temporal2",
        execution_mode="development",
    )
    try:
        backend_bad.generate(request)
        ok = False
    except UnsafeCommandError:
        ok = True
    except RuntimeError:
        ok = False
    c.check("unknown placeholder rejected explicitly", ok)

    prod_shell = SketchControlledVideoBackend(
        {"allow_shell": True, "unsafe_shell_command": "/bin/echo {output}"},
        base / "temporal3",
        execution_mode="production",
    )
    try:
        prod_shell.generate(request)
        ok = False
    except UnsafeCommandError:
        ok = True
    except RuntimeError:
        ok = False
    c.check("production rejects shell path without security override", ok)


def _test_job_runtime(c: Collector, base: Path) -> None:
    job_dir = base / "job_a"
    calls = {"n": 0}

    def stage():
        calls["n"] += 1
        return {"value": calls["n"]}

    runtime = ResumableJobRuntime(job_dir, job_id="J", topic="t", config={"k": 1})
    first = runtime.execute("s1", {"in": 1}, stage)
    c.check("stage executes", first == {"value": 1})
    again = runtime.execute("s1", {"in": 1}, stage)
    c.check("resume reuses completed output", again == {"value": 1} and calls["n"] == 1)

    # Checksum mismatch: tamper with the artifact -> stage re-runs.
    output_path = Path(runtime.manifest.stages["s1"].output_path)
    output_path.write_text('{"value": 999}', encoding="utf-8")
    rerun = runtime.execute("s1", {"in": 1}, stage)
    c.check("tampered artifact checksum forces re-run", rerun == {"value": 2})
    runtime.mark_completed()

    # Interruption recovery: simulate a stage left running by a dead worker.
    manifest = load_json(job_dir / "job_manifest.json")
    manifest["stages"]["s1"]["status"] = "running"
    save_json(job_dir / "job_manifest.json", manifest)
    runtime2 = ResumableJobRuntime(job_dir, job_id="J", topic="t", config={"k": 1})
    c.check(
        "running stage recovered as interrupted then re-runnable",
        runtime2.manifest.stages["s1"].status == "interrupted",
    )
    rerun2 = runtime2.execute("s1", {"in": 1}, stage)
    c.check("interrupted stage re-executes", rerun2 == {"value": 3})

    # Config change invalidation.
    runtime2.mark_completed()
    runtime3 = ResumableJobRuntime(job_dir, job_id="J", topic="t", config={"k": 2})
    c.check(
        "config change marks completed stages stale",
        runtime3.manifest.stages["s1"].status == "stale"
        and any("Configuration changed" in w for w in runtime3.manifest.warnings),
    )
    rerun3 = runtime3.execute("s1", {"in": 1}, stage)
    c.check("stale stage re-executes after config change", rerun3 == {"value": 4})

    # Concurrency: same-process lock is re-entrant.
    try:
        ResumableJobRuntime(job_dir, job_id="J", topic="t", config={"k": 2})
        conflict = False
    except JobConcurrencyError:
        conflict = True
    c.check("same-process lock is re-entrant", not conflict)
    runtime3.release_lock()

    # A lock held by a foreign LIVE pid must be rejected (pid 1 is init).
    (job_dir / "job.lock").write_text("1\n2020-01-01T00:00:00\n", encoding="utf-8")
    try:
        ResumableJobRuntime(job_dir, job_id="J", topic="t", config={"k": 2})
        rejected = False
    except JobConcurrencyError:
        rejected = True
    c.check("live foreign lock rejects concurrent worker", rejected)

    # A lock from a DEAD pid is stale and reclaimed.
    import subprocess as _sp

    child = _sp.Popen(["/bin/true"])
    child.wait()
    dead_pid = child.pid
    (job_dir / "job.lock").write_text(f"{dead_pid}\n2020-01-01T00:00:00\n", encoding="utf-8")
    runtime4 = ResumableJobRuntime(job_dir, job_id="J", topic="t", config={"k": 2})
    c.check("stale lock from dead pid reclaimed", runtime4.lock_path.exists())
    runtime4.release_lock()

    # Retry budget.
    budget_dir = base / "job_budget"

    def failing():
        raise ValueError("boom")

    budget_runtime = ResumableJobRuntime(budget_dir, job_id="B", topic="t", config={}, max_stage_attempts=2)
    for _ in range(2):
        try:
            budget_runtime.execute("f", {"x": 1}, failing)
        except ValueError:
            pass
    try:
        budget_runtime.execute("f", {"x": 1}, failing)
        exhausted = False
    except StageRetryExhaustedError:
        exhausted = True
    c.check("stage retry budget enforced", exhausted)

    # No tracebacks inside the manifest; errors are typed and redacted.
    record = budget_runtime.manifest.stages["f"]
    c.check(
        "manifest stores typed redacted error, no traceback",
        record.error_type == "ValueError" and "traceback" not in record.metadata,
    )
    budget_runtime.release_lock()


def _test_download_validation(c: Collector, base: Path) -> None:
    real_png = _png_bytes()
    ok_response = FakeHTTPResponse(content=real_png, headers={"Content-Type": "image/png"})
    saved = download_image(ok_response, base / "dl" / "ok.png")
    c.check("valid image downloads atomically", saved.exists() and not list(saved.parent.glob(".*.part")))

    for name, response in (
        ("html body", FakeHTTPResponse(content=b"<html>err</html>", headers={"Content-Type": "text/html"})),
        ("json error body", FakeHTTPResponse(content=b'{"error": "x"}', headers={"Content-Type": "application/json"})),
        ("empty body", FakeHTTPResponse(content=b"", headers={"Content-Type": "image/png"})),
        ("corrupt image", FakeHTTPResponse(content=b"\x89PNG\r\n\x1a\nGARBAGE", headers={"Content-Type": "image/png"})),
    ):
        try:
            download_image(response, base / "dl" / "bad.png")
            ok = False
        except DownloadPolicyError:
            ok = True
        c.check(f"download rejects {name}", ok and not (base / "dl" / "bad.png").exists())

    big = FakeHTTPResponse(content=real_png, headers={"Content-Type": "image/png", "Content-Length": str(10**9)})
    try:
        download_image(big, base / "dl" / "big.png", max_bytes=1024 * 1024)
        ok = False
    except DownloadPolicyError:
        ok = True
    c.check("download rejects oversized declared payload", ok)

    small_cap = FakeHTTPResponse(content=real_png, headers={"Content-Type": "image/png"})
    try:
        download_image(small_cap, base / "dl" / "cap.png", max_bytes=10)
        ok = False
    except DownloadPolicyError:
        ok = True
    c.check("download rejects payload above byte cap", ok)


# ---------------------------------------------------------------------------
# Provider-lock and router tests
# ---------------------------------------------------------------------------


def _test_provider_locks(c: Collector, base: Path) -> None:
    # Production text generation with no OpenAI key fails clearly.
    router = LLMRouter({"execution_mode": "production", "provider_order": ["openai"]}, {}, base / "router_cache")
    try:
        router.generate_json(system="s", prompt="p", namespace="lock", fallback={"fallback": True}, force=True)
        ok = False
    except ProviderUnavailableError:
        ok = True
    c.check("production text fails clearly without OpenAI", ok)

    # Production never touches Gemini/OpenRouter even when keys exist and the
    # (invalid) order lists them.
    called = {"gemini": 0, "openrouter": 0, "local": 0}

    class SpyRouter(LLMRouter):
        def _gemini_json(self, *a, **k):
            called["gemini"] += 1
            return '{"x": 1}'

        def _openrouter_json(self, *a, **k):
            called["openrouter"] += 1
            return '{"x": 1}'

        def _local_json(self, *a, **k):
            called["local"] += 1
            return '{"x": 1}'

    spy = SpyRouter(
        {
            "execution_mode": "production",
            "provider_order": ["gemini", "openrouter", "local", "openai"],
            "enable_local_fallback": True,
        },
        {"GEMINI_API_KEY": "g-key-123456", "OPENROUTER_API_KEY": "or-key-123456"},
        base / "router_cache2",
    )
    c.check("production filters provider order to openai", spy.provider_order == ["openai"])
    try:
        spy.generate_json(system="s", prompt="p", namespace="lock2", fallback={"f": 1}, force=True)
    except ProviderUnavailableError:
        pass
    c.check("production never calls unauthorized providers", called == {"gemini": 0, "openrouter": 0, "local": 0})

    # Production vision fails clearly instead of returning the fallback.
    image = _image_file(base / "router" / "img.png")
    try:
        spy.critique_image(image_path=image, prompt="review", fallback={"approve": True}, force=True)
        ok = False
    except ProviderUnavailableError:
        ok = True
    c.check("production vision fails clearly without OpenAI", ok)

    # Production image generation requires BFL; no fallback engines.
    try:
        spy.generate_reference_image(prompt="x", output_path=base / "router" / "gen.png", force=True)
        ok = False
    except ProviderUnavailableError:
        ok = True
    c.check("production image generation requires BFL", ok)

    # Injected fake providers are rejected by the pipeline in production.
    from .pipeline import ScientificMotionStudioV10
    from .tests import FakeLLM

    try:
        ScientificMotionStudioV10(
            {
                "workspace": str(base / "prod_studio"),
                "execution_mode": "production",
                "llm": {"provider_order": ["openai"], "vision_provider_order": ["openai"]},
            },
            llm_router=FakeLLM(),
        )
        ok = False
    except ProviderLockViolationError:
        ok = True
    c.check("production rejects injected fake providers", ok)

    # Development fallback: allowed, but marked + stored in the fallback
    # namespace, and honours its TTL.
    dev = LLMRouter(
        {"execution_mode": "development", "provider_order": [], "fallback_cache_ttl_s": 3600}, {}, base / "router_dev"
    )
    value = dev.generate_json(system="s", prompt="p", namespace="devns", fallback={"deterministic": True})
    c.check("development fallback returns the fallback value", value == {"deterministic": True})
    fallback_files = list((base / "router_dev" / "_fallback" / "devns").glob("*.json"))
    c.check(
        "fallback cached in separate namespace with marker",
        len(fallback_files) == 1 and load_json(fallback_files[0]).get(FALLBACK_MARKER) is True,
    )
    success_files = list((base / "router_dev" / "devns").glob("*.json"))
    c.check("fallback never cached as live success", not success_files)

    envelope = load_json(fallback_files[0])
    envelope["cached_at"] = time.time() - 999999
    save_json(fallback_files[0], envelope)
    calls = {"n": 0}

    def counted_fallback():
        calls["n"] += 1
        return {"deterministic": True}

    dev.generate_json(system="s", prompt="p", namespace="devns", fallback=counted_fallback)
    c.check("expired fallback cache is not reused", calls["n"] == 1)


def _test_vision_cache_content_hash(c: Collector, base: Path) -> None:
    approvals = iter(['{"status": "approve", "round": 1}', '{"status": "revise", "round": 2}'])

    class VisionRouter(LLMRouter):
        def _openai_vision(self, image_path, prompt):
            return next(approvals)

    router = VisionRouter(
        {"execution_mode": "development", "provider_order": ["openai"], "vision_provider_order": ["openai"]},
        {"OPENAI_API_KEY": "sk-test-abcdefghijklmnop"},
        base / "vision_cache",
    )
    image = base / "vision" / "frame.png"
    _image_file(image, "#2E77A6")
    first = router.critique_image(image_path=image, prompt="review", fallback=None)
    c.check("vision critique returns provider output", first == {"status": "approve", "round": 1})
    cached = router.critique_image(image_path=image, prompt="review", fallback=None)
    c.check("vision cache hit on identical content", cached == first)
    # Same file name, same byte size, different pixels -> different key.
    original_size = image.stat().st_size
    # PNG re-render with a different color, padded/truncated is unsafe; instead
    # draw a same-size image and pad to the same byte length via tEXt chunkless
    # rewrite: simply regenerate and then compare sizes; if they differ, the
    # test still exercises content-hash keys (name unchanged).
    _image_file(image, "#D8483E")
    second = router.critique_image(image_path=image, prompt="review", fallback=None)
    c.check(
        "changed pixels invalidate vision cache (content hash, not name/size)",
        second == {"status": "revise", "round": 2},
        f"size_before={original_size} size_after={image.stat().st_size}",
    )


def _test_openai_mocked_integration(c: Collector, base: Path) -> None:
    class FakeUsage:
        input_tokens = 10
        output_tokens = 5
        total_tokens = 15

    class FakeOpenAIResponse:
        def __init__(self, text: str, response_id: str = "resp_123"):
            self.output_text = text
            self.id = response_id
            self.usage = FakeUsage()
            self.output = []

    class ScriptedResponses:
        def __init__(self, script):
            self.script = list(script)
            self.calls = 0

        def create(self, **kwargs):
            self.calls += 1
            action = self.script.pop(0)
            if isinstance(action, Exception):
                raise action
            return FakeOpenAIResponse(action)

    class FakeOpenAIClient:
        def __init__(self, script):
            self.responses = ScriptedResponses(script)

    def make_router(script, **extra):
        router = LLMRouter(
            {
                "execution_mode": "development",
                "provider_order": ["openai"],
                "retry": {"max_attempts": 3, "base_delay_s": 0.01, "max_delay_s": 0.02, "jitter": 0.0},
                **extra,
            },
            {"OPENAI_API_KEY": "sk-test-abcdefghijklmnop"},
            base / f"openai_{len(list(base.glob('openai_*')))}",
        )
        router._openai_client = FakeOpenAIClient(script)
        return router

    # Success.
    router = make_router(['{"answer": 42}'])
    value = router.generate_json(system="s", prompt="p", namespace="ok", fallback={"f": 1}, force=True)
    c.check("openai success parses JSON", value == {"answer": 42})

    # Malformed JSON -> bounded repair round-trip succeeds.
    router = make_router(["not json at all", '{"repaired": true}'])
    value = router.generate_json(system="s", prompt="p", namespace="repair", fallback={"f": 1}, force=True)
    c.check(
        "malformed JSON repaired via bounded retry",
        value == {"repaired": True} and router._openai_client.responses.calls == 2,
    )

    # Empty output is retried (classified retryable) then succeeds.
    router = make_router(["", '{"second": true}'])
    value = router.generate_json(system="s", prompt="p", namespace="empty", fallback={"f": 1}, force=True)
    c.check("empty output retried then succeeds", value == {"second": True})

    # 429 then success.
    class Fake429(Exception):
        status_code = 429

    router = make_router([Fake429("rate limited"), '{"after429": true}'])
    value = router.generate_json(system="s", prompt="p", namespace="rate", fallback={"f": 1}, force=True)
    c.check("429 then success recovers", value == {"after429": True} and router._openai_client.responses.calls == 2)

    # 401 fails fast: exactly one call, falls to fallback in development.
    class Fake401(Exception):
        status_code = 401

    router = make_router([Fake401("bad key"), '{"never": true}'])
    value = router.generate_json(system="s", prompt="p", namespace="auth", fallback={"fell_back": True}, force=True)
    c.check("401 fails fast without retry", value == {"fell_back": True} and router._openai_client.responses.calls == 1)


def _test_bfl_mocked_integration(c: Collector, base: Path) -> None:
    real_png = _png_bytes()
    config = {
        "bfl_model": "flux-kontext-pro",
        "bfl_aspect_ratio": "9:16",
        "bfl_output_format": "png",
        "bfl_timeout": 5,
        "bfl_poll_interval": 0.01,
        "bfl_allowed_url_hosts": ["api.bfl.ai", "poll.local", "delivery.local"],
        "retry": {"max_attempts": 3, "base_delay_s": 0.01, "max_delay_s": 0.02, "jitter": 0.0},
    }

    def scripted(post_script, get_factory):
        posts = list(post_script)

        def fake_post(url, headers=None, json=None, timeout=None, **kwargs):
            action = posts.pop(0)
            if isinstance(action, Exception):
                raise action
            return action

        return fake_post, get_factory

    submit_ok = FakeHTTPResponse({"id": "REQ1", "polling_url": "https://poll.local/r"})

    # Submit + poll + download success.
    def get_ok(url, headers=None, params=None, timeout=None, **kwargs):
        if "poll.local" in url:
            return FakeHTTPResponse({"status": "Ready", "result": {"sample": "https://delivery.local/s.png"}})
        return FakeHTTPResponse(content=real_png, headers={"Content-Type": "image/png"})

    client = BFLClient(config, "test-bfl-key-000", base / "bfl_ok")
    with (
        patch("requests.post", side_effect=scripted([submit_ok], get_ok)[0]),
        patch("requests.get", side_effect=get_ok),
    ):
        out = client.generate("prompt", base / "bfl_ok" / "out.png", force=True)
    c.check("bfl submit/poll/download success", out.exists() and out.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n")
    meta = load_json(next((base / "bfl_ok" / "bfl_images").glob("*.meta.json")))
    c.check(
        "bfl metadata records request id + hashes, no base64",
        meta["request_id"] == "REQ1"
        and bool(meta["prompt_hash"])
        and meta.get("input_image") is None
        and "base64" not in json.dumps(meta),
    )

    # Cache hit: identical inputs never re-submit.
    posted = {"n": 0}

    def fail_post(url, **kwargs):
        posted["n"] += 1
        raise AssertionError("must not re-submit on cache hit")

    with patch("requests.post", side_effect=fail_post):
        cached = client.generate("prompt", base / "bfl_ok" / "out2.png")
    c.check("bfl content-addressed cache hit skips network", cached.exists() and posted["n"] == 0)

    # 429 then success on submit.
    submit_429 = FakeHTTPResponse({"error": "rate"}, status_code=429)
    client2 = BFLClient(config, "test-bfl-key-000", base / "bfl_429")
    posts = [submit_429, FakeHTTPResponse({"id": "REQ2", "polling_url": "https://poll.local/r"})]

    def post_429(url, headers=None, json=None, timeout=None, **kwargs):
        return posts.pop(0)

    with patch("requests.post", side_effect=post_429), patch("requests.get", side_effect=get_ok):
        out2 = client2.generate("prompt2", base / "bfl_429" / "out.png", force=True)
    c.check("bfl 429 then success recovers", out2.exists())

    # Timeout: poll never becomes Ready.
    def get_pending(url, headers=None, params=None, timeout=None, **kwargs):
        return FakeHTTPResponse({"status": "Pending"})

    client3 = BFLClient({**config, "bfl_timeout": 0.05}, "test-bfl-key-000", base / "bfl_to")
    with patch("requests.post", side_effect=lambda *a, **k: submit_ok), patch("requests.get", side_effect=get_pending):
        try:
            client3.generate("p3", base / "bfl_to" / "out.png", force=True)
            ok = False
        except BFLError as exc:
            ok = "timed out" in str(exc)
    c.check("bfl poll timeout fails explicitly", ok)

    # Moderated result.
    def get_moderated(url, headers=None, params=None, timeout=None, **kwargs):
        return FakeHTTPResponse({"status": "Content Moderated"})

    client4 = BFLClient(config, "test-bfl-key-000", base / "bfl_mod")
    with (
        patch("requests.post", side_effect=lambda *a, **k: submit_ok),
        patch("requests.get", side_effect=get_moderated),
    ):
        try:
            client4.generate("p4", base / "bfl_mod" / "out.png", force=True)
            ok = False
        except BFLError as exc:
            ok = "moderated" in str(exc).lower()
    c.check("bfl moderated result fails explicitly", ok)

    # Unknown status fails explicitly.
    def get_unknown(url, headers=None, params=None, timeout=None, **kwargs):
        return FakeHTTPResponse({"status": "Transmogrifying"})

    client5 = BFLClient(config, "test-bfl-key-000", base / "bfl_unknown")
    with patch("requests.post", side_effect=lambda *a, **k: submit_ok), patch("requests.get", side_effect=get_unknown):
        try:
            client5.generate("p5", base / "bfl_unknown" / "out.png", force=True)
            ok = False
        except BFLError as exc:
            ok = "unknown status" in str(exc)
    c.check("bfl unknown status fails explicitly", ok)

    # Corrupt image payload rejected; nothing cached.
    def get_corrupt(url, headers=None, params=None, timeout=None, **kwargs):
        if "poll.local" in url:
            return FakeHTTPResponse({"status": "Ready", "result": {"sample": "https://delivery.local/s.png"}})
        return FakeHTTPResponse(content=b"\x89PNG\r\n\x1a\nBROKEN", headers={"Content-Type": "image/png"})

    client6 = BFLClient(config, "test-bfl-key-000", base / "bfl_corrupt")
    with patch("requests.post", side_effect=lambda *a, **k: submit_ok), patch("requests.get", side_effect=get_corrupt):
        try:
            client6.generate("p6", base / "bfl_corrupt" / "out.png", force=True)
            ok = False
        except DownloadPolicyError:
            ok = True
    pngs = list((base / "bfl_corrupt" / "bfl_images").glob("*.png"))
    c.check("bfl corrupt image rejected and never cached", ok and not pngs)

    # Wrong content type rejected.
    def get_html(url, headers=None, params=None, timeout=None, **kwargs):
        if "poll.local" in url:
            return FakeHTTPResponse({"status": "Ready", "result": {"sample": "https://delivery.local/s.png"}})
        return FakeHTTPResponse(content=b"<html>x</html>", headers={"Content-Type": "text/html"})

    client7 = BFLClient(config, "test-bfl-key-000", base / "bfl_html")
    with patch("requests.post", side_effect=lambda *a, **k: submit_ok), patch("requests.get", side_effect=get_html):
        try:
            client7.generate("p7", base / "bfl_html" / "out.png", force=True)
            ok = False
        except DownloadPolicyError:
            ok = True
    c.check("bfl wrong content-type rejected", ok)

    # Oversized download rejected.
    def get_big(url, headers=None, params=None, timeout=None, **kwargs):
        if "poll.local" in url:
            return FakeHTTPResponse({"status": "Ready", "result": {"sample": "https://delivery.local/s.png"}})
        return FakeHTTPResponse(content=real_png, headers={"Content-Type": "image/png", "Content-Length": str(10**10)})

    client8 = BFLClient({**config, "bfl_max_download_bytes": 2048}, "test-bfl-key-000", base / "bfl_big")
    with patch("requests.post", side_effect=lambda *a, **k: submit_ok), patch("requests.get", side_effect=get_big):
        try:
            client8.generate("p8", base / "bfl_big" / "out.png", force=True)
            ok = False
        except DownloadPolicyError:
            ok = True
    c.check("bfl oversized download rejected", ok)

    # Unsafe result URL rejected (host outside the allowlist).
    def get_unsafe(url, headers=None, params=None, timeout=None, **kwargs):
        return FakeHTTPResponse({"status": "Ready", "result": {"sample": "https://attacker.example/x.png"}})

    client9 = BFLClient(config, "test-bfl-key-000", base / "bfl_ssrf")
    with patch("requests.post", side_effect=lambda *a, **k: submit_ok), patch("requests.get", side_effect=get_unsafe):
        try:
            client9.generate("p9", base / "bfl_ssrf" / "out.png", force=True)
            ok = False
        except DownloadPolicyError:
            ok = True
    c.check("bfl unsafe result URL rejected (SSRF)", ok)

    # Invalid configuration is rejected before any network call.
    try:
        BFLClient({**config, "bfl_model": "sdxl"}, "k-000000", base / "bfl_badmodel")
        ok = False
    except BFLError:
        ok = True
    c.check("bfl invalid model rejected pre-flight", ok)
    try:
        BFLClient({**config, "bfl_aspect_ratio": "1:10"}, "k-000000", base / "bfl_badar")
        ok = False
    except BFLError:
        ok = True
    c.check("bfl invalid aspect ratio rejected pre-flight", ok)


# ---------------------------------------------------------------------------
# LLM shape-drift coercion (systemic OpenModel)
# ---------------------------------------------------------------------------


def _test_llm_shape_coercion(c: Collector, base: Path) -> None:
    from .schemas import Beat, ResearchPack, ScriptPackage

    # dict-of-notes where a list is declared (first live production failure).
    pack = ResearchPack(topic="t", limitations={"coverage": "sources cited", "uncertainty": "varies"})
    c.check("dict limitations coerced to list", pack.limitations == ["sources cited", "varies"])
    c.check("scalar hook coerced to list", ResearchPack(topic="t", hooks="one").hooks == ["one"])
    c.check("list summary joined to str", ResearchPack(topic="t", summary=["a", "b"]).summary == "a; b")

    # dict/str where a float is declared (second live production failure).
    raw = {
        "topic": "t",
        "facts": [
            {"claim": "a", "confidence": {"label": "high", "score": 0.8}},
            {"claim": "b", "confidence": "about 0.55 (moderate)"},
            {"claim": "c", "confidence": [0.42]},
            {"claim": "d", "confidence": {"label": "unknown"}},
        ],
    }
    facts = ResearchPack.model_validate(raw).facts
    c.check("dict confidence extracts score", facts[0].confidence == 0.8)
    c.check("string confidence parses number", round(facts[1].confidence, 2) == 0.55)
    c.check("list confidence extracts number", facts[2].confidence == 0.42)
    c.check("non-numeric confidence falls back to default", facts[3].confidence == 0.6)

    # int field + dict-of-beats + Any preserved.
    beat = Beat.model_validate({"spoken_line": "x", "pause_after_ms": {"ms": 120}})
    c.check("dict int field extracts number", beat.pause_after_ms == 120)
    script = ScriptPackage.model_validate({"topic": "t", "beats": {"b1": {"spoken_line": "hi"}}})
    c.check("dict-of-beats coerced to list", len(script.beats) == 1 and script.beats[0].spoken_line == "hi")
    c.check(
        "Any field untouched by coercion",
        ResearchPack(topic="t", raw_llm_output={"keep": "dict"}).raw_llm_output == {"keep": "dict"},
    )


# ---------------------------------------------------------------------------
# FLUX prompt budget (bloated initial prompt must fit, not crash)
# ---------------------------------------------------------------------------


def _test_flux_prompt_budget(c: Collector, base: Path) -> None:
    from .flux_prompt_system import FluxPromptSystem
    from .schemas import (
        ContinuityCanon,
        DepthPlane,
        MotionSeam,
        ReferencePack,
        SceneIllustrationArchitecture,
    )
    from .style_canon import base_bible

    root = base / "flux_prompt"
    system = FluxPromptSystem(
        {"seed_base": 100, "aspect_ratio": "9:16", "min_initial_prompt_words": 80, "max_initial_prompt_words": 380},
        root,
    )
    bible = base_bible("Rain for a year", "ref.png")
    pack = ReferencePack(scene_id="scene_1", board_path=str(root / "board.png"))
    Image.new("RGB", (180, 320), "#FAFAF7").save(pack.board_path)
    long_text = (
        "a sprawling metropolis under relentless torrential rainfall where every street canal and "
        "rooftop overflows with churning grey water while exhausted residents wade through waist "
        "deep floods past submerged vehicles collapsing infrastructure and improvised barricades " * 6
    )
    architecture = SceneIllustrationArchitecture(
        scene_id="scene_1",
        beat_id="B01",
        visual_thesis=long_text,
        focal_subject=long_text,
        narrative_claim=long_text,
        secondary_subjects=[long_text, long_text],
        depth_planes=[DepthPlane(plane_id="city", depth="midground", contents=long_text)],
        motion_seams=[
            MotionSeam(
                seam_id="rain",
                subject="rain",
                method="texture_loop",
                region="rain field",
                resting_overlap_rule="masked",
                required_variants=["initial", "changed"],
            )
        ],
        animation_representation=["texture_loop"],
    )
    brief = system.beauty_frame(
        architecture,
        bible,
        ContinuityCanon(),
        pack,
        "narration",
        "HEADLINE",
        0,
        style_source_path=str(pack.board_path),
        init_strategy="reference_board",
    )
    diagnostics = brief.prompt_diagnostics
    c.check(
        "bloated initial prompt fits budget instead of crashing",
        diagnostics.valid and diagnostics.word_count <= 380,
        f"valid={diagnostics.valid} words={diagnostics.word_count} errors={diagnostics.errors}",
    )
    head = " ".join(brief.compiled_prompt.lower().split()[:55])
    c.check(
        "trimmed prompt still states style early",
        any(token in head for token in ("scientific editorial", "illustration")),
    )
    # A second identical build is deterministic (resume reproduces the prompt).
    system2 = FluxPromptSystem(
        {"seed_base": 100, "aspect_ratio": "9:16", "min_initial_prompt_words": 80, "max_initial_prompt_words": 380},
        base / "flux_prompt2",
    )
    pack2 = ReferencePack(scene_id="scene_1", board_path=str(pack.board_path))
    brief2 = system2.beauty_frame(
        architecture,
        bible,
        ContinuityCanon(),
        pack2,
        "narration",
        "HEADLINE",
        0,
        style_source_path=str(pack.board_path),
        init_strategy="reference_board",
    )
    c.check("prompt trimming is deterministic", brief2.compiled_prompt == brief.compiled_prompt)


# ---------------------------------------------------------------------------
# Offline end-to-end
# ---------------------------------------------------------------------------


def _test_e2e_offline(c: Collector, base: Path) -> None:
    import subprocess

    from .pipeline import ScientificMotionStudioV10
    from .tests import FakeLLM

    ref_frame = _image_file(base / "ref.png", "#20282D")
    ref_video = base / "reference.mp4"
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-loop",
            "1",
            "-i",
            str(ref_frame),
            "-t",
            "1",
            "-r",
            "12",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            str(ref_video),
        ],
        check=True,
    )

    def fake_flux(brief):
        out = Path(brief.output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        image = Image.new("RGB", (180, 320), "#FAFAF7")
        draw = ImageDraw.Draw(image)
        seed = brief.seed or 0
        colors = ["#475157", "#2E77A6", "#4190C3", "#D8483E", "#F3BD38"]
        draw.polygon([(15, 280), (45, 80), (90, 35), (160, 100), (170, 280)], fill=colors[seed % len(colors)])
        draw.text((10, 10), brief.scene_id, fill="#20282D")
        image.save(out)
        return out

    def fake_mask(beauty_path, region, output):
        beauty = Image.open(beauty_path)
        mask = Image.new("L", beauty.size, 0)
        ImageDraw.Draw(mask).rectangle((35, 55, 155, 285), fill=255)
        output.parent.mkdir(parents=True, exist_ok=True)
        mask.save(output)
        return output

    config = {
        "workspace": str(base / "studio"),
        "execution_mode": "test",
        "research_search": {"enabled": False},
        "llm": {"provider_order": [], "vision_provider_order": []},
        "drawing": {
            "seed_base": 123,
            "aspect_ratio": "9:16",
            "min_initial_prompt_words": 40,
            "max_initial_prompt_words": 500,
        },
        "candidate_tournament": {"candidate_count": 2},
        "audio": {"enabled": False},
        "temporal": {"enabled": False},
        "flux_studio": {"maximum_director_revisions": 1, "require_vision_director": True},
        "render": {"backend": "pil", "crf": 30},
    }
    studio = ScientificMotionStudioV10(
        config, llm_router=FakeLLM(), image_generator=fake_flux, mask_generator=fake_mask
    )
    result = studio.run(
        "What happens under nonstop rain?", ref_video, plan_only=False, render_video=True, job_id="e2e", force=True
    )
    c.check("e2e offline run renders", result["mode"] == "rendered")
    video = Path(result["video"])
    c.check("e2e video exists", video.exists() and video.stat().st_size > 1000)
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "stream=codec_name", "-of", "json", str(video)],
        check=True,
        capture_output=True,
        text=True,
    )
    c.check("e2e video is valid H.264", "h264" in json.loads(probe.stdout)["streams"][0]["codec_name"])

    run_dir = Path(result["run_dir"])
    manifest = load_json(run_dir / "job_manifest.json")
    c.check(
        "e2e manifest completed with schema version",
        manifest["status"] == "completed" and manifest["schema_version"] == "10.1",
    )

    provenance = load_json(run_dir / "provenance_manifest.json")
    c.check(
        "provenance manifest exists with providers locked",
        provenance["providers"]["reasoning"] == "openai-gpt"
        and provenance["providers"]["image_generation"] == "bfl-flux-kontext",
    )
    from .utils import sha256_file

    video_entry = provenance["artifacts"]["video"]
    c.check(
        "provenance video hash matches artifact", video_entry["exists"] and video_entry["sha256"] == sha256_file(video)
    )

    # Resume: a second run must reuse completed stages, not re-execute them.
    manifest_before = load_json(run_dir / "job_manifest.json")
    attempts_before = {k: v["attempt"] for k, v in manifest_before["stages"].items()}
    studio2 = ScientificMotionStudioV10(
        config, llm_router=FakeLLM(), image_generator=fake_flux, mask_generator=fake_mask
    )
    result2 = studio2.run(
        "What happens under nonstop rain?", ref_video, plan_only=False, render_video=False, job_id="e2e"
    )
    manifest_after = load_json(run_dir / "job_manifest.json")
    attempts_after = {k: v["attempt"] for k, v in manifest_after["stages"].items()}
    c.check(
        "rerun is idempotent (no stage re-execution)",
        result2["mode"] == "live_generation" and attempts_after == attempts_before,
        f"before={attempts_before} after={attempts_after}",
    )


def _test_cli(c: Collector, base: Path) -> None:
    import contextlib
    import subprocess

    from .cli import build_parser, main

    help_text = build_parser().format_help()
    c.check("cli --help documents plan-only default", "Plan-only is the default" in help_text)
    c.check("cli missing args exits 2", main([]) == 2)
    c.check("cli missing config exits 2", main(["t", "ref.mp4", "--config", str(base / "nope.json")]) == 2)

    # Plan-only smoke: full offline pipeline through the CLI entry point.
    ref_frame = _image_file(base / "cli_ref.png", "#20282D")
    ref_video = base / "cli_reference.mp4"
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-loop",
            "1",
            "-i",
            str(ref_frame),
            "-t",
            "1",
            "-r",
            "12",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            str(ref_video),
        ],
        check=True,
    )
    config_path = base / "cli_config.json"
    config_path.write_text(
        json.dumps(
            {
                "workspace": str(base / "cli_studio"),
                "execution_mode": "test",
                "research_search": {"enabled": False},
                "llm": {"provider_order": [], "vision_provider_order": []},
                "audio": {"build_in_plan_mode": False},
                "temporal": {"enabled": False},
            }
        ),
        encoding="utf-8",
    )
    stdout = io.StringIO()
    with contextlib.redirect_stdout(stdout):
        rc = main(
            ["What if rivers ran backward?", str(ref_video), "--config", str(config_path), "--job-id", "cli-smoke"]
        )
    c.check("cli plan-only smoke exits 0", rc == 0)
    result = json.loads(stdout.getvalue())
    c.check(
        "cli prints plan-only result manifest", result["mode"] == "plan_only" and Path(result["job_manifest"]).exists()
    )


def _test_service_api(c: Collector, base: Path) -> None:
    try:
        from fastapi.testclient import TestClient
    except ImportError as exc:
        raise AssertionError(
            "fastapi+httpx are required for the API tests (install the [api] and [test] extras)"
        ) from exc

    from .pipeline import ScientificMotionStudioV10
    from .service_api import create_app
    from .tests import FakeLLM

    import subprocess

    ref_frame = _image_file(base / "api_ref.png", "#20282D")
    ref_video = base / "api_reference.mp4"
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-loop",
            "1",
            "-i",
            str(ref_frame),
            "-t",
            "1",
            "-r",
            "12",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            str(ref_video),
        ],
        check=True,
    )

    def factory():
        return ScientificMotionStudioV10(
            {
                "workspace": str(base / "api_studio"),
                "execution_mode": "test",
                "research_search": {"enabled": False},
                "llm": {"provider_order": [], "vision_provider_order": []},
                "audio": {"build_in_plan_mode": False},
                "temporal": {"enabled": False},
            },
            llm_router=FakeLLM(),
        )

    client = TestClient(create_app(factory))
    c.check("api 404 for unknown job", client.get("/jobs/nope").status_code == 404)
    bad = client.post("/jobs", json={"topic": "test topic", "reference_video": str(base / "missing.mp4")})
    c.check("api 400 for missing reference video", bad.status_code == 400)
    invalid = client.post("/jobs", json={"topic": "x", "reference_video": str(ref_video)})
    c.check("api 422 for too-short topic", invalid.status_code == 422)

    accepted = client.post(
        "/jobs",
        json={
            "topic": "What if rivers ran backward?",
            "reference_video": str(ref_video),
            "plan_only": True,
            "job_id": "api-smoke",
        },
    )
    c.check("api 202 accepted with job id", accepted.status_code == 202 and accepted.json()["job_id"] == "api-smoke")
    deadline = time.time() + 120
    status = {}
    while time.time() < deadline:
        status = client.get("/jobs/api-smoke").json()
        if status.get("status") in {"completed", "failed"}:
            break
        time.sleep(0.2)
    c.check(
        "api job completes with plan-only mode",
        status.get("status") == "completed" and status.get("mode") == "plan_only",
        json.dumps(status),
    )


def _fresh_root(root: str | Path | None, prefix: str) -> Path:
    """Test scratch roots are owned by the suite: an existing root from an
    earlier run is removed so reruns are idempotent (attempt budgets, locks
    and manifests never leak between executions)."""
    if root is None:
        return Path(tempfile.mkdtemp(prefix=prefix))
    base = Path(root)
    if base.exists():
        import shutil

        shutil.rmtree(base, ignore_errors=True)
    base.mkdir(parents=True, exist_ok=True)
    return base


def run_final_validation_tests(root: str | Path | None = None) -> dict[str, Any]:
    base = _fresh_root(root, "scistudio_final_tests_")
    c = Collector()

    _test_config_validation(c, base / "config")
    _test_secret_redaction(c)
    _test_url_and_retry_policy(c)
    _test_atomic_and_cache(c, base / "io")
    _test_safe_subprocess(c, base / "subproc")
    _test_job_runtime(c, base / "jobs")
    _test_download_validation(c, base / "downloads")
    _test_provider_locks(c, base / "locks")
    _test_vision_cache_content_hash(c, base / "vision")
    _test_openai_mocked_integration(c, base / "openai")
    _test_bfl_mocked_integration(c, base / "bfl")
    _test_llm_shape_coercion(c, base / "coercion")
    _test_flux_prompt_budget(c, base / "flux_budget")
    _test_e2e_offline(c, base / "e2e")
    _test_cli(c, base / "cli")
    _test_service_api(c, base / "api")

    passed = all(item["passed"] for item in c.items)
    return {"passed": passed, "count": len(c.items), "tests": c.items, "root": str(base)}


def run_all_tests(root: str | Path | None = None) -> dict[str, Any]:
    """Aggregate runner: retained V9 + V10 suites plus the finalization suite.

    The given root is a dedicated test scratch directory owned by the suite;
    it is recreated from scratch on every run so reruns are idempotent.
    """
    from .tests_v10 import run_v10_regression_tests

    base = _fresh_root(root, "scistudio_all_tests_")
    legacy = run_v10_regression_tests(base / "regression")
    final = run_final_validation_tests(base / "final")
    from .tests_notebook import run_notebook_tests

    notebook = run_notebook_tests(base / "notebook")
    return {
        "passed": bool(legacy["passed"] and final["passed"] and notebook["passed"]),
        "count": legacy["count"] + final["count"] + notebook["count"],
        "regression_count": legacy["count"],
        "final_count": final["count"],
        "notebook_count": notebook["count"],
        "tests": [*legacy["tests"], *final["tests"], *notebook["tests"]],
        "root": str(base),
    }


if __name__ == "__main__":
    report = run_all_tests()
    print(json.dumps({k: v for k, v in report.items() if k != "tests"}, indent=2))
    raise SystemExit(0 if report["passed"] else 1)


In [ ]:
%%writefile scistudio_v10/tests_notebook.py
"""Control Center / notebook-runner test suite (fully offline).

Covers config round-trips, preset validity, the paid-run gate, secret
handling, preflight injection, action dispatch, quick smoke, resume,
force-rerun scoping, artifact ZIP, progress callbacks, cancellation and the
ipywidgets fallback. No test performs a live network call.
"""

from __future__ import annotations

import builtins
import json
import subprocess
import tempfile
from pathlib import Path
from typing import Any

from .config_models import StudioConfig
from .errors import (
    ConfigurationError,
    JobCancelledError,
    PreflightError,
    ProviderAuthenticationError,
    ReferenceVideoError,
)
from .notebook_runner import (
    CONFIG_PRESETS,
    PAID_CONFIRMATION_PHRASE,
    CancellationToken,
    NotebookRunController,
    PipelineAction,
    PipelineRunRequest,
    RunMonitorState,
    generate_job_id,
    security_source_sweep,
    stage_progress_fraction,
)
from .preflight import run_preflight
from .security import clear_registered_secrets, redact_secrets


class Collector:
    def __init__(self) -> None:
        self.items: list[dict[str, Any]] = []

    def check(self, name: str, condition: bool, detail: str = "") -> None:
        self.items.append({"name": name, "passed": bool(condition), "detail": detail})
        if not condition:
            raise AssertionError(f"{name}: {detail}")


def _reference_video(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-f",
            "lavfi",
            "-i",
            "color=c=gray:s=180x320:r=12",
            "-t",
            "1",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            str(path),
        ],
        check=True,
    )
    return path


def run_notebook_tests(root: str | Path | None = None) -> dict[str, Any]:
    base = Path(root) if root else Path(tempfile.mkdtemp(prefix="scistudio_nb_tests_"))
    base.mkdir(parents=True, exist_ok=True)
    c = Collector()
    controller = NotebookRunController(base / "workspace")
    reference = _reference_video(base / "ref.mp4")
    dev_config = dict(CONFIG_PRESETS["Safe Development"])

    # -- presets & config round-trip ---------------------------------------
    for name, preset in CONFIG_PRESETS.items():
        settings = StudioConfig.from_dict(preset)
        c.check(f"preset valid: {name}", settings is not None)
        if preset.get("execution_mode") == "production":
            c.check(
                f"preset provider-locked: {name}",
                settings.llm.ordered_providers() == ["openai"] and settings.llm.image_provider == "bfl",
            )
    round_trip = json.loads(json.dumps(dev_config))
    c.check(
        "ui config serialization round-trip",
        StudioConfig.from_dict(round_trip).as_runtime_dict()["execution_mode"] == "development",
    )

    # -- job id generator ----------------------------------------------------
    job_id = generate_job_id("What if oceans doubled in depth?", str(reference))
    c.check(
        "job id readable and directory-safe",
        job_id.startswith("what-if-oceans-doubled-in-depth") and "/" not in job_id and " " not in job_id,
    )
    c.check("job id stable", job_id == generate_job_id("What if oceans doubled in depth?", str(reference)))

    # -- reference validation -------------------------------------------------
    for name, bad in (("missing file", str(base / "nope.mp4")), ("invalid extension", str(base / "ref.txt"))):
        (base / "ref.txt").write_text("x", encoding="utf-8")
        try:
            controller.validate_reference_video(bad)
            ok = False
        except ReferenceVideoError:
            ok = True
        c.check(f"reference rejected: {name}", ok)

    # -- invalid config JSON → field errors ----------------------------------
    try:
        controller.validate_config({"llm": {"bfl_model": "sdxl"}})
        ok = False
    except ConfigurationError as exc:
        ok = bool(getattr(exc, "field_errors", []))
    c.check("invalid config reports field errors", ok)

    # -- paid-run gate ---------------------------------------------------------
    live_request = PipelineRunRequest(
        action=PipelineAction.live_generation,
        topic="t",
        reference_video=str(reference),
        config=CONFIG_PRESETS["Production Live Generation"],
    )
    try:
        controller.validate_request(live_request)
        ok = False
    except PreflightError:
        ok = True
    c.check("paid run blocked without checkbox", ok)
    try:
        controller.validate_request(live_request.model_copy(update={"confirm_paid_run": True}))
        ok = False
    except PreflightError:
        ok = True
    c.check("paid run blocked without confirmation phrase", ok)
    try:
        controller.validate_request(
            live_request.model_copy(
                update={"confirm_paid_run": True, "paid_confirmation_phrase": PAID_CONFIRMATION_PHRASE}
            )
        )
        ok = False
    except ProviderAuthenticationError:
        ok = True  # gate passed, missing API key caught BEFORE any pipeline start
    c.check("missing API key handled before pipeline", ok)
    try:
        controller.run_provider_smoke(live_request.model_copy(update={"action": PipelineAction.provider_live_smoke}))
        ok = False
    except PreflightError:
        ok = True
    c.check("live smoke blocked without confirmation", ok)

    # -- preflight with injected missing tools ----------------------------------
    report = run_preflight(
        config=dev_config, reference_video=reference, workspace=base / "workspace", which=lambda tool: None
    )
    failed = {check["check"] for check in report["checks"] if check["status"] == "FAIL"}
    c.check("preflight FAILs on missing ffmpeg", "ffmpeg" in failed and "ffprobe" in failed)
    remotion_config = {**dev_config, "render": {"backend": "remotion"}}
    report = run_preflight(config=remotion_config, which=lambda tool: None)
    failed = {check["check"] for check in report["checks"] if check["status"] == "FAIL"}
    c.check("preflight FAILs on missing npm/npx for remotion", {"node", "npm", "npx"} <= failed)
    report = run_preflight(config=dev_config, live=False)
    statuses = {check["check"]: check["status"] for check in report["checks"]}
    c.check("network check skipped offline", statuses["network"] == "SKIPPED")

    # -- stage progress mapping ---------------------------------------------------
    c.check(
        "stage progress maps shot-state wildcard",
        0 < stage_progress_fraction("09_shot_state_SC01") < stage_progress_fraction("13_flux_studio"),
    )

    # -- action dispatch completeness (spy controller: routing only, no execution)
    routed: list[str] = []

    class SpyController(NotebookRunController):
        def run_preflight(self, request):
            routed.append("preflight")
            return {"summary": {"PASS": 0, "WARNING": 0, "FAIL": 0, "SKIPPED": 0}, "checks": []}

        def run_quick_smoke(self, request=None):
            routed.append("quick_smoke")
            return {"passed": True}

        def run_offline_tests(self, request=None):
            routed.append("offline_tests")
            return {"passed": True}

        def run_provider_smoke(self, request):
            routed.append("provider_smoke")
            return {}

        def run_pipeline(self, request, **kwargs):
            routed.append(f"pipeline:{request.action.value}")
            return {"mode": "stub"}

        def package_artifacts(self, job_id):
            routed.append("package")
            return base / "stub.zip"

        def validate_config(self, config):
            routed.append("validate_config")
            return StudioConfig.from_dict(dev_config)

    spy = SpyController(base / "spy_workspace")
    for action in PipelineAction:
        request = PipelineRunRequest(action=action, topic="t", reference_video=str(reference), config=dev_config)
        spy.dispatch(request)
    c.check(
        "all actions dispatchable through one controller",
        len(routed) >= len(PipelineAction)
        and any(r.startswith("pipeline:") for r in routed)
        and {"preflight", "quick_smoke", "offline_tests", "provider_smoke", "package"} <= set(routed),
        str(routed),
    )

    # -- quick smoke (offline, no keys) ----------------------------------------------
    smoke = controller.run_quick_smoke()
    c.check("quick smoke passes without API keys", smoke["passed"], json.dumps(smoke))
    c.check("quick smoke resume used cache", smoke["resume_used_cache"])

    # -- pipeline runs: plan-only, progress callback, resume, cancel, force -----------
    workspace_config = {**dev_config, "workspace": str(controller.paths["base"])}
    plan_request = PipelineRunRequest(
        action=PipelineAction.plan_only,
        topic="What if rivers ran backward?",
        reference_video=str(reference),
        config=workspace_config,
        job_id="nb-plan",
    )
    monitor = RunMonitorState()
    result = controller.run_pipeline(plan_request, progress_callback=monitor.callback)
    snapshot = monitor.snapshot()
    c.check("plan-only run completes via controller", result["mode"] == "plan_only")
    c.check(
        "progress callback captured stage events", snapshot["event_count"] > 5 and "03_script" in snapshot["completed"]
    )
    run_dir = Path(result["run_dir"])
    c.check(
        "run_request + resolved_config persisted (redacted)",
        (run_dir / "run_request.json").exists() and (run_dir / "resolved_config.json").exists(),
    )

    resume_request = plan_request.model_copy(update={"action": PipelineAction.resume_job})
    monitor2 = RunMonitorState()
    result2 = controller.run_pipeline(resume_request, progress_callback=monitor2.callback)
    snapshot2 = monitor2.snapshot()
    c.check(
        "resume reuses manifest and cache",
        result2["mode"] == "plan_only" and len(snapshot2["cached"]) > 0 and not snapshot2["completed"],
        json.dumps(snapshot2),
    )

    token = CancellationToken()
    token.cancel()
    try:
        controller.run_pipeline(plan_request.model_copy(update={"job_id": "nb-cancel"}), cancellation_token=token)
        ok = False
    except JobCancelledError:
        ok = True
    c.check("cancellation stops before next safe stage", ok)
    c.check("cancelled job left resumable manifest", isinstance(controller.job_status("nb-cancel"), dict))

    # Force rerun scoped to a selected stage: only that stage + downstream re-run.
    force_request = plan_request.model_copy(
        update={
            "action": PipelineAction.force_rerun,
            "force_scope": "selected_and_downstream",
            "force_stage": "04_storyboard",
        }
    )
    monitor3 = RunMonitorState()
    controller.run_pipeline(force_request, progress_callback=monitor3.callback)
    snapshot3 = monitor3.snapshot()
    c.check(
        "scoped force rerun re-executes selected stage",
        "04_storyboard" in snapshot3["completed"],
        json.dumps(snapshot3),
    )
    c.check(
        "scoped force rerun keeps earlier stages cached",
        "02_research" in snapshot3["cached"] and "03_script" in snapshot3["cached"],
    )

    # -- artifact zip + safe paths ------------------------------------------------------
    archive = controller.package_artifacts("nb-plan")
    c.check("artifact ZIP generated", archive.exists() and archive.stat().st_size > 500)
    try:
        controller.package_artifacts("../../etc")
        ok = False
    except (ValueError, ConfigurationError):
        ok = True
    c.check("zip rejects path traversal job ids", ok)

    # -- secrets ---------------------------------------------------------------------------
    clear_registered_secrets()
    controller.secrets.set_manual("OPENAI_API_KEY", "sk-manual-test-key-000111222333")
    c.check("manual secret reported configured", controller.secrets.status()["OPENAI_API_KEY"] == "Configured")
    saved_texts = "".join(
        path.read_text(encoding="utf-8") for path in controller.paths["base"].rglob("*.json") if path.is_file()
    )
    c.check("no secret written into reports/configs", "sk-manual-test-key-000111222333" not in saved_texts)
    c.check(
        "manual secret redacted in errors",
        "sk-manual-test-key-000111222333" not in redact_secrets("boom sk-manual-test-key-000111222333"),
    )
    controller.secrets.clear_manual()
    c.check("clear manual secrets works", controller.secrets.status()["OPENAI_API_KEY"] in ("Missing", "Configured"))
    clear_registered_secrets()

    # -- security sweep + no /content -----------------------------------------------------
    sweep = security_source_sweep(Path(__file__).parent)
    c.check("package security sweep clean (incl. no /content)", sweep["ok"], json.dumps(sweep["findings"]))

    # -- backward compatibility ------------------------------------------------------------
    from .pipeline import ScientificMotionStudioV10

    legacy = ScientificMotionStudioV10({**dev_config, "workspace": str(base / "legacy")}).run(
        "Legacy call still works?", reference, plan_only=True, render_video=False, force=False, job_id="legacy"
    )
    c.check("legacy run() signature unchanged", legacy["mode"] == "plan_only")

    # -- notebook UI import & ipywidgets fallback --------------------------------------------
    from . import notebook_ui

    c.check("notebook_ui imports outside Colab", hasattr(notebook_ui, "launch_control_center"))
    real_import = builtins.__import__

    def blocked_import(name, *args, **kwargs):
        if name.startswith("ipywidgets"):
            raise ImportError("ipywidgets blocked for fallback test")
        return real_import(name, *args, **kwargs)

    builtins.__import__ = blocked_import
    try:
        center = notebook_ui.launch_control_center(base / "workspace_fallback")
    finally:
        builtins.__import__ = real_import
    c.check("headless fallback when ipywidgets missing", getattr(center, "headless", False) is True)
    if notebook_ui._widgets_available():
        center2 = notebook_ui.launch_control_center(base / "workspace_widgets")
        c.check("widget control center builds", hasattr(center2, "tabs"))
        c.check(
            "paid gate disables primary button until confirmed", center2.primary.disabled is False
        )  # default offline action enabled
        center2.action.value = PipelineAction.live_generation
        c.check("live action disables primary without confirmation", center2.primary.disabled is True)
        center2.paid_checkbox.value = True
        center2.paid_phrase.value = PAID_CONFIRMATION_PHRASE
        c.check("confirmation enables primary", center2.primary.disabled is False)
        exported = center2._config_from_form()
        c.check("form config validates", StudioConfig.from_dict(exported) is not None)
        center2._apply_config_to_form(CONFIG_PRESETS["Production Full Render"])
        c.check("form round-trips preset", center2._config_from_form()["render"]["backend"] == "remotion")

    passed = all(item["passed"] for item in c.items)
    return {"passed": passed, "count": len(c.items), "tests": c.items, "root": str(base)}


if __name__ == "__main__":
    report = run_notebook_tests()
    print(json.dumps({k: v for k, v in report.items() if k != "tests"}, indent=2))
    raise SystemExit(0 if report["passed"] else 1)


In [ ]:
%%writefile scistudio_v10/tests_v10.py
from __future__ import annotations

import json
import subprocess
import tempfile
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw

from .asset_registry import AssetRegistry
from .cache_store import ArtifactCache
from .candidate_tournament import CandidateTournament
from .character_registry import CharacterRegistry
from .hybrid_package import HybridPackageBuilder
from .hybrid_render import RemotionHybridExporter
from .job_runtime import ResumableJobRuntime
from .pipeline import ScientificMotionStudioV10
from .provider_registry import ProviderRegistry
from .reference_selector import DynamicReferenceSelector
from .schemas import (
    AnimationPlan,
    AssetQuery,
    AssetRecord,
    CharacterProfile,
    CharacterView,
    DepthPlane,
    DrawingBrief,
    ProviderCapability,
    ProviderSpec,
    SceneIllustrationArchitecture,
    SceneRequest,
    SemanticLayer,
    SemanticLayerContract,
    ShotState,
    TemporalRequest,
)
from .shot_state import ShotStatePlanner
from .sketch_control import ControlSketchBuilder
from .temporal_backends import (
    DeterministicCompositorBackend,
    SketchControlledVideoBackend,
    TemporalBackendRouter,
)
from .tests import FakeLLM, run_regression_tests as run_v9_regression_tests
from .utils import load_json


class Collector:
    def __init__(self):
        self.items = []

    def check(self, name, condition, detail=""):
        self.items.append({"name": name, "passed": bool(condition), "detail": detail})
        if not condition:
            raise AssertionError(f"{name}: {detail}")


class TournamentLLM(FakeLLM):
    def critique_image(self, **kwargs):
        fallback = kwargs.get("fallback", {})
        # Preserve deterministic fallback ranking for candidate boards, but
        # approve the existing V9 art-director tests.
        if "candidate_rank" in str(kwargs.get("namespace", "")):
            return fallback
        return super().critique_image(**kwargs)


def _image(path: Path, color: str, detail: int = 1) -> str:
    image = Image.new("RGB", (180, 320), "#FAFAF7")
    d = ImageDraw.Draw(image)
    d.rectangle((25, 40, 155, 280), fill=color)
    for i in range(detail):
        d.line((30, 70 + i * 12, 150, 50 + i * 17), fill="#20282D", width=2)
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path)
    return str(path)


def _ffprobe(path: str | Path) -> dict[str, Any]:
    raw = subprocess.check_output(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "stream=codec_name,width,height,r_frame_rate",
            "-of",
            "json",
            str(path),
        ]
    )
    return json.loads(raw)


def run_v10_regression_tests(root: str | Path | None = None) -> dict[str, Any]:
    base = Path(root) if root else Path(tempfile.mkdtemp(prefix="scistudio_v10_tests_"))
    base.mkdir(parents=True, exist_ok=True)
    c = Collector()

    prior = run_v9_regression_tests(base / "v9")
    c.check("V9.1 regression retained", prior["passed"] and prior["count"] >= 53)

    providers = ProviderRegistry(base / "providers")
    providers.register(
        ProviderSpec(
            provider_id="slow",
            provider_type="image",
            implementation="x",
            priority=50,
            capabilities=[ProviderCapability(name="reference_edit", supports_images=True)],
        ),
        object(),
    )
    fast_impl = object()
    providers.register(
        ProviderSpec(
            provider_id="fast",
            provider_type="image",
            implementation="y",
            priority=10,
            capabilities=[ProviderCapability(name="reference_edit", supports_images=True)],
        ),
        fast_impl,
    )
    spec, impl = providers.resolve("image", require="reference_edit")
    c.check("provider registry resolves priority", spec.provider_id == "fast" and impl is fast_impl)
    c.check("provider snapshot exists", Path(providers.snapshot()).exists())
    try:
        providers.resolve("publisher")
        missing = False
    except RuntimeError:
        missing = True
    c.check("missing provider fails explicitly", missing)

    cache = ArtifactCache(base / "cache")
    key = cache.key("brief", {"a": 1})
    cache.put_json(key, {"ok": True})
    c.check("content cache JSON roundtrip", cache.get_json(key) == {"ok": True})
    src = base / "source.bin"
    src.write_bytes(b"abc")
    cached = cache.put_file(key, src)
    c.check("content cache file roundtrip", Path(cached).read_bytes() == b"abc")

    calls = {"n": 0}
    runtime = ResumableJobRuntime(base / "job", job_id="J1", topic="topic", config={"x": 1})

    def stage():
        calls["n"] += 1
        return {"value": 7}

    c.check("job stage first execution", runtime.execute("stage", {"x": 1}, stage) == {"value": 7})
    c.check(
        "job stage resume reuses output", runtime.execute("stage", {"x": 1}, stage) == {"value": 7} and calls["n"] == 1
    )
    c.check(
        "job input change reruns stage", runtime.execute("stage", {"x": 2}, stage) == {"value": 7} and calls["n"] == 2
    )
    runtime.mark_completed()
    c.check("job manifest completed", load_json(runtime.manifest_path)["status"] == "completed")

    registry = AssetRegistry(base / "assets.sqlite")
    style_path = _image(base / "style.png", "#475157", 2)
    subject_path = _image(base / "subject.png", "#2E77A6", 5)
    future_path = _image(base / "future.png", "#D8483E", 4)
    registry.upsert(
        AssetRecord(
            asset_id="style",
            path=style_path,
            asset_type="style_anchor",
            role_tags=["style"],
            style_hash="S",
            quality_score=1,
        )
    )
    registry.upsert(
        AssetRecord(
            asset_id="subject",
            path=subject_path,
            asset_type="subject_view",
            subject_ids=["human"],
            camera_view="side",
            perspective="moderate",
            chronology_index=1,
            role_tags=["approved"],
            style_hash="S",
            quality_score=0.9,
        )
    )
    registry.upsert(
        AssetRecord(
            asset_id="future",
            path=future_path,
            asset_type="approved_scene",
            subject_ids=["human"],
            camera_view="side",
            perspective="moderate",
            chronology_index=9,
            role_tags=["approved"],
            style_hash="S",
            quality_score=1,
        )
    )
    c.check("asset registry get", registry.get("subject").subject_ids == ["human"])
    selector = DynamicReferenceSelector(registry, base / "selector")
    selection = selector.select(
        AssetQuery(
            scene_id="SC02",
            subject_ids=["human"],
            camera_view="side",
            perspective="moderate",
            style_hash="S",
            chronology_index=2,
            desired_types=["style_anchor", "subject_view", "approved_scene"],
            maximum_results=8,
        )
    )
    selected_ids = [x.asset.asset_id for x in selection.selected]
    c.check("dynamic selector retrieves identity reference", "subject" in selected_ids)
    c.check("dynamic selector excludes future continuity", "future" not in selected_ids)
    c.check("reference board built", Path(selection.board_path).exists())

    characters = CharacterRegistry(base / "characters")
    characters.upsert(
        CharacterProfile(subject_id="human", display_name="Adult subject", immutable_traits=["adult proportions"])
    )
    characters.add_view("human", CharacterView(view_id="side-1", angle="side", path=subject_path))
    c.check(
        "character registry persists views",
        CharacterRegistry(base / "characters").get("human").views[0].angle == "side",
    )

    architecture = SceneIllustrationArchitecture(
        scene_id="SC01",
        beat_id="B01",
        visual_thesis="integrated rain city",
        depth_planes=[DepthPlane(plane_id="city", depth="midground", contents="city", movement_role="hold")],
        motion_seams=[],
    )
    scene = SceneRequest(scene_id="SC01", beat_id="B01", duration_s=1, narration="Hold.", visual_event="city")
    state = ShotStatePlanner(FakeLLM(), {}, base / "shot").plan(scene, architecture, force=True)
    c.check("shot state defines first and last", bool(state.first_frame_description and state.last_frame_description))
    c.check(
        "hold shot does not request control sketch",
        state.temporal_complexity == "hold" and not state.control_sketch_required,
    )

    brief = DrawingBrief(
        brief_id="SC01-beauty",
        scene_id="SC01",
        positive_prompt="One integrated mature scientific editorial city scene with fluid authored linework and coherent perspective.",
        negative_prompt="isolated icons",
        kontext_instruction="Preserve the studio identity.",
        output_path=str(base / "candidate.png"),
        seed=100,
    )
    tournament = CandidateTournament(TournamentLLM(), {"candidate_count": 4}, base / "tournament")

    def generate(candidate_brief):
        color = ["#475157", "#2E77A6", "#4190C3", "#D8483E"][candidate_brief.request_metadata["candidate_index"]]
        return _image(Path(candidate_brief.output_path), color, candidate_brief.request_metadata["candidate_index"] + 1)

    result = tournament.run(brief, architecture, state, generate, force=True)
    c.check("candidate tournament generates four", len(result.candidates) == 4)
    c.check("candidate tournament unique seeds", len({x.seed for x in result.candidates}) == 4)
    c.check(
        "candidate tournament selects valid winner", result.winner_id in {x.candidate_id for x in result.candidates}
    )
    c.check("candidate comparison board exists", Path(result.comparison_board).exists())

    sketch = ControlSketchBuilder(base / "sketch")
    first, last = sketch.build_pair(
        "SC01",
        subject_path,
        future_path,
        ShotState(
            scene_id="SC01",
            first_frame_description="a",
            last_frame_description="b",
            control_sketch_regions=["arm"],
            motion_bridge=["arm extends"],
            control_sketch_required=True,
            temporal_complexity="articulated",
        ),
    )
    c.check("control sketch pair generated", Path(first.output_path).exists() and Path(last.output_path).exists())

    det = DeterministicCompositorBackend(base / "temporal_det")
    request = TemporalRequest(
        scene_id="SC01",
        beauty_start=subject_path,
        duration_frames=15,
        fps=15,
        complexity="simple",
        output_path=str(base / "simple.mp4"),
    )
    temporal = det.generate(request)
    info = _ffprobe(temporal.output_path)
    c.check("deterministic temporal clip encoded", info["streams"][0]["codec_name"] == "h264")
    sketch_backend = SketchControlledVideoBackend({"command": ""}, base / "sketch_backend")
    c.check("unconfigured sketch backend unavailable", not sketch_backend.available())
    router = TemporalBackendRouter([sketch_backend, det], base / "router")
    c.check("router chooses deterministic for simple", router.select(request).backend_id == "deterministic-compositor")
    organic = request.model_copy(
        update={"scene_id": "SC02", "complexity": "organic", "control_sketch_start": first.output_path}
    )
    try:
        router.select(organic)
        wrongly_fell_back = True
    except RuntimeError:
        wrongly_fell_back = False
    c.check("organic motion never degrades to still compositor", not wrongly_fell_back)

    contract = SemanticLayerContract(
        scene_id="SC01",
        beauty_frame_path=subject_path,
        layers=[
            SemanticLayer(
                layer_id="beauty-base",
                description="base",
                source_region="full",
                extraction_method="full_frame",
                locked=True,
            )
        ],
    )
    animation = AnimationPlan(scene_id="SC01", duration_frames=15, fps=15, camera_locked=True, events=[])
    overlay = base / "overlay.svg"
    overlay.write_text('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 180 320"></svg>')
    package = HybridPackageBuilder({"width": 180, "height": 320}, base / "package").build(
        scene,
        contract,
        animation,
        str(overlay),
        temporal_clip_path=temporal.output_path,
        temporal_backend=temporal.backend_id,
    )
    c.check(
        "temporal clip replaces beauty layer",
        package.layers[0].kind == "video_clip" and package.temporal_backend == "deterministic-compositor",
    )
    project = RemotionHybridExporter({}, base / "remotion").create_project([package])
    ts = (project / "src" / "index.tsx").read_text()
    c.check("Remotion supports temporal video layer", "OffthreadVideo" in ts and "ScientificMotionV10" in ts)

    # Full plan-only execution from the integrated V10 pipeline.
    ref_frame = _image(base / "ref_frame.png", "#20282D", 8)
    ref_video = base / "reference.mp4"
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-loop",
            "1",
            "-i",
            ref_frame,
            "-t",
            "1",
            "-r",
            "12",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            str(ref_video),
        ],
        check=True,
    )
    config = {
        "workspace": str(base / "studio"),
        "research_search": {"enabled": False},
        "llm": {"provider_order": [], "vision_provider_order": []},
        "drawing": {
            "seed_base": 123,
            "aspect_ratio": "9:16",
            "min_initial_prompt_words": 40,
            "max_initial_prompt_words": 500,
        },
        "candidate_tournament": {"candidate_count": 3},
        "references": {"maximum_retrieved_assets": 8},
        "audio": {"build_in_plan_mode": False},
        "temporal": {"enabled": True},
        "providers": [],
    }
    pipeline_result = ScientificMotionStudioV10(config).run(
        "What happens under nonstop rain?", ref_video, plan_only=True, job_id="plan-test", force=True
    )
    c.check("full V10 plan-only pipeline completed", pipeline_result["mode"] == "plan_only")
    c.check("full pipeline emits shot states", Path(pipeline_result["shot_states"]).exists())
    c.check("full pipeline emits reference selections", Path(pipeline_result["reference_selections"]).exists())
    c.check("full pipeline emits candidate plans", Path(pipeline_result["candidate_plans"]).exists())
    c.check(
        "full pipeline emits persistent job manifest",
        load_json(pipeline_result["job_manifest"])["status"] == "completed",
    )

    # Full live-generation integration with injected deterministic providers.
    live_root = base / "live_studio"

    def fake_flux(brief):
        out = Path(brief.output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        image = Image.new("RGB", (180, 320), "#FAFAF7")
        draw = ImageDraw.Draw(image)
        seed = brief.seed or 0
        colors = ["#475157", "#2E77A6", "#4190C3", "#D8483E", "#F3BD38"]
        draw.polygon([(15, 280), (45, 80), (90, 35), (160, 100), (170, 280)], fill=colors[seed % len(colors)])
        for i in range(18):
            draw.line((20, 50 + i * 11, 160, 40 + i * 13), fill="#20282D", width=1 + (i % 3))
        draw.text((10, 10), brief.scene_id, fill="#20282D")
        image.save(out)
        return out

    def fake_mask(beauty_path, region, output):
        beauty = Image.open(beauty_path)
        mask = Image.new("L", beauty.size, 0)
        ImageDraw.Draw(mask).rectangle((35, 55, 155, 285), fill=255)
        output.parent.mkdir(parents=True, exist_ok=True)
        mask.save(output)
        return output

    live_config = {
        **config,
        "workspace": str(live_root),
        "audio": {"enabled": False},
        "candidate_tournament": {"candidate_count": 2},
        "temporal": {"enabled": False},
        "flux_studio": {"maximum_director_revisions": 1, "require_vision_director": True},
        "semantic": {"require_approved_beauty": True},
    }
    live_result = ScientificMotionStudioV10(
        live_config, llm_router=FakeLLM(), image_generator=fake_flux, mask_generator=fake_mask
    ).run(
        "What happens under nonstop rain?",
        ref_video,
        plan_only=False,
        render_video=False,
        job_id="live-test",
        force=True,
    )
    c.check("full live fixture reaches live_generation", live_result["mode"] == "live_generation")
    c.check("full live fixture creates beauty frames", Path(live_result["beauty_frames"]).exists())
    c.check("full live fixture creates semantic contracts", Path(live_result["semantic_contracts"]).exists())
    c.check(
        "full live fixture creates candidate tournaments",
        any(Path(live_result["candidate_tournaments"]).glob("SC*.json")),
    )
    c.check("full live fixture persists approved assets", len(AssetRegistry(live_result["asset_registry"]).list()) > 1)
    registry.close()

    passed = all(x["passed"] for x in c.items)
    return {
        "passed": passed,
        "count": len(c.items) + prior["count"],
        "v10_count": len(c.items),
        "v9_count": prior["count"],
        "tests": [*prior["tests"], *c.items],
        "root": str(base),
    }


In [ ]:
%%writefile scistudio_v10/utils.py
"""Filesystem, hashing, JSON and subprocess utilities.

All JSON/file writes performed through this module are **atomic** (temp file
in the destination directory + ``os.replace``), and all JSON reads are
**corruption-safe**: a truncated or invalid file is quarantined with a
``.corrupt`` suffix and the caller receives the default value instead of a
crash that would permanently break job resume.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from datetime import datetime, UTC
from pathlib import Path
from typing import Any, Callable, Iterable

from .security import redact_secrets

_COMMAND_ERROR_TEXT_LIMIT = 4000


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def slugify(value: str, max_len: int = 80) -> str:
    text = re.sub(r"[^a-zA-Z0-9]+", "-", str(value).strip().lower()).strip("-")
    return (text or "untitled")[:max_len].rstrip("-")


def canonical_json(value: Any) -> str:
    if hasattr(value, "model_dump"):
        value = value.model_dump(mode="json")
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str)


def hash_value(value: Any, length: int = 20) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()[:length]


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _jsonable(value: Any) -> Any:
    """Recursively serialize nested Pydantic models and path-like values."""
    if hasattr(value, "model_dump"):
        return _jsonable(value.model_dump(mode="json"))
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_jsonable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_bytes(path: str | Path, data: bytes) -> Path:
    """Write bytes atomically: temp file in the same directory, then rename."""
    path = Path(path)
    ensure_dir(path.parent)
    handle = tempfile.NamedTemporaryFile(dir=str(path.parent), prefix=f".{path.name}.", suffix=".part", delete=False)
    try:
        with handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(handle.name, path)
    except BaseException:
        try:
            os.unlink(handle.name)
        except OSError:
            pass
        raise
    return path


def atomic_copy(src: str | Path, dst: str | Path) -> Path:
    """Copy a file atomically (temp + rename); never leaves partial targets."""
    src, dst = Path(src), Path(dst)
    ensure_dir(dst.parent)
    handle = tempfile.NamedTemporaryFile(dir=str(dst.parent), prefix=f".{dst.name}.", suffix=".part", delete=False)
    handle.close()
    try:
        shutil.copy2(src, handle.name)
        os.replace(handle.name, dst)
    except BaseException:
        try:
            os.unlink(handle.name)
        except OSError:
            pass
        raise
    return dst


def save_json(path: str | Path, value: Any) -> Path:
    path = Path(path)
    payload = json.dumps(_jsonable(value), ensure_ascii=False, indent=2, default=str)
    return atomic_write_bytes(path, payload.encode("utf-8"))


def quarantine_corrupt_file(path: Path) -> Path | None:
    """Move a corrupt file aside so subsequent runs regenerate it."""
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S")
    target = path.with_name(f"{path.name}.corrupt-{stamp}")
    try:
        os.replace(path, target)
        return target
    except OSError:
        return None


def load_json(path: str | Path, default: Any = None) -> Any:
    """Read JSON; quarantine corrupt files and return *default* instead of raising."""
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError, OSError):
        quarantine_corrupt_file(path)
        return default


def ensure_within(base: str | Path, candidate: str | Path) -> Path:
    """Resolve *candidate* and require it to stay inside *base* (anti-traversal)."""
    base = Path(base).resolve()
    resolved = Path(candidate).resolve()
    if base != resolved and base not in resolved.parents:
        raise ValueError(f"Path {resolved} escapes the allowed base directory {base}")
    return resolved


def extract_json(text: str | bytes | dict | list | None, fallback: Any = None) -> Any:
    if isinstance(text, (dict, list)):
        return text
    if text is None:
        return fallback
    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="replace")
    text = str(text).strip()
    if not text:
        return fallback
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        pass

    fenced = re.findall(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)
    for candidate in fenced:
        try:
            return json.loads(candidate.strip())
        except (json.JSONDecodeError, ValueError):
            continue

    starts = [idx for idx in (text.find("{"), text.find("[")) if idx >= 0]
    if not starts:
        return fallback
    start = min(starts)
    opening = text[start]
    closing = "}" if opening == "{" else "]"
    depth = 0
    in_string = False
    escaped = False
    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue
        if char == '"':
            in_string = True
        elif char == opening:
            depth += 1
        elif char == closing:
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start : index + 1])
                except (json.JSONDecodeError, ValueError):
                    break
    return fallback


def run_command(
    command: Iterable[str],
    *,
    cwd: str | Path | None = None,
    env: dict[str, str] | None = None,
    timeout: int | float | None = None,
    check: bool = True,
    capture: bool = True,
) -> subprocess.CompletedProcess:
    """Run an argv-list command (never a shell string).

    Failure messages are redacted and length-capped so provider errors that
    leak through stderr cannot carry secrets into logs or manifests.
    """
    merged_env = os.environ.copy()
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        timeout=timeout,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if check and result.returncode != 0:
        stdout = redact_secrets(str(result.stdout or ""))[:_COMMAND_ERROR_TEXT_LIMIT]
        stderr = redact_secrets(str(result.stderr or ""))[:_COMMAND_ERROR_TEXT_LIMIT]
        printable = " ".join(redact_secrets(str(part)) for part in command)
        raise RuntimeError(f"Command failed ({result.returncode}): {printable}\nSTDOUT:\n{stdout}\nSTDERR:\n{stderr}")
    return result


def ffprobe_duration(path: str | Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    try:
        result = run_command(
            [
                "ffprobe",
                "-v",
                "error",
                "-show_entries",
                "format=duration",
                "-of",
                "default=noprint_wrappers=1:nokey=1",
                str(path),
            ]
        )
        return float((result.stdout or "0").strip())
    except (RuntimeError, ValueError, subprocess.SubprocessError):
        return 0.0


def copy_into(src: str | Path, dst: str | Path) -> Path:
    return atomic_copy(src, dst)


def retry(
    fn: Callable[[], Any],
    attempts: int = 3,
    delay: float = 1.0,
    exceptions: tuple[type[Exception], ...] = (Exception,),
    *,
    max_delay: float = 30.0,
    jitter: float = 0.25,
) -> Any:
    """Exponential-backoff retry with jitter for the given exception types."""
    from .http_safety import backoff_delays

    delays = backoff_delays(attempts, base_delay=delay, max_delay=max_delay, jitter=jitter)
    last: Exception | None = None
    for index in range(attempts):
        try:
            return fn()
        except exceptions as exc:
            last = exc
            if index + 1 < attempts:
                time.sleep(delays[min(index, len(delays) - 1)])
    if last:
        raise last
    raise RuntimeError("retry() exhausted without an exception")


In [ ]:
%%writefile scistudio_v10/webui.py
"""Streamlit WebUI.

Uses the same :class:`NotebookRunController` as the notebook Control Center —
one implementation of validation, preflight, the paid-run gate and execution.
Plan-only is the safe default; failures show short redacted errors.
"""

from __future__ import annotations

import json
from pathlib import Path

from .notebook_runner import (
    ACTION_DESCRIPTIONS,
    LIVE_ACTIONS,
    PAID_CONFIRMATION_PHRASE,
    NotebookRunController,
    PipelineAction,
    PipelineRunRequest,
    RunMonitorState,
)
from .preflight import VIDEO_SUFFIXES
from .security import redact_secrets, redacted_exception_text


def main() -> None:
    try:
        import streamlit as st
    except ImportError as exc:
        raise RuntimeError("Install streamlit to use the WebUI") from exc

    st.set_page_config(page_title="Scientific Motion Studio V10", layout="wide")
    st.title("Scientific Motion Studio V10")

    workspace = st.sidebar.text_input("Workspace", "./scientific_motion_studio_v10")
    controller = NotebookRunController(workspace)

    st.sidebar.subheader("Secrets")
    for name, status in controller.secrets.status().items():
        st.sidebar.write(f"{'✓' if status == 'Configured' else '✕'} {name}: {status}")

    config_source = st.sidebar.radio("Config source", ["Import JSON file", "Paste JSON"])
    config: dict = {}
    if config_source == "Import JSON file":
        uploaded = st.sidebar.file_uploader("Config JSON", type=["json"])
        if uploaded is not None:
            config = json.loads(uploaded.read().decode("utf-8"))
    else:
        pasted = st.sidebar.text_area("Config JSON", "{}")
        try:
            config = json.loads(pasted or "{}")
        except json.JSONDecodeError as exc:
            st.sidebar.error(f"Invalid JSON: {exc}")

    topic = st.text_area("Topic", "What happens if it rains nonstop for one year?")
    upload = st.file_uploader("Reference video", type=[s.lstrip(".") for s in VIDEO_SUFFIXES])
    reference_path = st.text_input("…or existing reference path")
    if upload is not None:
        target = controller.paths["base"] / "uploads" / upload.name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(upload.read())
        reference_path = str(target)
        st.info(f"Uploaded to {target}")

    action = st.selectbox(
        "Action",
        [
            PipelineAction.environment_check,
            PipelineAction.config_validation,
            PipelineAction.quick_smoke,
            PipelineAction.plan_only,
            PipelineAction.live_generation,
            PipelineAction.full_production_render,
            PipelineAction.resume_job,
            PipelineAction.package_artifacts,
        ],
        format_func=lambda a: a.value.replace("_", " ").title(),
    )
    st.caption(ACTION_DESCRIPTIONS[action])
    job_id = st.text_input("Job ID (optional)")

    confirm = phrase = ""
    if action in LIVE_ACTIONS:
        st.warning("This action performs PAID provider calls.")
        confirm = st.checkbox("I understand this action may create paid API requests.")
        phrase = st.text_input(f"Type {PAID_CONFIRMATION_PHRASE!r} to confirm")

    if st.button("Run preflight"):
        request = PipelineRunRequest(
            action=action, topic=topic, reference_video=reference_path, config=config, job_id=job_id or None
        )
        report = controller.run_preflight(request)
        summary = report["summary"]
        st.write(
            f"PASS {summary['PASS']} | WARNING {summary['WARNING']} | "
            f"FAIL {summary['FAIL']} | SKIPPED {summary['SKIPPED']}"
        )
        st.dataframe(report["checks"])

    if st.button("Run action", type="primary"):
        request = PipelineRunRequest(
            action=action,
            topic=topic,
            reference_video=reference_path,
            config=config,
            job_id=job_id or None,
            confirm_paid_run=bool(confirm),
            paid_confirmation_phrase=phrase,
        )
        monitor = RunMonitorState()
        progress_bar = st.progress(0.0, text="Waiting for first stage…")

        def on_progress(stage_id: str, event: str, record: dict) -> None:
            monitor.callback(stage_id, event, record)
            snapshot = monitor.snapshot()
            progress_bar.progress(min(1.0, snapshot["progress"]), text=f"{stage_id} ({event})")

        try:
            result = (
                controller.dispatch(request, progress_callback=on_progress)
                if action
                in {
                    PipelineAction.plan_only,
                    PipelineAction.live_generation,
                    PipelineAction.full_production_render,
                    PipelineAction.resume_job,
                }
                else controller.dispatch(request)
            )
            st.success("Completed")
            with st.expander("Result (redacted)"):
                st.json(redact_secrets(result))
            video = result.get("video") if isinstance(result, dict) else ""
            if video and Path(video).exists():
                st.video(video)
        except Exception as exc:
            st.error(f"[{type(exc).__name__}] {redacted_exception_text(exc, 400)}")
            for item in getattr(exc, "field_errors", []) or []:
                st.write(f"• {item['field']}: {item['message']}")


if __name__ == "__main__":
    main()


## 3. Dependency Installation (grouped, idempotent)

Optional dependency groups. Only missing packages are installed; nothing is reinstalled on re-run. FFmpeg/Node are **checked**, never auto-installed — use the Preflight tab's install button or your platform's package manager. Offline; safe for Run all.

In [ ]:
DEPENDENCY_GROUPS = {
    "Core": ["pydantic>=2.7", "Pillow>=10.0", "requests>=2.31", "openai>=1.68"],
    "Notebook UI": ["ipywidgets>=8.0"],
    "Tests": ["nbclient>=0.10", "nbformat>=5.10", "fastapi>=0.115", "httpx>=0.27"],
    "API": ["fastapi>=0.115", "uvicorn>=0.30"],
    "Render": ["cairosvg>=2.7"],
}
DEPENDENCY_GROUPS["All Production Dependencies"] = sorted(
    {pkg for group in DEPENDENCY_GROUPS.values() for pkg in group})

def install_group(name: str) -> None:
    """Install one dependency group; skips packages that already import."""
    import importlib, re, subprocess, sys
    todo = []
    for spec in DEPENDENCY_GROUPS[name]:
        module = re.split(r"[><=]", spec)[0].replace("Pillow", "PIL").replace("-", "_")
        try:
            importlib.import_module(module)
        except ImportError:
            todo.append(spec)
    if todo:
        print(f"[{name}] installing:", ", ".join(todo))
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *todo], check=True)
        print("If imports appear stale after installs, restart the runtime once.")
    else:
        print(f"[{name}] already satisfied.")

install_group("Core")
install_group("Notebook UI")
install_group("Tests")


## 4. Package Import and Reload

Imports (or reloads) the freshly generated package so the notebook uses exactly the source above. Offline; safe for Run all.

In [ ]:
import importlib
import sys

for module_name in sorted(list(sys.modules)):
    if module_name == "scistudio_v10" or module_name.startswith("scistudio_v10."):
        del sys.modules[module_name]
import scistudio_v10

print("scistudio_v10", scistudio_v10.__version__, "loaded from", scistudio_v10.__file__)


## 5. Configuration Presets

Validates every built-in preset (including the production provider locks) and writes `config.example.json`. Offline; safe for Run all.

In [ ]:
import json
from pathlib import Path
from scistudio_v10.config_models import StudioConfig
from scistudio_v10.notebook_runner import CONFIG_PRESETS

for name, preset in CONFIG_PRESETS.items():
    settings = StudioConfig.from_dict(preset)
    lock = ""
    if settings.execution_mode.value == "production":
        assert settings.llm.ordered_providers() == ["openai"]
        assert settings.llm.image_provider == "bfl"
        lock = " [provider-locked]"
    print(f"✓ PASS  {name:28s} mode={settings.execution_mode.value}{lock}")
print("\nExample production config:", Path("config.example.json").resolve())


## 6. Colab Secrets and Storage

Secrets resolve from **Colab Secrets → environment variables → the Control Center's password fields** and stay in memory only; every value is registered for redaction. This cell prints status only — never values. Offline; safe for Run all.

In [ ]:
from scistudio_v10.notebook_runner import SecretsManager

secrets = SecretsManager()
for name, status in secrets.status().items():
    icon = "✓" if status == "Configured" else "○"
    print(f"{icon} {name}: {status}")
print("\nIn Colab: add keys under the 🔑 'Secrets' sidebar, then rerun this cell "
      "or press 'Refresh Secret Status' in the Control Center.")


## 7. Notebook Control Center

The full production console: Project, Execution, Providers & Secrets, Model & Generation, Temporal & Rendering, Advanced Config, Preflight & Tests, Run Monitor, Results & Artifacts. Rendering it performs **no API calls**; paid actions stay disabled until the paid-run gate (checkbox + `RUN LIVE` + keys + clean preflight) is satisfied. Safe for Run all.

In [ ]:
from scistudio_v10.notebook_ui import launch_control_center

control_center = launch_control_center(WORKSPACE / "scientific_motion_studio_v10")


## 8. Headless Python Usage

The same controller drives headless code, the CLI (`scistudio-v10 "topic" reference.mp4 --config config.json`), FastAPI and Streamlit (`streamlit run -m scistudio_v10.webui`). The demo below runs a fully offline development plan-only pipeline. No paid calls; safe for Run all.

In [ ]:
import json
import subprocess
from pathlib import Path
from scistudio_v10.notebook_runner import (
    CONFIG_PRESETS, NotebookRunController, PipelineAction, PipelineRunRequest, RunMonitorState,
)

controller = control_center.controller if hasattr(control_center, "controller") else \
    NotebookRunController(WORKSPACE / "scientific_motion_studio_v10")
demo_ref = controller.paths["base"] / "uploads" / "demo_reference.mp4"
demo_ref.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i",
                "color=c=gray:s=180x320:r=12", "-t", "1", "-c:v", "libx264",
                "-pix_fmt", "yuv420p", str(demo_ref)], check=True)

monitor = RunMonitorState()
result = controller.dispatch(
    PipelineRunRequest(
        action=PipelineAction.plan_only,
        topic="What happens if it rains nonstop for one year?",
        reference_video=str(demo_ref),
        config={**CONFIG_PRESETS["Safe Development"]},
        job_id="notebook-headless-demo",
    ),
    progress_callback=monitor.callback,
)
snapshot = monitor.snapshot()
print("mode:", result["mode"], "| stages completed:", len(snapshot["completed"]),
      "| cached:", len(snapshot["cached"]))
print("run dir:", result["run_dir"])


## 9. Offline Validation

Runs the retained regression suites, the finalization suite and the new Control-Center suite (260+ checks), plus the shared security source sweep. **No paid calls**; safe for Run all (takes several minutes).

In [ ]:
import json
from pathlib import Path
from scistudio_v10.tests_final import run_all_tests

report = run_all_tests(Path("v10_final_validation"))
Path("v10_final_test_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"PASS={report['passed']} | TOTAL={report['count']} "
      f"| REGRESSION={report['regression_count']} | FINALIZATION={report['final_count']} "
      f"| CONTROL_CENTER={report['notebook_count']}")
assert report["passed"], "offline validation failed — see v10_final_test_report.json"


In [ ]:
import json
from pathlib import Path
from scistudio_v10.notebook_runner import security_source_sweep

sweep = security_source_sweep(Path("scistudio_v10"))
Path("v10_security_sweep.json").write_text(json.dumps(sweep, indent=2), encoding="utf-8")
print(json.dumps(sweep, indent=2))
assert sweep["ok"], "security sweep found issues"
if "/content" in Path("config.example.json").read_text(encoding="utf-8"):
    raise AssertionError("config.example.json contains a hard-coded /content path")
print("config.example.json free of hard-coded /content paths")


## 10. Optional Live Smoke Tests (paid, explicit opt-in)

**These cells can incur API charges.** They run only when `SCISTUDIO_RUN_LIVE_TESTS=1` and the key is set; otherwise they record `NOT RUN`. The Control Center's *Provider Live Smoke Test* action offers the same checks behind its interactive gate. Never triggered by Run all.

In [ ]:
import json
import os
from pathlib import Path

LIVE = os.environ.get("SCISTUDIO_RUN_LIVE_TESTS") == "1"
live_results = {"openai": "NOT RUN", "bfl": "NOT RUN"}

if LIVE and os.environ.get("OPENAI_API_KEY"):
    from scistudio_v10.llm import LLMRouter
    router = LLMRouter({"execution_mode": "production", "provider_order": ["openai"],
                        "openai_max_output_tokens": 512,
                        "retry": {"max_attempts": 2, "base_delay_s": 1.0}},
                       {}, Path("live_smoke/openai_cache"))
    value = router.generate_json(
        system="You return strict JSON.",
        prompt='Return exactly {"status": "ok", "provider": "openai"}.',
        namespace="live_smoke", fallback=None, force=True)
    assert isinstance(value, dict) and value.get("status") == "ok", value
    live_results["openai"] = "PASS"
    print("OpenAI live smoke: PASS | request id:", getattr(router, "_last_request_id", ""))
else:
    print("OpenAI live smoke: NOT RUN (set SCISTUDIO_RUN_LIVE_TESTS=1 and OPENAI_API_KEY; may incur charges)")

if LIVE and os.environ.get("BFL_API_KEY"):
    from scistudio_v10.bfl_client import BFLClient
    client = BFLClient({"bfl_model": "flux-kontext-pro", "bfl_aspect_ratio": "1:1",
                        "bfl_output_format": "jpeg", "bfl_timeout": 120,
                        "retry": {"max_attempts": 2}},
                       os.environ["BFL_API_KEY"], Path("live_smoke/bfl_cache"))
    out = client.generate("Minimal test: one small blue circle on white, flat vector style.",
                          Path("live_smoke/bfl_smoke.jpg"))
    assert out.exists() and out.stat().st_size > 1024
    live_results["bfl"] = "PASS"
    print("BFL live smoke: PASS ->", out)
else:
    print("BFL live smoke: NOT RUN (set SCISTUDIO_RUN_LIVE_TESTS=1 and BFL_API_KEY; may incur charges)")

Path("v10_live_smoke.json").write_text(json.dumps(live_results, indent=2), encoding="utf-8")


## 11. Validation Summary

Aggregates every gate into `final_validation_report.json`. `NOT RUN` entries are honest — they were not executed in this environment. Offline; safe for Run all.

In [ ]:
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

tests = json.loads(Path("v10_final_test_report.json").read_text(encoding="utf-8"))
sweep = json.loads(Path("v10_security_sweep.json").read_text(encoding="utf-8"))
live = json.loads(Path("v10_live_smoke.json").read_text(encoding="utf-8")) if Path("v10_live_smoke.json").exists() else {"openai": "NOT RUN", "bfl": "NOT RUN"}

matrix = {
    "offline_test_suite": "PASS" if tests["passed"] else "FAIL",
    "retained_regression_tests": f"PASS ({tests['regression_count']})" if tests["passed"] else "FAIL",
    "finalization_tests": f"PASS ({tests['final_count']})" if tests["passed"] else "FAIL",
    "control_center_tests": f"PASS ({tests['notebook_count']})" if tests["passed"] else "FAIL",
    "security_source_sweep": "PASS" if sweep["ok"] else "FAIL",
    "provider_lock_tests": "PASS" if tests["passed"] else "FAIL",
    "live_openai_smoke": live["openai"],
    "live_bfl_smoke": live["bfl"],
    "docker_build": "NOT RUN (requires Docker: docker build -t scistudio-v10 .)",
}
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "package_version": __import__("scistudio_v10").__version__,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "total_checks": tests["count"],
    "matrix": matrix,
    "providers": {
        "reasoning": "OpenAI GPT (exclusive in production)",
        "vision_review": "OpenAI GPT (exclusive in production)",
        "image_generation": "BFL FLUX Kontext (exclusive in production)",
        "anthropic_runtime_dependency": False,
    },
}
Path("final_validation_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
overall = tests["passed"] and sweep["ok"]
print("\nOVERALL:", "PASS" if overall else "FAIL")
assert overall


## 12. Export and Download

Lists the run artifacts and shows the ZIP helper (the Control Center's *Results & Artifacts* tab has one-click buttons for the same operations). Offline; safe for Run all.

In [ ]:
from pathlib import Path

base = controller.paths["base"]
print("Workspace layout:")
for sub in ("configs", "jobs", "reports", "smoke_tests", "exports", "logs"):
    entries = sorted((base / sub).glob("*"))
    print(f"  {sub}/ ({len(entries)} entries)")
archive = controller.package_artifacts("notebook-headless-demo")
print("\nDemo run packaged:", archive, f"({archive.stat().st_size / 1024:.0f} KiB)")
print("In Colab, download via the Files sidebar or files.download(str(archive)).")
